---
# 4. GAN Model Training for Non-Augmented EMNIST Data

In this section, we will be training various Generative Adversarial Network (GAN) Models from scratch. Subsequently, we will compare all the GAN Models against each other to obtain the Best Model which will subsequently be used to compare to the EMNIST Training Data with Augmentation. Subsequently, we will perform further comparison and analysis with the best GAN Model.

---
## 4.1 Vanilla GAN Model Training

In this sub-section, we will be training a Vanilla GAN for the EMNIST Dataset. The Vanilla GAN will also provide a strong and suitable baseline for the future GAN Models that we will be training. Due to the Simplicity of Vanilla GAN, it will most likely be able to handle the EMNIST Dataset well. A Vanilla GAN is the most basic GAN which mainly consist of Two Neural Networks trained in opposition, namely the Generator and the Discriminator. Both of the Neural Network will compete in a 'Minimax Game' where the Generator tries to 'fool' the Discriminator with realistic generated images and the Discriminator tries to tell apart real and generated images. The relevant formulas for the Vanilla GAN is indicated below:

---
**Minimax Objective (Vanilla GAN Loss):**

Purpose: Original Adversarial Objective where Generator $G$ tries to Fool Discriminator $D$, while $D$ tries to Distinguish Real Samples from Fake Ones.

$$
\min_G \max_D V(D, G) = \mathbb{E}_{x \sim p_{\text{data}}(x)}[\log D(x)] + \mathbb{E}_{z \sim p_z(z)}[\log (1 - D(G(z)))]
$$

Where:
- $G$ = Generator Network
- $D$ = Discriminator Network
- $x$ = Real Data Sample
- $z$ = Random Noise Vector
- $p_{\text{data}}(x)$ = Distribution of Real Data
- $p_z(z)$ = Prior Distribution of Noise (e.g., $\mathcal{N}(0, I)$)
- $D(x)$ = Probability that $x$ is Real
- $D(G(z))$ = Probability that Generated Image is Real

---
**Discriminator Loss (Optimisation Form):**

Purpose: Loss used to Train the Discriminator to Assign High Probability to Real Data and Low Probability to Fake Data.

$$
L_D = -\mathbb{E}_{x \sim p_{\text{data}}(x)}[\log D(x)] - \mathbb{E}_{z \sim p_z(z)}[\log (1 - D(G(z)))]
$$

Where:
- $L_D$ = Discriminator Loss
- First Term Penalises Misclassifying Real Data as Fake
- Second Term Penalises Misclassifying Fake Data as Real

---
**Generator Loss (Original Formulation):**

Purpose: Commonly used in Practice due to Stronger Gradients to Generator, helping it Train more Effectively when Discriminator is Performing Well.


$$
L_G = \mathbb{E}_{z \sim p_z(z)}[\log (1 - D(G(z)))]
$$

Where:
- $L_G$ = Generator Loss
- $G$ = Generator Network
- $D$ = Discriminator Network

---
**Noise Vector Sampling:**

Purpose: The Generator takes a Random Noise Vector $z$ from a Known Distribution as Input and Transforms it into a Realistic Image.

$$
z \sim \mathcal{N}(0, I) \quad \text{or} \quad z \sim U(-1, 1)
$$

Where:
- $z$ = Latent Noise Vector Sampled from Standard Normal or Uniform Distribution
- $G(z)$ = Generator Transforms $z$ into Synthetic Image

---
With the relevant mathematical formulas indicated above, we will proceed to train the Vanilla GAN in this sub-section for the purpose of it's simplicity and also to serve as a Baseline for our other GAN Models.

---
### 4.1.1 Defining Data Pre-Processing Function

In this sub-section, we will be defining the Data Pre-Processing Function which will allow for consistent Data Pipeline to be built and modifed accordingly should the need arise such as when we need to add Spectral Normalisation. This will ensure that the Data's integrity is not affected and we are able to make fair and un-biased assumptions. The Data Pre-Processing Function is defined in the Code Cell below:

In [ ]:
# ========== Date Pre-Processing Function ========== #
def load_data():
    global X_train

    batch_size = 32
    buffer_size = X_train.shape[0]

    dataset = tf.data.Dataset.from_tensor_slices(X_train)
    dataset = dataset.shuffle(buffer_size).batch(batch_size).prefetch(tf.data.AUTOTUNE)

    return dataset

With reference to the Code Cell above, we are able to determine that the Data Pre-Processing Function have been defined successfully and we are able to move on to Defining Callback Functions in the next sub-section.

---
### 4.1.2 Defining Callback Functions

In this sub-section, we will be pre-defining the various Callbacks. This is to ensure consistency throughout the model and also to increase the GAN Accuracy and optimise the Computation Cost for training each GAN Model. These Callbacks will be main used during the GAN Training in the next sub-section. The formulas and logic for these Callbacks are indicated below:

---
**Learning Rate Scheduler:**

Purpose: Reduces Learning Rate During Training to Fine-tune Model Convergence.

$$\text{If } epoch \mod 10 = 0 \Rightarrow \eta_{new} = \frac{1}{2} \cdot \eta$$

Where:

- $\eta$ = current learning rate

- $\eta_{\text{new}}$ = updated learning rate

- $epoch \bmod 10$ checks if the epoch is a multiple of 10

---
**ReduceLROnPlateau:**

Purpose: Automatically Reduces the Learning Rate when Validation Performance Plateaus.

$$
\text{If no improvement in } val\_loss \text{ for 10 epochs: } \eta_{\text{new}} = 0.5 \cdot \eta
$$

Where:

- $\eta_{\text{new}}$ = reduced learning rate

---
**EarlyStopping:**

Purpose: Prevents Wastage of Computational Power if Model Does Not Improve.

$$
\text{If } val\_loss \text{ does not improve for 25 epochs, stop training and restore best weights}
$$

---
With the formulas and Various Callbacks listed, we will proceed to Define the Callbacks in the Code Cell below in preparation for the GAN Model Training.

In [ ]:
# ========== Define Learning Rate Scheduler ========== #
lr_scheduler = LearningRateScheduler(
    lambda epoch, lr: lr * 0.95 if epoch % 10 == 0 else lr,
    verbose=1
)

# ========== Define Reduce Learning Rate on Plateau ========== #
reduce_lr = ReduceLROnPlateau(
    monitor='loss',
    factor=0.5,
    patience=10,
    verbose=1,
    min_lr=1e-6
)

# ========== Define Early Stopping ========== #
early_stop = EarlyStopping(
    monitor='loss',
    patience=25,
    verbose=1,
    restore_best_weights=False
)

# ========== Define Generated Image ========== #
def display_generated_images(generator, latent_dim, n=7):
    noise = tf.random.normal([n * n, latent_dim])
    generated_images = generator(noise, training=False)

    # Rescale from [-1, 1] to [0, 1] if using tanh
    generated_images = (generated_images + 1.0) / 2.0

    fig, axes = plt.subplots(n, n, figsize=(n, n))
    for i in range(n):
        for j in range(n):
            img = generated_images[i * n + j, :, :, 0]
            axes[i, j].imshow(img, cmap='gray')
            axes[i, j].axis('off')

    plt.tight_layout()
    plt.show()

# ========== Define FID Calculator ========== #
_inception_model = InceptionV3(
    include_top=False,
    pooling='avg',
    input_shape=(299,299,3)
)

def calculate_fid(real_images, fake_images, batch_size=10):

    # ----- Scale to [0,255] and Resize to 299×299 ----- #
    real = (real_images + 1.0) * 127.5
    fake = (fake_images + 1.0) * 127.5
    real = tf.image.resize(real, (299,299))
    fake = tf.image.resize(fake, (299,299))

    # ----- If Grayscale, Convert to RGB ----- #
    if real.shape[-1] == 1:
        real = tf.image.grayscale_to_rgb(real)
        fake = tf.image.grayscale_to_rgb(fake)

    # ----- Preprocess for Inception (–1 to +1) ----- #
    real_pp = preprocess_input(real)
    fake_pp = preprocess_input(fake)

    # ----- Extract Features in Batches ----- #
    def _get_acts(x):
        acts = []
        n = x.shape[0]
        for i in range(0, n, batch_size):
            chunk = x[i:i+batch_size]
            acts.append(_inception_model(chunk, training=False).numpy())
        return np.vstack(acts)

    act_real = _get_acts(real_pp)
    act_fake = _get_acts(fake_pp)

    # ----- Compute Statistics ----- #
    mu1, sigma1 = act_real.mean(axis=0), np.cov(act_real, rowvar=False)
    mu2, sigma2 = act_fake.mean(axis=0), np.cov(act_fake, rowvar=False)
    diff    = mu1 - mu2
    covmean = sqrtm(sigma1.dot(sigma2))
    if np.iscomplexobj(covmean):
        covmean = covmean.real
    fid = diff.dot(diff) + np.trace(sigma1 + sigma2 - 2*covmean)
    return float(np.round(fid,4))

# ========== Confirmation Message ========== #
print("Callbacks Defined Successfully")

With reference to the Code Cell above, we are able to verify that that the Callbacks have been successfully defined and the various parameters are set so as to ensure a High Accuracy and Computing Efficient GAN Model Training.

---
### 4.1.3 Defining Vanilla GAN Generator Architecture

In this section, we will be defining the Vanilla GAN's Generator Architecture. The Generator will be trying to make realistic generated images in the attempt to bypass the Discriminator. Subsequently, the generated images from the Generator will be evaluated based on a few evaluation metrics such as Inception Score. The relevent mathematical formulas for the Loss Function of the Vanilla GAN Generator is indicated below:

---

**Generator Loss Formula**

Purpose: Non-saturating Generator Loss encouraging Generator $G$ to Produce Outputs $G(z)$ that Discriminator $D$ Classifies as Real (i.e., closer to 1).

$$
L_G = -\mathbb{E}_{z \sim p_z(z)}[\log D(G(z))]
$$

Where:
- $L_G$ = Generator Loss  
- $z$ = Latent Vector Drawn from Noise Distribution $p_z(z)$ (e.g. Normal or Uniform)  
- $G(z)$ = Generated Fake Image from Noise  
- $D(G(z))$ = Probability Assigned by the Discriminator that the Generated Image is Real  
- $\log D(G(z))$ = Log-likelihood that Fake Image is Considered Real
---
With the relevant Generator Loss Formula indicated above, we will proceed to define the GAN Generator's Architecture in the Code Cell below and the Lost Function.

In [ ]:
# ========== Generator Function ========== #
def build_generator(latent_dim=100, units=256, output_shape=(28, 28, 1)):
    model = tf.keras.Sequential(name="generator")

    model.add(tf.keras.layers.Dense(units, activation='relu', input_dim=latent_dim))
    model.add(tf.keras.layers.BatchNormalization())

    model.add(tf.keras.layers.Dense(units * 2, activation='relu'))
    model.add(tf.keras.layers.BatchNormalization())

    model.add(tf.keras.layers.Dense(np.prod(output_shape), activation='tanh'))
    model.add(tf.keras.layers.Reshape(output_shape))

    return model

# ========== Initialise Loss ========== #
cross_entropy = tf.keras.losses.BinaryCrossentropy(from_logits=False)

# ========== Generator Loss Function ========== #
def generator_loss(fake_output):
    return cross_entropy(tf.ones_like(fake_output), fake_output)

With reference to the Code Cell above, we are able to verify that the GAN's Generator Architecture have been successfully defined and we are able to proceed to define the GAN's Discriminator Architecture.

---
### 4.1.4 Defining Vanilla GAN Discriminator Architecture

In this section, we will be defining the Architecture of the Discriminator of the GAN. The Discriminator will be the Neural Network that will attempt to tell the Generated Images apart from the Actual Images. This will allow the GAN to improve overtime and also improve itself overtime. The mathematical function for the Discriminator Loss Function is indicated below:

---

Purpose: Train Discriminator $D$ to Distinguish between Real and Fake Samples, Rewarding High Confidence in Real Images ($D(x) \to 1$) and Low Confidence in Fake Ones ($D(G(z)) \to 0$).

$$
L_D = -\mathbb{E}_{x \sim p_{\text{data}}(x)}[\log D(x)] - \mathbb{E}_{z \sim p_z(z)}[\log (1 - D(G(z)))]
$$

Where:
- $L_D$ = Discriminator Loss  
- $x$ = Real Image Sampled from the Dataset Distribution $p_{\text{data}}(x)$  
- $z$ = Latent Vector from Prior Noise Distribution $p_z(z)$  
- $G(z)$ = Fake Image Generated by $G$
- $D(x)$ = Discriminator's Predicted Probability that $x$ is Real  
- $D(G(z))$ = Discriminator's Predicted Probability that $G(z)$ is Real  
- $\log D(x)$ = Log-likelihood that Real Image is Correctly Classified  
- $\log(1 - D(G(z)))$ = Log-likelihood that Fake Image is Correctly Rejected
---
With the mathematical formula for the Loss Function indicated above, we will proceed to define the GAN's Discriminator Architecture and the Loss Function in the Code Cell below.

In [ ]:
# ========== Discriminator Function ========== #
def build_discriminator(input_shape=(28, 28, 1), units=256):
    model = tf.keras.Sequential(name="discriminator")

    model.add(tf.keras.layers.Flatten(input_shape=input_shape))

    model.add(tf.keras.layers.Dense(units))
    model.add(tf.keras.layers.LeakyReLU(alpha=0.2))

    model.add(tf.keras.layers.Dense(units // 2))
    model.add(tf.keras.layers.LeakyReLU(alpha=0.2))

    model.add(tf.keras.layers.Dense(1, activation='sigmoid'))

    return model

# ========== Discriminator Loss Function ========== #
def discriminator_loss(real_output, fake_output):
    real_loss = cross_entropy(tf.ones_like(real_output), real_output)
    fake_loss = cross_entropy(tf.zeros_like(fake_output), fake_output)
    return real_loss + fake_loss

With reference to the Code Cell above, we are able to verify that the GAN's Discriminator Architecture have been successfully defined and we are able to proceed to define the GAN's Training Steps.

---
### 4.1.5 Defining Training Step

In this section, we will be Defining the Training Step function so as to train both the Generator and the Discriminator at the same time. The Training Step will compute the gradients for both the Generator and the Discriminator at the same time. The relevant mathematical formulas for the Training Step is indicated below:

---
**Generator Loss:**

Purpose: Encourages the Generator to Create Samples that the Discriminator believes are Real.

$$
L_G = -\mathbb{E}_{z \sim p_z(z)}[\log D(G(z))]
$$

Where:
- $L_G$ = Generator Loss  
- $z$ = Latent Noise Vector Sampled from Prior Distribution $p_z(z)$  
- $G(z)$ = Fake Image Generated from $z$  
- $D(G(z))$ = Discriminator's Probability that the Generated Image is Real
---

Purpose: Train Discriminator $D$ to Distinguish between Real and Fake Samples, Rewarding High Confidence in Real Images ($D(x) \to 1$) and Low Confidence in Fake Ones ($D(G(z)) \to 0$).

$$
L_D = -\mathbb{E}_{x \sim p_{\text{data}}(x)}[\log D(x)] - \mathbb{E}_{z \sim p_z(z)}[\log (1 - D(G(z)))]
$$

Where:
- $L_D$ = Discriminator Loss  
- $x$ = Real Image Sampled from the Dataset Distribution $p_{\text{data}}(x)$  
- $z$ = Latent Vector from Prior Noise Distribution $p_z(z)$  
- $G(z)$ = Fake Image Generated by $G$
- $D(x)$ = Discriminator's Predicted Probability that $x$ is Real  
- $D(G(z))$ = Discriminator's Predicted Probability that $G(z)$ is Real  
- $\log D(x)$ = Log-likelihood that Real Image is Correctly Classified  
- $\log(1 - D(G(z)))$ = Log-likelihood that Fake Image is Correctly Rejected
---

**Gradient Update:**

Purpose: Updates the Generator and Discriminator Parameters using Gradient Descent.

$$
\theta_G \leftarrow \theta_G - \eta \cdot \nabla_{\theta_G} L_G \\
\theta_D \leftarrow \theta_D - \eta \cdot \nabla_{\theta_D} L_D
$$

Where:
- $\theta_G$ = Generator Parameters  
- $\theta_D$ = Discriminator Parameters  
- $\eta$ = Learning Rate  
- $\nabla_{\theta} L$ = Gradient of Loss with Respect to Model Weights
---

With the relevant mathematical formulas indicated above, we will proceed to define the Training Step in the Code Cell below.

In [ ]:
# ========== Train Step Function ========== #
@tf.function
def train_step(real_images, generator, discriminator, gen_optimizer, disc_optimizer, latent_dim):
    batch_size = tf.shape(real_images)[0]
    noise = tf.random.normal([batch_size, latent_dim])

    with tf.GradientTape() as gen_tape, tf.GradientTape() as disc_tape:
        # ----- Forward Pass ----- #
        generated_images = generator(noise, training=True)

        real_output = discriminator(real_images, training=True)
        fake_output = discriminator(generated_images, training=True)

        # ----- Compute Losses ----- #
        gen_loss = generator_loss(fake_output)
        disc_loss = discriminator_loss(real_output, fake_output)

    # ----- Compute Gradients ----- #
    gradients_of_generator = gen_tape.gradient(gen_loss, generator.trainable_variables)
    gradients_of_discriminator = disc_tape.gradient(disc_loss, discriminator.trainable_variables)

    # ----- Apply Gradients ----- #
    gen_optimizer.apply_gradients(zip(gradients_of_generator, generator.trainable_variables))
    disc_optimizer.apply_gradients(zip(gradients_of_discriminator, discriminator.trainable_variables))

    return gen_loss, disc_loss

With reference to the Code Cell above, we are able to determine that the Training Step have been successfully defined and we are able to proceed to prepare the Training Loop in the next section.

---
### 4.1.6 Defining Training Loop

In this section, we will be defining the Training Loop for the GAN Training. The Training Loop will go through each of the Epoch and train both the Generator and Discriminator simultaneously. The Training Loop will also track the Loss for both the Generator and the Discriminator for each of the Epoch trained. The relevant mathematical formulas are indicated below:

---
**Epoch Loss Averaging:**

Purpose: These are the Average Generator and Discriminator Losses over all Batches in Epoch $t$.

$$
\bar{L}_G^{(t)} = \frac{1}{N} \sum_{i=1}^{N} L_G^{(i)} \\
\bar{L}_D^{(t)} = \frac{1}{N} \sum_{i=1}^{N} L_D^{(i)}
$$

Where:
- $\bar{L}_G^{(t)}$ = Average Generator Loss at Epoch $t$  
- $\bar{L}_D^{(t)}$ = Average Discriminator Loss at Epoch $t$  
- $N$ = Number of Batches in the Dataset  
- $L_G^{(i)}$ = Generator Loss on Batch $i$  
- $L_D^{(i)}$ = Discriminator Loss on Batch $i$

---
**Epoch Iteration:**

Purpose: Describes the Training Loop Logic: for Each Epoch $t$, perform `train_step` for Every Mini-batch.

$$
\text{for } t = 1 \text{ to } T:
\quad \text{for each batch } (x^{(i)}):
\quad \text{train_step}(x^{(i)})
$$

Where:
- $T$: Total Number of Epochs  
- $x^{(i)}$: Real Batch $i$ from the Dataset  
- `train_step`: Function that Updates $G$ and $D$ using that Batch

---
**Generated Image Output:**

Purpose: Sample Random Noise $z$ and Generate Synthetic Image $\hat{x}$ from the Generator. This is used for Visual Monitoring of Model Quality.

$$
z \sim p_z(z) \\
\hat{x} = G(z)
$$

Where:
- $z$: Random Latent Vector  
- $G(z)$: Generated Image from Generator

---
With the mathematical formulas indicated above, we will proceed to define the Training Loop in the Code Cell below in preparation of the GAN Training.

In [ ]:
# ========== Training Loop Function ========== #
def train(dataset, epochs, generator, discriminator, gen_optimizer, disc_optimizer,
          latent_dim, save_path, use_early_stopping=True, steps_per_epoch=None):

    # ----- Prepare FID Reference Sets ----- #
    real_val  = tf.data.Dataset.from_tensor_slices(X_val).batch(50).take(1)
    noise_val = tf.random.uniform([50, latent_dim], minval=-1.0, maxval=1.0)
    noise_val = tf.math.l2_normalize(noise_val, axis=1)

    # ----- FID Early Stopping Settings ----- #
    best_fid    = float('inf')
    no_improve  = 0
    patience    = 10
    fid_interval = 1

    # ----- Create & Compile Dummy Model ----- #
    dummy_input  = Input(shape=(1,))
    dummy_output = Lambda(lambda x: x)(dummy_input)
    dummy_model  = Model(dummy_input, dummy_output)
    dummy_model.compile(optimizer=Adam(1e-4), loss='mse')

    # ----- Attach Dummy Model to Callbacks ----- #
    early_stop.set_model(dummy_model)
    reduce_lr.set_model(dummy_model)
    lr_scheduler.set_model(dummy_model)

    # ----- Simulate Keras Callback Start ----- #
    logs = {}
    early_stop.on_train_begin(logs=logs)
    reduce_lr.on_train_begin(logs=logs)
    lr_scheduler.on_train_begin(logs=logs)

    # ----- Initialise Loss Trackers ----- #
    gen_loss_history  = []
    disc_loss_history = []

    # ----- Begin Training Loop ----- #
    for epoch in range(1, epochs + 1):
        print(f"\nEpoch {epoch}/{epochs}")

        gen_loss_epoch  = []
        disc_loss_epoch = []

        for i, real_images in enumerate(dataset):
            if steps_per_epoch and i >= steps_per_epoch:
                break

            # ----- One step of generator & discriminator updates ----- #
            gen_loss, disc_loss = train_step(
                real_images, generator, discriminator,
                gen_optimizer, disc_optimizer, latent_dim
            )
            gen_loss_epoch.append(gen_loss)
            disc_loss_epoch.append(disc_loss)

        # ----- Epoch Statistics ----- #
        avg_gen_loss  = tf.reduce_mean(gen_loss_epoch)
        avg_disc_loss = tf.reduce_mean(disc_loss_epoch)

        print(f"Generator Loss: {avg_gen_loss:.4f} | Discriminator Loss: {avg_disc_loss:.4f}")

        # ----- Track Loss History ----- #
        gen_loss_history.append(float(avg_gen_loss))
        disc_loss_history.append(float(avg_disc_loss))

        # ----- Simulate Callback Updates on Generator Loss ----- #
        logs = {'loss': float(avg_gen_loss)}
        if use_early_stopping:
            early_stop.on_epoch_end(epoch=epoch, logs=logs)
            if early_stop.stopped_epoch > 0:
                print(f"Early stopping triggered at epoch {epoch}")
                break
        reduce_lr.on_epoch_end(epoch=epoch, logs=logs)
        lr_scheduler.on_epoch_end(epoch=epoch, logs=logs)

        # ----- FID Evaluation & Early Stopping ----- #
        if epoch % fid_interval == 0:
            fake_val   = generator(noise_val, training=False)
            real_batch = next(iter(real_val))

            real_for_fid = tf.image.resize(real_batch, [299,299]) * 0.5 + 0.5
            fake_for_fid = tf.image.resize(fake_val,    [299,299]) * 0.5 + 0.5

            fid_value = calculate_fid(real_for_fid, fake_for_fid)
            print(f"Epoch {epoch} → Val FID: {fid_value:.2f}")

            if fid_value < best_fid:
                best_fid, no_improve = fid_value, 0
                if save_path:
                    os.makedirs(save_path, exist_ok=True)
                    generator.save_weights(os.path.join(save_path, "best_gen.weights.h5"))
                    discriminator.save_weights(os.path.join(save_path, "best_disc.weights.h5"))
            else:
                no_improve += 1

            if no_improve >= patience:
                print(f"No FID improvement for {patience} epochs, stopping.")
                break

        # ----- Image Visualisation Every 10 Epochs ----- #
        if save_path and epoch % 10 == 0:
            display_generated_images(generator, latent_dim)

    # ----- Restore Best Weights After Training ----- #
    if save_path:
        best_gen_path  = os.path.join(save_path, "best_gen.weights.h5")
        best_disc_path = os.path.join(save_path, "best_disc.weights.h5")
        if os.path.exists(best_gen_path) and os.path.exists(best_disc_path):
            generator.load_weights(best_gen_path)
            discriminator.load_weights(best_disc_path)
            print("Restored best generator and discriminator weights based on lowest FID.")
        else:
            print("Best weights not found. Skipping restore step.")

    return avg_gen_loss, gen_loss_history, disc_loss_history


With reference to the Code Cell above, we are able to determine that the Training Loop have been successfully defined and we are able to proceed to define the Optimisers with Tunable Parameter in preparation for the Optuna Tuning.

---
### 4.1.7 Defining Optimisers with Tunable Parameters

In this section, we will be defining the Common Optimisers that will be Tuned using the Optuna Tuning. The Optimisers will each have a range of values in a list so as to allow for the Optuna to search for the best Hyperparameter for the GAN. Through these parameters, it will affect how effective and efficiently the GAN will learn from the provided EMNIST Data. The relevent mathematical formula is indicated below:

---
**Optimiser Update Rule**

Purpose: Parameter Update Step in the Adam Optimiser, using Adaptive Learning Rates with Momentum.

$$
\theta \leftarrow \theta - \eta \cdot \frac{m_t}{\sqrt{v_t} + \epsilon}
$$

Where:
- $\theta$ = Model Parameters (Generator or Discriminator)
- $\eta$ = Learning Rate (Tuned)
- $m_t$ = First Moment Estimate (Mean of Gradients)
- $v_t$ = Second Moment Estimate (Variance of Gradients)
- $\epsilon$ = Small Constant for Numerical Stability

---
With the mathematical formula indicated above, we will proceed to define the Optimisers in the Code Cell.

In [ ]:
def objective(trial):
    # Hyperparameters
    latent_dim    = trial.suggest_categorical('latent_dim', [64, 100, 128])
    gen_units     = trial.suggest_categorical('gen_units', [128, 256, 512])
    disc_units    = trial.suggest_categorical('disc_units', [128, 256, 512])
    learning_rate = trial.suggest_float('learning_rate', 1e-5, 1e-3, log=True)
    beta_1        = trial.suggest_float('beta_1', 0.5, 0.9)

    # Build Models
    generator     = build_generator(latent_dim=latent_dim, units=gen_units)
    discriminator = build_discriminator(units=disc_units)

    # Optimisers
    gen_optimizer = Adam(learning_rate=learning_rate, beta_1=beta_1)
    disc_optimizer= Adam(learning_rate=learning_rate, beta_1=beta_1)

    # Init optimisers
    _ = gen_optimizer.apply_gradients([(tf.zeros_like(v), v) for v in generator.trainable_variables])
    _ = disc_optimizer.apply_gradients([(tf.zeros_like(v), v) for v in discriminator.trainable_variables])

    # Load data
    dataset = load_data()
    steps   = min(1000, len(X_train)//32)

    # Train
    _ = train(dataset, epochs=100,
              generator=generator,
              discriminator=discriminator,
              gen_optimizer=gen_optimizer,
              disc_optimizer=disc_optimizer,
              latent_dim=latent_dim,
              steps_per_epoch=steps,
              save_path=False
              )

    # Evaluate with FID
    z = tf.random.uniform([1000, latent_dim], -1.0, 1.0)
    fake = generator(z, training=False)
    fake = (fake + 1.0)/2.0
    real = X_val[:1000]
    fid_score = calculate_fid(real, fake)
    return fid_score

With reference to the Code Cell above, we are able to determine that the Optimiser have been successfully defined with a range of different Parameter Values so as to provide a robust Optuna Tuning in the subsequent sections.

---
### 4.1.8 Optuna Tuning Study

In this sub-section, we will be conducting the Optuna Tuning Study on the GAN Model so as to obtain the best Hyperparameter for the GAN Model. It is expected that the Tuning takes a prolonged duration due to the nature of how the GAN works and the 2 Neural Networks involved. However, the training data will be stored so as to prevent re-running of repetitive codes. The Optuna Tuning Study will be conducted in the Code Cell below.

In [ ]:
# ========== Create or Load Study with SQLite Backend ========== #
study = optuna.create_study(
    direction="minimize",
    study_name="vanilla_gan_tuning",
    storage="sqlite:////content/drive/MyDrive/Colab Notebooks/DELE CA2 A/Non-Augmented GAN Tunings/vanilla_gan_optuna.db",
    load_if_exists=True
)

# ========== Trial Management ========== #
MAX_TRIALS = 50
completed_trials = len([t for t in study.trials if t.state == optuna.trial.TrialState.COMPLETE])
remaining_trials = MAX_TRIALS - completed_trials

if remaining_trials > 0:
    print(f"Resuming study: {completed_trials} completed, running {remaining_trials} more.")
    study.optimize(objective, n_trials=remaining_trials)
else:
    print(f"Study already completed {MAX_TRIALS} trials. Skipping optimisation.")

# ========== Output Best Trial ========== #
best_trial = study.best_trial
best_params_df = pd.DataFrame([best_trial.params])
best_params_df["Final FID"] = best_trial.value
best_params_df.style.background_gradient(cmap="Blues")

With reference to the Code Cell above, we are able to view the Best Hyperparameter obtained during the Optuna Tuning Study. This Hyperparameter will be extracted and be trained for a longer period of time so as to attempt to increase the performance and limit the loss for the GAN.

---
### 4.1.9 GAN Re-Training Operation

As obtained from the previous sub-section, we will be re-training the GAN using the best Hyperparameter obtained during the Optuna Tuning Study. This is to push the GAN Model to its limits and also to save Computational Power as we are not taking up prolonged periods of time tuning GAN Models which may not have any clear signs of good performance. The retraining of the GAN Model will be conducted in the Code Cell below.

In [ ]:
# =========== Extract Best Parameters from Study =========== #
best_params = study.best_trial.params

latent_dim = best_params['latent_dim']
gen_units = best_params['gen_units']
disc_units = best_params['disc_units']
learning_rate = best_params['learning_rate']
beta_1 = best_params['beta_1']

print("Using best trial parameters:")
print(best_params)

# =========== Rebuild Models =========== #
generator = build_generator(latent_dim=latent_dim, units=gen_units)
discriminator = build_discriminator(units=disc_units)

gen_optimizer  = tf.keras.optimizers.Adam(learning_rate=learning_rate, beta_1=beta_1)
disc_optimizer = tf.keras.optimizers.Adam(learning_rate=learning_rate, beta_1=beta_1)

# =========== Prebuild Optimisers =========== #
_ = gen_optimizer.apply_gradients([(tf.zeros_like(var), var) for var in generator.trainable_variables])
_ = disc_optimizer.apply_gradients([(tf.zeros_like(var), var) for var in discriminator.trainable_variables])

# =========== Reload Dataset =========== #
dataset = load_data()

# =========== Define Model Name =========== #
model_name = "vanilla_gan"  # Change this per model type
output_dir = f"/content/drive/MyDrive/Colab Notebooks/DELE CA2 A/final_outputs_{model_name}_noAugment"
os.makedirs(output_dir, exist_ok=True)

# =========== Define Save Paths =========== #
gen_weights_path   = os.path.join(output_dir, "best_gen.weights.h5")
disc_weights_path  = os.path.join(output_dir, "best_disc.weights.h5")
loss_csv_path      = os.path.join(output_dir, "loss_history.csv")

# =========== Check for Existing Files =========== #
if os.path.exists(gen_weights_path) and os.path.exists(disc_weights_path) and os.path.exists(loss_csv_path):
    print("All outputs found. Skipping training...")

    # ----- Load Weights ----- #
    generator.load_weights(gen_weights_path)
    discriminator.load_weights(disc_weights_path)
    print("Generator and Discriminator Weights Loaded.")

    # ----- Load Loss History ----- #
    loss_df = pd.read_csv(loss_csv_path)
    gen_loss_history = loss_df['gen_loss'].tolist()
    disc_loss_history = loss_df['disc_loss'].tolist()
    final_gen_loss = gen_loss_history[-1]

else:
    # =========== Train =========== #
    final_gen_loss, gen_loss_history, disc_loss_history = train(
        dataset=dataset,
        epochs=1200,
        generator=generator,
        discriminator=discriminator,
        gen_optimizer=gen_optimizer,
        disc_optimizer=disc_optimizer,
        latent_dim=latent_dim,
        use_early_stopping=False,
        save_path=output_dir
    )

    # =========== Save Final Weights =========== #
    generator.save_weights(gen_weights_path)
    discriminator.save_weights(disc_weights_path)
    print("Generator and Discriminator Weights Saved.")

    # =========== Save Loss History =========== #
    loss_df = pd.DataFrame({
        'epoch': list(range(1, len(gen_loss_history) + 1)),
        'gen_loss': gen_loss_history,
        'disc_loss': disc_loss_history
    })
    loss_df.to_csv(loss_csv_path, index=False)
    print(f"Loss history saved to {loss_csv_path}")

With reference to the Code Cell above, we are able to determine that the best GAN Model have been trained and saved successfully and we are able to proceed with the Model Evaluation in the next sub-sections where we determine how well the GAN Model performed based on various metrics and evaluation methodologies.

---
### 4.1.10 GAN Model Evaluation

In this sub-section, we will be evaluating the GAN's Generator performance on various aspects. We will be utilising different methodologies and metrics to attempt to quantify the performance of the GAN Generator while also conducting Visual Inspection on the result output of the GAN. The Evaluation will be conducted in the following sections.

---
#### 4.1.10.1 Visual Grid of Samples

In this sub-section, we will be visualising the Grid of Samples of the Synthesised Data generated by the GAN Model. A list of observations will be made before the utilisation of Quantitative Metrics to evaluate the GAN Generator's general performance. As the observations made are purely from professional opinion, it will most likely not be used as a Basis of Comparison between GAN Models due to differing opinions unless in extreme cases. The visualisation will be conducted in the Code Cell below.

In [ ]:
# ========== Sample Generation Function ========== #
def plot_generated_images(generator, latent_dim, n_rows=5, n_cols=5, save_path=None, title="Vanilla GAN Generated Samples (Non-Augmented)"):
    noise = tf.random.normal([n_rows * n_cols, latent_dim])
    gen_images = generator(noise, training=False)

    # ----- Rescale from [-1, 1] to [0, 1] ----- #
    gen_images = (gen_images + 1.0) / 2.0
    gen_images = tf.clip_by_value(gen_images, 0.0, 1.0)

    # ----- Check Shape ----- #
    assert gen_images.shape[-1] == 1, "Expected single-channel (grayscale) output"

    # ----- Plot Generated Image Grid ----- #
    fig, axes = plt.subplots(n_rows, n_cols, figsize=(n_cols, n_rows))
    for i in range(n_rows * n_cols):
        ax = axes[i // n_cols, i % n_cols]
        ax.imshow(gen_images[i, :, :, 0], cmap='gray')
        ax.axis('off')

    # ----- Title & Layout ----- #
    plt.suptitle(title, fontsize=16)
    plt.tight_layout(rect=[0, 0.03, 1, 0.95])

    # ----- Save if Requested ----- #
    if save_path:
        plt.savefig(save_path, dpi=300)
        print(f"Saved Generated Image Grid to: {save_path}")

    plt.show()

# ========== Plot Function Call ========== #
plot_generated_images(generator, latent_dim=100, n_rows=10, n_cols=16, save_path=None, title="Vanilla GAN Generated Samples")

With reference to the output of the Samples above, we will proceed to list the various observations below:

**Character Recognisability**

- Several generated characters such as 'A', 'E', 'F', 'J', 'O', 'P', 'X', and 'Z' are generally recognisable.

- The majority of characters maintain basic stroke structure and symmetry.

**Variation in Image Quality**

- Some images are well-formed and clean, while others exhibit noise, distortions, or incomplete strokes.

- Certain samples display irregularities such as jagged edges or uneven contrast.

**Evidence of Mode Collapse**

- Repetition of visually similar characters is noticeable in certain rows, indicating potential mode collapse.

- Diversity of generated outputs may be limited for specific classes.

**Positioning and Scaling Issues**

- Most characters are reasonably centred, but a few appear shifted, misaligned, or improperly scaled.

- Inconsistent stroke thickness is observed across the image grid.

**Synthetic Appearance**

- A subset of characters presents a visibly synthetic look, with unrealistic curves or fragmented strokes.

- This suggests the model may require further training or architectural adjustments.

With the observations indicated above, we will proceed to conduct the next Model Evaluation operation in the next sub-section.

---
#### 4.1.10.2 Loss Curve Over Epochs

In this sub-section, we will be plotting the GAN Loss Curve over Epochs to evaluate the training behaviour of the GAN. This curve provides insight into whether the GAN is experiencing underfitting, overfitting, or achieving a stable training dynamic between the generator and discriminator.

Although GANs do not minimise a single unified loss in the traditional sense, the trajectory of the generator and discriminator losses can indicate whether the adversarial training process is converging appropriately.

If the generator loss remains high while discriminator loss quickly drops to near-zero, this may indicate **underfitting**, where the generator is not learning effectively to produce plausible images.

If the discriminator loss increases while generator loss sharply decreases, this may suggest **overfitting** or **mode collapse**, where the generator exploits narrow weaknesses in the discriminator without general improvement.

A relatively stable oscillation or convergence between both losses typically signifies a **well-balanced** training process, where the generator and discriminator are learning in tandem.

The Potential Observations and Corresponding Action Plans are outlined below:

**Underfitting Loss Curve**

- Increase training epochs to allow generator more time to learn.

- Simplify the discriminator to reduce overpowering the generator.

- Adjust the learning rate or apply label smoothing.

- Consider adding batch normalisation in the generator.

**Overfitting or Mode Collapse Loss Curve**

- Introduce or increase dropout in the discriminator.

- Apply input noise or label flipping to the discriminator.

- Evaluate diversity of generated samples regularly.

**Balanced/Ideal Loss Curve**

- Maintain current architecture and hyperparameters.

- Proceed with full-scale image generation.

- Evaluate both qualitative (visual inspection) and quantitative metrics (e.g. Inception Score or FID).

With these potential training behaviours and action plan outlined, we will proceed to visualise the GAN loss dynamics in the code cell below.



In [ ]:
# ========== Plot Size Configuration ========== #
plt.figure(figsize=(10, 6))

# ========== Loss Curve Plot ========== #
plt.plot(gen_loss_history, label="Generator Loss", linewidth=2, color='blue')
plt.plot(disc_loss_history, label="Discriminator Loss", linewidth=2, color='orange')

# ========== Labels and Title ========== #
plt.xlabel("Epoch", fontsize=12)
plt.ylabel("Loss", fontsize=12)
plt.title("Vanilla GAN Training Loss Curve (Non-Augmented)", fontsize=14)

# ========== Grid and Size Configurarion ========== #
plt.grid(True, linestyle='--', alpha=0.6)
plt.legend(fontsize=12)
plt.xticks(fontsize=10)
plt.yticks(fontsize=10)

# ========== Plot Display ========== #
plt.tight_layout()
plt.show()

With reference to the output of the Learning Curve visualisation above, we are able to make the following observations below:

**Initial Instability**

- Both Generator and Discriminator losses exhibit sharp fluctuations in the initial epochs, which is typical during early adversarial training.

**Discriminator Dominance**

- The Discriminator Loss remains significantly higher than the generator loss throughout training, suggesting the Discriminator is stronger and more confident in distinguishing real from fake images.

**Gradual Increase in Generator Loss**

- The Generator loss shows a slow upward trend across epochs, indicating that it is struggling to improve image quality or effectively fool the Discriminator.

**Stable but Divergent Loss Patterns**

- After the initial phase, both losses stabilise but do not converge. This suggests a suboptimal equilibrium, where the Generator is not learning as effectively as desired.

**No Clear Oscillatory Behaviour**

- The expected adversarial oscillations between generator and discriminator are largely absent, implying limited competition between the two networks.

**Potential Underfitting**

- The persistent gap between the Generator and Discriminator Losses, along with the increasing Generator loss, suggests the Generator may be Underfitting or unable to keep pace with the Discriminator.

With the observations indicated above, we will proceed to conduct the next GAN Model Evaluation in the next sub-section.

---
#### 4.1.10.3 Quantitative Metrics Evaluation

In this sub-section we will be utilising Quantitative Metrics such as FID and KID to tabulate the results of the performance and quantify the performance of the GAN Generator. We will subsequently be use these metrics to conduct inter-model evaluation after tuning and training all the GANs. The metrics that we will be utilising and their respective formulas are indicated below:

---
**Fréchet Inception Distance**

Purpose: Measures the Distance between the Real-Image and Generated-Image Distributions in Inception Feature Space.

$$
\mathrm{FID} \;=\;\|\mu_r - \mu_f\|^2
\;+\;\mathrm{Tr}\Bigl(\Sigma_r + \Sigma_f - 2\,(\Sigma_r\,\Sigma_f)^{\tfrac12}\Bigr)
$$


Where:
- $\mu_r = \mathbb{E}[f(x)]$, $\Sigma_r = \mathrm{Cov}[f(x)]$ for Real Images $x$.  
- $\mu_f = \mathbb{E}[f(\hat x)]$, $\Sigma_f = \mathrm{Cov}[f(\hat x)]$ for Generated Images $\hat x$.  
- $f(\cdot)$ = Map from Image to its InceptionV3 'pooling=avg' Features.  

---
**Diversity (t-SNE Spread)**

Purpose: Quantifies how 'wide' the 2D t-SNE Embedding of Generated Samples.

$$
\mathrm{Spread}
\;=\;
\bigl(\max_i\,z_i^{(1)} - \min_i\,z_i^{(1)}\bigr)
\;\times\;
\bigl(\max_i\,z_i^{(2)} - \min_i\,z_i^{(2)}\bigr)
$$

Where:
- $z_i = (z_i^{(1)}, z_i^{(2)})$ = 2-dimensional t-SNE Embedding of $i$th Generated Image.

---
**Mode Collapse Risk**

Purpose: Flags when too many Generated Images are Nearly Identical (mode collapse).

$$
\text{ModeCollapseRisk} =
\begin{cases}
\text{Low}, & \dfrac{\bigl|\{\mathrm{unique\_rounded}(x_i)\}\bigr|}{N} > \tau,\\
\text{High}, & \text{otherwise}.
\end{cases}
$$

Where:
- $x_i$ = $N$ Generated Samples.  
- $\mathrm{unique\_rounded}(x_i)$ = Rounds Pixels to Detect Duplicates.  
- $\tau=0.9$ = Uniqueness Threshold (90%).

---
**Perceptual Path Length**

Purpose: Measures Sensitively of Generator's Outputs (in VGG16 Feature Space) move when the Latent Code $z$ is Perturbed.

$$
\mathrm{PPL}
\;=\;
\mathbb{E}_{z,\delta z}\Bigl[\,
\|\phi\bigl(G(z + \epsilon\,\delta z)\bigr)\;-\;\phi\bigl(G(z)\bigr)\|_2^2
\Bigr]
$$

Where:  
- $G(z)$ = GAN Generator Mapping $z\in\mathbb{R}^{\mathrm{latent\_dim}}$ to an Image.  
- $\phi(\cdot)$ = VGG16 'pooling=avg' Feature Extractor.  
- $\delta z\sim\mathcal{N}(0,I)$, $\epsilon$ = Small Constant.

---
**Kernel Inception Distance**

Purpose: An Unbiased Estimator of the Squared Maximum Mean Discrepancy (MMD) between Real and Generated Inception Features.

$$
\mathrm{KID}
\;=\;
\frac{1}{m(m-1)}\sum_{i\neq j} k\bigl(\phi(x_i),\phi(x_j)\bigr)
\;+\;
\frac{1}{n(n-1)}\sum_{i\neq j} k\bigl(\phi(\hat x_i),\phi(\hat x_j)\bigr)
\;-\;
\frac{2}{mn}\sum_{i=1}^m\sum_{j=1}^n k\bigl(\phi(x_i),\phi(\hat x_j)\bigr)
$$

Where:  
- $x_i$ ($i=1\ldots m$) = Real Images.
- $\hat x_j$ ($j=1\ldots n$) = Generated Images.  
- $\phi(\cdot)$ = InceptionV3 Feature Extractor (pooling='avg').  
- $k(u,v) = \bigl(\frac{u^\top v}{d}+1\bigr)^3$ = degree-3 Polynomial Kernel on $d$-dimensional Features.

---

With the mathematical and metrics indicated above, we will proceed to conduct the Quantitave Metrics Evalution in the Code Cell below.

In [ ]:
# ========== Preload InceptionV3 and VGG16 Models ========== #
inception = InceptionV3(include_top=False, pooling='avg', input_shape=(299, 299, 3))
vgg_model = VGG16(include_top=False, weights='imagenet', input_shape=(128, 128, 3))

# ========== Preprocess for Inception ========== #
def preprocess_images_for_inception(images):
    images = (images + 1.0) * 127.5  # Scale from [-1, 1] to [0, 255]
    images = tf.image.resize(images, [299, 299])
    if images.shape[-1] == 1:
        images = tf.image.grayscale_to_rgb(images)
    return preprocess_input(images)

# ========== Compute FID ========== #
def calculate_fid(real_images, fake_images, batch_size=50):
    real_pp = preprocess_images_for_inception(real_images)
    fake_pp = preprocess_images_for_inception(fake_images)

    act1 = inception.predict(real_pp, batch_size=batch_size, verbose=0)
    act2 = inception.predict(fake_pp, batch_size=batch_size, verbose=0)

    mu1, sigma1 = np.mean(act1, axis=0), np.cov(act1, rowvar=False)
    mu2, sigma2 = np.mean(act2, axis=0), np.cov(act2, rowvar=False)

    diff = mu1 - mu2
    covmean = sqrtm(sigma1 @ sigma2)
    if np.iscomplexobj(covmean):
        covmean = covmean.real

    fid = diff @ diff + np.trace(sigma1 + sigma2 - 2 * covmean)
    return round(fid, 4)

# ========== Compute KID ========== #
def polynomial_kernel(X, Y):
    d = X.shape[1]
    return (np.dot(X, Y.T) / d + 1) ** 3

def calculate_kid(real_images, fake_images, batch_size=128):
    real_pp = preprocess_images_for_inception(real_images)
    fake_pp = preprocess_images_for_inception(fake_images)

    real_features = inception.predict(real_pp, batch_size=batch_size, verbose=0)
    fake_features = inception.predict(fake_pp, batch_size=batch_size, verbose=0)

    m = real_features.shape[0]
    n = fake_features.shape[0]

    k_rr = polynomial_kernel(real_features, real_features)
    k_gg = polynomial_kernel(fake_features, fake_features)
    k_rg = polynomial_kernel(real_features, fake_features)

    np.fill_diagonal(k_rr, 0)
    np.fill_diagonal(k_gg, 0)

    mmd = (k_rr.sum() / (m * (m - 1)) +
           k_gg.sum() / (n * (n - 1)) -
           2 * k_rg.mean())
    return round(mmd, 4)

# ========== Compute t-SNE Spread (Diversity) ========== #
def calculate_tsne_spread(images):
    flat = images.reshape(images.shape[0], -1)
    tsne = TSNE(n_components=2, random_state=42)
    proj = tsne.fit_transform(flat)
    x_range = proj[:, 0].max() - proj[:, 0].min()
    y_range = proj[:, 1].max() - proj[:, 1].min()
    return round(x_range * y_range, 2)

# ========== Assess Mode Collapse ========== #
def assess_mode_collapse(images):
    unique = np.unique(np.round(images), axis=0).shape[0]
    return "Low" if unique > 0.9 * images.shape[0] else "High"

# ========== Compute Perceptual Path Length (PPL) ========== #
def calculate_ppl(generator, latent_dim=100, num_samples=50, epsilon=1e-2):
    distances = []
    for _ in range(num_samples):
        z1 = tf.random.normal([1, latent_dim])
        z2 = z1 + epsilon * tf.random.normal([1, latent_dim])
        img1 = generator(z1, training=False)
        img2 = generator(z2, training=False)

        img1 = tf.image.resize(tf.image.grayscale_to_rgb((img1 + 1.0) * 127.5), (128, 128))
        img2 = tf.image.resize(tf.image.grayscale_to_rgb((img2 + 1.0) * 127.5), (128, 128))

        f1 = vgg_model(vgg_preprocess(img1))
        f2 = vgg_model(vgg_preprocess(img2))
        d = tf.reduce_mean(tf.square(f1 - f2)).numpy()
        distances.append(d)
    return round(np.mean(distances), 4)

# ========== Evaluate GAN Performance ========== #
def evaluate_gan_model(generator, latent_dim, X_val, gen_loss_history, disc_loss_history, model_name="Vanilla GAN"):
    # ----- Generate Fake Images ----- #
    noise = tf.random.normal([1000, latent_dim])
    fake_images = generator(noise, training=False)
    fake_images = tf.clip_by_value((fake_images + 1) / 2.0, 0.0, 1.0)
    fake_np = fake_images.numpy()

    # ----- Select and preprocess 1000 Real Images from Validation Set ----- #
    real_images = tf.convert_to_tensor(X_val[:1000], dtype=tf.float32)
    real_images = tf.clip_by_value(real_images, 0.0, 1.0)

    # ----- Compute Metrics ----- #
    fid_value = calculate_fid(real_images, fake_images)
    kid_value = calculate_kid(real_images, fake_images)
    tsne_spread_value = calculate_tsne_spread(fake_np)
    collapse_risk_value = assess_mode_collapse(fake_np)
    ppl_score_value = calculate_ppl(generator, latent_dim)
    mean_gen_loss_value = round(np.mean(gen_loss_history), 4)
    std_gen_loss_value = round(np.std(gen_loss_history), 4)
    mean_disc_loss_value = round(np.mean(disc_loss_history), 4)
    std_disc_loss_value = round(np.std(disc_loss_history), 4)

    # ----- Store as Global Vars (Optional) ----- #
    prefix = model_name.upper()
    globals()[f"{prefix}_FID"] = fid_value
    globals()[f"{prefix}_KID"] = kid_value
    globals()[f"{prefix}_TSNE"] = tsne_spread_value
    globals()[f"{prefix}_MODE_COLLAPSE"] = collapse_risk_value
    globals()[f"{prefix}_PPL"] = ppl_score_value
    globals()[f"{prefix}_GEN_LOSS_MEAN"] = mean_gen_loss_value
    globals()[f"{prefix}_GEN_LOSS_STD"] = std_gen_loss_value
    globals()[f"{prefix}_DISC_LOSS_MEAN"] = mean_disc_loss_value
    globals()[f"{prefix}_DISC_LOSS_STD"] = std_disc_loss_value

    # ----- Final Summary Row ----- #
    row = {
        "Model": model_name,
        "FID Score": fid_value,
        "KID Score": kid_value,
        "Mean Generator Loss": mean_gen_loss_value,
        "Std Generator Loss": std_gen_loss_value,
        "Mean Discriminator Loss": mean_disc_loss_value,
        "Std Discriminator Loss": std_disc_loss_value,
        "Mode Collapse Risk": collapse_risk_value,
        "Visual Quality": "Acceptable",
        "Diversity (t-SNE Spread)": tsne_spread_value,
        "PPL": ppl_score_value
    }

    return pd.DataFrame([row])[[
        "Model", "FID Score", "KID Score",
        "Mean Generator Loss", "Std Generator Loss",
        "Mean Discriminator Loss", "Std Discriminator Loss",
        "Mode Collapse Risk", "Visual Quality",
        "Diversity (t-SNE Spread)", "PPL"
    ]]

# ========== Create DataFrame ========== #
df = evaluate_gan_model(generator, latent_dim, X_val, gen_loss_history, disc_loss_history, model_name="Vanilla GAN (Non-Augmented)")

# ========== Display DataFrame ========== #
df.style.background_gradient(cmap="Blues")

With reference to the output of the Quantitative Metrics above, we are able to determine the performance of the GAN. This will be used as a basis of comparison to the other GAN Models.

---
#### 4.1.10.4 t-SNE of Generated Samples

In this sub-section, we visualise the t-SNE projection of generated samples to evaluate the latent diversity learned by the unconditional GAN. Although our GAN is not class-conditioned, t-SNE remains a valuable tool to assess whether the generator is producing varied outputs that reflect meaningful use of the latent space. While GANs are trained in high-dimensional spaces, t-SNE enables us to observe 2D structural patterns such as sample clustering, separation, and density, which provide indirect insights into the diversity and generalisation capabilities of the generator. The visualisation will be conducted in the Code Cell below.

In [ ]:
# ========== Generate New Images From Random Noise ========== #
latent_dim = 100
noise = tf.random.normal([500, latent_dim])
generated_images = generator(noise, training=False)
generated_images = (generated_images + 1) / 2.0

def plot_tsne_embeddings(generated_images, save_path=None, title="t-SNE of Vanilla GAN Generated Samples (Non-Augmented)"):
    # ----- Flatten Generated Images ----- #
    flat_images = generated_images.numpy().reshape(generated_images.shape[0], -1)

    # ----- Apply t-SNE ----- #
    tsne = TSNE(n_components=2, perplexity=30, learning_rate='auto', init='pca', random_state=42)
    tsne_proj = tsne.fit_transform(flat_images)

    # ----- Plot t-SNE ----- #
    plt.figure(figsize=(12, 6))
    sns.scatterplot(
        x=tsne_proj[:, 0],
        y=tsne_proj[:, 1],
        s=20,
        alpha=0.9,
        edgecolor='none'
    )
    plt.title(title, fontsize=14, weight='bold')
    plt.xlabel("t-SNE Dimension 1")
    plt.ylabel("t-SNE Dimension 2")
    plt.grid(True, linestyle='--', alpha=0.3)
    plt.tight_layout()
    plt.show()

# ========== Call Plot Function ========== #
plot_tsne_embeddings(generated_images=generated_images)

With reference to the t-SNE plot of generated samples (Non-Augmented) above, we are able to make the following observations:

**Spread Across Latent Space**

- The data points are broadly distributed across the 2D plane.

- There is no excessive clustering in any one region, suggesting reasonable coverage of the latent space.

**Lack of Clear Clustering**

- No obvious clusters or tight groupings are visible.

- This implies limited semantic separability, i.e. generated samples do not naturally group into distinct character types.

**Mild Uniformity**

- The spread appears relatively balanced, with few voids or dense regions.

- Indicates the generator is sampling diverse regions of the latent space, even if output diversity is visually limited.

**Absence of Outliers**

- There are few, if any, extreme outliers.

- Suggests stable generation without major breakdowns or malformed data points in the embedding space.

With the observations indicated above, we will proceed to train the next GAN Model in the next sub-section.

---
## 4.2 Deep Convolutional GAN Model Training

In this sub-section, we will be training a Deep Convolutional GAN (DCGAN) for the EMNIST Dataset. The DCGAN mainly replaces the Fully Connected Layers for Generating Images with the Conv2D and Conv2DTranspose Layers similar to a Convolutional Neural Network (CNN). Through these layers, the DCGAN will be able to Capture Spatial Hierarchies, Preserve Image Locality and most importantly, produce Sharper and More Structured Output. Therefore, it is expected that DCGAN perform better than all the other GAN Models. The relevant mathematical formulas are indicated below:

---
**Minimax Objective (Vanilla GAN Loss):**

Purpose: Original Adversarial Objective where Generator $G$ tries to Fool Discriminator $D$, while $D$ tries to Distinguish Real Samples from Fake Ones.

$$
\min_G \max_D V(D, G) = \mathbb{E}_{x \sim p_{\text{data}}(x)}[\log D(x)] + \mathbb{E}_{z \sim p_z(z)}[\log (1 - D(G(z)))]
$$

Where:
- $G$ = Generator Network
- $D$ = Discriminator Network
- $x$ = Real Data Sample
- $z$ = Random Noise Vector
- $p_{\text{data}}(x)$ = Distribution of Real Data
- $p_z(z)$ = Prior Distribution of Noise (e.g., $\mathcal{N}(0, I)$)
- $D(x)$ = Probability that $x$ is Real
- $D(G(z))$ = Probability that Generated Image is Real

---
**Generator Loss (Binary Crossentropy):**

Purpose: The Generator is Rewarded when Discriminator Assigns High Probability to Fake Images.

$$
\mathcal{L}_{G} = -\mathbb{E}_{z \sim p_z}[\log(D(G(z)))]
$$

Where:
- $z$ = Random Noise Input
- $G(z)$ = Fake Image
- $D(G(z))$ = Discriminator's Probability that $G(z)$ is Real

---
**Discriminator Loss (Binary Crossentropy):**

Purpose: The Discriminator Assign 1 to Real Images and 0 to Generated Images.

$$
\mathcal{L}_{D} = -\mathbb{E}_{x \sim p_{\text{data}}}[\log D(x)] - \mathbb{E}_{z \sim p_z}[\log(1 - D(G(z)))]
$$

**Where:**
- $D(x)$: Output for Real EMNIST Images
- $D(G(z))$: Output for Generated Images

---
**LeakyReLU Activation Function:**

Purpose: Prevents 'dying ReLU' Problem in the Generator/Discriminator by allowing a Small Gradient for Negative Inputs, Ensuring Neurons Stay Active.

$$
f(x) = \begin{cases}
x & \text{if } x > 0 \\
\alpha x & \text{if } x \leq 0
\end{cases}
$$

Where:
- $x$: Input Value
- $\alpha$: Small Slope Constant

---
**Tanh Activation Function (Output of Generator):**

Purpose: The Generator uses $\tanh$ to Output Pixel Values between -1 and 1, which matches the Normalised EMNIST Dataset Format.

$$
\tanh(x) = \frac{e^x - e^{-x}}{e^x + e^{-x}}
$$

Where:
- $x$: Output of Final Generator Layer

---
**Normalisation Formula (for Input Scaling):**

Purpose: Scaling is Required because DCGAN uses $\tanh$ Activation

$$
x_{\text{scaled}} = \frac{x - 127.5}{127.5}
$$

Where:
- $x$: Original Grayscale Pixel Value in [0, 255]

---
With the relevant mathematical formulas indicated above, we will proceed to train the DCGAN in this sub-section for the purpose of it's potential for high accuracy due to its close relation with CNN and also to serve as a comparison for our other GAN Models.

---
### 4.2.1 Defining Data Pre-Processing Function

In this sub-section, we will be defining the Data Pre-Processing Function which will allow for consistent Data Pipeline to be built and modifed accordingly should the need arise such as when we need to add Spectral Normalisation. This will ensure that the Data's integrity is not affected and we are able to make fair and un-biased assumptions. The Data Pre-Processing Function is defined in the Code Cell below:

In [ ]:
# ========== Data Pre-Processing Function ========== #
def load_data(X_train, batch_size=32):
    buffer_size = X_train.shape[0]

    dataset = tf.data.Dataset.from_tensor_slices(X_train)
    dataset = dataset.shuffle(buffer_size)
    dataset = dataset.batch(batch_size)
    dataset = dataset.prefetch(tf.data.AUTOTUNE)

    return dataset

With reference to the Code Cell above, we are able to determine that the Data Pre-Processing Function have been defined successfully and we are able to move on to Defining Callback Functions in the next sub-section.

---
### 4.2.2 Defining Callback Functions

In this sub-section, we will be pre-defining the various Callbacks. This is to ensure consistency throughout the model and also to increase the GAN Accuracy and optimise the Computation Cost for training each GAN Model. These Callbacks will be main used during the GAN Training in the next sub-section. The formulas and logic for these Callbacks are indicated below:

---
**Learning Rate Scheduler:**

Purpose: Reduces Learning Rate During Training to Fine-tune Model Convergence.

$$\text{If } epoch \mod 10 = 0 \Rightarrow \eta_{new} = \frac{1}{2} \cdot \eta$$

Where:

- $\eta$ = current learning rate

- $\eta_{\text{new}}$ = updated learning rate

- $epoch \bmod 10$ checks if the epoch is a multiple of 10

---
**ReduceLROnPlateau:**

Purpose: Automatically Reduces the Learning Rate when Validation Performance Plateaus.

$$
\text{If no improvement in } val\_loss \text{ for 10 epochs: } \eta_{\text{new}} = 0.5 \cdot \eta
$$

Where:

- $\eta_{\text{new}}$ = reduced learning rate

---
**EarlyStopping:**

Purpose: Prevents Wastage of Computational Power if Model Does Not Improve.

$$
\text{If } val\_loss \text{ does not improve for 25 epochs, stop training and restore best weights}
$$

---
With the formulas and Various Callbacks listed, we will proceed to Define the Callbacks in the Code Cell below in preparation for the GAN Model Training.

In [ ]:
# ========== Define Learning Rate Scheduler ========== #
lr_scheduler = LearningRateScheduler(
    lambda epoch, lr: lr * 0.95 if epoch % 10 == 0 else lr,
    verbose=1
)

# ========== Define Reduce LR on Plateau ========== #
reduce_lr = ReduceLROnPlateau(
    monitor='loss',
    factor=0.5,
    patience=10,
    verbose=1,
    min_lr=1e-6
)

# ========== Define Early Stopping ========== #
early_stop = EarlyStopping(
    monitor='loss',
    patience=25,
    verbose=1,
    restore_best_weights=False
)

# ========== Define Display Generated Images ========== #
def display_generated_images(generator, latent_dim, n=7):
    noise = tf.random.normal([n * n, latent_dim])
    generated_images = generator(noise, training=False)
    generated_images = (generated_images + 1.0) / 2.0

    fig, axes = plt.subplots(n, n, figsize=(n, n))
    for i in range(n):
        for j in range(n):
            img = generated_images[i * n + j, :, :, 0]
            axes[i, j].imshow(img, cmap='gray')
            axes[i, j].axis('off')

    plt.tight_layout()
    plt.show()

# ========== Define FID Calculator ========== #
_inception_model = InceptionV3(
    include_top=False,
    pooling='avg',
    input_shape=(299,299,3)
)

def calculate_fid(real_images, fake_images, batch_size=10):

    # ----- Scale to [0,255] and Resize to 299×299 ----- #
    real = (real_images + 1.0) * 127.5
    fake = (fake_images + 1.0) * 127.5
    real = tf.image.resize(real, (299,299))
    fake = tf.image.resize(fake, (299,299))

    # ----- If Grayscale, Convert to RGB ----- #
    if real.shape[-1] == 1:
        real = tf.image.grayscale_to_rgb(real)
        fake = tf.image.grayscale_to_rgb(fake)

    # ----- Preprocess for Inception (–1 to +1) ----- #
    real_pp = preprocess_input(real)
    fake_pp = preprocess_input(fake)

    # ----- Extract Features in Batches ----- #
    def _get_acts(x):
        acts = []
        n = x.shape[0]
        for i in range(0, n, batch_size):
            chunk = x[i:i+batch_size]
            acts.append(_inception_model(chunk, training=False).numpy())
        return np.vstack(acts)

    act_real = _get_acts(real_pp)
    act_fake = _get_acts(fake_pp)

    # ----- Compute Statistics ----- #
    mu1, sigma1 = act_real.mean(axis=0), np.cov(act_real, rowvar=False)
    mu2, sigma2 = act_fake.mean(axis=0), np.cov(act_fake, rowvar=False)
    diff    = mu1 - mu2
    covmean = sqrtm(sigma1.dot(sigma2))
    if np.iscomplexobj(covmean):
        covmean = covmean.real
    fid = diff.dot(diff) + np.trace(sigma1 + sigma2 - 2*covmean)
    return float(np.round(fid,4))

# ========== Confirmation Message ========== #
print("DCGAN Callbacks and Visualisation Setup Complete")

With reference to the Code Cell above, we are able to verify that that the Callbacks have been successfully defined and the various parameters are set so as to ensure a High Accuracy and Computing Efficient GAN Model Training.

---
### 4.2.3 Defining DCGAN Generator Architecture

In this section, we will be defining the Vanilla GAN's Generator Architecture. The Generator will be trying to make realistic generated images in the attempt to bypass the Discriminator. Subsequently, the generated images from the Generator will be evaluated based on a few evaluation metrics such as Inception Score. The relevent mathematical formulas for the Loss Function of the Vanilla GAN Generator is indicated below:

---

**Generator Loss (Binary Crossentropy):**

Purpose: The Generator is Rewarded when Discriminator Assigns High Probability to Fake Images.

$$
\mathcal{L}_{G} = -\mathbb{E}_{z \sim p_z}[\log(D(G(z)))]
$$

Where:
- $z$ = Random Noise Input
- $G(z)$ = Fake Image
- $D(G(z))$ = Discriminator's Probability that $G(z)$ is Real

---
With the relevant Generator Loss Formula indicated above, we will proceed to define the GAN Generator's Architecture in the Code Cell below and the Lost Function.

In [ ]:
def build_generator(latent_dim=100, use_dropout=True):
    model = tf.keras.Sequential(name="generator")

    # ----- Project and Reshape ----- #
    model.add(tf.keras.layers.Dense(7 * 7 * 256, use_bias=False, input_shape=(latent_dim,)))
    model.add(tf.keras.layers.BatchNormalization())
    model.add(tf.keras.layers.ReLU())
    model.add(tf.keras.layers.Reshape((7, 7, 256)))

    # ----- Upsample 1 ----- #
    model.add(tf.keras.layers.Conv2DTranspose(128, kernel_size=5, strides=1, padding='same', use_bias=False))
    model.add(tf.keras.layers.BatchNormalization())
    model.add(tf.keras.layers.ReLU())
    if use_dropout:
        model.add(tf.keras.layers.Dropout(0.3))

    # ----- Upsample 2 ----- #
    model.add(tf.keras.layers.Conv2DTranspose(64, kernel_size=5, strides=2, padding='same', use_bias=False))
    model.add(tf.keras.layers.BatchNormalization())
    model.add(tf.keras.layers.ReLU())
    if use_dropout:
        model.add(tf.keras.layers.Dropout(0.3))

    # ----- Final Output to 28x28x1 ----- #
    model.add(tf.keras.layers.Conv2DTranspose(1, kernel_size=5, strides=2, padding='same', activation='tanh'))

    return model

# ========== Define Generator Loss ========== #
loss_fn = tf.keras.losses.BinaryCrossentropy(from_logits=False)

def generator_loss(fake_output):
    labels = tf.ones_like(fake_output) * tf.random.uniform(tf.shape(fake_output), minval=0.9, maxval=1.0)
    return loss_fn(labels, fake_output)

With reference to the Code Cell above, we are able to verify that the GAN's Generator Architecture have been successfully defined and we are able to proceed to define the GAN's Discriminator Architecture.

---
### 4.2.4 Defining Spectral Normalisation

In this section, we will be defining the Spectral Normalisation layer for the Discriminator of the GAN. The Spectral Normalisation controls the Lipschitz constant of the Discriminator by constraining the Spectral Norm of each weight matrix. This will prevent the Discriminator from being too powerful and dominating the Generator which can cause various issues such as Mode Collaspe, Training Instability and Exploding Gradients. The relevant mathematical formula is indicated below:

---
**Spectral Normalisation (SN)**

Purpose: Controls the Lipschitz Constant of the Discriminator.

$$
\bar{W} = \frac{W}{\sigma(W)}
$$

Where:
- $W$ = Original Weight Matrix
- $\bar{W}$ = Spectrally Normalised Weight Matrix used in the Forward Pass
- $\sigma(W)$ = Spectral Norm of $W$
---
**Spectral Norm Approximation using Power Iteration**

Purpose: Controls the Lipschitz Constant of the Discriminator.

$$
\sigma(W) \approx \mathbf{u}^\top W \mathbf{v}
$$

Where:
- $\mathbf{v} \leftarrow \frac{W^\top \mathbf{u}}{\|W^\top \mathbf{u}\|_2}$
- $\mathbf{u} \leftarrow \frac{W \mathbf{v}}{\|W \mathbf{v}\|_2}$
- $\mathbf{u}, \mathbf{v}$: Approximated Unit Vectors
---
With the relevant mathematical formulas indicated, we will proceed to define the Spectral Normalisation in the Code Cell below.

In [ ]:
# ========== Spectral Normalisation Convolution Layer ========== #
class SpectralConv2D(tf.keras.layers.Layer):
    def __init__(self, filters, kernel_size, strides=1, padding='same', **kwargs):
        super().__init__()
        self.conv = tf.keras.layers.Conv2D(filters, kernel_size, strides=strides, padding=padding, use_bias=False, **kwargs)
        self.u = None

    def build(self, input_shape):
        self.conv.build(input_shape)
        self.w = self.conv.kernel
        self.u = self.add_weight(shape=(1, self.w.shape[-1]), initializer='random_normal', trainable=False, name='sn_u')

    def compute_spectral_norm(self, w):
        w_reshaped = tf.reshape(w, [-1, w.shape[-1]])
        u = self.u

        for _ in range(1):
            v = tf.linalg.l2_normalize(tf.matmul(u, tf.transpose(w_reshaped)))
            u = tf.linalg.l2_normalize(tf.matmul(v, w_reshaped))

        sigma = tf.matmul(tf.matmul(v, w_reshaped), tf.transpose(u))
        self.u.assign(u)
        return w / sigma

    def call(self, inputs):
        self.conv.kernel.assign(self.compute_spectral_norm(self.w))
        return self.conv(inputs)

# ========== Spectral Normalisation Dense Layer ========== #
class SpectralDense(tf.keras.layers.Layer):
    def __init__(self, units, activation=None):
        super().__init__()
        self.dense = tf.keras.layers.Dense(units, activation=activation, use_bias=False)
        self.u = None

    def build(self, input_shape):
        self.dense.build(input_shape)
        self.w = self.dense.kernel
        self.u = self.add_weight(shape=(1, self.w.shape[-1]), initializer='random_normal', trainable=False, name='sn_u_dense')

    def compute_spectral_norm(self, w):
        w_reshaped = tf.reshape(w, [-1, w.shape[-1]])
        u = self.u

        for _ in range(1):
            v = tf.linalg.l2_normalize(tf.matmul(u, tf.transpose(w_reshaped)))
            u = tf.linalg.l2_normalize(tf.matmul(v, w_reshaped))

        sigma = tf.matmul(tf.matmul(v, w_reshaped), tf.transpose(u))
        self.u.assign(u)
        return w / sigma

    def call(self, inputs):
        self.dense.kernel.assign(self.compute_spectral_norm(self.w))
        return self.dense(inputs)

With reference to the Code Cell above, we are able to verify that the Spectral Normalisation have been successfully defined and we are able to proceed to include this layer in the Discriminator Architecture in the next section.

---
### 4.2.5 Defining DCGAN Discriminator Architecture

In this section, we will be defining the Architecture of the Discriminator of the GAN. The Discriminator will be the Neural Network that will attempt to tell the Generated Images apart from the Actual Images. This will allow the GAN to improve overtime and also improve itself overtime. The mathematical function for the Discriminator Loss Function is indicated below:

---
**Discriminator Loss (Binary Crossentropy):**

Purpose: The Discriminator Assign 1 to Real Images and 0 to Generated Images.

$$
\mathcal{L}_{D} = -\mathbb{E}_{x \sim p_{\text{data}}}[\log D(x)] - \mathbb{E}_{z \sim p_z}[\log(1 - D(G(z)))]
$$

**Where:**
- $D(x)$: Output for Real EMNIST Images
- $D(G(z))$: Output for Generated Images

---
With the mathematical formula for the Loss Function indicated above, we will proceed to define the GAN's Discriminator Architecture and the Loss Function in the Code Cell below.

In [ ]:
# ========== Discriminator Function ========== #
def build_discriminator(input_shape=(28, 28, 1)):
    model = tf.keras.Sequential(name="discriminator")

    model.add(SpectralConv2D(64, kernel_size=5, strides=2, padding='same', input_shape=input_shape))
    model.add(tf.keras.layers.LeakyReLU(alpha=0.2))
    model.add(tf.keras.layers.Dropout(0.3))

    model.add(SpectralConv2D(128, kernel_size=5, strides=2, padding='same'))
    model.add(tf.keras.layers.LeakyReLU(alpha=0.2))
    model.add(tf.keras.layers.Dropout(0.3))

    model.add(tf.keras.layers.Flatten())
    model.add(SpectralDense(1, activation='sigmoid'))

    return model

# ========== Discriminator Loss Function ========== #
def discriminator_loss(real_output, fake_output):
    real_labels = tf.ones_like(real_output) * 0.9
    fake_labels = tf.zeros_like(fake_output)

    real_loss = loss_fn(real_labels, real_output)
    fake_loss = loss_fn(fake_labels, fake_output)
    return real_loss + fake_loss

With reference to the Code Cell above, we are able to verify that the GAN's Discriminator Architecture have been successfully defined and we are able to proceed to define the GAN's Training Steps.

---
### 4.2.6 Defining Training Step

In this section, we will be Defining the Training Step function so as to train both the Generator and the Discriminator at the same time. The Training Step will compute the gradients for both the Generator and the Discriminator at the same time. The relevant mathematical formulas for the Training Step is indicated below:

---
**Generator Loss:**

Purpose: Encourages the Generator to Create Samples that the Discriminator believes are Real.

$$
L_G = -\mathbb{E}_{z \sim p_z(z)}[\log D(G(z))]
$$

Where:
- $L_G$ = Generator Loss  
- $z$ = Latent Noise Vector Sampled from Prior Distribution $p_z(z)$  
- $G(z)$ = Fake Image Generated from $z$  
- $D(G(z))$ = Discriminator's Probability that the Generated Image is Real
---
**Discriminator Loss:**

Purpose: Train Discriminator $D$ to Distinguish between Real and Fake Samples, Rewarding High Confidence in Real Images ($D(x) \to 1$) and Low Confidence in Fake Ones ($D(G(z)) \to 0$).

$$
L_D = -\mathbb{E}_{x \sim p_{\text{data}}(x)}[\log D(x)] - \mathbb{E}_{z \sim p_z(z)}[\log (1 - D(G(z)))]
$$

Where:
- $L_D$ = Discriminator Loss  
- $x$ = Real Image Sampled from the Dataset Distribution $p_{\text{data}}(x)$  
- $z$ = Latent Vector from Prior Noise Distribution $p_z(z)$  
- $G(z)$ = Fake Image Generated by $G$
- $D(x)$ = Discriminator's Predicted Probability that $x$ is Real  
- $D(G(z))$ = Discriminator's Predicted Probability that $G(z)$ is Real  
- $\log D(x)$ = Log-likelihood that Real Image is Correctly Classified  
- $\log(1 - D(G(z)))$ = Log-likelihood that Fake Image is Correctly Rejected
---

**Gradient Update:**

Purpose: Updates the Generator and Discriminator Parameters using Gradient Descent.

$$
\theta_G \leftarrow \theta_G - \eta \cdot \nabla_{\theta_G} L_G \\
\theta_D \leftarrow \theta_D - \eta \cdot \nabla_{\theta_D} L_D
$$

Where:
- $\theta_G$ = Generator Parameters  
- $\theta_D$ = Discriminator Parameters  
- $\eta$ = Learning Rate  
- $\nabla_{\theta} L$ = Gradient of Loss with Respect to Model Weights
---

With the relevant mathematical formulas indicated above, we will proceed to define the Training Step in the Code Cell below.

In [ ]:
# ========== Training Step for LSGAN ========== #
@tf.function
def train_step(real_images, generator, discriminator, gen_optimizer, disc_optimizer, latent_dim):
    batch_size = tf.shape(real_images)[0]
    noise = tf.random.normal([batch_size, latent_dim])

    # ----- Add Stabilisation Noise to Inputs ----- #
    real_images += 0.05 * tf.random.normal(tf.shape(real_images))

    with tf.GradientTape() as gen_tape, tf.GradientTape() as disc_tape:
        # ----- Generate Fake Images ----- #
        fake_images = generator(noise, training=True)

        # ----- Discriminator Predictions ----- #
        real_output = discriminator(real_images, training=True)
        fake_output = discriminator(fake_images, training=True)

        # ----- LSGAN Losses (MSE) ----- #
        real_loss = loss_fn(tf.ones_like(real_output), real_output)
        fake_loss = loss_fn(tf.zeros_like(fake_output), fake_output)
        disc_loss = real_loss + fake_loss

        gen_loss = loss_fn(tf.ones_like(fake_output), fake_output)

    # ----- Compute & Apply Gradients ----- #
    gen_grads = gen_tape.gradient(gen_loss, generator.trainable_variables)
    disc_grads = disc_tape.gradient(disc_loss, discriminator.trainable_variables)
    gen_optimizer.apply_gradients(zip(gen_grads, generator.trainable_variables))
    disc_optimizer.apply_gradients(zip(disc_grads, discriminator.trainable_variables))

    return gen_loss, disc_loss, real_output, fake_output

With reference to the Code Cell above, we are able to determine that the Training Step have been successfully defined and we are able to proceed to prepare the Training Loop in the next section.

---
### 4.2.7 Defining Training Loop

In this section, we will be defining the Training Loop for the GAN Training. The Training Loop will go through each of the Epoch and train both the Generator and Discriminator simultaneously. The Training Loop will also track the Loss for both the Generator and the Discriminator for each of the Epoch trained. The relevant mathematical formulas are indicated below:

---
**Epoch Loss Averaging:**

Purpose: These are the Average Generator and Discriminator Losses over all Batches in Epoch $t$.

$$
\bar{L}_G^{(t)} = \frac{1}{N} \sum_{i=1}^{N} L_G^{(i)} \\
\bar{L}_D^{(t)} = \frac{1}{N} \sum_{i=1}^{N} L_D^{(i)}
$$

Where:
- $\bar{L}_G^{(t)}$ = Average Generator Loss at Epoch $t$  
- $\bar{L}_D^{(t)}$ = Average Discriminator Loss at Epoch $t$  
- $N$ = Number of Batches in the Dataset  
- $L_G^{(i)}$ = Generator Loss on Batch $i$  
- $L_D^{(i)}$ = Discriminator Loss on Batch $i$

---
**Epoch Iteration:**

Purpose: Describes the Training Loop Logic: for Each Epoch $t$, perform `train_step` for Every Mini-batch.

$$
\text{for } t = 1 \text{ to } T:
\quad \text{for each batch } (x^{(i)}):
\quad \text{train_step}(x^{(i)})
$$

Where:
- $T$: Total Number of Epochs  
- $x^{(i)}$: Real Batch $i$ from the Dataset  
- `train_step`: Function that Updates $G$ and $D$ using that Batch

---
**Generated Image Output:**

Purpose: Sample Random Noise $z$ and Generate Synthetic Image $\hat{x}$ from the Generator. This is used for Visual Monitoring of Model Quality.

$$
z \sim p_z(z) \\
\hat{x} = G(z)
$$

Where:
- $z$: Random Latent Vector  
- $G(z)$: Generated Image from Generator

---
With the mathematical formulas indicated above, we will proceed to define the Training Loop in the Code Cell below in preparation of the GAN Training.

In [ ]:
def train(dataset, epochs, generator, discriminator, gen_optimizer, disc_optimizer, latent_dim, save_path=None, use_early_stopping=True, steps_per_epoch=None):

    # ----- Prepare FID Reference Sets ----- #
    real_val  = tf.data.Dataset.from_tensor_slices(X_val).batch(50).take(1)
    noise_val = tf.random.normal([50, latent_dim])
    noise_val = tf.math.l2_normalize(noise_val, axis=1)

    # ----- FID Early Stopping Settings for DCGAN ----- #
    best_fid = float('inf')
    no_improve = 0
    patience = 10
    fid_interval = 1

    # ----- Dummy Model for Callbacks ----- #
    dummy_input  = Input(shape=(1,))
    dummy_output = Lambda(lambda x: x)(dummy_input)
    dummy_model  = Model(dummy_input, dummy_output)
    dummy_model.compile(optimizer=Adam(1e-4), loss='mse')

    early_stop.set_model(dummy_model)
    reduce_lr.set_model(dummy_model)
    lr_scheduler.set_model(dummy_model)
    early_stop.on_train_begin({})
    reduce_lr.on_train_begin({})
    lr_scheduler.on_train_begin({})

    # ----- Track Loss History ------ #
    gen_loss_history  = []
    disc_loss_history = []

    for epoch in range(1, epochs + 1):
        print(f"\nEpoch {epoch}/{epochs}")
        gen_losses, disc_losses, real_outs, fake_outs = [], [], [], []

        for i, real_images in enumerate(dataset):
            if steps_per_epoch and i >= steps_per_epoch:
                break

            gen_loss, disc_loss, ro, fo = train_step(
                real_images,
                generator,
                discriminator,
                gen_optimizer,
                disc_optimizer,
                latent_dim
            )
            gen_losses.append(gen_loss)
            disc_losses.append(disc_loss)
            real_outs.append(ro)
            fake_outs.append(fo)

        avg_gen_loss  = tf.reduce_mean(gen_losses)
        avg_disc_loss = tf.reduce_mean(disc_losses)

        # ----- Compute Real/Fake accuracy ----- #
        if real_outs and fake_outs:
            acc_real = tf.reduce_mean(tf.cast(tf.concat(real_outs, axis=0) > 0.5, tf.float32))
            acc_fake = tf.reduce_mean(tf.cast(tf.concat(fake_outs, axis=0) < 0.5, tf.float32))
        else:
            acc_real = tf.constant(0.0)
            acc_fake = tf.constant(0.0)

        print(f"Generator Loss: {avg_gen_loss:.4f} | "
              f"Discriminator Loss: {avg_disc_loss:.4f} | "
              f"Acc R/F: {acc_real:.3f}/{acc_fake:.3f}")

        gen_loss_history.append(float(avg_gen_loss))
        disc_loss_history.append(float(avg_disc_loss))

        # ----- Callback Updates on Generator Loss ----- #
        logs = {'loss': float(avg_gen_loss)}
        if use_early_stopping:
            early_stop.on_epoch_end(epoch=epoch, logs=logs)
            if early_stop.stopped_epoch > 0:
                print(f"Early stopping triggered at epoch {epoch}")
                break
        reduce_lr.on_epoch_end(epoch=epoch, logs=logs)
        lr_scheduler.on_epoch_end(epoch=epoch, logs=logs)

        # ----- FID Evaluation & Early Stopping ----- #
        if epoch % fid_interval == 0:
            fake_val   = generator(noise_val, training=False)
            real_batch = next(iter(real_val))

            real_for_fid = tf.image.resize(real_batch, [299,299]) * 0.5 + 0.5
            fake_for_fid = tf.image.resize(fake_val,    [299,299]) * 0.5 + 0.5

            fid_value = calculate_fid(real_for_fid, fake_for_fid)
            print(f"Epoch {epoch} → Val FID: {fid_value:.2f}")

            if fid_value < best_fid:
                best_fid   = fid_value
                no_improve = 0
                if save_path:
                    os.makedirs(save_path, exist_ok=True)
                    generator.save_weights(os.path.join(save_path, "best_gen.weights.h5"))
                    discriminator.save_weights(os.path.join(save_path, "best_disc.weights.h5"))
            else:
                no_improve += 1

            if no_improve >= patience:
                print(f"No FID improvement for {patience} epochs, stopping.")
                break

        # ----- Display Every 5 Epochs ----- #
        if epoch % 5 == 0:
            display_generated_images(generator, latent_dim)

    # ----- Restore Best Weights After Training ----- #
    if save_path:
        best_gen_path = os.path.join(save_path, "best_gen.weights.h5")
        best_disc_path = os.path.join(save_path, "best_disc.weights.h5")
        if os.path.exists(best_gen_path) and os.path.exists(best_disc_path):
            generator.load_weights(best_gen_path)
            discriminator.load_weights(best_disc_path)
            print("Restored best generator and discriminator weights based on lowest FID.")
        else:
            print("Best weight files not found. Skipping restore.")

    return avg_gen_loss, gen_loss_history, disc_loss_history

With reference to the Code Cell above, we are able to determine that the Training Loop have been successfully defined and we are able to proceed to define the Optimisers with Tunable Parameter in preparation for the Optuna Tuning.

---
### 4.2.8 Defining Optimisers with Tunable Parameters

In this section, we will be defining the Common Optimisers that will be Tuned using the Optuna Tuning. The Optimisers will each have a range of values in a list so as to allow for the Optuna to search for the best Hyperparameter for the GAN. Through these parameters, it will affect how effective and efficiently the GAN will learn from the provided EMNIST Data. The relevent mathematical formula is indicated below:

---
**Optimiser Update Rule**

Purpose: Parameter Update Step in the Adam Optimiser, using Adaptive Learning Rates with Momentum.

$$
\theta \leftarrow \theta - \eta \cdot \frac{m_t}{\sqrt{v_t} + \epsilon}
$$

Where:
- $\theta$ = Model Parameters (Generator or Discriminator)
- $\eta$ = Learning Rate (Tuned)
- $m_t$ = First Moment Estimate (Mean of Gradients)
- $v_t$ = Second Moment Estimate (Variance of Gradients)
- $\epsilon$ = Small Constant for Numerical Stability

---
With the mathematical formula indicated above, we will proceed to define the Optimisers in the Code Cell.

In [ ]:
def objective(trial):
    # ----- Hyperparameters ----- #
    latent_dim = trial.suggest_categorical('latent_dim', [100, 128, 160])
    learning_rate = trial.suggest_float('learning_rate', 5e-5, 2e-4, log=True)
    beta_1 = trial.suggest_float('beta_1', 0.4, 0.6)

    # ----- Build Models ----- #
    generator = build_generator(latent_dim=latent_dim)
    discriminator = build_discriminator()

    # ----- Trigger Weight Creation ----- #
    _ = generator(tf.random.uniform([1, latent_dim], minval=-1.0, maxval=1.0))
    _ = discriminator(tf.random.normal([1, 28, 28, 1]))

    # ----- Define Optimizers ----- #
    gen_optimizer = tf.keras.optimizers.Adam(learning_rate=learning_rate, beta_1=beta_1)
    disc_optimizer = tf.keras.optimizers.Adam(learning_rate=learning_rate, beta_1=beta_1)

    # ----- Prebuild Optimizers for tf.function compatibility ----- #
    _ = gen_optimizer.apply_gradients([(tf.zeros_like(v), v) for v in generator.trainable_variables])
    _ = disc_optimizer.apply_gradients([(tf.zeros_like(v), v) for v in discriminator.trainable_variables])

    # ----- Load Data ----- #
    dataset = load_data(X_train, batch_size=64)
    steps = min(1000, len(X_train) // 64)  # dynamic step limit

    # ----- Train ----- #
    _, _, _ = train(
        dataset=dataset,
        epochs=50,
        generator=generator,
        discriminator=discriminator,
        gen_optimizer=gen_optimizer,
        disc_optimizer=disc_optimizer,
        latent_dim=latent_dim,
        steps_per_epoch=steps,
        use_early_stopping=False,
        save_path=None
    )

    # ----- Generate Fake Images ----- #
    z = tf.random.uniform([1000, latent_dim], minval=-1.0, maxval=1.0)
    fake_images = generator(z, training=False)
    fake_images = (fake_images + 1.0) / 2.0  # scale from [-1,1] to [0,1]

    # ----- Get Real Images ----- #
    real_images = X_val[:1000]

    # ----- Compute FID Score ----- #
    fid_score = calculate_fid(real_images, fake_images)

    # ----- Return FID to Minimise ----- #
    return fid_score

With reference to the Code Cell above, we are able to determine that the Optimiser have been successfully defined with a range of different Parameter Values so as to provide a robust Optuna Tuning in the subsequent sections.

---
### 4.2.9 Optuna Tuning Study

In this sub-section, we will be conducting the Optuna Tuning Study on the GAN Model so as to obtain the best Hyperparameter for the GAN Model. It is expected that the Tuning takes a prolonged duration due to the nature of how the GAN works and the 2 Neural Networks involved. However, the training data will be stored so as to prevent re-running of repetitive codes. The Optuna Tuning Study will be conducted in the Code Cell below.

In [ ]:
# ========== Create or Load Study with SQLite Backend ========== #
study = optuna.create_study(
    direction="minimize",
    study_name="dcgan_tuning",
    storage="sqlite:////content/drive/MyDrive/Colab Notebooks/DELE CA2 A/Non-Augmented GAN Tunings/dcgan_optuna.db",
    load_if_exists=True
)

# ========== Trial Management ========== #
MAX_TRIALS = 50
completed_trials = len([t for t in study.trials if t.state == optuna.trial.TrialState.COMPLETE])
remaining_trials = MAX_TRIALS - completed_trials

if remaining_trials > 0:
    print(f"Resuming DCGAN study: {completed_trials} completed, running {remaining_trials} more.")
    study.optimize(objective, n_trials=remaining_trials)
else:
    print(f"DCGAN study already completed {MAX_TRIALS} trials. Skipping optimization.")

# ========== Output Best Trial ========== #
best_trial = study.best_trial
best_params_df = pd.DataFrame([best_trial.params])
best_params_df["Final FID"] = best_trial.value
best_params_df.style.background_gradient(cmap="Blues")

With reference to the Code Cell above, we are able to view the Best Hyperparameter obtained during the Optuna Tuning Study. This Hyperparameter will be extracted and be trained for a longer period of time so as to attempt to increase the performance and limit the loss for the GAN.

---
### 4.2.10 GAN Re-Training Operation

As obtained from the previous sub-section, we will be re-training the GAN using the best Hyperparameter obtained during the Optuna Tuning Study. This is to push the GAN Model to its limits and also to save Computational Power as we are not taking up prolonged periods of time tuning GAN Models which may not have any clear signs of good performance. The retraining of the GAN Model will be conducted in the Code Cell below.

In [ ]:
# ========== Extract Best Parameters from Study ========== #
best_params = study.best_trial.params

latent_dim  = best_params['latent_dim']
learning_rate = best_params['learning_rate']
beta_1 = best_params['beta_1']

print("Using Best Trial Parameters for DCGAN:")
print(best_params)

# ========== Rebuild DCGAN Models ========== #
generator = build_generator(latent_dim=latent_dim)
discriminator = build_discriminator()

# ========== Trigger Variable Creation ========== #
_ = generator(tf.random.normal([1, latent_dim]))
_ = discriminator(tf.random.normal([1, 28, 28, 1]))

gen_optimizer  = tf.keras.optimizers.Adam(learning_rate=learning_rate, beta_1=beta_1)
disc_optimizer = tf.keras.optimizers.Adam(learning_rate=learning_rate, beta_1=beta_1)

# ========== Initialise Optimisers ========== #
_ = gen_optimizer.apply_gradients([(tf.zeros_like(var), var) for var in generator.trainable_variables])
_ = disc_optimizer.apply_gradients([(tf.zeros_like(var), var) for var in discriminator.trainable_variables])

# ========== Reload Preprocessed Dataset ========== #
dataset = load_data(X_train)

# ========== Define Output Paths ========== #
model_name   = "dcgan"
output_dir   = f"/content/drive/MyDrive/Colab Notebooks/DELE CA2 A/final_outputs_{model_name}_noAugment"
weights_path = os.path.join(output_dir, f"{model_name}_generator_final.weights.h5")
gen_ckpt     = os.path.join(output_dir, "best_gen.weights.h5")
disc_ckpt    = os.path.join(output_dir, "best_disc.weights.h5")
loss_csv     = os.path.join(output_dir, "loss_history.csv")

os.makedirs(output_dir, exist_ok=True)

# ========== Skip Training if Everything Exists ========== #
if os.path.exists(gen_ckpt) and os.path.exists(disc_ckpt) and os.path.exists(loss_csv):
    print("All outputs found. Skipping training...")

    # ----- Load weights ----- #
    generator.load_weights(gen_ckpt)
    discriminator.load_weights(disc_ckpt)
    print("Generator and Discriminator Weights Loaded.")

    # ----- Load loss history ----- #
    loss_df = pd.read_csv(loss_csv)
    gen_loss_history  = loss_df['gen_loss'].tolist()
    disc_loss_history = loss_df['disc_loss'].tolist()
    final_gen_loss    = gen_loss_history[-1]

else:
    # ========== Train DCGAN ========== #
    final_gen_loss, gen_loss_history, disc_loss_history = train(
        dataset=dataset,
        epochs=100,
        steps_per_epoch=1546,
        generator=generator,
        discriminator=discriminator,
        gen_optimizer=gen_optimizer,
        disc_optimizer=disc_optimizer,
        latent_dim=latent_dim,
        use_early_stopping=False,
        save_path=output_dir
    )

    # ========== Save Final Weights ========== #
    generator.save_weights(weights_path)
    print("Generator Weights Saved.")

    # ========== Save Loss History to CSV ========== #
    loss_df = pd.DataFrame({
        'epoch': list(range(1, len(gen_loss_history) + 1)),
        'gen_loss': gen_loss_history,
        'disc_loss': disc_loss_history
    })
    loss_df.to_csv(loss_csv, index=False)
    print(f"Loss history saved to {loss_csv}")

With reference to the Code Cell above, we are able to determine that the best GAN Model have been trained and saved successfully and we are able to proceed with the Model Evaluation in the next sub-sections where we determine how well the GAN Model performed based on various metrics and evaluation methodologies.

---
### 4.2.11 GAN Model Evaluation

In this sub-section, we will be evaluating the GAN's Generator performance on various aspects. We will be utilising different methodologies and metrics to attempt to quantify the performance of the GAN Generator while also conducting Visual Inspection on the result output of the GAN. The Evaluation will be conducted in the following sections.

---
#### 4.2.11.1 Visual Grid of Samples

In this sub-section, we will be visualising the Grid of Samples of the Synthesised Data generated by the GAN Model. A list of observations will be made before the utilisation of Quantitative Metrics to evaluate the GAN Generator's general performance. As the observations made are purely from professional opinion, it will most likely not be used as a Basis of Comparison between GAN Models due to differing opinions unless in extreme cases. The visualisation will be conducted in the Code Cell below.

In [ ]:
def plot_generated_images(generator, latent_dim, n_rows=5, n_cols=5, save_path=None, title="DCGAN Generated Images (Non-Augmented)"):
    noise = tf.random.normal([n_rows * n_cols, latent_dim])
    gen_images = generator(noise, training=False)

    # ----- Rescale from [-1, 1] to [0, 1] ----- #
    gen_images = (gen_images + 1.0) / 2.0
    gen_images = tf.clip_by_value(gen_images, 0.0, 1.0)

    # ----- Check Shape ----- #
    assert gen_images.shape[-1] == 1, "Expected single-channel (grayscale) output"

    # ----- Plot Generated Image ----- #
    fig, axes = plt.subplots(n_rows, n_cols, figsize=(n_cols, n_rows))
    for i in range(n_rows * n_cols):
        ax = axes[i // n_cols, i % n_cols]
        ax.imshow(gen_images[i, :, :, 0], cmap='gray')
        ax.axis('off')

    # ----- Display Plot ----- #
    plt.suptitle(title, fontsize=16)
    plt.tight_layout()
    if save_path:
        plt.savefig(save_path, dpi=300)
        print(f"Saved Generated Image Grid to: {save_path}")
    plt.show()

plot_generated_images(generator, latent_dim=128, n_rows=10, n_cols=16)

With reference to the output of the Visual Grid of Samples above, we are able to make the following observations:

**Basic Structure of Letters is Somewhat Captured**
- Some characters resemble letters like "C", "B", "L", "E", or "P".
- Overall shape and stroke thickness is consistent with EMNIST style.

**High Blur and Noise**
- Many letters are fuzzy, incomplete, or warped, making them hard to classify by eye.

**Poor Character Consistency**
- Characters like "A", "D", "G" often appear as loops or blobs rather than structured glyphs.
- Sharp corners and straight edges are missing.

**Lack of Diversity**
- Several characters repeat with minor variations — may indicate mode collapse.

With the observations indicated above, we will proceed to conduct the Loss Curve Analysis in the next sub-section.


---
#### 4.2.11.2 Loss Curve Over Epochs

In this sub-section, we will be plotting the GAN Loss Curve over Epochs to evaluate the training behaviour of the GAN. This curve provides insight into whether the GAN is experiencing underfitting, overfitting, or achieving a stable training dynamic between the generator and discriminator.

Although GANs do not minimise a single unified loss in the traditional sense, the trajectory of the generator and discriminator losses can indicate whether the adversarial training process is converging appropriately.

If the generator loss remains high while discriminator loss quickly drops to near-zero, this may indicate **underfitting**, where the generator is not learning effectively to produce plausible images.

If the discriminator loss increases while generator loss sharply decreases, this may suggest **overfitting** or **mode collapse**, where the generator exploits narrow weaknesses in the discriminator without general improvement.

A relatively stable oscillation or convergence between both losses typically signifies a **well-balanced** training process, where the generator and discriminator are learning in tandem.

The Potential Observations and Corresponding Action Plans are outlined below:

**Underfitting Loss Curve**

- Increase training epochs to allow generator more time to learn.

- Simplify the discriminator to reduce overpowering the generator.

- Adjust the learning rate or apply label smoothing.

- Consider adding batch normalisation in the generator.

**Overfitting or Mode Collapse Loss Curve**

- Introduce or increase dropout in the discriminator.

- Apply input noise or label flipping to the discriminator.

- Evaluate diversity of generated samples regularly.

**Balanced/Ideal Loss Curve**

- Maintain current architecture and hyperparameters.

- Proceed with full-scale image generation.

- Evaluate both qualitative (visual inspection) and quantitative metrics (e.g. Inception Score or FID).

With these potential training behaviours and action plan outlined, we will proceed to visualise the GAN loss dynamics in the code cell below.

In [ ]:
# ========== Figure Size ========== #
plt.figure(figsize=(10, 6))

# ========== Plot Training Losses ========== #
plt.plot(gen_loss_history, label="Generator Loss", linewidth=2, color='blue')
plt.plot(disc_loss_history, label="Discriminator Loss", linewidth=2, color='green')

# ========== Labels and Title ========== #
plt.xlabel("Epoch", fontsize=12)
plt.ylabel("Loss", fontsize=12)
plt.title("DCGAN Training Loss Curve (Non-Augmented)", fontsize=16)

# ========== Grid, Legend, and Styling ========== #
plt.grid(True, linestyle='--', alpha=0.6)
plt.legend(fontsize=12)
plt.xticks(fontsize=10)
plt.yticks(fontsize=10)

# ========== Display the Plot ========== #
plt.tight_layout()
plt.show()

With reference to the output of the Loss CUrve above, we are able to make the following observations:

- Stable training achieved, with both losses plateauing.

- Lack of augmentation limits generator's challenge, possibly leading to less realistic or diverse outputs.

- Suggests augmentation was beneficial in making the training task more robust and improving generalisation.

With the observations indicated above, we will porceed to conduct the Quantitative Metrics Evaluation in the next sub-section.

---
#### 4.2.11.3 Quantitative Metrics Evaluation

In this sub-section we will be utilising Quantitative Metrics such as FID and KID to tabulate the results of the performance and quantify the performance of the GAN Generator. We will subsequently be use these metrics to conduct inter-model evaluation after tuning and training all the GANs. The metrics that we will be utilising and their respective formulas are indicated below:

---
**Fréchet Inception Distance**

Purpose: Measures the Distance between the Real-Image and Generated-Image Distributions in Inception Feature Space.

$$
\mathrm{FID} \;=\;\|\mu_r - \mu_f\|^2
\;+\;\mathrm{Tr}\Bigl(\Sigma_r + \Sigma_f - 2\,(\Sigma_r\,\Sigma_f)^{\tfrac12}\Bigr)
$$


Where:
- $\mu_r = \mathbb{E}[f(x)]$, $\Sigma_r = \mathrm{Cov}[f(x)]$ for Real Images $x$.  
- $\mu_f = \mathbb{E}[f(\hat x)]$, $\Sigma_f = \mathrm{Cov}[f(\hat x)]$ for Generated Images $\hat x$.  
- $f(\cdot)$ = Map from Image to its InceptionV3 'pooling=avg' Features.  

---
**Diversity (t-SNE Spread)**

Purpose: Quantifies how 'wide' the 2D t-SNE Embedding of Generated Samples.

$$
\mathrm{Spread}
\;=\;
\bigl(\max_i\,z_i^{(1)} - \min_i\,z_i^{(1)}\bigr)
\;\times\;
\bigl(\max_i\,z_i^{(2)} - \min_i\,z_i^{(2)}\bigr)
$$

Where:
- $z_i = (z_i^{(1)}, z_i^{(2)})$ = 2-dimensional t-SNE Embedding of $i$th Generated Image.

---
**Mode Collapse Risk**

Purpose: Flags when too many Generated Images are Nearly Identical (mode collapse).

$$
\text{ModeCollapseRisk} =
\begin{cases}
\text{Low}, & \dfrac{\bigl|\{\mathrm{unique\_rounded}(x_i)\}\bigr|}{N} > \tau,\\
\text{High}, & \text{otherwise}.
\end{cases}
$$

Where:
- $x_i$ = $N$ Generated Samples.  
- $\mathrm{unique\_rounded}(x_i)$ = Rounds Pixels to Detect Duplicates.  
- $\tau=0.9$ = Uniqueness Threshold (90%).

---
**Perceptual Path Length**

Purpose: Measures Sensitively of Generator's Outputs (in VGG16 Feature Space) move when the Latent Code $z$ is Perturbed.

$$
\mathrm{PPL}
\;=\;
\mathbb{E}_{z,\delta z}\Bigl[\,
\|\phi\bigl(G(z + \epsilon\,\delta z)\bigr)\;-\;\phi\bigl(G(z)\bigr)\|_2^2
\Bigr]
$$

Where:  
- $G(z)$ = GAN Generator Mapping $z\in\mathbb{R}^{\mathrm{latent\_dim}}$ to an Image.  
- $\phi(\cdot)$ = VGG16 'pooling=avg' Feature Extractor.  
- $\delta z\sim\mathcal{N}(0,I)$, $\epsilon$ = Small Constant.

---
**Kernel Inception Distance**

Purpose: An Unbiased Estimator of the Squared Maximum Mean Discrepancy (MMD) between Real and Generated Inception Features.

$$
\mathrm{KID}
\;=\;
\frac{1}{m(m-1)}\sum_{i\neq j} k\bigl(\phi(x_i),\phi(x_j)\bigr)
\;+\;
\frac{1}{n(n-1)}\sum_{i\neq j} k\bigl(\phi(\hat x_i),\phi(\hat x_j)\bigr)
\;-\;
\frac{2}{mn}\sum_{i=1}^m\sum_{j=1}^n k\bigl(\phi(x_i),\phi(\hat x_j)\bigr)
$$

Where:  
- $x_i$ ($i=1\ldots m$) = Real Images.
- $\hat x_j$ ($j=1\ldots n$) = Generated Images.  
- $\phi(\cdot)$ = InceptionV3 Feature Extractor (pooling='avg').  
- $k(u,v) = \bigl(\frac{u^\top v}{d}+1\bigr)^3$ = degree-3 Polynomial Kernel on $d$-dimensional Features.

---

With the mathematical and metrics indicated above, we will proceed to conduct the Quantitave Metrics Evalution in the Code Cell below.

In [ ]:
# ========== Preload InceptionV3 and VGG16 Models ========== #
inception = InceptionV3(include_top=False, pooling='avg', input_shape=(299, 299, 3))
vgg_model = VGG16(include_top=False, weights='imagenet', input_shape=(128, 128, 3))

# ========== Preprocess for Inception ========== #
def preprocess_images_for_inception(images):
    images = (images + 1.0) * 127.5  # Scale from [-1, 1] to [0, 255]
    images = tf.image.resize(images, [299, 299])
    if images.shape[-1] == 1:
        images = tf.image.grayscale_to_rgb(images)
    return preprocess_input(images)

# ========== Compute FID ========== #
def calculate_fid(real_images, fake_images, batch_size=50):
    real_pp = preprocess_images_for_inception(real_images)
    fake_pp = preprocess_images_for_inception(fake_images)

    act1 = inception.predict(real_pp, batch_size=batch_size, verbose=0)
    act2 = inception.predict(fake_pp, batch_size=batch_size, verbose=0)

    mu1, sigma1 = np.mean(act1, axis=0), np.cov(act1, rowvar=False)
    mu2, sigma2 = np.mean(act2, axis=0), np.cov(act2, rowvar=False)

    diff = mu1 - mu2
    covmean = sqrtm(sigma1 @ sigma2)
    if np.iscomplexobj(covmean):
        covmean = covmean.real

    fid = diff @ diff + np.trace(sigma1 + sigma2 - 2 * covmean)
    return round(fid, 4)

# ========== Compute KID ========== #
def polynomial_kernel(X, Y):
    d = X.shape[1]
    return (np.dot(X, Y.T) / d + 1) ** 3

def calculate_kid(real_images, fake_images, batch_size=128):
    real_pp = preprocess_images_for_inception(real_images)
    fake_pp = preprocess_images_for_inception(fake_images)

    real_features = inception.predict(real_pp, batch_size=batch_size, verbose=0)
    fake_features = inception.predict(fake_pp, batch_size=batch_size, verbose=0)

    m = real_features.shape[0]
    n = fake_features.shape[0]

    k_rr = polynomial_kernel(real_features, real_features)
    k_gg = polynomial_kernel(fake_features, fake_features)
    k_rg = polynomial_kernel(real_features, fake_features)

    np.fill_diagonal(k_rr, 0)
    np.fill_diagonal(k_gg, 0)

    mmd = (k_rr.sum() / (m * (m - 1)) +
           k_gg.sum() / (n * (n - 1)) -
           2 * k_rg.mean())
    return round(mmd, 4)

# ========== Compute t-SNE Spread (Diversity) ========== #
def calculate_tsne_spread(images):
    flat = images.reshape(images.shape[0], -1)
    tsne = TSNE(n_components=2, random_state=42)
    proj = tsne.fit_transform(flat)
    x_range = proj[:, 0].max() - proj[:, 0].min()
    y_range = proj[:, 1].max() - proj[:, 1].min()
    return round(x_range * y_range, 2)

# ========== Assess Mode Collapse ========== #
def assess_mode_collapse(images):
    unique = np.unique(np.round(images), axis=0).shape[0]
    return "Low" if unique > 0.9 * images.shape[0] else "High"

# ========== Compute Perceptual Path Length (PPL) ========== #
def calculate_ppl(generator, latent_dim=128, num_samples=50, epsilon=1e-2):
    distances = []
    for _ in range(num_samples):
        z1 = tf.random.normal([1, latent_dim])
        z2 = z1 + epsilon * tf.random.normal([1, latent_dim])
        img1 = generator(z1, training=False)
        img2 = generator(z2, training=False)

        img1 = tf.image.resize(tf.image.grayscale_to_rgb((img1 + 1.0) * 127.5), (128, 128))
        img2 = tf.image.resize(tf.image.grayscale_to_rgb((img2 + 1.0) * 127.5), (128, 128))

        f1 = vgg_model(vgg_preprocess(img1))
        f2 = vgg_model(vgg_preprocess(img2))
        d = tf.reduce_mean(tf.square(f1 - f2)).numpy()
        distances.append(d)
    return round(np.mean(distances), 4)

# ========== Evaluate GAN Performance ========== #
def evaluate_gan_model(generator, latent_dim, X_val, gen_loss_history, disc_loss_history, model_name="DCGAN"):
    # ----- Generate Fake Images ----- #
    noise = tf.random.normal([1000, latent_dim])
    fake_images = generator(noise, training=False)
    fake_images = tf.clip_by_value((fake_images + 1) / 2.0, 0.0, 1.0)
    fake_np = fake_images.numpy()

    # ----- Select and preprocess 1000 Real Images from Validation Set ----- #
    real_images = tf.convert_to_tensor(X_val[:1000], dtype=tf.float32)
    real_images = tf.clip_by_value(real_images, 0.0, 1.0)

    # ----- Compute Metrics ----- #
    fid_value = calculate_fid(real_images, fake_images)
    kid_value = calculate_kid(real_images, fake_images)
    tsne_spread_value = calculate_tsne_spread(fake_np)
    collapse_risk_value = assess_mode_collapse(fake_np)
    ppl_score_value = calculate_ppl(generator, latent_dim)
    mean_gen_loss_value = round(np.mean(gen_loss_history), 4)
    std_gen_loss_value = round(np.std(gen_loss_history), 4)
    mean_disc_loss_value = round(np.mean(disc_loss_history), 4)
    std_disc_loss_value = round(np.std(disc_loss_history), 4)

    # ----- Store as Global Vars (Optional) ----- #
    prefix = model_name.upper()
    globals()[f"{prefix}_FID"] = fid_value
    globals()[f"{prefix}_KID"] = kid_value
    globals()[f"{prefix}_TSNE"] = tsne_spread_value
    globals()[f"{prefix}_MODE_COLLAPSE"] = collapse_risk_value
    globals()[f"{prefix}_PPL"] = ppl_score_value
    globals()[f"{prefix}_GEN_LOSS_MEAN"] = mean_gen_loss_value
    globals()[f"{prefix}_GEN_LOSS_STD"] = std_gen_loss_value
    globals()[f"{prefix}_DISC_LOSS_MEAN"] = mean_disc_loss_value
    globals()[f"{prefix}_DISC_LOSS_STD"] = std_disc_loss_value

    # ----- Final Summary Row ----- #
    row = {
        "Model": model_name,
        "FID Score": fid_value,
        "KID Score": kid_value,
        "Mean Generator Loss": mean_gen_loss_value,
        "Std Generator Loss": std_gen_loss_value,
        "Mean Discriminator Loss": mean_disc_loss_value,
        "Std Discriminator Loss": std_disc_loss_value,
        "Mode Collapse Risk": collapse_risk_value,
        "Visual Quality": "Acceptable",
        "Diversity (t-SNE Spread)": tsne_spread_value,
        "PPL": ppl_score_value
    }

    return pd.DataFrame([row])[[
        "Model", "FID Score", "KID Score",
        "Mean Generator Loss", "Std Generator Loss",
        "Mean Discriminator Loss", "Std Discriminator Loss",
        "Mode Collapse Risk", "Visual Quality",
        "Diversity (t-SNE Spread)", "PPL"
    ]]

# ========== Create DataFrame ========== #
df = evaluate_gan_model(generator, latent_dim, X_val, gen_loss_history, disc_loss_history, model_name="DCGAN (Non-Augmented)")

# ========== Display DataFrame ========== #
df.style.background_gradient(cmap="Blues")

With reference to the output of the Quantitative Metrics above, we are able to determine the performance of the GAN. This will be used as a basis of comparison to the other GAN Models.

---
#### 4.2.11.4 t-SNE of Generated Samples

In this sub-section, we visualise the t-SNE projection of generated samples to evaluate the latent diversity learned by the unconditional GAN. Although our GAN is not class-conditioned, t-SNE remains a valuable tool to assess whether the generator is producing varied outputs that reflect meaningful use of the latent space. While GANs are trained in high-dimensional spaces, t-SNE enables us to observe 2D structural patterns such as sample clustering, separation, and density, which provide indirect insights into the diversity and generalisation capabilities of the generator. The visualisation will be conducted in the Code Cell below.

In [ ]:
# ========== Generate New Images from Random Noise ========== #
latent_dim = 128
noise = tf.random.normal([500, latent_dim])
generated_images = generator(noise, training=False)
generated_images = (generated_images + 1) / 2.0

def plot_tsne_embeddings(generated_images, save_path=None, title="t-SNE of DCGAN Generated Samples (Non-Augmented)"):
    # ----- Convert to [0, 1] Range ----- #
    if tf.reduce_max(generated_images).numpy() > 1.0:
        raise ValueError("Images should be in [-1, 1] range before scaling.")

    images_rescaled = (generated_images + 1.0) / 2.0
    flat_images = images_rescaled.numpy().reshape(images_rescaled.shape[0], -1)

    # ----- Run t-SNE ----- #
    tsne = TSNE(n_components=2, perplexity=30, learning_rate='auto', init='pca', random_state=42)
    tsne_proj = tsne.fit_transform(flat_images)

    # ----- Plot t-SNE ----- #
    plt.figure(figsize=(12, 6))
    sns.scatterplot(
        x=tsne_proj[:, 0],
        y=tsne_proj[:, 1],
        s=20,
        alpha=0.9,
        edgecolor='none'
    )
    plt.title(title, fontsize=14, weight='bold')
    plt.xlabel("t-SNE Dimension 1")
    plt.ylabel("t-SNE Dimension 2")
    plt.grid(True, linestyle='--', alpha=0.3)
    plt.tight_layout()

    if save_path:
        plt.savefig(save_path, dpi=300)
        print(f"t-SNE plot saved to: {save_path}")

    plt.show()

plot_tsne_embeddings(generated_images=generated_images)

With reference to the t-SNE plot of DCGAN generated samples (Non-Augmented) above, we are able to make the following observations:

**Spread Across Latent Space**

- The points are distributed broadly, covering a circular or elliptical region.

- This suggests that the DCGAN has learned to generate samples across a wide and balanced latent space.

**Lack of Clear Clustering**

- No dense clusters or segmentations are visible, indicating minimal semantic grouping of characters.

- The embeddings do not reflect strong class separability, which is expected from an unconditional GAN.

**Mild Uniformity**

- The distribution appears relatively even, with no large voids or dense concentrations.

- This reflects a reasonable sample variety, even if character-level diversity may be visually lacking.

**Absence of Outliers**

- No extreme points lie far outside the main distribution.

- Implies stable latent generation without major instability in the feature space.

With the observations indicated above, we will proceed to train the next GAN Model in the next sub-section.

---
## 4.3 Wasserstein GAN with Gradient Penalty Model Training

In this sub-section, we will be training a Wasserstein GAN with Gradient Penalty (WGAN-GP) for the EMNIST Dataset. The WGAN-GP mainly replaces the Jensen-Shannon Divergence with Wasserstein (Earth-Mover) Distance. Through this replacement, the WGAN-GP is capable of providing a smoother and more meaningful loss metric despite the Generator being far from the real Data Distrubution. This will ultimately lead to Better Gradients, Stability, Convergence and Reduce Mode Collapse. The relevant mathematical formulas are indicated below:

---
**Generator Loss:**

Purpose: The Generator Maximizes the Critic's Score on Fake Images

$$
L_G = -\mathbb{E}_{\tilde{x} \sim \mathbb{P}_g} \left[D(\tilde{x})\right]
$$

Where:
- $L_G$ = Generator Loss  
- $\tilde{x} \sim \mathbb{P}_g$ = Fake Data Sampled from Generator Distribution  
- $D(\tilde{x})$ = Critic's Prediction Score for Fake Input $\tilde{x}$  
- $\mathbb{E}$ = Expectation Operator

---
**Discriminator (Critic) Loss:**

Purpose: Trains Critic to give Higher Scores to Real Data and Lower Scores to Fake Data.

$$
L_D = \mathbb{E}_{\tilde{x} \sim \mathbb{P}_g} \left[D(\tilde{x})\right] - \mathbb{E}_{x \sim \mathbb{P}_r} \left[D(x)\right] + \lambda \cdot \mathbb{E}_{\hat{x} \sim \mathbb{P}_{\hat{x}}} \left[\left(\| \nabla_{\hat{x}} D(\hat{x}) \|_2 - 1\right)^2\right]
$$

Where:
- $D(x)$ = Critic Output for Real Input  
- $\mathbb{P}_r$ = Real Data Distribution  
- $\mathbb{P}_g$ = Generator (fake) Distribution  
- $\tilde{x}$ = Fake Sample from Generator
- $\hat{x}$ = Interpolated Sample Between Real and Fake  
- $\lambda$ = Gradient Penalty Coefficient (typically $\lambda = 10$)

---
**Gradient Penalty Term:**

Purpose: Enforces 1-Lipschitz Constraint via Soft Regularisation.

$$
GP = \mathbb{E}_{\hat{x} \sim \mathbb{P}_{\hat{x}}} \left[ \left(\| \nabla_{\hat{x}} D(\hat{x}) \|_2 - 1 \right)^2 \right]
$$

**Where:**
- $GP$ = Gradient Penalty Term  
- $\hat{x} = \epsilon x + (1 - \epsilon)\tilde{x}$ = Random Point along the Line Between Real and Fake Images  
- $\epsilon \sim \mathcal{U}(0, 1)$ = Random Number from Uniform Distribution  
- $\nabla_{\hat{x}} D(\hat{x})$ = Gradient of Critic Output with Respect to Interpolated Input  
- $\|\cdot\|_2$ = Euclidean Norm (L2 norm) of Gradient Vector

---
With the relevant mathematical formulas indicated above, we will proceed to train the WGAN-GP in this sub-section for the purpose of it's potential for smoother training process and also to serve as a comparison for our other GAN Models.

---
### 4.3.1 Defining Data Pre-Processing Function

In this sub-section, we will be defining the Data Pre-Processing Function which will allow for consistent Data Pipeline to be built and modifed accordingly should the need arise such as when we need to add Spectral Normalisation. This will ensure that the Data's integrity is not affected and we are able to make fair and un-biased assumptions. The Data Pre-Processing Function is defined in the Code Cell below:

In [ ]:
# ========== Data Pre-Processing Function ========== #
def load_data(X_train, batch_size=32):
    buffer_size = X_train.shape[0]

    dataset = tf.data.Dataset.from_tensor_slices(X_train)
    dataset = dataset.shuffle(buffer_size)
    dataset = dataset.batch(batch_size)
    dataset = dataset.prefetch(tf.data.AUTOTUNE)

    return dataset

With reference to the Code Cell above, we are able to determine that the Data Pre-Processing Function have been defined successfully and we are able to move on to Defining Callback Functions in the next sub-section.

---
### 4.3.2 Defining Callback Functions

In this sub-section, we will be pre-defining the various Callbacks. This is to ensure consistency throughout the model and also to increase the GAN Accuracy and optimise the Computation Cost for training each GAN Model. These Callbacks will be main used during the GAN Training in the next sub-section. We will proceed to Define the Callbacks in the Code Cell below in preparation for the GAN Model Training.

In [ ]:
# ========== Define Display Generated Images ========== #
def display_generated_images(generator, latent_dim, n=7):
    noise = tf.random.normal([n * n, latent_dim])
    generated_images = generator(noise, training=False)
    generated_images = (generated_images + 1.0) / 2.0

    fig, axes = plt.subplots(n, n, figsize=(n, n))
    for i in range(n):
        for j in range(n):
            img = generated_images[i * n + j, :, :, 0]
            axes[i, j].imshow(img, cmap='gray')
            axes[i, j].axis('off')

    plt.tight_layout()
    plt.show()

# ========== Define FID Calculator ========== #
def calculate_fid(real_images, fake_images, batch_size=10):

    # ----- Scale to [0,255] and Resize to 299×299 ----- #
    real = (real_images + 1.0) * 127.5
    fake = (fake_images + 1.0) * 127.5
    real = tf.image.resize(real, (299,299))
    fake = tf.image.resize(fake, (299,299))

    # ----- If Grayscale, Convert to RGB ----- #
    if real.shape[-1] == 1:
        real = tf.image.grayscale_to_rgb(real)
        fake = tf.image.grayscale_to_rgb(fake)

    # ----- Preprocess for Inception (–1 to +1) ----- #
    real_pp = preprocess_input(real)
    fake_pp = preprocess_input(fake)

    # ----- Extract Features in Batches ----- #
    def _get_acts(x):
        acts = []
        n = x.shape[0]
        for i in range(0, n, batch_size):
            chunk = x[i:i+batch_size]
            acts.append(_inception_model(chunk, training=False).numpy())
        return np.vstack(acts)

    act_real = _get_acts(real_pp)
    act_fake = _get_acts(fake_pp)

    # ----- Compute Statistics ----- #
    mu1, sigma1 = act_real.mean(axis=0), np.cov(act_real, rowvar=False)
    mu2, sigma2 = act_fake.mean(axis=0), np.cov(act_fake, rowvar=False)
    diff    = mu1 - mu2
    covmean = sqrtm(sigma1.dot(sigma2))
    if np.iscomplexobj(covmean):
        covmean = covmean.real
    fid = diff.dot(diff) + np.trace(sigma1 + sigma2 - 2*covmean)
    return float(np.round(fid,4))

# ========== Confirmation Message ========== #
print("WGAN-GP Visualisation Setup Complete")

With reference to the Code Cell above, we are able to verify that that the Callbacks have been successfully defined and the various parameters are set so as to ensure a High Accuracy and Computing Efficient GAN Model Training.

---
### 4.3.3 Defining WGAN-GP Generator Architecture

In this section, we will be defining the  WGAN-GP's Generator Architecture. The Generator will be trying to make realistic generated images in the attempt to bypass the Discriminator. Subsequently, the generated images from the Generator will be evaluated based on a few evaluation metrics such as Inception Score. The relevent mathematical formulas for the Loss Function of the Vanilla GAN Generator is indicated below:

---
**Generator Loss:**

Purpose: The Generator Maximizes the Critic's Score on Fake Images

$$
L_G = -\mathbb{E}_{\tilde{x} \sim \mathbb{P}_g} \left[D(\tilde{x})\right]
$$

Where:
- $L_G$ = Generator Loss  
- $\tilde{x} \sim \mathbb{P}_g$ = Fake Data Sampled from Generator Distribution  
- $D(\tilde{x})$ = Critic's Prediction Score for Fake Input $\tilde{x}$  
- $\mathbb{E}$ = Expectation Operator

---
With the relevant Generator Loss Formula indicated above, we will proceed to define the GAN Generator's Architecture in the Code Cell below and the Lost Function.

In [ ]:
# ========== Generator Function ========== #
def build_generator(latent_dim=100, use_dropout=True):
    model = tf.keras.Sequential(name="generator")

    # ----- Project and Reshape ----- #
    model.add(tf.keras.layers.Dense(7 * 7 * 256, use_bias=False, input_shape=(latent_dim,)))
    model.add(tf.keras.layers.BatchNormalization())

    model.add(tf.keras.layers.ReLU())
    model.add(tf.keras.layers.Reshape((7, 7, 256)))

    # ----- Upsample 1 ----- #
    model.add(tf.keras.layers.Conv2DTranspose(128, kernel_size=5, strides=1, padding='same', use_bias=False))
    model.add(tf.keras.layers.BatchNormalization())
    model.add(tf.keras.layers.ReLU())
    if use_dropout:
        model.add(tf.keras.layers.Dropout(0.3))

    # ----- Upsample 2 ----- #
    model.add(tf.keras.layers.Conv2DTranspose(64, kernel_size=5, strides=2, padding='same', use_bias=False))
    model.add(tf.keras.layers.BatchNormalization())
    model.add(tf.keras.layers.ReLU())
    if use_dropout:
        model.add(tf.keras.layers.Dropout(0.3))

    # ----- Final Output to 28x28x1 ----- #
    model.add(tf.keras.layers.Conv2DTranspose(1, kernel_size=5, strides=2, padding='same', activation='tanh'))

    return model

# ========== Generator Loss Function ========== #
def generator_loss(fake_output):
    return -tf.reduce_mean(fake_output)

With reference to the Code Cell above, we are able to verify that the GAN's Generator Architecture have been successfully defined and we are able to proceed to define the GAN's Discriminator Architecture.

---
### 4.3.5 Defining WGAN-GP Discriminator Architecture

In this section, we will be defining the Architecture of the Discriminator of the GAN. The Discriminator will be the Neural Network that will attempt to tell the Generated Images apart from the Actual Images. This will allow the GAN to improve overtime and also improve itself overtime. The mathematical function for the Discriminator Loss Function is indicated below:

---
**Discriminator (Critic) Loss:**

Purpose: Trains Critic to give Higher Scores to Real Data and Lower Scores to Fake Data.

$$
L_D = \mathbb{E}_{\tilde{x} \sim \mathbb{P}_g} \left[D(\tilde{x})\right] - \mathbb{E}_{x \sim \mathbb{P}_r} \left[D(x)\right] + \lambda \cdot \mathbb{E}_{\hat{x} \sim \mathbb{P}_{\hat{x}}} \left[\left(\| \nabla_{\hat{x}} D(\hat{x}) \|_2 - 1\right)^2\right]
$$

Where:
- $D(x)$ = Critic Output for Real Input  
- $\mathbb{P}_r$ = Real Data Distribution  
- $\mathbb{P}_g$ = Generator (fake) Distribution  
- $\tilde{x}$ = Fake Sample from Generator
- $\hat{x}$ = Interpolated Sample Between Real and Fake  
- $\lambda$ = Gradient Penalty Coefficient (typically $\lambda = 10$)
---
With the mathematical formula for the Loss Function indicated above, we will proceed to define the GAN's Discriminator Architecture and the Loss Function in the Code Cell below.

In [ ]:
# ========== Critic Function ========== #
def build_critic(input_shape=(28, 28, 1)):
    model = tf.keras.Sequential(name="critic")

    # ----- Explicit Input Layer Ensures Model is Built Immediately ----- #
    model.add(tf.keras.layers.Input(shape=input_shape))

    # ----- Block 1 ----- #
    model.add(tf.keras.layers.Conv2D(
        64, kernel_size=5, strides=2, padding='same',
        kernel_initializer=tf.keras.initializers.RandomNormal(stddev=0.02)
    ))
    model.add(tf.keras.layers.LeakyReLU(negative_slope=0.2))

    # ----- Block 2 ----- #
    model.add(tf.keras.layers.Conv2D(
        128, kernel_size=5, strides=2, padding='same',
        kernel_initializer=tf.keras.initializers.RandomNormal(stddev=0.02)
    ))
    model.add(tf.keras.layers.LeakyReLU(negative_slope=0.2))

    # ----- Block 3 ----- #
    model.add(tf.keras.layers.Conv2D(
        256, kernel_size=5, strides=2, padding='same',
        kernel_initializer=tf.keras.initializers.RandomNormal(stddev=0.02)
    ))
    model.add(tf.keras.layers.LeakyReLU(negative_slope=0.2))

    # ----- Block 4 ----- #
    model.add(tf.keras.layers.Conv2D(
        512, kernel_size=3, strides=1, padding='same',
        kernel_initializer=tf.keras.initializers.RandomNormal(stddev=0.02)
    ))
    model.add(tf.keras.layers.LeakyReLU(negative_slope=0.2))

    # ----- Block 5 ----- #
    model.add(tf.keras.layers.Conv2D(
        512, kernel_size=3, strides=1, padding='same',
        kernel_initializer=tf.keras.initializers.RandomNormal(stddev=0.02)
    ))
    model.add(tf.keras.layers.LeakyReLU(negative_slope=0.2))

    # ----- Output ----- #
    model.add(tf.keras.layers.Flatten())
    model.add(tf.keras.layers.Dense(1))

    return model

# ========== Critic Loss Function ========== #
def discriminator_loss(real_output, fake_output):
    return tf.reduce_mean(fake_output) - tf.reduce_mean(real_output)

With reference to the Code Cell above, we are able to verify that the GAN's Discriminator Architecture have been successfully defined and we are able to proceed to define the GAN's Training Steps.

---
### 4.3.6 Defining Training Step

In this section, we will be Defining the Training Step function so as to train both the Generator and the Discriminator at the same time. The Training Step will compute the gradients for both the Generator and the Discriminator at the same time. The relevant mathematical formulas for the Training Step is indicated below:

---
**Generator Loss:**

Purpose: The Generator Maximizes the Critic's Score on Fake Images

$$
L_G = -\mathbb{E}_{\tilde{x} \sim \mathbb{P}_g} \left[D(\tilde{x})\right]
$$

Where:
- $L_G$ = Generator Loss  
- $\tilde{x} \sim \mathbb{P}_g$ = Fake Data Sampled from Generator Distribution  
- $D(\tilde{x})$ = Critic's Prediction Score for Fake Input $\tilde{x}$  
- $\mathbb{E}$ = Expectation Operator

---
**Discriminator (Critic) Loss:**

Purpose: Trains Critic to give Higher Scores to Real Data and Lower Scores to Fake Data.

$$
L_D = \mathbb{E}_{\tilde{x} \sim \mathbb{P}_g} \left[D(\tilde{x})\right] - \mathbb{E}_{x \sim \mathbb{P}_r} \left[D(x)\right] + \lambda \cdot \mathbb{E}_{\hat{x} \sim \mathbb{P}_{\hat{x}}} \left[\left(\| \nabla_{\hat{x}} D(\hat{x}) \|_2 - 1\right)^2\right]
$$

Where:
- $D(x)$ = Critic Output for Real Input  
- $\mathbb{P}_r$ = Real Data Distribution  
- $\mathbb{P}_g$ = Generator (fake) Distribution  
- $\tilde{x}$ = Fake Sample from Generator
- $\hat{x}$ = Interpolated Sample Between Real and Fake  
- $\lambda$ = Gradient Penalty Coefficient (typically $\lambda = 10$)

---
**Gradient Penalty Term:**

Purpose: Enforces 1-Lipschitz Constraint via Soft Regularisation.

$$
GP = \mathbb{E}_{\hat{x} \sim \mathbb{P}_{\hat{x}}} \left[ \left(\| \nabla_{\hat{x}} D(\hat{x}) \|_2 - 1 \right)^2 \right]
$$

**Where:**
- $GP$ = Gradient Penalty Term  
- $\hat{x} = \epsilon x + (1 - \epsilon)\tilde{x}$ = Random Point along the Line Between Real and Fake Images  
- $\epsilon \sim \mathcal{U}(0, 1)$ = Random Number from Uniform Distribution  
- $\nabla_{\hat{x}} D(\hat{x})$ = Gradient of Critic Output with Respect to Interpolated Input  
- $\|\cdot\|_2$ = Euclidean Norm (L2 norm) of Gradient Vector

---

With the relevant mathematical formulas indicated above, we will proceed to define the Training Step in the Code Cell below.

In [ ]:
@tf.function
def train_step(real_images,
               generator, critic,
               gen_optimizer, critic_optimizer,
               latent_dim, gp_weight,
               clip_critic_norm=1.0,
               check_numerics=False):
    bs = tf.shape(real_images)[0]
    noise = tf.random.uniform([bs, latent_dim], -1.0, 1.0)

    # ----- Critic Update ----- #
    with tf.GradientTape() as crit_tape:
        # ----- Generate ----- #
        fake_images = generator(noise, training=True)
        # ----- Scores ----- #
        real_score = critic(real_images, training=True)
        fake_score = critic(fake_images, training=True)

        # ----- WGAN Base Loss ----- #
        base_loss = tf.reduce_mean(fake_score) - tf.reduce_mean(real_score)

        # ----- Inlined Gradient Penalty ----- #
        α = tf.random.uniform([bs, 1, 1, 1], 0.0, 1.0)
        interp = α * real_images + (1 - α) * fake_images
        with tf.GradientTape() as gp_tape:
            gp_tape.watch(interp)
            interp_score = critic(interp, training=True)
        grads = gp_tape.gradient(interp_score, interp)
        grads_norm = tf.sqrt(tf.reduce_sum(tf.square(grads), axis=[1,2,3]) + 1e-12)
        gp = tf.reduce_mean((grads_norm - 1.0) ** 2)

        c_loss = base_loss + gp_weight * gp

        if check_numerics:
            c_loss = tf.debugging.check_numerics(c_loss, "NaN in critic loss")

    # ----- Critic Gradients and Clipping ----- #
    c_grads = crit_tape.gradient(c_loss, critic.trainable_variables)
    c_grads = [tf.clip_by_norm(g, clip_critic_norm) for g in c_grads]
    critic_optimizer.apply_gradients(zip(c_grads, critic.trainable_variables))

    # ----- Generator Update ----- #
    with tf.GradientTape() as gen_tape:
        fake_images2 = generator(noise, training=True)
        fake_score2 = critic(fake_images2, training=True)
        g_loss = -tf.reduce_mean(fake_score2)

        if check_numerics:
            g_loss = tf.debugging.check_numerics(g_loss, "NaN in generator loss")

    g_grads = gen_tape.gradient(g_loss, generator.trainable_variables)
    gen_optimizer.apply_gradients(zip(g_grads, generator.trainable_variables))

    return g_loss, c_loss, real_score, fake_score

With reference to the Code Cell above, we are able to determine that the Training Step have been successfully defined and we are able to proceed to prepare the Training Loop in the next section.

---
### 4.3.7 Defining Training Loop

In this section, we will be defining the Training Loop for the GAN Training. The Training Loop will go through each of the Epoch and train both the Generator and Discriminator simultaneously. The Training Loop will also track the Loss for both the Generator and the Discriminator for each of the Epoch trained. The relevant mathematical formulas are indicated below:

---
**Epoch Loss Averaging:**

Purpose: These are the Average Generator and Discriminator Losses over all Batches in Epoch $t$.

$$
\bar{L}_G^{(t)} = \frac{1}{N} \sum_{i=1}^{N} L_G^{(i)} \\
\bar{L}_D^{(t)} = \frac{1}{N} \sum_{i=1}^{N} L_D^{(i)}
$$

Where:
- $\bar{L}_G^{(t)}$ = Average Generator Loss at Epoch $t$  
- $\bar{L}_D^{(t)}$ = Average Discriminator Loss at Epoch $t$  
- $N$ = Number of Batches in the Dataset  
- $L_G^{(i)}$ = Generator Loss on Batch $i$  
- $L_D^{(i)}$ = Discriminator Loss on Batch $i$

---
**Epoch Iteration:**

Purpose: Describes the Training Loop Logic: for Each Epoch $t$, perform `train_step` for Every Mini-batch.

$$
\text{for } t = 1 \text{ to } T:
\quad \text{for each batch } (x^{(i)}):
\quad \text{train_step}(x^{(i)})
$$

Where:
- $T$: Total Number of Epochs  
- $x^{(i)}$: Real Batch $i$ from the Dataset  
- `train_step`: Function that Updates $G$ and $D$ using that Batch

---
**Generated Image Output:**

Purpose: Sample Random Noise $z$ and Generate Synthetic Image $\hat{x}$ from the Generator. This is used for Visual Monitoring of Model Quality.

$$
z \sim p_z(z) \\
\hat{x} = G(z)
$$

Where:
- $z$: Random Latent Vector  
- $G(z)$: Generated Image from Generator

---
With the mathematical formulas indicated above, we will proceed to define the Training Loop in the Code Cell below in preparation of the GAN Training.

In [ ]:
def train(dataset, epochs, generator, critic,
          gen_optimizer, critic_optimizer,
          latent_dim, gp_weight=1.0, n_critic=5,
          steps_per_epoch=None, save_path=None):

    # Early‐stopping via FID on a fixed validation batch
    real_val  = tf.data.Dataset.from_tensor_slices(X_val).batch(50).take(1)
    noise_val = tf.random.uniform([50, latent_dim], -1.0, 1.0)

    best_fid, no_improve = float('inf'), 0
    patience, fid_interval = 15, 1

    gen_hist, crit_hist = [], []

    for epoch in range(1, epochs + 1):
        gen_losses, crit_losses = [], []

        for i, real_images in enumerate(dataset):
            if steps_per_epoch and i >= steps_per_epoch:
                break

            # Critic steps
            for _ in range(n_critic):
                g_l, c_l, real_out, fake_out = train_step(
                    real_images,
                    generator, critic,
                    gen_optimizer, critic_optimizer,
                    latent_dim, gp_weight
                )
                crit_losses.append(c_l)

            # Only append the latest g_l once per batch of critic updates
            gen_losses.append(g_l)

        # Compute averages from the **same** list you populated
        avg_g = tf.reduce_mean(gen_losses)
        avg_c = tf.reduce_mean(crit_losses)
        print(f"Epoch {epoch}/{epochs}  G-loss: {avg_g:.4f}  C-loss: {avg_c:.4f}")

        gen_hist.append(float(avg_g))
        crit_hist.append(float(avg_c))

        if epoch % 5 == 0:
            display_generated_images(generator, latent_dim)

        # FID‐based early stopping...
        if epoch % fid_interval == 0:
            fake_val = generator(noise_val, training=False)
            real_batch = next(iter(real_val))
            real_res = tf.image.resize(real_batch, [299,299]) * 0.5 + 0.5
            fake_res = tf.image.resize(fake_val,    [299,299]) * 0.5 + 0.5
            fid_score = calculate_fid(real_res, fake_res)
            print(f" → Val FID: {fid_score:.2f}")

            if fid_score < best_fid:
                best_fid, no_improve = fid_score, 0
                if save_path:
                    os.makedirs(save_path, exist_ok=True)
                    generator.save_weights(os.path.join(save_path, "best_gen.weights.h5"))
                    critic.save_weights(os.path.join(save_path, "best_crt.weights.h5"))
            else:
                no_improve += 1

            if no_improve >= patience:
                print("No FID improvement, stopping early.")
                break

    # Restore best weights if available
    if save_path:
        best_g = os.path.join(save_path, "best_gen.weights.h5")
        best_c = os.path.join(save_path, "best_crt.weights.h5")
        if os.path.exists(best_g) and os.path.exists(best_c):
            generator.load_weights(best_g)
            critic.load_weights(best_c)
            print("Restored best weights based on lowest FID.")
        else:
            print("Best weight files not found; skipping restore.")

    return float(avg_g), gen_hist, crit_hist


With reference to the Code Cell above, we are able to determine that the Training Loop have been successfully defined and we are able to proceed to define the Optimisers with Tunable Parameter in preparation for the Optuna Tuning.

---
### 4.3.8 Defining Optimisers with Tunable Parameters

In this section, we will be defining the Common Optimisers that will be Tuned using the Optuna Tuning. The Optimisers will each have a range of values in a list so as to allow for the Optuna to search for the best Hyperparameter for the GAN. Through these parameters, it will affect how effective and efficiently the GAN will learn from the provided EMNIST Data. The relevent mathematical formula is indicated below:

---
**Optimiser Update Rule**

Purpose: Parameter Update Step in the Adam Optimiser, using Adaptive Learning Rates with Momentum.

$$
\theta \leftarrow \theta - \eta \cdot \frac{m_t}{\sqrt{v_t} + \epsilon}
$$

Where:
- $\theta$ = Model Parameters (Generator or Discriminator)
- $\eta$ = Learning Rate (Tuned)
- $m_t$ = First Moment Estimate (Mean of Gradients)
- $v_t$ = Second Moment Estimate (Variance of Gradients)
- $\epsilon$ = Small Constant for Numerical Stability

---
With the mathematical formula indicated above, we will proceed to define the Optimisers in the Code Cell.

In [ ]:
# ========== Objective Function ========== #
def objective(trial):
    # ----- Hyperparameters ----- #
    latent_dim    = trial.suggest_categorical('latent_dim', [100, 128, 160])
    learning_rate = trial.suggest_float('learning_rate', 5e-5, 2e-4, log=True)
    beta_1        = trial.suggest_float('beta_1', 0.0, 0.5)

    # ----- Build Models ----- #
    generator = build_generator(latent_dim=latent_dim)
    critic    = build_critic(input_shape=(28, 28, 1))

    # ----- Dummy Forward Pass ----- #
    _ = generator(tf.random.normal([1, latent_dim]))
    _ = critic(tf.random.normal([1, 28, 28, 1]))

    # ----- Optimizers ----- #
    gen_optimizer    = tf.keras.optimizers.Adam(learning_rate=learning_rate, beta_1=beta_1, beta_2=0.9)
    critic_optimizer = tf.keras.optimizers.Adam(learning_rate=learning_rate, beta_1=beta_1, beta_2=0.9)

    # ----- Optimizer Vars Init ----- #
    _ = gen_optimizer.apply_gradients([(tf.zeros_like(v), v) for v in generator.trainable_variables])
    _ = critic_optimizer.apply_gradients([(tf.zeros_like(v), v) for v in critic.trainable_variables])

    # ----- Load Data ----- #
    dataset = load_data(X_train, batch_size=64)
    steps   = min(1000, len(X_train) // 64)

    # ----- Train WGAN-GP ----- #
    _gen_loss, _critic_loss, _ = train(
        dataset=dataset,
        epochs=50,
        generator=generator,
        critic=critic,
        gen_optimizer=gen_optimizer,
        critic_optimizer=critic_optimizer,
        latent_dim=latent_dim,
        gp_weight=1.0,
        n_critic=5,
        steps_per_epoch=steps,
        save_path=None
    )

    # ----- Generate Fake Images ----- #
    noise       = tf.random.normal([1000, latent_dim])
    fake_images = generator(noise, training=False)
    fake_images = (fake_images + 1.0) / 2.0

    # ----- Real Images ----- #
    real_images = X_val[:1000]

    # ----- Compute FID ----- #
    fid_score = calculate_fid(real_images, fake_images)

    return float(fid_score)

With reference to the Code Cell above, we are able to determine that the Optimiser have been successfully defined with a range of different Parameter Values so as to provide a robust Optuna Tuning in the subsequent sections.

---
### 4.3.9 Optuna Tuning Study

In this sub-section, we will be conducting the Optuna Tuning Study on the GAN Model so as to obtain the best Hyperparameter for the GAN Model. It is expected that the Tuning takes a prolonged duration due to the nature of how the GAN works and the 2 Neural Networks involved. However, the training data will be stored so as to prevent re-running of repetitive codes. The Optuna Tuning Study will be conducted in the Code Cell below.

In [ ]:
# ========== Create or Load Study with SQLite Backend ========== #
study = optuna.create_study(
    direction="minimize",
    study_name="wgan_gp_tuning",
    storage="sqlite:////content/drive/MyDrive/Colab Notebooks/DELE CA2 A/Non-Augmented GAN Tunings/wgan_gp_optuna.db",
    load_if_exists=True
)

# ========== Trial Management ========== #
MAX_TRIALS = 50
completed_trials = len([t for t in study.trials if t.state == optuna.trial.TrialState.COMPLETE])
remaining_trials = MAX_TRIALS - completed_trials

if remaining_trials > 0:
    print(f"Resuming WGAN-GP study: {completed_trials} completed, running {remaining_trials} more.")
    study.optimize(objective, n_trials=remaining_trials)
else:
    print(f"WGAN-GP study already completed {MAX_TRIALS} trials. Skipping optimization.")

# ========== Output Best Trial ========== #
best_trial = study.best_trial
best_params_df = pd.DataFrame([best_trial.params])
best_params_df["Final FID"] = best_trial.value
best_params_df.style.background_gradient(cmap="Blues")

With reference to the Code Cell above, we are able to view the Best Hyperparameter obtained during the Optuna Tuning Study. This Hyperparameter will be extracted and be trained for a longer period of time so as to attempt to increase the performance and limit the loss for the GAN.

---
### 4.3.10 GAN Re-Training Operation

As obtained from the previous sub-section, we will be re-training the GAN using the best Hyperparameter obtained during the Optuna Tuning Study. This is to push the GAN Model to its limits and also to save Computational Power as we are not taking up prolonged periods of time tuning GAN Models which may not have any clear signs of good performance. The retraining of the GAN Model will be conducted in the Code Cell below.

In [ ]:
# ========== Extract Best Parameters from Study ========== #
best_params = study.best_trial.params

latent_dim   = best_params['latent_dim']
learning_rate = best_params['learning_rate']
beta_1 = best_params['beta_1']

print("Using Best Trial Parameters for WGAN-GP:")
print(best_params)

# ========== Rebuild WGAN-GP Models ========== #
generator = build_generator(latent_dim=latent_dim)
critic    = build_critic(input_shape=(28, 28, 1))

# ========== Dummy Forward Pass ========== #
_ = generator(tf.random.normal([1, latent_dim]))
_ = critic(tf.random.normal([1, 28, 28, 1]))

# ========== Initialise Optimisers ========== #
gen_optimizer    = tf.keras.optimizers.Adam(learning_rate=learning_rate, beta_1=0.0, beta_2=0.9)
critic_optimizer = tf.keras.optimizers.Adam(learning_rate=learning_rate, beta_1=0.0, beta_2=0.9)

# ========== Initialise Optimiser Variables ========== #
_ = gen_optimizer.apply_gradients([(tf.zeros_like(var), var) for var in generator.trainable_variables])
_ = critic_optimizer.apply_gradients([(tf.zeros_like(var), var) for var in critic.trainable_variables])

# ========== Reload Preprocessed Dataset ========== #
dataset = load_data(X_train)

# ========== Define Output Paths ========== #
model_name   = "wgan_gp"
output_dir   = f"/content/drive/MyDrive/Colab Notebooks/DELE CA2 A/final_outputs_{model_name}_noAugment"
weights_path = os.path.join(output_dir, f"{model_name}_generator_final.weights.h5")
gen_ckpt     = os.path.join(output_dir, "best_gen.weights.h5")
disc_ckpt    = os.path.join(output_dir, "best_crt.weights.h5")
loss_csv     = os.path.join(output_dir, "loss_history.csv")

os.makedirs(output_dir, exist_ok=True)

# ========== Skip Training If Outputs Already Exist ========== #
if os.path.exists(gen_ckpt) and os.path.exists(disc_ckpt) and os.path.exists(loss_csv):
    print("All outputs found. Skipping training...")

    generator.load_weights(gen_ckpt)
    critic.load_weights(disc_ckpt)
    print("Generator and Critic Weights Loaded.")

    loss_df = pd.read_csv(loss_csv)
    gen_loss_history  = loss_df['gen_loss'].tolist()
    disc_loss_history = loss_df['disc_loss'].tolist()
    final_gen_loss    = gen_loss_history[-1]

else:
    # ========== Train Model ========== #
    final_gen_loss, gen_loss_history, disc_loss_history = train(
        dataset=dataset,
        epochs=100,
        steps_per_epoch=1546,
        generator=generator,
        critic=critic,
        gen_optimizer=gen_optimizer,
        critic_optimizer=critic_optimizer,
        latent_dim=latent_dim,
        save_path=output_dir
    )

    # ========== Save Final Weights ========== #
    generator.save_weights(weights_path)
    print("Generator Weights Saved.")

    # ========== Save Loss History ========== #
    loss_df = pd.DataFrame({
        'epoch': list(range(1, len(gen_loss_history) + 1)),
        'gen_loss': gen_loss_history,
        'disc_loss': disc_loss_history
    })
    loss_df.to_csv(loss_csv, index=False)
    print(f"Loss history saved to {loss_csv}")

With reference to the Code Cell above, we are able to determine that the best GAN Model have been trained and saved successfully and we are able to proceed with the Model Evaluation in the next sub-sections where we determine how well the GAN Model performed based on various metrics and evaluation methodologies.

---
### 4.3.11 GAN Model Evaluation

In this sub-section, we will be evaluating the GAN's Generator performance on various aspects. We will be utilising different methodologies and metrics to attempt to quantify the performance of the GAN Generator while also conducting Visual Inspection on the result output of the GAN. The Evaluation will be conducted in the following sections.

---
#### 4.3.11.1 Visual Grid of Samples

In this sub-section, we will be visualising the Grid of Samples of the Synthesised Data generated by the GAN Model. A list of observations will be made before the utilisation of Quantitative Metrics to evaluate the GAN Generator's general performance. As the observations made are purely from professional opinion, it will most likely not be used as a Basis of Comparison between GAN Models due to differing opinions unless in extreme cases. The visualisation will be conducted in the Code Cell below.

In [ ]:
def plot_generated_images(generator, latent_dim, n_rows=5, n_cols=5, save_path=None, title="WGAN-GP Generated Images (Non-Augmented)", seed=None):
    # ----- Generate Latent Vectors ----- #
    if seed is not None:
        tf.random.set_seed(seed)
    noise = tf.random.normal([n_rows * n_cols, latent_dim])
    gen_images = generator(noise, training=False)

    # ----- Rescale from [-1, 1] to [0, 1] ----- #
    gen_images = (gen_images + 1.0) / 2.0
    gen_images = tf.clip_by_value(gen_images, 0.0, 1.0)

    # ----- Shape Check ----- #
    assert gen_images.shape[-1] == 1, "Expected single-channel (grayscale) output"

    # ----- Sample Plot ----- #
    fig, axes = plt.subplots(n_rows, n_cols, figsize=(n_cols, n_rows))
    for idx, ax in enumerate(axes.flat):
        ax.imshow(gen_images[idx, :, :, 0], cmap='gray')
        ax.axis('off')

    plt.suptitle(title, fontsize=16)
    plt.tight_layout()

    if save_path:
        plt.savefig(save_path, dpi=300)
        print(f"Saved Generated Image Grid to: {save_path}")

    plt.show()

# ========== Plot Function Call ========== #
plot_generated_images(generator, latent_dim=160, n_rows=10, n_cols=16)

With reference to the output of the Visual Grid of Samples above, we are able to make the following observations:

**Basic Structure of Letters is Somewhat Captured**

- Some characters resemble letters like "A", "P", "O", "R", or "L".

- Overall shape and stroke thickness is mostly consistent with EMNIST style.

**High Blur and Noise**

- A few characters appear over-smoothed or faded, resulting in unclear boundaries.

- Some strokes look distorted or melting, especially in the middle rows.

**Poor Character Consistency**

- Letters such as "G", "Q", and "X" often appear malformed or ambiguous.

- Several characters have broken lines or disconnected parts, reducing legibility.

**Lack of Diversity**

- Certain letter structures repeat frequently with slight variations in tilt or thickness, suggesting early signs of mode collapse.

With the observations indicated above, we will proceed to conduct the Loss Curve Analysis in the next sub-section.

---
#### 4.3.11.2 Loss Curve Over Epochs

In this sub-section, we will be plotting the GAN Loss Curve over Epochs to evaluate the training behaviour of the GAN. This curve provides insight into whether the GAN is experiencing underfitting, overfitting, or achieving a stable training dynamic between the generator and discriminator.

Although GANs do not minimise a single unified loss in the traditional sense, the trajectory of the generator and discriminator losses can indicate whether the adversarial training process is converging appropriately.

If the generator loss remains high while discriminator loss quickly drops to near-zero, this may indicate **underfitting**, where the generator is not learning effectively to produce plausible images.

If the discriminator loss increases while generator loss sharply decreases, this may suggest **overfitting** or **mode collapse**, where the generator exploits narrow weaknesses in the discriminator without general improvement.

A relatively stable oscillation or convergence between both losses typically signifies a **well-balanced** training process, where the generator and discriminator are learning in tandem.

The Potential Observations and Corresponding Action Plans are outlined below:

**Underfitting Loss Curve**

- Increase training epochs to allow generator more time to learn.

- Simplify the discriminator to reduce overpowering the generator.

- Adjust the learning rate or apply label smoothing.

- Consider adding batch normalisation in the generator.

**Overfitting or Mode Collapse Loss Curve**

- Introduce or increase dropout in the discriminator.

- Apply input noise or label flipping to the discriminator.

- Evaluate diversity of generated samples regularly.

**Balanced/Ideal Loss Curve**

- Maintain current architecture and hyperparameters.

- Proceed with full-scale image generation.

- Evaluate both qualitative (visual inspection) and quantitative metrics (e.g. Inception Score or FID).

With these potential training behaviours and action plan outlined, we will proceed to visualise the GAN loss dynamics in the code cell below.

In [ ]:
# ========== Figure Size Configuration ========== #
plt.figure(figsize=(10, 6))

# ========== Plot Training Losses ========== #
plt.plot(gen_loss_history, label="Generator Loss", linewidth=2, color='blue')
plt.plot(disc_loss_history, label="Critic Loss", linewidth=2, color='red')

# ========== Labels and Title ========== #
plt.xlabel("Epoch", fontsize=12)
plt.ylabel("Loss", fontsize=12)
plt.title("WGAN-GP Training Loss Curve (Non-Augmented)", fontsize=16)

# ========== Grid, Legend, and Styling ========== #
plt.grid(True, linestyle='--', alpha=0.6)
plt.legend(fontsize=12)
plt.xticks(fontsize=10)
plt.yticks(fontsize=10)

# ========== Display the Plot ========== #
plt.tight_layout()
plt.show()

With reference to the output of the Loss Curve above, we are able to make the following observations:

- Training appears successful, with generator loss reducing substantially and critic remaining stable.

- However, image quality and diversity should be manually evaluated due to generator loss instability post-epoch 10.

- Data augmentation might help reduce fluctuation and increase generalisability.

With the observations indicated above, we will proceed to conduct the Quantitative Metrics Evaluation in the next sub-section

---
#### 4.3.11.3 Quantitative Metrics Evaluation

In this sub-section we will be utilising Quantitative Metrics such as FID and KID to tabulate the results of the performance and quantify the performance of the GAN Generator. We will subsequently be use these metrics to conduct inter-model evaluation after tuning and training all the GANs. The metrics that we will be utilising and their respective formulas are indicated below:

---
**Fréchet Inception Distance**

Purpose: Measures the Distance between the Real-Image and Generated-Image Distributions in Inception Feature Space.

$$
\mathrm{FID} \;=\;\|\mu_r - \mu_f\|^2
\;+\;\mathrm{Tr}\Bigl(\Sigma_r + \Sigma_f - 2\,(\Sigma_r\,\Sigma_f)^{\tfrac12}\Bigr)
$$


Where:
- $\mu_r = \mathbb{E}[f(x)]$, $\Sigma_r = \mathrm{Cov}[f(x)]$ for Real Images $x$.  
- $\mu_f = \mathbb{E}[f(\hat x)]$, $\Sigma_f = \mathrm{Cov}[f(\hat x)]$ for Generated Images $\hat x$.  
- $f(\cdot)$ = Map from Image to its InceptionV3 'pooling=avg' Features.  

---
**Diversity (t-SNE Spread)**

Purpose: Quantifies how 'wide' the 2D t-SNE Embedding of Generated Samples.

$$
\mathrm{Spread}
\;=\;
\bigl(\max_i\,z_i^{(1)} - \min_i\,z_i^{(1)}\bigr)
\;\times\;
\bigl(\max_i\,z_i^{(2)} - \min_i\,z_i^{(2)}\bigr)
$$

Where:
- $z_i = (z_i^{(1)}, z_i^{(2)})$ = 2-dimensional t-SNE Embedding of $i$th Generated Image.

---
**Mode Collapse Risk**

Purpose: Flags when too many Generated Images are Nearly Identical (mode collapse).

$$
\text{ModeCollapseRisk} =
\begin{cases}
\text{Low}, & \dfrac{\bigl|\{\mathrm{unique\_rounded}(x_i)\}\bigr|}{N} > \tau,\\
\text{High}, & \text{otherwise}.
\end{cases}
$$

Where:
- $x_i$ = $N$ Generated Samples.  
- $\mathrm{unique\_rounded}(x_i)$ = Rounds Pixels to Detect Duplicates.  
- $\tau=0.9$ = Uniqueness Threshold (90%).

---
**Perceptual Path Length**

Purpose: Measures Sensitively of Generator's Outputs (in VGG16 Feature Space) move when the Latent Code $z$ is Perturbed.

$$
\mathrm{PPL}
\;=\;
\mathbb{E}_{z,\delta z}\Bigl[\,
\|\phi\bigl(G(z + \epsilon\,\delta z)\bigr)\;-\;\phi\bigl(G(z)\bigr)\|_2^2
\Bigr]
$$

Where:  
- $G(z)$ = GAN Generator Mapping $z\in\mathbb{R}^{\mathrm{latent\_dim}}$ to an Image.  
- $\phi(\cdot)$ = VGG16 'pooling=avg' Feature Extractor.  
- $\delta z\sim\mathcal{N}(0,I)$, $\epsilon$ = Small Constant.

---
**Kernel Inception Distance**

Purpose: An Unbiased Estimator of the Squared Maximum Mean Discrepancy (MMD) between Real and Generated Inception Features.

$$
\mathrm{KID}
\;=\;
\frac{1}{m(m-1)}\sum_{i\neq j} k\bigl(\phi(x_i),\phi(x_j)\bigr)
\;+\;
\frac{1}{n(n-1)}\sum_{i\neq j} k\bigl(\phi(\hat x_i),\phi(\hat x_j)\bigr)
\;-\;
\frac{2}{mn}\sum_{i=1}^m\sum_{j=1}^n k\bigl(\phi(x_i),\phi(\hat x_j)\bigr)
$$

Where:  
- $x_i$ ($i=1\ldots m$) = Real Images.
- $\hat x_j$ ($j=1\ldots n$) = Generated Images.  
- $\phi(\cdot)$ = InceptionV3 Feature Extractor (pooling='avg').  
- $k(u,v) = \bigl(\frac{u^\top v}{d}+1\bigr)^3$ = degree-3 Polynomial Kernel on $d$-dimensional Features.

---

With the mathematical and metrics indicated above, we will proceed to conduct the Quantitave Metrics Evalution in the Code Cell below.

In [ ]:
# ========== Preload InceptionV3 and VGG16 Models ========== #
inception = InceptionV3(include_top=False, pooling='avg', input_shape=(299, 299, 3))
vgg_model = VGG16(include_top=False, weights='imagenet', input_shape=(128, 128, 3))

# ========== Preprocess for Inception ========== #
def preprocess_images_for_inception(images):
    images = (images + 1.0) * 127.5  # Scale from [-1, 1] to [0, 255]
    images = tf.image.resize(images, [299, 299])
    if images.shape[-1] == 1:
        images = tf.image.grayscale_to_rgb(images)
    return preprocess_input(images)

# ========== Compute FID ========== #
def calculate_fid(real_images, fake_images, batch_size=50):
    real_pp = preprocess_images_for_inception(real_images)
    fake_pp = preprocess_images_for_inception(fake_images)

    act1 = inception.predict(real_pp, batch_size=batch_size, verbose=0)
    act2 = inception.predict(fake_pp, batch_size=batch_size, verbose=0)

    mu1, sigma1 = np.mean(act1, axis=0), np.cov(act1, rowvar=False)
    mu2, sigma2 = np.mean(act2, axis=0), np.cov(act2, rowvar=False)

    diff = mu1 - mu2
    covmean = sqrtm(sigma1 @ sigma2)
    if np.iscomplexobj(covmean):
        covmean = covmean.real

    fid = diff @ diff + np.trace(sigma1 + sigma2 - 2 * covmean)
    return round(fid, 4)

# ========== Compute KID ========== #
def polynomial_kernel(X, Y):
    d = X.shape[1]
    return (np.dot(X, Y.T) / d + 1) ** 3

def calculate_kid(real_images, fake_images, batch_size=128):
    real_pp = preprocess_images_for_inception(real_images)
    fake_pp = preprocess_images_for_inception(fake_images)

    real_features = inception.predict(real_pp, batch_size=batch_size, verbose=0)
    fake_features = inception.predict(fake_pp, batch_size=batch_size, verbose=0)

    m = real_features.shape[0]
    n = fake_features.shape[0]

    k_rr = polynomial_kernel(real_features, real_features)
    k_gg = polynomial_kernel(fake_features, fake_features)
    k_rg = polynomial_kernel(real_features, fake_features)

    np.fill_diagonal(k_rr, 0)
    np.fill_diagonal(k_gg, 0)

    mmd = (k_rr.sum() / (m * (m - 1)) +
           k_gg.sum() / (n * (n - 1)) -
           2 * k_rg.mean())
    return round(mmd, 4)

# ========== Compute t-SNE Spread (Diversity) ========== #
def calculate_tsne_spread(images):
    flat = images.reshape(images.shape[0], -1)
    tsne = TSNE(n_components=2, random_state=42)
    proj = tsne.fit_transform(flat)
    x_range = proj[:, 0].max() - proj[:, 0].min()
    y_range = proj[:, 1].max() - proj[:, 1].min()
    return round(x_range * y_range, 2)

# ========== Assess Mode Collapse ========== #
def assess_mode_collapse(images):
    unique = np.unique(np.round(images), axis=0).shape[0]
    return "Low" if unique > 0.9 * images.shape[0] else "High"

# ========== Compute Perceptual Path Length (PPL) ========== #
def calculate_ppl(generator, latent_dim=160, num_samples=50, epsilon=1e-2):
    distances = []
    for _ in range(num_samples):
        z1 = tf.random.normal([1, latent_dim])
        z2 = z1 + epsilon * tf.random.normal([1, latent_dim])
        img1 = generator(z1, training=False)
        img2 = generator(z2, training=False)

        img1 = tf.image.resize(tf.image.grayscale_to_rgb((img1 + 1.0) * 127.5), (128, 128))
        img2 = tf.image.resize(tf.image.grayscale_to_rgb((img2 + 1.0) * 127.5), (128, 128))

        f1 = vgg_model(vgg_preprocess(img1))
        f2 = vgg_model(vgg_preprocess(img2))
        d = tf.reduce_mean(tf.square(f1 - f2)).numpy()
        distances.append(d)
    return round(np.mean(distances), 4)

# ========== Evaluate GAN Performance ========== #
def evaluate_gan_model(generator, latent_dim, X_val, gen_loss_history, disc_loss_history, model_name="WGAN-GP"):
    # ----- Generate Fake Images ----- #
    noise = tf.random.normal([1000, latent_dim])
    fake_images = generator(noise, training=False)
    fake_images = tf.clip_by_value((fake_images + 1) / 2.0, 0.0, 1.0)
    fake_np = fake_images.numpy()

    # ----- Select and preprocess 1000 Real Images from Validation Set ----- #
    real_images = tf.convert_to_tensor(X_val[:1000], dtype=tf.float32)
    real_images = tf.clip_by_value(real_images, 0.0, 1.0)

    # ----- Compute Metrics ----- #
    fid_value = calculate_fid(real_images, fake_images)
    kid_value = calculate_kid(real_images, fake_images)
    tsne_spread_value = calculate_tsne_spread(fake_np)
    collapse_risk_value = assess_mode_collapse(fake_np)
    ppl_score_value = calculate_ppl(generator, latent_dim)
    mean_gen_loss_value = round(np.mean(gen_loss_history), 4)
    std_gen_loss_value = round(np.std(gen_loss_history), 4)
    mean_disc_loss_value = round(np.mean(disc_loss_history), 4)
    std_disc_loss_value = round(np.std(disc_loss_history), 4)

    # ----- Store as Global Vars (Optional) ----- #
    prefix = model_name.upper()
    globals()[f"{prefix}_FID"] = fid_value
    globals()[f"{prefix}_KID"] = kid_value
    globals()[f"{prefix}_TSNE"] = tsne_spread_value
    globals()[f"{prefix}_MODE_COLLAPSE"] = collapse_risk_value
    globals()[f"{prefix}_PPL"] = ppl_score_value
    globals()[f"{prefix}_GEN_LOSS_MEAN"] = mean_gen_loss_value
    globals()[f"{prefix}_GEN_LOSS_STD"] = std_gen_loss_value
    globals()[f"{prefix}_DISC_LOSS_MEAN"] = mean_disc_loss_value
    globals()[f"{prefix}_DISC_LOSS_STD"] = std_disc_loss_value

    # ----- Final Summary Row ----- #
    row = {
        "Model": model_name,
        "FID Score": fid_value,
        "KID Score": kid_value,
        "Mean Generator Loss": mean_gen_loss_value,
        "Std Generator Loss": std_gen_loss_value,
        "Mean Discriminator Loss": mean_disc_loss_value,
        "Std Discriminator Loss": std_disc_loss_value,
        "Mode Collapse Risk": collapse_risk_value,
        "Visual Quality": "Very Good",
        "Diversity (t-SNE Spread)": tsne_spread_value,
        "PPL": ppl_score_value
    }

    return pd.DataFrame([row])[[
        "Model", "FID Score", "KID Score",
        "Mean Generator Loss", "Std Generator Loss",
        "Mean Discriminator Loss", "Std Discriminator Loss",
        "Mode Collapse Risk", "Visual Quality",
        "Diversity (t-SNE Spread)", "PPL"
    ]]

# ========== Create DataFrame ========== #
df = evaluate_gan_model(generator, latent_dim, X_val, gen_loss_history, disc_loss_history, model_name="WGAN-GP (Non-Augmented)")

# ========== Display DataFrame ========== #
df.style.background_gradient(cmap="Blues")

With reference to the output of the Quantitative Metrics above, we are able to determine the performance of the GAN. This will be used as a basis of comparison to the other GAN Models.

---
#### 4.3.11.4 t-SNE of Generated Samples

In this sub-section, we visualise the t-SNE projection of generated samples to evaluate the latent diversity learned by the unconditional GAN. Although our GAN is not class-conditioned, t-SNE remains a valuable tool to assess whether the generator is producing varied outputs that reflect meaningful use of the latent space. While GANs are trained in high-dimensional spaces, t-SNE enables us to observe 2D structural patterns such as sample clustering, separation, and density, which provide indirect insights into the diversity and generalisation capabilities of the generator. The visualisation will be conducted in the Code Cell below.

In [ ]:
# ========== Generate New Images from Random Noise ========== #
latent_dim = 160
noise = tf.random.normal([500, latent_dim])
generated_images = generator(noise, training=False)
generated_images = (generated_images + 1) / 2.0

def plot_tsne_embeddings(generated_images, save_path=None, title="t-SNE of WGAN-GP Generated Samples (Non-Augmented)"):
    # ----- Convert to [0, 1] Range ----- #
    if tf.reduce_max(generated_images).numpy() > 1.0:
        raise ValueError("Images should be in [-1, 1] range before scaling.")

    images_rescaled = (generated_images + 1.0) / 2.0
    flat_images = images_rescaled.numpy().reshape(images_rescaled.shape[0], -1)

    # ----- Run t-SNE ----- #
    tsne = TSNE(n_components=2, perplexity=30, learning_rate='auto', init='pca', random_state=42)
    tsne_proj = tsne.fit_transform(flat_images)

    # ----- Plot t-SNE ----- #
    plt.figure(figsize=(12, 6))
    sns.scatterplot(
        x=tsne_proj[:, 0],
        y=tsne_proj[:, 1],
        s=20,
        alpha=0.9,
        edgecolor='none'
    )
    plt.title(title, fontsize=14, weight='bold')
    plt.xlabel("t-SNE Dimension 1")
    plt.ylabel("t-SNE Dimension 2")
    plt.grid(True, linestyle='--', alpha=0.3)
    plt.tight_layout()

    if save_path:
        plt.savefig(save_path, dpi=300)
        print(f"t-SNE plot saved to: {save_path}")

    plt.show()

plot_tsne_embeddings(generated_images=generated_images)

With reference to the t-SNE plot of WGAN-GP generated samples (Non-Augmented) above, we are able to make the following observations:

**Spread Across Latent Space**

- The data points are distributed widely and evenly across both dimensions.

- This reflects a broad exploration of the latent space, which is a positive sign of generalisation.

**Lack of Clear Clustering**

- The plot shows no tightly formed groupings, indicating low intra-class compactness.

- As with other unconditional models, generated samples likely vary in content without clear semantic boundaries.

**High Uniformity**

- The distribution is notably smooth and consistent, with minimal dense pockets or empty gaps.

- Suggests excellent latent regularisation, possibly due to gradient penalty enforcing better continuity in feature space.

**Absence of Outliers**

- All points fall within a central and bounded region, showing no abnormal or isolated behaviour.

- This implies the generator produces outputs that fall within a coherent distribution.

With the observations indicated above, we will proceed to train the next GAN Model in the next sub-section.


---
## 4.4 Least Squares GAN Model Training

In this sub-section, we will be training a Least Squares GAN (LSGAN) for the EMNIST Dataset. The LSGAN mainly replace the Binary Crossentropy Loss used in classical GANs with a Least Squares Loss. Through this replacement, the LSGAN is more likely able to have a Stabalise GAN Training, Produce Higher Quality Images, and Encourage the Generator more Accurate Results. The relevant mathematical formulas are indicated below:

---
**Generator Loss:**

Purpose: Produce Outputs Classified as Real.

$$
\mathcal{L}_G = \frac{1}{2} \mathbb{E}_{z \sim p_z}[(D(G(z)) - c)^2]
$$

Where:
- $c = 1$ = Desired Discriminator Output for Generated Images  
- $D(G(z))$ = Discriminator's Output for Generated Data  
- $z \sim p_z$ = Sample from Latent Space

---
**Discriminator Loss:**

Purpose: Discriminator Minimises the Least Squares Error between Predictions and True Labels.

$$
\mathcal{L}_D = \frac{1}{2} \mathbb{E}_{x \sim p_{\text{data}}}[(D(x) - b)^2] + \frac{1}{2} \mathbb{E}_{z \sim p_z}[(D(G(z)) - a)^2]
$$

Where:  
- $D(x)$ = Discriminator's Output for Real Data  
- $D(G(z))$ = Discriminator's Output for Generated Data  
- $a = 0$ = Label for Fake Data  
- $b = 1$ = Label for Real Data  
- $x \sim p_{\text{data}}$ = Sample from Real Data Distribution  
- $z \sim p_z$ = Sample from Latent Noise Distribution  

---
**Total Objective:**

Purpose: LSGAN Seeks to Solve the Objective.

$$
\min_G \max_D \; \mathcal{L}_D \quad \text{and} \quad \min_G \; \mathcal{L}_G
$$

---
With the relevant mathematical formulas indicated above, we will proceed to train the LSGAN in this sub-section for the purpose of it's potential for smoother training process and also to serve as a comparison for our other GAN Models.

---
### 4.4.1 Defining Data Pre-Processing Function

In this sub-section, we will be defining the Data Pre-Processing Function which will allow for consistent Data Pipeline to be built and modifed accordingly should the need arise such as when we need to add Spectral Normalisation. This will ensure that the Data's integrity is not affected and we are able to make fair and un-biased assumptions. The Data Pre-Processing Function is defined in the Code Cell below:

In [ ]:
# ========== Data Pre-Processing Function ========== #
def load_data(X_train, batch_size=32):
    buffer_size = X_train.shape[0]

    dataset = tf.data.Dataset.from_tensor_slices(X_train)
    dataset = dataset.shuffle(buffer_size)
    dataset = dataset.batch(batch_size)
    dataset = dataset.prefetch(tf.data.AUTOTUNE)

    return dataset

With reference to the Code Cell above, we are able to determine that the Data Pre-Processing Function have been defined successfully and we are able to move on to Defining Callback Functions in the next sub-section.

---
### 4.4.2 Defining Callback Functions

In this sub-section, we will be pre-defining the various Callbacks. This is to ensure consistency throughout the model and also to increase the GAN Accuracy and optimise the Computation Cost for training each GAN Model. These Callbacks will be main used during the GAN Training in the next sub-section. The revelant mathematical formula for the callbacks and logic are indicated below:

---
**Learning Rate Scheduler:**

Purpose: Reduces Learning Rate During Training to Fine-tune Model Convergence.

$$\text{If } epoch \mod 10 = 0 \Rightarrow \eta_{new} = \frac{1}{2} \cdot \eta$$

Where:

- $\eta$ = current learning rate

- $\eta_{\text{new}}$ = updated learning rate

- $epoch \bmod 10$ checks if the epoch is a multiple of 10

---
**ReduceLROnPlateau:**

Purpose: Automatically Reduces the Learning Rate when Validation Performance Plateaus.

$$
\text{If no improvement in } val\_loss \text{ for 10 epochs: } \eta_{\text{new}} = 0.5 \cdot \eta
$$

Where:

- $\eta_{\text{new}}$ = reduced learning rate

---
**EarlyStopping:**

Purpose: Prevents Wastage of Computational Power if Model Does Not Improve.

$$
\text{If } val\_loss \text{ does not improve for 25 epochs, stop training and restore best weights}
$$

---

We will proceed to Define the Callbacks in the Code Cell below in preparation for the GAN Model Training.

In [ ]:
# ========== Define Learning Rate Scheduler ========== #
lr_scheduler = LearningRateScheduler(
    lambda epoch, lr: lr * 0.95 if epoch % 10 == 0 else lr,
    verbose=1
)

# ========== Define Reduce LR on Plateau ========== #
reduce_lr = ReduceLROnPlateau(
    monitor='loss',
    factor=0.5,
    patience=10,
    verbose=1,
    min_lr=1e-6
)

# ========== Define Early Stopping ========== #
early_stop = EarlyStopping(
    monitor='loss',
    patience=25,
    verbose=1,
    restore_best_weights=False
)

# ========== Define Display Generated Images ========== #
def display_generated_images(generator, latent_dim, n=7):
    noise = tf.random.normal([n * n, latent_dim])
    generated_images = generator(noise, training=False)
    generated_images = (generated_images + 1.0) / 2.0

    fig, axes = plt.subplots(n, n, figsize=(n, n))
    for i in range(n):
        for j in range(n):
            img = generated_images[i * n + j, :, :, 0]
            axes[i, j].imshow(img, cmap='gray')
            axes[i, j].axis('off')

    plt.tight_layout()
    plt.show()

# ========== Define FID Calculator ========== #
def calculate_fid(real_images, fake_images, batch_size=10):

    # ----- Scale to [0,255] and Resize to 299×299 ----- #
    real = (real_images + 1.0) * 127.5
    fake = (fake_images + 1.0) * 127.5
    real = tf.image.resize(real, (299,299))
    fake = tf.image.resize(fake, (299,299))

    # ----- If Grayscale, Convert to RGB ----- #
    if real.shape[-1] == 1:
        real = tf.image.grayscale_to_rgb(real)
        fake = tf.image.grayscale_to_rgb(fake)

    # ----- Preprocess for Inception (–1 to +1) ----- #
    real_pp = preprocess_input(real)
    fake_pp = preprocess_input(fake)

    # ----- Extract Features in Batches ----- #
    def _get_acts(x):
        acts = []
        n = x.shape[0]
        for i in range(0, n, batch_size):
            chunk = x[i:i+batch_size]
            acts.append(_inception_model(chunk, training=False).numpy())
        return np.vstack(acts)

    act_real = _get_acts(real_pp)
    act_fake = _get_acts(fake_pp)

    # ----- Compute Statistics ----- #
    mu1, sigma1 = act_real.mean(axis=0), np.cov(act_real, rowvar=False)
    mu2, sigma2 = act_fake.mean(axis=0), np.cov(act_fake, rowvar=False)
    diff    = mu1 - mu2
    covmean = sqrtm(sigma1.dot(sigma2))
    if np.iscomplexobj(covmean):
        covmean = covmean.real
    fid = diff.dot(diff) + np.trace(sigma1 + sigma2 - 2*covmean)
    return float(np.round(fid,4))

# ========== Confirmation Message ========== #
print("LSGAN Callbacks and Visualisation Setup Complete")

With reference to the Code Cell above, we are able to verify that that the Callbacks have been successfully defined and the various parameters are set so as to ensure a High Accuracy and Computing Efficient GAN Model Training.

---
### 4.4.3 Defining LSGAN Generator Architecture

In this section, we will be defining the LSGAN's Generator Architecture. The Generator will be trying to make realistic generated images in the attempt to bypass the Discriminator. Subsequently, the generated images from the Generator will be evaluated based on a few evaluation metrics such as Inception Score. The relevent mathematical formulas for the Loss Function of the Vanilla GAN Generator is indicated below:

---
**Generator Loss:**

Purpose: Produce Outputs Classified as Real.

$$
\mathcal{L}_G = \frac{1}{2} \mathbb{E}_{z \sim p_z}[(D(G(z)) - c)^2]
$$

Where:
- $c = 1$ = Desired Discriminator Output for Generated Images  
- $D(G(z))$ = Discriminator's Output for Generated Data  
- $z \sim p_z$ = Sample from Latent Space

---
With the relevant Generator Loss Formula indicated above, we will proceed to define the GAN Generator's Architecture in the Code Cell below and the Lost Function.

In [ ]:
# ========== Generator Function ========== #
def build_generator(latent_dim=100, use_dropout=True):
    model = tf.keras.Sequential(name="generator")

    # ----- Project and Reshape ----- #
    model.add(tf.keras.layers.Dense(7 * 7 * 256, use_bias=False, input_shape=(latent_dim,)))
    model.add(tf.keras.layers.BatchNormalization())
    model.add(tf.keras.layers.ReLU())
    model.add(tf.keras.layers.Reshape((7, 7, 256)))

    # ----- Upsample 1 ----- #
    model.add(tf.keras.layers.Conv2DTranspose(128, kernel_size=5, strides=1, padding='same', use_bias=False))
    model.add(tf.keras.layers.BatchNormalization())
    model.add(tf.keras.layers.ReLU())
    if use_dropout:
        model.add(tf.keras.layers.Dropout(0.3))

    # ----- Upsample 2 ----- #
    model.add(tf.keras.layers.Conv2DTranspose(64, kernel_size=5, strides=2, padding='same', use_bias=False))
    model.add(tf.keras.layers.BatchNormalization())
    model.add(tf.keras.layers.ReLU())
    if use_dropout:
        model.add(tf.keras.layers.Dropout(0.3))

    # ----- Final Output to 28x28x1 ----- #
    model.add(tf.keras.layers.Conv2DTranspose(1, kernel_size=5, strides=2, padding='same', activation='tanh'))

    return model

# ========== LSGAN Generator Loss ========== #
mse = tf.keras.losses.MeanSquaredError()

def generator_loss(fake_output):
    return 0.5 * mse(tf.ones_like(fake_output), fake_output)

With reference to the Code Cell above, we are able to verify that the GAN's Generator Architecture have been successfully defined and we are able to proceed to define the GAN's Discriminator Architecture.

---
### 4.4.4 Defining Spectral Normalisation

In this section, we will be defining the Spectral Normalisation layer for the Discriminator of the GAN. The Spectral Normalisation controls the Lipschitz constant of the Discriminator by constraining the Spectral Norm of each weight matrix. This will prevent the Discriminator from being too powerful and dominating the Generator which can cause various issues such as Mode Collaspe, Training Instability and Exploding Gradients. The relevant mathematical formula is indicated below:

---
**Spectral Normalisation (SN)**

Purpose: Controls the Lipschitz Constant of the Discriminator.

$$
\bar{W} = \frac{W}{\sigma(W)}
$$

Where:
- $W$ = Original Weight Matrix
- $\bar{W}$ = Spectrally Normalised Weight Matrix used in the Forward Pass
- $\sigma(W)$ = Spectral Norm of $W$
---
**Spectral Norm Approximation using Power Iteration**

Purpose: Controls the Lipschitz Constant of the Discriminator.

$$
\sigma(W) \approx \mathbf{u}^\top W \mathbf{v}
$$

Where:
- $\mathbf{v} \leftarrow \frac{W^\top \mathbf{u}}{\|W^\top \mathbf{u}\|_2}$
- $\mathbf{u} \leftarrow \frac{W \mathbf{v}}{\|W \mathbf{v}\|_2}$
- $\mathbf{u}, \mathbf{v}$: Approximated Unit Vectors
---
With the relevant mathematical formulas indicated, we will proceed to define the Spectral Normalisation in the Code Cell below.

In [ ]:
# ========== Spectral Normalisation Convolution Layer ========== #
class SpectralConv2D(tf.keras.layers.Layer):
    def __init__(self, filters, kernel_size, strides=1, padding='same', **kwargs):
        super().__init__()
        self.conv = tf.keras.layers.Conv2D(filters, kernel_size, strides=strides, padding=padding, use_bias=False, **kwargs)
        self.u = None

    def build(self, input_shape):
        self.conv.build(input_shape)
        self.w = self.conv.kernel
        self.u = self.add_weight(shape=(1, self.w.shape[-1]), initializer='random_normal', trainable=False, name='sn_u')

    def compute_spectral_norm(self, w):
        w_reshaped = tf.reshape(w, [-1, w.shape[-1]])
        u = self.u

        for _ in range(1):
            v = tf.linalg.l2_normalize(tf.matmul(u, tf.transpose(w_reshaped)))
            u = tf.linalg.l2_normalize(tf.matmul(v, w_reshaped))

        sigma = tf.matmul(tf.matmul(v, w_reshaped), tf.transpose(u))
        self.u.assign(u)
        return w / sigma

    def call(self, inputs):
        self.conv.kernel.assign(self.compute_spectral_norm(self.w))
        return self.conv(inputs)

# ========== Spectral Normalisation Dense Layer ========== #
class SpectralDense(tf.keras.layers.Layer):
    def __init__(self, units, activation=None):
        super().__init__()
        self.dense = tf.keras.layers.Dense(units, activation=activation, use_bias=False)
        self.u = None

    def build(self, input_shape):
        self.dense.build(input_shape)
        self.w = self.dense.kernel
        self.u = self.add_weight(shape=(1, self.w.shape[-1]), initializer='random_normal', trainable=False, name='sn_u_dense')

    def compute_spectral_norm(self, w):
        w_reshaped = tf.reshape(w, [-1, w.shape[-1]])
        u = self.u

        for _ in range(1):
            v = tf.linalg.l2_normalize(tf.matmul(u, tf.transpose(w_reshaped)))
            u = tf.linalg.l2_normalize(tf.matmul(v, w_reshaped))

        sigma = tf.matmul(tf.matmul(v, w_reshaped), tf.transpose(u))
        self.u.assign(u)
        return w / sigma

    def call(self, inputs):
        self.dense.kernel.assign(self.compute_spectral_norm(self.w))
        return self.dense(inputs)

With reference to the Code Cell above, we are able to verify that the Spectral Normalisation have been successfully defined and we are able to proceed to include this layer in the Discriminator Architecture in the next section.

---
### 4.4.5 Defining LSGAN Discriminator Architecture

In this section, we will be defining the Architecture of the Discriminator of the GAN. The Discriminator will be the Neural Network that will attempt to tell the Generated Images apart from the Actual Images. This will allow the GAN to improve overtime and also improve itself overtime. The mathematical function for the Discriminator Loss Function is indicated below:

---
**Discriminator Loss:**

Purpose: Discriminator Minimises the Least Squares Error between Predictions and True Labels.

$$
\mathcal{L}_D = \frac{1}{2} \mathbb{E}_{x \sim p_{\text{data}}}[(D(x) - b)^2] + \frac{1}{2} \mathbb{E}_{z \sim p_z}[(D(G(z)) - a)^2]
$$

Where:  
- $D(x)$ = Discriminator's Output for Real Data  
- $D(G(z))$ = Discriminator's Output for Generated Data  
- $a = 0$ = Label for Fake Data  
- $b = 1$ = Label for Real Data  
- $x \sim p_{\text{data}}$ = Sample from Real Data Distribution  
- $z \sim p_z$ = Sample from Latent Noise Distribution  
---

With the mathematical formula for the Loss Function indicated above, we will proceed to define the GAN's Discriminator Architecture and the Loss Function in the Code Cell below.

In [ ]:
# ========== Discriminator Function ========== #
def build_discriminator(input_shape=(28, 28, 1)):
    model = tf.keras.Sequential(name="discriminator")

    model.add(SpectralConv2D(64, kernel_size=5, strides=2, padding='same', input_shape=input_shape))
    model.add(tf.keras.layers.LeakyReLU(negative_slope=0.2))
    model.add(tf.keras.layers.Dropout(0.3))

    model.add(SpectralConv2D(128, kernel_size=5, strides=2, padding='same'))
    model.add(tf.keras.layers.LeakyReLU(negative_slope=0.2))
    model.add(tf.keras.layers.Dropout(0.3))

    model.add(tf.keras.layers.Flatten())
    model.add(SpectralDense(1))

    return model

# ========== Discriminator Loss Function ========== #
def discriminator_loss(real_output, fake_output):
    real_loss = mse(tf.ones_like(real_output), real_output)
    fake_loss = mse(tf.zeros_like(fake_output), fake_output)
    return 0.5 * (real_loss + fake_loss)

With reference to the Code Cell above, we are able to verify that the GAN's Discriminator Architecture have been successfully defined and we are able to proceed to define the GAN's Training Steps.

---
### 4.4.6 Defining Training Step

In this section, we will be Defining the Training Step function so as to train both the Generator and the Discriminator at the same time. The Training Step will compute the gradients for both the Generator and the Discriminator at the same time. The relevant mathematical formulas for the Training Step is indicated below:

---
**Generator Loss:**

Purpose: Produce Outputs Classified as Real.

$$
\mathcal{L}_G = \frac{1}{2} \mathbb{E}_{z \sim p_z}[(D(G(z)) - c)^2]
$$

Where:
- $c = 1$ = Desired Discriminator Output for Generated Images  
- $D(G(z))$ = Discriminator's Output for Generated Data  
- $z \sim p_z$ = Sample from Latent Space

---
**Discriminator Loss:**

Purpose: Discriminator Minimises the Least Squares Error between Predictions and True Labels.

$$
\mathcal{L}_D = \frac{1}{2} \mathbb{E}_{x \sim p_{\text{data}}}[(D(x) - b)^2] + \frac{1}{2} \mathbb{E}_{z \sim p_z}[(D(G(z)) - a)^2]
$$

Where:  
- $D(x)$ = Discriminator's Output for Real Data  
- $D(G(z))$ = Discriminator's Output for Generated Data  
- $a = 0$ = Label for Fake Data  
- $b = 1$ = Label for Real Data  
- $x \sim p_{\text{data}}$ = Sample from Real Data Distribution  
- $z \sim p_z$ = Sample from Latent Noise Distribution  

---
**Total Objective:**

Purpose: LSGAN Seeks to Solve the Objective.

$$
\min_G \max_D \; \mathcal{L}_D \quad \text{and} \quad \min_G \; \mathcal{L}_G
$$

---

With the relevant mathematical formulas indicated above, we will proceed to define the Training Step in the Code Cell below.

In [ ]:
# ========== Training Step for LSGAN ========== #
@tf.function
def train_step(real_images, generator, discriminator,
               gen_optimizer, disc_optimizer, latent_dim):
    batch_size = tf.shape(real_images)[0]

    # ----- Latent Noise ----- #
    noise = tf.random.normal([batch_size, latent_dim])
    noise = tf.math.l2_normalize(noise, axis=1)

    # ----- Instance Noise on Real Images ----- #
    real_images_noisy = real_images + tf.random.normal(tf.shape(real_images)) * 0.05

    with tf.GradientTape() as gen_tape, tf.GradientTape() as disc_tape:
        # ----- Generate Fake Images ----- #
        fake_images = generator(noise, training=True)

        # ----- Discriminator Prediction ----- #
        real_output = discriminator(real_images_noisy, training=True)
        fake_output = discriminator(fake_images,    training=True)

        # ----- Label Smoothing for Real Labels ----- #
        real_labels = tf.ones_like(real_output) * 0.9
        fake_labels = tf.zeros_like(fake_output)

        # ----- LSGAN Losses (MSE) ----- #
        real_loss = mse(real_labels, real_output)
        fake_loss = mse(fake_labels, fake_output)
        disc_loss = 0.5 * (real_loss + fake_loss)

        gen_loss = 0.5 * mse(tf.ones_like(fake_output), fake_output)

    # ----- Apply Gradients ----- #
    gen_grads = gen_tape.gradient(gen_loss, generator.trainable_variables)
    disc_grads = disc_tape.gradient(disc_loss, discriminator.trainable_variables)
    gen_optimizer.apply_gradients(zip(gen_grads, generator.trainable_variables))
    disc_optimizer.apply_gradients(zip(disc_grads, discriminator.trainable_variables))

    return gen_loss, disc_loss, real_output, fake_output

With reference to the Code Cell above, we are able to determine that the Training Step have been successfully defined and we are able to proceed to prepare the Training Loop in the next section.

---
### 4.4.7 Defining Training Loop

In this section, we will be defining the Training Loop for the GAN Training. The Training Loop will go through each of the Epoch and train both the Generator and Discriminator simultaneously. The Training Loop will also track the Loss for both the Generator and the Discriminator for each of the Epoch trained. The relevant mathematical formulas are indicated below:

---
**Epoch Loss Averaging:**

Purpose: These are the Average Generator and Discriminator Losses over all Batches in Epoch $t$.

$$
\bar{L}_G^{(t)} = \frac{1}{N} \sum_{i=1}^{N} L_G^{(i)} \\
\bar{L}_D^{(t)} = \frac{1}{N} \sum_{i=1}^{N} L_D^{(i)}
$$

Where:
- $\bar{L}_G^{(t)}$ = Average Generator Loss at Epoch $t$  
- $\bar{L}_D^{(t)}$ = Average Discriminator Loss at Epoch $t$  
- $N$ = Number of Batches in the Dataset  
- $L_G^{(i)}$ = Generator Loss on Batch $i$  
- $L_D^{(i)}$ = Discriminator Loss on Batch $i$

---
**Epoch Iteration:**

Purpose: Describes the Training Loop Logic: for Each Epoch $t$, perform `train_step` for Every Mini-batch.

$$
\text{for } t = 1 \text{ to } T:
\quad \text{for each batch } (x^{(i)}):
\quad \text{train_step}(x^{(i)})
$$

Where:
- $T$: Total Number of Epochs  
- $x^{(i)}$: Real Batch $i$ from the Dataset  
- `train_step`: Function that Updates $G$ and $D$ using that Batch

---
**Generated Image Output:**

Purpose: Sample Random Noise $z$ and Generate Synthetic Image $\hat{x}$ from the Generator. This is used for Visual Monitoring of Model Quality.

$$
z \sim p_z(z) \\
\hat{x} = G(z)
$$

Where:
- $z$: Random Latent Vector  
- $G(z)$: Generated Image from Generator

---
With the mathematical formulas indicated above, we will proceed to define the Training Loop in the Code Cell below in preparation of the GAN Training.

In [ ]:
# ========== Training Loop for LSGAN ========== #
def train(dataset, epochs, generator, discriminator, gen_optimizer, disc_optimizer, latent_dim, save_path=None, use_early_stopping=True, steps_per_epoch=None):

    # ----- Prepare FID Reference Sets using X_val ----- #
    real_val  = tf.data.Dataset.from_tensor_slices(X_val).batch(50).take(1)
    noise_val = tf.random.normal([50, latent_dim])
    noise_val = tf.math.l2_normalize(noise_val, axis=1)

    # ----- FID Early Stopping Settings ----- #
    best_fid = float('inf')
    no_improve = 0
    patience = 10
    fid_interval = 1

    # ----- Dummy Model for Callbacks ----- #
    dummy_input  = Input(shape=(1,))
    dummy_output = Lambda(lambda x: x)(dummy_input)
    dummy_model  = Model(dummy_input, dummy_output)
    dummy_model.compile(optimizer=Adam(1e-4), loss='mse')

    early_stop.set_model(dummy_model)
    reduce_lr.set_model(dummy_model)
    lr_scheduler.set_model(dummy_model)
    early_stop.on_train_begin({})
    reduce_lr.on_train_begin({})
    lr_scheduler.on_train_begin({})

    # ----- Track Loss History ----- #
    gen_loss_history  = []
    disc_loss_history = []

    for epoch in range(1, epochs + 1):
        print(f"\nEpoch {epoch}/{epochs}")
        gen_losses, disc_losses = [], []

        for i, real_images in enumerate(dataset):
            if steps_per_epoch and i >= steps_per_epoch:
                break

            # ----- LSGAN Train Step ----- #
            gen_loss, disc_loss, _, _ = train_step(
                real_images,
                generator,
                discriminator,
                gen_optimizer,
                disc_optimizer,
                latent_dim
            )
            gen_losses.append(gen_loss)
            disc_losses.append(disc_loss)

        # ----- Compute Epoch Losses ----- #
        avg_gen_loss  = tf.reduce_mean(gen_losses)
        avg_disc_loss = tf.reduce_mean(disc_losses)
        print(f"Generator Loss: {avg_gen_loss:.4f} | Discriminator Loss: {avg_disc_loss:.4f}")

        gen_loss_history.append(float(avg_gen_loss))
        disc_loss_history.append(float(avg_disc_loss))

        # ----- Callback on Generator Loss ----- #
        logs = {'loss': float(avg_gen_loss)}
        if use_early_stopping:
            early_stop.on_epoch_end(epoch=epoch, logs=logs)
            if early_stop.stopped_epoch > 0:
                print(f"Early stopping triggered at epoch {epoch}")
                break
        reduce_lr.on_epoch_end(epoch=epoch, logs=logs)
        lr_scheduler.on_epoch_end(epoch=epoch, logs=logs)

        # ----- FID Evaluation & Early Stopping ----- #
        if epoch % fid_interval == 0:
            fake_val   = generator(noise_val, training=False)
            real_batch = next(iter(real_val))
            real_for_fid = tf.image.resize(real_batch, [299,299]) * 0.5 + 0.5
            fake_for_fid = tf.image.resize(fake_val,    [299,299]) * 0.5 + 0.5

            fid_value = calculate_fid(real_for_fid, fake_for_fid)
            print(f"Epoch {epoch} → Val FID: {fid_value:.2f}")

            if fid_value < best_fid:
                best_fid   = fid_value
                no_improve = 0
                if save_path:
                    os.makedirs(save_path, exist_ok=True)
                    generator.save_weights(os.path.join(save_path, "best_gen.weights.h5"))
                    discriminator.save_weights(os.path.join(save_path, "best_disc.weights.h5"))
            else:
                no_improve += 1

            if no_improve >= patience:
                print(f"No FID improvement for {patience} epochs, stopping.")
                break

        # ----- Display Every 5 Epochs ----- #
        if epoch % 5 == 0:
            display_generated_images(generator, latent_dim)

    # ----- Restore Best Weights After Training ----- #
    if save_path:
        best_gen_path = os.path.join(save_path, "best_gen.weights.h5")
        best_disc_path = os.path.join(save_path, "best_disc.weights.h5")
        if os.path.exists(best_gen_path) and os.path.exists(best_disc_path):
            generator.load_weights(best_gen_path)
            discriminator.load_weights(best_disc_path)
            print("Restored best generator and discriminator weights based on lowest FID.")
        else:
            print("Best weight files not found. Skipping restore.")

    return avg_gen_loss, gen_loss_history, disc_loss_history

With reference to the Code Cell above, we are able to determine that the Training Loop have been successfully defined and we are able to proceed to define the Optimisers with Tunable Parameter in preparation for the Optuna Tuning.

---
### 4.4.8 Defining Optimisers with Tunable Parameters

In this section, we will be defining the Common Optimisers that will be Tuned using the Optuna Tuning. The Optimisers will each have a range of values in a list so as to allow for the Optuna to search for the best Hyperparameter for the GAN. Through these parameters, it will affect how effective and efficiently the GAN will learn from the provided EMNIST Data. The relevent mathematical formula is indicated below:

---
**Optimiser Update Rule**

Purpose: Parameter Update Step in the Adam Optimiser, using Adaptive Learning Rates with Momentum.

$$
\theta \leftarrow \theta - \eta \cdot \frac{m_t}{\sqrt{v_t} + \epsilon}
$$

Where:
- $\theta$ = Model Parameters (Generator or Discriminator)
- $\eta$ = Learning Rate (Tuned)
- $m_t$ = First Moment Estimate (Mean of Gradients)
- $v_t$ = Second Moment Estimate (Variance of Gradients)
- $\epsilon$ = Small Constant for Numerical Stability

---
With the mathematical formula indicated above, we will proceed to define the Optimisers in the Code Cell.

In [ ]:
# ========== Objective Function ========== #
def objective(trial):
    # ----- Hyperparameters ----- #
    latent_dim = trial.suggest_categorical('latent_dim', [100, 128, 160])
    learning_rate = trial.suggest_float('learning_rate', 5e-5, 2e-4, log=True)
    beta_1 = trial.suggest_float('beta_1', 0.4, 0.6)

    # ----- Build Models ----- #
    generator = build_generator(latent_dim=latent_dim)
    discriminator = build_discriminator()

    # ----- Trigger Weight Creation ----- #
    _ = generator(tf.random.uniform([1, latent_dim], minval=-1.0, maxval=1.0))
    _ = discriminator(tf.random.normal([1, 28, 28, 1]))

    # ----- Define Optimizers ----- #
    gen_optimizer = tf.keras.optimizers.Adam(learning_rate=learning_rate, beta_1=beta_1)
    disc_optimizer = tf.keras.optimizers.Adam(learning_rate=learning_rate, beta_1=beta_1)

    # ----- Prebuild Optimizers ----- #
    _ = gen_optimizer.apply_gradients([(tf.zeros_like(v), v) for v in generator.trainable_variables])
    _ = disc_optimizer.apply_gradients([(tf.zeros_like(v), v) for v in discriminator.trainable_variables])

    # ----- Load Data ----- #
    dataset = load_data(X_train, batch_size=64)
    steps = min(1000, len(X_train) // 64)

    # ----- Train ----- #
    _ = train(
        dataset=dataset,
        epochs=100,
        generator=generator,
        discriminator=discriminator,
        gen_optimizer=gen_optimizer,
        disc_optimizer=disc_optimizer,
        latent_dim=latent_dim,
        steps_per_epoch=steps,
        use_early_stopping=False,
        save_path=None
    )

    # ----- Generate Fake Images ----- #
    z = tf.random.uniform([1000, latent_dim], minval=-1.0, maxval=1.0)
    fake_images = generator(z, training=False)
    fake_images = (fake_images + 1.0) / 2.0

    # ----- Get Real Images ----- #
    real_images = X_val[:1000]

    # ----- Compute FID Score ----- #
    fid_score = calculate_fid(real_images, fake_images)

    return fid_score

With reference to the Code Cell above, we are able to determine that the Optimiser have been successfully defined with a range of different Parameter Values so as to provide a robust Optuna Tuning in the subsequent sections.

---
### 4.4.9 Optuna Tuning Study

In this sub-section, we will be conducting the Optuna Tuning Study on the GAN Model so as to obtain the best Hyperparameter for the GAN Model. It is expected that the Tuning takes a prolonged duration due to the nature of how the GAN works and the 2 Neural Networks involved. However, the training data will be stored so as to prevent re-running of repetitive codes. The Optuna Tuning Study will be conducted in the Code Cell below.

In [ ]:
# ========== Create or Load Study with SQLite Backend ========== #
study = optuna.create_study(
    direction="minimize",
    study_name="lsgan_tuning",
    storage="sqlite:////content/drive/MyDrive/Colab Notebooks/DELE CA2 A/Non-Augmented GAN Tunings/lsgan_optuna.db",
    load_if_exists=True
)

# ========== Trial Management ========== #
MAX_TRIALS = 50
completed_trials = len([t for t in study.trials if t.state == optuna.trial.TrialState.COMPLETE])
remaining_trials = MAX_TRIALS - completed_trials

if remaining_trials > 0:
    print(f"Resuming LSGAN study: {completed_trials} completed, running {remaining_trials} more.")
    study.optimize(objective, n_trials=remaining_trials)
else:
    print(f"LSGAN study already completed {MAX_TRIALS} trials. Skipping optimization.")

# ========== Output Best Trial ========== #
best_trial = study.best_trial
best_params_df = pd.DataFrame([best_trial.params])
best_params_df["Final FID"] = best_trial.value
best_params_df.style.background_gradient(cmap="Blues")

With reference to the Code Cell above, we are able to view the Best Hyperparameter obtained during the Optuna Tuning Study. This Hyperparameter will be extracted and be trained for a longer period of time so as to attempt to increase the performance and limit the loss for the GAN.

---
### 4.4.10 GAN Re-Training Operation

As obtained from the previous sub-section, we will be re-training the GAN using the best Hyperparameter obtained during the Optuna Tuning Study. This is to push the GAN Model to its limits and also to save Computational Power as we are not taking up prolonged periods of time tuning GAN Models which may not have any clear signs of good performance. The retraining of the GAN Model will be conducted in the Code Cell below.

In [ ]:
# ========== Extract Best Parameters from Study ========== #
best_params = study.best_trial.params

latent_dim   = best_params['latent_dim']
learning_rate = best_params['learning_rate']
beta_1 = best_params['beta_1']

print("Using Best Trial Parameters for LSGAN:")
print(best_params)

# ========== Rebuild LSGAN Models ========== #
generator     = build_generator(latent_dim=latent_dim)
discriminator = build_discriminator()

# ========== Optimisers ========== #
gen_optimizer  = tf.keras.optimizers.Adam(learning_rate=learning_rate, beta_1=beta_1)
disc_optimizer = tf.keras.optimizers.Adam(learning_rate=learning_rate, beta_1=beta_1)

# ========== Dummy Forward Pass to Trigger Variables ========== #
_ = generator(tf.random.normal([1, latent_dim]))
_ = discriminator(tf.random.normal([1, 28, 28, 1]))

if generator.trainable_variables:
    _ = gen_optimizer.apply_gradients([(tf.zeros_like(v), v) for v in generator.trainable_variables])
if discriminator.trainable_variables:
    _ = disc_optimizer.apply_gradients([(tf.zeros_like(v), v) for v in discriminator.trainable_variables])

# ========== Reload Preprocessed Dataset ========== #
dataset = load_data(X_train)

# ========== Define Paths ========== #
model_name   = "lsgan"
output_dir   = f"/content/drive/MyDrive/Colab Notebooks/DELE CA2 A/final_outputs_{model_name}_noAugment"
weights_path = os.path.join(output_dir, f"{model_name}_generator_final.weights.h5")
gen_ckpt     = os.path.join(output_dir, "best_gen.weights.h5")
disc_ckpt    = os.path.join(output_dir, "best_disc.weights.h5")
loss_csv     = os.path.join(output_dir, "loss_history.csv")

os.makedirs(output_dir, exist_ok=True)

# ========== Skip Training If Artifacts Already Exist ========== #
if os.path.exists(gen_ckpt) and os.path.exists(disc_ckpt) and os.path.exists(loss_csv):
    print("All outputs found. Skipping training...")

    generator.load_weights(gen_ckpt)
    discriminator.load_weights(disc_ckpt)
    print("Generator and Discriminator Weights Loaded.")

    loss_df = pd.read_csv(loss_csv)
    gen_loss_history  = loss_df['gen_loss'].tolist()
    disc_loss_history = loss_df['disc_loss'].tolist()
    final_gen_loss    = gen_loss_history[-1]

else:
    # ========== Train Model ========== #
    final_gen_loss, gen_loss_history, disc_loss_history = train(
        dataset=dataset,
        epochs=100,
        steps_per_epoch=1546,
        generator=generator,
        discriminator=discriminator,
        gen_optimizer=gen_optimizer,
        disc_optimizer=disc_optimizer,
        latent_dim=latent_dim,
        use_early_stopping=False,
        save_path=output_dir
    )

    # ========== Save Final Generator Weights ========== #
    generator.save_weights(weights_path)
    print("Generator Weights Saved.")

    # ========== Save Loss History ========== #
    loss_df = pd.DataFrame({
        'epoch': list(range(1, len(gen_loss_history) + 1)),
        'gen_loss': gen_loss_history,
        'disc_loss': disc_loss_history
    })
    loss_df.to_csv(loss_csv, index=False)
    print(f"Loss history saved to {loss_csv}")

With reference to the Code Cell above, we are able to determine that the best GAN Model have been trained and saved successfully and we are able to proceed with the Model Evaluation in the next sub-sections where we determine how well the GAN Model performed based on various metrics and evaluation methodologies.

---
### 4.4.11 GAN Model Evaluation

In this sub-section, we will be evaluating the GAN's Generator performance on various aspects. We will be utilising different methodologies and metrics to attempt to quantify the performance of the GAN Generator while also conducting Visual Inspection on the result output of the GAN. The Evaluation will be conducted in the following sections.

---
#### 4.4.11.1 Visual Grid of Samples

In this sub-section, we will be visualising the Grid of Samples of the Synthesised Data generated by the GAN Model. A list of observations will be made before the utilisation of Quantitative Metrics to evaluate the GAN Generator's general performance. As the observations made are purely from professional opinion, it will most likely not be used as a Basis of Comparison between GAN Models due to differing opinions unless in extreme cases. The visualisation will be conducted in the Code Cell below.

In [ ]:
def plot_generated_images(generator, latent_dim, n_rows=5, n_cols=5, save_path=None, title="LSGAN Generated Images (Non-Augmented)", seed=42):
    # ----- Generate Latent Vectors ----- #
    if seed is not None:
        tf.random.set_seed(seed)
    noise = tf.random.normal([n_rows * n_cols, latent_dim])
    gen_images = generator(noise, training=False)

    # ----- Rescale from [-1, 1] to [0, 1] ----- #
    gen_images = (gen_images + 1.0) / 2.0
    gen_images = tf.clip_by_value(gen_images, 0.0, 1.0)

    # ----- Shape Check ----- #
    assert gen_images.shape[-1] == 1, "Expected single-channel (grayscale) output"

    # ----- Sample Plot ----- #
    fig, axes = plt.subplots(n_rows, n_cols, figsize=(n_cols, n_rows))
    for idx, ax in enumerate(axes.flat):
        ax.imshow(gen_images[idx, :, :, 0], cmap='gray')
        ax.axis('off')

    plt.suptitle(title, fontsize=16)
    plt.tight_layout()

    if save_path:
        plt.savefig(save_path, dpi=300)
        print(f"Saved Generated Image Grid to: {save_path}")

    plt.show()

# ========== Call Plot Function ========== #
plot_generated_images(generator, latent_dim=100, n_rows=10, n_cols=16)

With reference to the output of the Visual Grid of Samples above, we are able to make the following observations:

**Basic Structure of Letters is Somewhat Captured**

- Some generated characters resemble "A", "J", "G", "P", or "Q" with identifiable loops and strokes.

- Many outputs have consistent stroke directionality and alignment.

**High Blur and Noise**

- A large number of characters appear over-smoothed, with glowing or smudged edges.

- Noise and artefacts are visible in the background of several images.

**Poor Character Consistency**

- Certain letters (e.g. "R", "Z", "M") appear warped or incomplete.

- Many samples show disconnected lines or malformed geometry, making them hard to interpret.

**Lack of Diversity**

- Several characters (especially loop-based ones) appear repeatedly with only minor shape distortion.

- The same style of stroke is reused often, which may point to limited generative diversity.

With the observations indicated above, we will proceed to conduct the Loss Curve Analysis in the next sub-section.

---
#### 4.4.11.2 Loss Curve Over Epochs

In this sub-section, we will be plotting the GAN Loss Curve over Epochs to evaluate the training behaviour of the GAN. This curve provides insight into whether the GAN is experiencing underfitting, overfitting, or achieving a stable training dynamic between the generator and discriminator.

Although GANs do not minimise a single unified loss in the traditional sense, the trajectory of the generator and discriminator losses can indicate whether the adversarial training process is converging appropriately.

If the generator loss remains high while discriminator loss quickly drops to near-zero, this may indicate **underfitting**, where the generator is not learning effectively to produce plausible images.

If the discriminator loss increases while generator loss sharply decreases, this may suggest **overfitting** or **mode collapse**, where the generator exploits narrow weaknesses in the discriminator without general improvement.

A relatively stable oscillation or convergence between both losses typically signifies a **well-balanced** training process, where the generator and discriminator are learning in tandem.

The Potential Observations and Corresponding Action Plans are outlined below:

**Underfitting Loss Curve**

- Increase training epochs to allow generator more time to learn.

- Simplify the discriminator to reduce overpowering the generator.

- Adjust the learning rate or apply label smoothing.

- Consider adding batch normalisation in the generator.

**Overfitting or Mode Collapse Loss Curve**

- Introduce or increase dropout in the discriminator.

- Apply input noise or label flipping to the discriminator.

- Evaluate diversity of generated samples regularly.

**Balanced/Ideal Loss Curve**

- Maintain current architecture and hyperparameters.

- Proceed with full-scale image generation.

- Evaluate both qualitative (visual inspection) and quantitative metrics (e.g. Inception Score or FID).

With these potential training behaviours and action plan outlined, we will proceed to visualise the GAN loss dynamics in the code cell below.

In [ ]:
# ========== Figure Size Configuration ========== #
plt.figure(figsize=(10, 6))

# ========== Plot Training Losses ========== #
plt.plot(gen_loss_history, label="Generator Loss", linewidth=2, color='blue')
plt.plot(disc_loss_history, label="Discriminator Loss", linewidth=2, color='red')

# ========== Labels and Title ========== #
plt.xlabel("Epoch", fontsize=12)
plt.ylabel("Loss", fontsize=12)
plt.title("LSGAN Training Loss Curve (Non-Augmented)", fontsize=16)

# ========== Grid, Legend, and Styling ========== #
plt.grid(True, linestyle='--', alpha=0.6)
plt.legend(fontsize=12)
plt.xticks(fontsize=10)
plt.yticks(fontsize=10)

# ========== Display the Plot ========== #
plt.tight_layout()
plt.show()

With reference to the output of the Loss Curve above, we are able to make the following observations:

- Excellent training dynamics: fast initial learning and stable long-term performance.

- LSGAN appears to be stable and effective, even without augmentation.

- Suggests that non-augmented data was sufficient for LSGAN's smoother loss formulation to generalise reasonably well.

With the observations indicated above, we will proceed to conduct the Quantitative Metircs Evaluation in the next sub-section.

---
#### 4.4.11.3 Quantitative Metrics Evaluation

In this sub-section we will be utilising Quantitative Metrics such as FID and KID to tabulate the results of the performance and quantify the performance of the GAN Generator. We will subsequently be use these metrics to conduct inter-model evaluation after tuning and training all the GANs. The metrics that we will be utilising and their respective formulas are indicated below:

---
**Fréchet Inception Distance**

Purpose: Measures the Distance between the Real-Image and Generated-Image Distributions in Inception Feature Space.

$$
\mathrm{FID} \;=\;\|\mu_r - \mu_f\|^2
\;+\;\mathrm{Tr}\Bigl(\Sigma_r + \Sigma_f - 2\,(\Sigma_r\,\Sigma_f)^{\tfrac12}\Bigr)
$$


Where:
- $\mu_r = \mathbb{E}[f(x)]$, $\Sigma_r = \mathrm{Cov}[f(x)]$ for Real Images $x$.  
- $\mu_f = \mathbb{E}[f(\hat x)]$, $\Sigma_f = \mathrm{Cov}[f(\hat x)]$ for Generated Images $\hat x$.  
- $f(\cdot)$ = Map from Image to its InceptionV3 'pooling=avg' Features.  

---
**Diversity (t-SNE Spread)**

Purpose: Quantifies how 'wide' the 2D t-SNE Embedding of Generated Samples.

$$
\mathrm{Spread}
\;=\;
\bigl(\max_i\,z_i^{(1)} - \min_i\,z_i^{(1)}\bigr)
\;\times\;
\bigl(\max_i\,z_i^{(2)} - \min_i\,z_i^{(2)}\bigr)
$$

Where:
- $z_i = (z_i^{(1)}, z_i^{(2)})$ = 2-dimensional t-SNE Embedding of $i$th Generated Image.

---
**Mode Collapse Risk**

Purpose: Flags when too many Generated Images are Nearly Identical (mode collapse).

$$
\text{ModeCollapseRisk} =
\begin{cases}
\text{Low}, & \dfrac{\bigl|\{\mathrm{unique\_rounded}(x_i)\}\bigr|}{N} > \tau,\\
\text{High}, & \text{otherwise}.
\end{cases}
$$

Where:
- $x_i$ = $N$ Generated Samples.  
- $\mathrm{unique\_rounded}(x_i)$ = Rounds Pixels to Detect Duplicates.  
- $\tau=0.9$ = Uniqueness Threshold (90%).

---
**Perceptual Path Length**

Purpose: Measures Sensitively of Generator's Outputs (in VGG16 Feature Space) move when the Latent Code $z$ is Perturbed.

$$
\mathrm{PPL}
\;=\;
\mathbb{E}_{z,\delta z}\Bigl[\,
\|\phi\bigl(G(z + \epsilon\,\delta z)\bigr)\;-\;\phi\bigl(G(z)\bigr)\|_2^2
\Bigr]
$$

Where:  
- $G(z)$ = GAN Generator Mapping $z\in\mathbb{R}^{\mathrm{latent\_dim}}$ to an Image.  
- $\phi(\cdot)$ = VGG16 'pooling=avg' Feature Extractor.  
- $\delta z\sim\mathcal{N}(0,I)$, $\epsilon$ = Small Constant.

---
**Kernel Inception Distance**

Purpose: An Unbiased Estimator of the Squared Maximum Mean Discrepancy (MMD) between Real and Generated Inception Features.

$$
\mathrm{KID}
\;=\;
\frac{1}{m(m-1)}\sum_{i\neq j} k\bigl(\phi(x_i),\phi(x_j)\bigr)
\;+\;
\frac{1}{n(n-1)}\sum_{i\neq j} k\bigl(\phi(\hat x_i),\phi(\hat x_j)\bigr)
\;-\;
\frac{2}{mn}\sum_{i=1}^m\sum_{j=1}^n k\bigl(\phi(x_i),\phi(\hat x_j)\bigr)
$$

Where:  
- $x_i$ ($i=1\ldots m$) = Real Images.
- $\hat x_j$ ($j=1\ldots n$) = Generated Images.  
- $\phi(\cdot)$ = InceptionV3 Feature Extractor (pooling='avg').  
- $k(u,v) = \bigl(\frac{u^\top v}{d}+1\bigr)^3$ = degree-3 Polynomial Kernel on $d$-dimensional Features.

---

With the mathematical and metrics indicated above, we will proceed to conduct the Quantitave Metrics Evalution in the Code Cell below.

In [ ]:
# ========== Preload InceptionV3 and VGG16 Models ========== #
inception = InceptionV3(include_top=False, pooling='avg', input_shape=(299, 299, 3))
vgg_model = VGG16(include_top=False, weights='imagenet', input_shape=(128, 128, 3))

# ========== Preprocess for Inception ========== #
def preprocess_images_for_inception(images):
    images = (images + 1.0) * 127.5  # Scale from [-1, 1] to [0, 255]
    images = tf.image.resize(images, [299, 299])
    if images.shape[-1] == 1:
        images = tf.image.grayscale_to_rgb(images)
    return preprocess_input(images)

# ========== Compute FID ========== #
def calculate_fid(real_images, fake_images, batch_size=50):
    real_pp = preprocess_images_for_inception(real_images)
    fake_pp = preprocess_images_for_inception(fake_images)

    act1 = inception.predict(real_pp, batch_size=batch_size, verbose=0)
    act2 = inception.predict(fake_pp, batch_size=batch_size, verbose=0)

    mu1, sigma1 = np.mean(act1, axis=0), np.cov(act1, rowvar=False)
    mu2, sigma2 = np.mean(act2, axis=0), np.cov(act2, rowvar=False)

    diff = mu1 - mu2
    covmean = sqrtm(sigma1 @ sigma2)
    if np.iscomplexobj(covmean):
        covmean = covmean.real

    fid = diff @ diff + np.trace(sigma1 + sigma2 - 2 * covmean)
    return round(fid, 4)

# ========== Compute KID ========== #
def polynomial_kernel(X, Y):
    d = X.shape[1]
    return (np.dot(X, Y.T) / d + 1) ** 3

def calculate_kid(real_images, fake_images, batch_size=128):
    real_pp = preprocess_images_for_inception(real_images)
    fake_pp = preprocess_images_for_inception(fake_images)

    real_features = inception.predict(real_pp, batch_size=batch_size, verbose=0)
    fake_features = inception.predict(fake_pp, batch_size=batch_size, verbose=0)

    m = real_features.shape[0]
    n = fake_features.shape[0]

    k_rr = polynomial_kernel(real_features, real_features)
    k_gg = polynomial_kernel(fake_features, fake_features)
    k_rg = polynomial_kernel(real_features, fake_features)

    np.fill_diagonal(k_rr, 0)
    np.fill_diagonal(k_gg, 0)

    mmd = (k_rr.sum() / (m * (m - 1)) +
           k_gg.sum() / (n * (n - 1)) -
           2 * k_rg.mean())
    return round(mmd, 4)

# ========== Compute t-SNE Spread (Diversity) ========== #
def calculate_tsne_spread(images):
    flat = images.reshape(images.shape[0], -1)
    tsne = TSNE(n_components=2, random_state=42)
    proj = tsne.fit_transform(flat)
    x_range = proj[:, 0].max() - proj[:, 0].min()
    y_range = proj[:, 1].max() - proj[:, 1].min()
    return round(x_range * y_range, 2)

# ========== Assess Mode Collapse ========== #
def assess_mode_collapse(images):
    unique = np.unique(np.round(images), axis=0).shape[0]
    return "Low" if unique > 0.9 * images.shape[0] else "High"

# ========== Compute Perceptual Path Length (PPL) ========== #
def calculate_ppl(generator, latent_dim=100, num_samples=50, epsilon=1e-2):
    distances = []
    for _ in range(num_samples):
        z1 = tf.random.normal([1, latent_dim])
        z2 = z1 + epsilon * tf.random.normal([1, latent_dim])
        img1 = generator(z1, training=False)
        img2 = generator(z2, training=False)

        img1 = tf.image.resize(tf.image.grayscale_to_rgb((img1 + 1.0) * 127.5), (128, 128))
        img2 = tf.image.resize(tf.image.grayscale_to_rgb((img2 + 1.0) * 127.5), (128, 128))

        f1 = vgg_model(vgg_preprocess(img1))
        f2 = vgg_model(vgg_preprocess(img2))
        d = tf.reduce_mean(tf.square(f1 - f2)).numpy()
        distances.append(d)
    return round(np.mean(distances), 4)

# ========== Evaluate GAN Performance ========== #
def evaluate_gan_model(generator, latent_dim, X_val, gen_loss_history, disc_loss_history, model_name="LSGAN"):
    # ----- Generate Fake Images ----- #
    noise = tf.random.normal([1000, latent_dim])
    fake_images = generator(noise, training=False)
    fake_images = tf.clip_by_value((fake_images + 1) / 2.0, 0.0, 1.0)
    fake_np = fake_images.numpy()

    # ----- Select and preprocess 1000 Real Images from Validation Set ----- #
    real_images = tf.convert_to_tensor(X_val[:1000], dtype=tf.float32)
    real_images = tf.clip_by_value(real_images, 0.0, 1.0)

    # ----- Compute Metrics ----- #
    fid_value = calculate_fid(real_images, fake_images)
    kid_value = calculate_kid(real_images, fake_images)
    tsne_spread_value = calculate_tsne_spread(fake_np)
    collapse_risk_value = assess_mode_collapse(fake_np)
    ppl_score_value = calculate_ppl(generator, latent_dim)
    mean_gen_loss_value = round(np.mean(gen_loss_history), 4)
    std_gen_loss_value = round(np.std(gen_loss_history), 4)
    mean_disc_loss_value = round(np.mean(disc_loss_history), 4)
    std_disc_loss_value = round(np.std(disc_loss_history), 4)

    # ----- Store as Global Vars (Optional) ----- #
    prefix = model_name.upper()
    globals()[f"{prefix}_FID"] = fid_value
    globals()[f"{prefix}_KID"] = kid_value
    globals()[f"{prefix}_TSNE"] = tsne_spread_value
    globals()[f"{prefix}_MODE_COLLAPSE"] = collapse_risk_value
    globals()[f"{prefix}_PPL"] = ppl_score_value
    globals()[f"{prefix}_GEN_LOSS_MEAN"] = mean_gen_loss_value
    globals()[f"{prefix}_GEN_LOSS_STD"] = std_gen_loss_value
    globals()[f"{prefix}_DISC_LOSS_MEAN"] = mean_disc_loss_value
    globals()[f"{prefix}_DISC_LOSS_STD"] = std_disc_loss_value

    # ----- Final Summary Row ----- #
    row = {
        "Model": model_name,
        "FID Score": fid_value,
        "KID Score": kid_value,
        "Mean Generator Loss": mean_gen_loss_value,
        "Std Generator Loss": std_gen_loss_value,
        "Mean Discriminator Loss": mean_disc_loss_value,
        "Std Discriminator Loss": std_disc_loss_value,
        "Mode Collapse Risk": collapse_risk_value,
        "Visual Quality": "Good",
        "Diversity (t-SNE Spread)": tsne_spread_value,
        "PPL": ppl_score_value
    }

    return pd.DataFrame([row])[[
        "Model", "FID Score", "KID Score",
        "Mean Generator Loss", "Std Generator Loss",
        "Mean Discriminator Loss", "Std Discriminator Loss",
        "Mode Collapse Risk", "Visual Quality",
        "Diversity (t-SNE Spread)", "PPL"
    ]]

# ========== Create DataFrame ========== #
df = evaluate_gan_model(generator, latent_dim, X_val, gen_loss_history, disc_loss_history, model_name="LSGAN (Non-Augmented)")

# ========== Display DataFrame ========== #
df.style.background_gradient(cmap="Blues")

With reference to the output of the Quantitative Metrics above, we are able to determine the performance of the GAN. This will be used as a basis of comparison to the other GAN Models.

---
#### 4.4.11.4 t-SNE of Generated Samples

In this sub-section, we visualise the t-SNE projection of generated samples to evaluate the latent diversity learned by the unconditional GAN. Although our GAN is not class-conditioned, t-SNE remains a valuable tool to assess whether the generator is producing varied outputs that reflect meaningful use of the latent space. While GANs are trained in high-dimensional spaces, t-SNE enables us to observe 2D structural patterns such as sample clustering, separation, and density, which provide indirect insights into the diversity and generalisation capabilities of the generator. The visualisation will be conducted in the Code Cell below.

In [ ]:
# ========== Generate New Images from Random Noise ========== #
latent_dim = 100
noise = tf.random.normal([500, latent_dim])
generated_images = generator(noise, training=False)
generated_images = (generated_images + 1) / 2.0

def plot_tsne_embeddings(generated_images, save_path=None, title="t-SNE of LSGAN Generated Samples (Non-Augmented)"):
    # ----- Convert to [0, 1] Range ----- #
    if tf.reduce_max(generated_images).numpy() > 1.0:
        raise ValueError("Images should be in [-1, 1] range before scaling.")

    images_rescaled = (generated_images + 1.0) / 2.0
    flat_images = images_rescaled.numpy().reshape(images_rescaled.shape[0], -1)

    # ----- Run t-SNE ----- #
    tsne = TSNE(n_components=2, perplexity=30, learning_rate='auto', init='pca', random_state=42)
    tsne_proj = tsne.fit_transform(flat_images)

    # ----- Plot t-SNE ----- #
    plt.figure(figsize=(12, 6))
    sns.scatterplot(
        x=tsne_proj[:, 0],
        y=tsne_proj[:, 1],
        s=20,
        alpha=0.9,
        edgecolor='none'
    )
    plt.title(title, fontsize=14, weight='bold')
    plt.xlabel("t-SNE Dimension 1")
    plt.ylabel("t-SNE Dimension 2")
    plt.grid(True, linestyle='--', alpha=0.3)
    plt.tight_layout()

    if save_path:
        plt.savefig(save_path, dpi=300)
        print(f"t-SNE plot saved to: {save_path}")

    plt.show()

plot_tsne_embeddings(generated_images=generated_images)

With reference to the t-SNE plot of LSGAN generated samples (Non-Augmented) above, we are able to make the following observations:

**Spread Across Latent Space**

- The points are well distributed along both dimensions, forming a rectangular region.

- This suggests that LSGAN is sampling from a diverse range of latent representations.

**Lack of Clear Clustering**

- There is no strong visual indication of groupings or semantic clusters.

- This is typical for unconditional generation, where class labels are not embedded.

**Balanced Uniformity**

- The distribution shows even coverage without over-concentration or empty voids.

- Indicates that LSGAN achieves stable latent traversal across the dataset.

**Absence of Outliers**

- All data points lie within a contained boundary, with no major anomalies or outlying samples.

- This reflects consistent generation quality, even without augmentation.

With the observations indicated above, we will proceed to train the next GAN Model in the next sub-section.

---
## 4.5 Wasserstein GAN Model Training

In this sub-section, we will be training a Wasserstein Squares GAN (WGAN) for the EMNIST Dataset. The WGAN mainly replace the Binary Crossentropy Loss used in classical GANs with a Wasserstein Distance. Through this replacement, the WGAN is more likely able to have a Stabalise GAN Training, Improve Image Quality for Small Dataset, and provide Loss Values that are Interpretable. The relevant mathematical formulas are indicated below:

---
**Generator Loss:**

Purpose: The Generator Maximizes the Critic's Score on Fake Images

$$
L_G = -\mathbb{E}_{\tilde{x} \sim \mathbb{P}_g} \left[D(\tilde{x})\right]
$$

Where:
- $L_G$ = Generator Loss  
- $\tilde{x} \sim \mathbb{P}_g$ = Fake Data Sampled from Generator Distribution  
- $D(\tilde{x})$ = Critic's Prediction Score for Fake Input $\tilde{x}$  
- $\mathbb{E}$ = Expectation Operator

---
**Discriminator (Critic) Loss:**

Purpose: Trains Critic to give Higher Scores to Real Data and Lower Scores to Fake Data.

$$
L_D = \mathbb{E}_{\tilde{x} \sim \mathbb{P}_g} \left[D(\tilde{x})\right] - \mathbb{E}_{x \sim \mathbb{P}_r} \left[D(x)\right] + \lambda \cdot \mathbb{E}_{\hat{x} \sim \mathbb{P}_{\hat{x}}} \left[\left(\| \nabla_{\hat{x}} D(\hat{x}) \|_2 - 1\right)^2\right]
$$

Where:
- $D(x)$ = Critic Output for Real Input  
- $\mathbb{P}_r$ = Real Data Distribution  
- $\mathbb{P}_g$ = Generator (fake) Distribution  
- $\tilde{x}$ = Fake Sample from Generator
- $\hat{x}$ = Interpolated Sample Between Real and Fake  
- $\lambda$ = Gradient Penalty Coefficient (typically $\lambda = 10$)

---
With the relevant mathematical formulas indicated above, we will proceed to train the WGAN in this sub-section for the purpose of it's potential for smoother training process and also to serve as a comparison for our other GAN Models.

---
### 4.5.1 Defining Data Pre-Processing Function

In this sub-section, we will be defining the Data Pre-Processing Function which will allow for consistent Data Pipeline to be built and modifed accordingly should the need arise such as when we need to add Spectral Normalisation. This will ensure that the Data's integrity is not affected and we are able to make fair and un-biased assumptions. The Data Pre-Processing Function is defined in the Code Cell below:

In [ ]:
# ========== Data Pre-Processing Function ========== #
def load_data(X_train, batch_size=32):
    buffer_size = X_train.shape[0]

    dataset = tf.data.Dataset.from_tensor_slices(X_train)
    dataset = dataset.shuffle(buffer_size)
    dataset = dataset.batch(batch_size)
    dataset = dataset.prefetch(tf.data.AUTOTUNE)

    return dataset

With reference to the Code Cell above, we are able to determine that the Data Pre-Processing Function have been defined successfully and we are able to move on to Defining Callback Functions in the next sub-section.

---
### 4.5.2 Defining Callback Functions

In this sub-section, we will be pre-defining the various Callbacks. This is to ensure consistency throughout the model and also to increase the GAN Accuracy and optimise the Computation Cost for training each GAN Model. These Callbacks will be main used during the GAN Training in the next sub-section. The relevant mathematical formula and logic are indicated below:

---
**Learning Rate Scheduler:**

Purpose: Reduces Learning Rate During Training to Fine-tune Model Convergence.

$$\text{If } epoch \mod 10 = 0 \Rightarrow \eta_{new} = \frac{1}{2} \cdot \eta$$

Where:

- $\eta$ = current learning rate

- $\eta_{\text{new}}$ = updated learning rate

- $epoch \bmod 10$ checks if the epoch is a multiple of 10

---
**EarlyStopping:**

Purpose: Prevents Wastage of Computational Power if Model Does Not Improve.

$$
\text{If } val\_loss \text{ does not improve for 25 epochs, stop training and restore best weights}
$$

---

We will proceed to Define the Callbacks in the Code Cell below in preparation for the GAN Model Training.

In [ ]:
# ========== Define Learning Rate Scheduler ========== #
lr_scheduler = LearningRateScheduler(
    lambda epoch, lr: lr * 0.95 if epoch % 10 == 0 else lr,
    verbose=1
)

# ========== Define Early Stopping for WGAN ========== #
early_stop = EarlyStopping(
    monitor='critic_loss',
    patience=25,
    verbose=1,
    restore_best_weights=False
)

# ========== Define Display Generated Images for WGAN ========== #
def display_generated_images(generator, latent_dim, n=7):
    noise = tf.random.normal([n * n, latent_dim])
    generated_images = generator(noise, training=False)
    generated_images = (generated_images + 1.0) / 2.0

    fig, axes = plt.subplots(n, n, figsize=(n, n))
    for i in range(n):
        for j in range(n):
            img = generated_images[i * n + j, :, :, 0]
            axes[i, j].imshow(img, cmap='gray')
            axes[i, j].axis('off')

    plt.tight_layout()
    plt.show()

# ========== Define FID Calculator ========== #
def calculate_fid(real_images, fake_images, batch_size=10):

    # ----- Scale to [0,255] and Resize to 299×299 ----- #
    real = (real_images + 1.0) * 127.5
    fake = (fake_images + 1.0) * 127.5
    real = tf.image.resize(real, (299,299))
    fake = tf.image.resize(fake, (299,299))

    # ----- If Grayscale, Convert to RGB ----- #
    if real.shape[-1] == 1:
        real = tf.image.grayscale_to_rgb(real)
        fake = tf.image.grayscale_to_rgb(fake)

    # ----- Preprocess for Inception (–1 to +1) ----- #
    real_pp = preprocess_input(real)
    fake_pp = preprocess_input(fake)

    # ----- Extract Features in Batches ----- #
    def _get_acts(x):
        acts = []
        n = x.shape[0]
        for i in range(0, n, batch_size):
            chunk = x[i:i+batch_size]
            acts.append(_inception_model(chunk, training=False).numpy())
        return np.vstack(acts)

    act_real = _get_acts(real_pp)
    act_fake = _get_acts(fake_pp)

    # ----- Compute Statistics ----- #
    mu1, sigma1 = act_real.mean(axis=0), np.cov(act_real, rowvar=False)
    mu2, sigma2 = act_fake.mean(axis=0), np.cov(act_fake, rowvar=False)
    diff    = mu1 - mu2
    covmean = sqrtm(sigma1.dot(sigma2))
    if np.iscomplexobj(covmean):
        covmean = covmean.real
    fid = diff.dot(diff) + np.trace(sigma1 + sigma2 - 2*covmean)
    return float(np.round(fid,4))

# ========== Confirmation Message ========== #
print("WGAN Callbacks and Visualisation Setup Complete")

With reference to the Code Cell above, we are able to verify that that the Callbacks have been successfully defined and the various parameters are set so as to ensure a High Accuracy and Computing Efficient GAN Model Training.

---
### 4.5.3 Defining WGAN Generator Architecture

In this section, we will be defining the WGAN's Generator Architecture. The Generator will be trying to make realistic generated images in the attempt to bypass the Discriminator. Subsequently, the generated images from the Generator will be evaluated based on a few evaluation metrics such as Inception Score. The relevent mathematical formulas for the Loss Function of the Vanilla GAN Generator is indicated below:

---
**Generator Loss:**

Purpose: The Generator Maximizes the Critic's Score on Fake Images

$$
L_G = -\mathbb{E}_{\tilde{x} \sim \mathbb{P}_g} \left[D(\tilde{x})\right]
$$

Where:
- $L_G$ = Generator Loss  
- $\tilde{x} \sim \mathbb{P}_g$ = Fake Data Sampled from Generator Distribution  
- $D(\tilde{x})$ = Critic's Prediction Score for Fake Input $\tilde{x}$  
- $\mathbb{E}$ = Expectation Operator

---
With the relevant Generator Loss Formula indicated above, we will proceed to define the GAN Generator's Architecture in the Code Cell below and the Lost Function.

In [ ]:
# ========== Generator Function ========== #
def build_generator(latent_dim=100, use_dropout=True):
    model = tf.keras.Sequential(name="generator")

    # ----- Project and Reshape ----- #
    model.add(tf.keras.layers.Dense(7 * 7 * 256, use_bias=False, input_shape=(latent_dim,)))
    model.add(tf.keras.layers.BatchNormalization())
    model.add(tf.keras.layers.ReLU())
    model.add(tf.keras.layers.Reshape((7, 7, 256)))

    # ----- Upsample 1 ----- #
    model.add(tf.keras.layers.Conv2DTranspose(128, kernel_size=5, strides=1, padding='same', use_bias=False))
    model.add(tf.keras.layers.BatchNormalization())
    model.add(tf.keras.layers.ReLU())
    if use_dropout:
        model.add(tf.keras.layers.Dropout(0.3))

    # ----- Upsample 2 ----- #
    model.add(tf.keras.layers.Conv2DTranspose(64, kernel_size=5, strides=2, padding='same', use_bias=False))
    model.add(tf.keras.layers.BatchNormalization())
    model.add(tf.keras.layers.ReLU())
    if use_dropout:
        model.add(tf.keras.layers.Dropout(0.3))

    # ----- Final Output to 28x28x1 ----- #
    model.add(tf.keras.layers.Conv2DTranspose(1, kernel_size=5, strides=2, padding='same', activation='tanh'))

    return model

# ========== WGAN Generator Loss ========== #
def generator_loss(fake_output):
    return -tf.reduce_mean(fake_output)

With reference to the Code Cell above, we are able to verify that the GAN's Generator Architecture have been successfully defined and we are able to proceed to define the GAN's Discriminator Architecture.

---
### 4.5.4 Defining Spectral Normalisation

In this section, we will be defining the Spectral Normalisation layer for the Discriminator of the GAN. The Spectral Normalisation controls the Lipschitz constant of the Discriminator by constraining the Spectral Norm of each weight matrix. This will prevent the Discriminator from being too powerful and dominating the Generator which can cause various issues such as Mode Collaspe, Training Instability and Exploding Gradients. The relevant mathematical formula is indicated below:

---
**Spectral Normalisation (SN)**

Purpose: Controls the Lipschitz Constant of the Discriminator.

$$
\bar{W} = \frac{W}{\sigma(W)}
$$

Where:
- $W$ = Original Weight Matrix
- $\bar{W}$ = Spectrally Normalised Weight Matrix used in the Forward Pass
- $\sigma(W)$ = Spectral Norm of $W$
---
**Spectral Norm Approximation using Power Iteration**

Purpose: Controls the Lipschitz Constant of the Discriminator.

$$
\sigma(W) \approx \mathbf{u}^\top W \mathbf{v}
$$

Where:
- $\mathbf{v} \leftarrow \frac{W^\top \mathbf{u}}{\|W^\top \mathbf{u}\|_2}$
- $\mathbf{u} \leftarrow \frac{W \mathbf{v}}{\|W \mathbf{v}\|_2}$
- $\mathbf{u}, \mathbf{v}$: Approximated Unit Vectors
---
With the relevant mathematical formulas indicated, we will proceed to define the Spectral Normalisation in the Code Cell below.

In [ ]:
# ========== Spectral Normalisation Convolution Layer ========== #
class SpectralConv2D(tf.keras.layers.Layer):
    def __init__(self, filters, kernel_size, strides=1, padding='same', **kwargs):
        super().__init__()
        self.conv = tf.keras.layers.Conv2D(filters, kernel_size, strides=strides, padding=padding, use_bias=False, **kwargs)
        self.u = None

    def build(self, input_shape):
        self.conv.build(input_shape)
        self.w = self.conv.kernel
        self.u = self.add_weight(shape=(1, self.w.shape[-1]), initializer='random_normal', trainable=False, name='sn_u')

    def compute_spectral_norm(self, w):
        w_reshaped = tf.reshape(w, [-1, w.shape[-1]])
        u = self.u

        for _ in range(1):
            v = tf.linalg.l2_normalize(tf.matmul(u, tf.transpose(w_reshaped)))
            u = tf.linalg.l2_normalize(tf.matmul(v, w_reshaped))

        sigma = tf.matmul(tf.matmul(v, w_reshaped), tf.transpose(u))
        self.u.assign(u)
        return w / sigma

    def call(self, inputs):
        self.conv.kernel.assign(self.compute_spectral_norm(self.w))
        return self.conv(inputs)

# ========== Spectral Normalisation Dense Layer ========== #
class SpectralDense(tf.keras.layers.Layer):
    def __init__(self, units, activation=None):
        super().__init__()
        self.dense = tf.keras.layers.Dense(units, activation=activation, use_bias=False)
        self.u = None

    def build(self, input_shape):
        self.dense.build(input_shape)
        self.w = self.dense.kernel
        self.u = self.add_weight(shape=(1, self.w.shape[-1]), initializer='random_normal', trainable=False, name='sn_u_dense')

    def compute_spectral_norm(self, w):
        w_reshaped = tf.reshape(w, [-1, w.shape[-1]])
        u = self.u

        for _ in range(1):
            v = tf.linalg.l2_normalize(tf.matmul(u, tf.transpose(w_reshaped)))
            u = tf.linalg.l2_normalize(tf.matmul(v, w_reshaped))

        sigma = tf.matmul(tf.matmul(v, w_reshaped), tf.transpose(u))
        self.u.assign(u)
        return w / sigma

    def call(self, inputs):
        self.dense.kernel.assign(self.compute_spectral_norm(self.w))
        return self.dense(inputs)

With reference to the Code Cell above, we are able to verify that the Spectral Normalisation have been successfully defined and we are able to proceed to include this layer in the Discriminator Architecture in the next section.

---
### 4.5.5 Defining WGAN Discriminator Architecture

In this section, we will be defining the Architecture of the Discriminator of the GAN. The Discriminator will be the Neural Network that will attempt to tell the Generated Images apart from the Actual Images. This will allow the GAN to improve overtime and also improve itself overtime. The mathematical function for the Discriminator Loss Function is indicated below:

---
**Discriminator (Critic) Loss:**

Purpose: Trains Critic to give Higher Scores to Real Data and Lower Scores to Fake Data.

$$
L_D = \mathbb{E}_{\tilde{x} \sim \mathbb{P}_g} \left[D(\tilde{x})\right] - \mathbb{E}_{x \sim \mathbb{P}_r} \left[D(x)\right] + \lambda \cdot \mathbb{E}_{\hat{x} \sim \mathbb{P}_{\hat{x}}} \left[\left(\| \nabla_{\hat{x}} D(\hat{x}) \|_2 - 1\right)^2\right]
$$

Where:
- $D(x)$ = Critic Output for Real Input  
- $\mathbb{P}_r$ = Real Data Distribution  
- $\mathbb{P}_g$ = Generator (fake) Distribution  
- $\tilde{x}$ = Fake Sample from Generator
- $\hat{x}$ = Interpolated Sample Between Real and Fake  
- $\lambda$ = Gradient Penalty Coefficient (typically $\lambda = 10$)
---

With the mathematical formula for the Loss Function indicated above, we will proceed to define the GAN's Discriminator Architecture and the Loss Function in the Code Cell below.

In [ ]:
# ========== Critic Function ========== #
def build_critic(input_shape=(28, 28, 1)):
    model = tf.keras.Sequential(name="critic")

    model.add(SpectralConv2D(64, kernel_size=5, strides=2, padding='same', input_shape=input_shape))
    model.add(tf.keras.layers.LeakyReLU(0.2))

    model.add(SpectralConv2D(128, kernel_size=5, strides=2, padding='same'))
    model.add(tf.keras.layers.LeakyReLU(0.2))

    model.add(tf.keras.layers.Flatten())
    model.add(SpectralDense(1))

    return model

# ========== Critic Loss ========== #
def critic_loss(real_output, fake_output):
    return tf.reduce_mean(fake_output) - tf.reduce_mean(real_output)

With reference to the Code Cell above, we are able to verify that the GAN's Discriminator Architecture have been successfully defined and we are able to proceed to define the GAN's Training Steps.

---
### 4.5.6 Defining Training Step

In this section, we will be Defining the Training Step function so as to train both the Generator and the Discriminator at the same time. The Training Step will compute the gradients for both the Generator and the Discriminator at the same time. The relevant mathematical formulas for the Training Step is indicated below:

---
**Generator Loss:**

Purpose: The Generator Maximizes the Critic's Score on Fake Images

$$
L_G = -\mathbb{E}_{\tilde{x} \sim \mathbb{P}_g} \left[D(\tilde{x})\right]
$$

Where:
- $L_G$ = Generator Loss  
- $\tilde{x} \sim \mathbb{P}_g$ = Fake Data Sampled from Generator Distribution  
- $D(\tilde{x})$ = Critic's Prediction Score for Fake Input $\tilde{x}$  
- $\mathbb{E}$ = Expectation Operator

---
**Discriminator (Critic) Loss:**

Purpose: Trains Critic to give Higher Scores to Real Data and Lower Scores to Fake Data.

$$
L_D = \mathbb{E}_{\tilde{x} \sim \mathbb{P}_g} \left[D(\tilde{x})\right] - \mathbb{E}_{x \sim \mathbb{P}_r} \left[D(x)\right] + \lambda \cdot \mathbb{E}_{\hat{x} \sim \mathbb{P}_{\hat{x}}} \left[\left(\| \nabla_{\hat{x}} D(\hat{x}) \|_2 - 1\right)^2\right]
$$

Where:
- $D(x)$ = Critic Output for Real Input  
- $\mathbb{P}_r$ = Real Data Distribution  
- $\mathbb{P}_g$ = Generator (fake) Distribution  
- $\tilde{x}$ = Fake Sample from Generator
- $\hat{x}$ = Interpolated Sample Between Real and Fake  
- $\lambda$ = Gradient Penalty Coefficient (typically $\lambda = 10$)

---

With the relevant mathematical formulas indicated above, we will proceed to define the Training Step in the Code Cell below.

In [ ]:
# ========== Training Step ========== #
@tf.function
def train_step(real_images, generator, critic, gen_optimizer, critic_optimizer, latent_dim, n_critic=5):
    # ----- Batch Size ----- #
    bs = tf.shape(real_images)[0]

    # ----- Uniform Latent Noise [-1, 1] ----- #
    noise = tf.random.uniform([bs, latent_dim], minval=-1.0, maxval=1.0)

    # ----- Update Critic n_critic Times ----- #
    for _ in range(n_critic):
        with tf.GradientTape() as tape_c:
            fake_images = generator(noise, training=True)
            real_score = critic(real_images, training=True)
            fake_score = critic(fake_images, training=True)

            # ----- Critic Loss ----- #
            c_loss = tf.reduce_mean(fake_score) - tf.reduce_mean(real_score)

        # ----- Apply Critic Gradients ----- #
        c_grads = tape_c.gradient(c_loss, critic.trainable_variables)
        critic_optimizer.apply_gradients(zip(c_grads, critic.trainable_variables))

    # ----- Update Generator ----- #
    with tf.GradientTape() as tape_g:
        fake_images = generator(noise, training=True)
        fake_score = critic(fake_images, training=True)

        # ----- Generator Loss ----- #
        g_loss = -tf.reduce_mean(fake_score)

    # ----- Apply Generator Gradients ----- #
    g_grads = tape_g.gradient(g_loss, generator.trainable_variables)
    gen_optimizer.apply_gradients(zip(g_grads, generator.trainable_variables))

    return g_loss, c_loss, real_score, fake_score

With reference to the Code Cell above, we are able to determine that the Training Step have been successfully defined and we are able to proceed to prepare the Training Loop in the next section.

---
### 4.5.7 Defining Training Loop

In this section, we will be defining the Training Loop for the GAN Training. The Training Loop will go through each of the Epoch and train both the Generator and Discriminator simultaneously. The Training Loop will also track the Loss for both the Generator and the Discriminator for each of the Epoch trained. The relevant mathematical formulas are indicated below:

---
**Epoch Loss Averaging:**

Purpose: These are the Average Generator and Discriminator Losses over all Batches in Epoch $t$.

$$
\bar{L}_G^{(t)} = \frac{1}{N} \sum_{i=1}^{N} L_G^{(i)} \\
\bar{L}_D^{(t)} = \frac{1}{N} \sum_{i=1}^{N} L_D^{(i)}
$$

Where:
- $\bar{L}_G^{(t)}$ = Average Generator Loss at Epoch $t$  
- $\bar{L}_D^{(t)}$ = Average Discriminator Loss at Epoch $t$  
- $N$ = Number of Batches in the Dataset  
- $L_G^{(i)}$ = Generator Loss on Batch $i$  
- $L_D^{(i)}$ = Discriminator Loss on Batch $i$

---
**Epoch Iteration:**

Purpose: Describes the Training Loop Logic: for Each Epoch $t$, perform `train_step` for Every Mini-batch.

$$
\text{for } t = 1 \text{ to } T:
\quad \text{for each batch } (x^{(i)}):
\quad \text{train_step}(x^{(i)})
$$

Where:
- $T$: Total Number of Epochs  
- $x^{(i)}$: Real Batch $i$ from the Dataset  
- `train_step`: Function that Updates $G$ and $D$ using that Batch

---
**Generated Image Output:**

Purpose: Sample Random Noise $z$ and Generate Synthetic Image $\hat{x}$ from the Generator. This is used for Visual Monitoring of Model Quality.

$$
z \sim p_z(z) \\
\hat{x} = G(z)
$$

Where:
- $z$: Random Latent Vector  
- $G(z)$: Generated Image from Generator

---
With the mathematical formulas indicated above, we will proceed to define the Training Loop in the Code Cell below in preparation of the GAN Training.

In [ ]:
# ========== Trainning Loop Function ========== #
def train(dataset, epochs, generator, critic, gen_optimizer, critic_optimizer, latent_dim, save_path=None, use_early_stopping=True, steps_per_epoch=None, n_critic=5):
    # ----- Prepare FID Reference Sets using X_val ----- #
    real_val  = tf.data.Dataset.from_tensor_slices(X_val).batch(50).take(1)
    noise_val = tf.random.uniform([50, latent_dim], minval=-1.0, maxval=1.0)

    # ----- FID Early Stopping Settings ----- #
    best_fid = float('inf')
    no_improve = 0
    patience = 12
    fid_interval = 1

    # ----- Track Loss History ----- #
    gen_loss_history    = []
    critic_loss_history = []

    for epoch in range(1, epochs + 1):
        print(f"Epoch {epoch}/{epochs}")
        gen_losses, critic_losses = [], []

        for i, real_images in enumerate(dataset):
            if steps_per_epoch and i >= steps_per_epoch:
                break

            # ----- One Training Step (includes n_critic updates) ----- #
            g_l, c_l, _, _ = train_step(
                real_images,
                generator,
                critic,
                gen_optimizer,
                critic_optimizer,
                latent_dim,
                n_critic
            )
            gen_losses.append(g_l)
            critic_losses.append(c_l)

        # ----- Epoch Statistics ----- #
        avg_gen_loss    = tf.reduce_mean(gen_losses)
        avg_critic_loss = tf.reduce_mean(critic_losses)
        print(f"Generator Loss: {avg_gen_loss:.4f} | Critic Loss: {avg_critic_loss:.4f}")

        # ----- Log Losses ----- #
        gen_loss_history.append(float(avg_gen_loss))
        critic_loss_history.append(float(avg_critic_loss))

        # ----- FID Evaluation & Early Stopping ----- #
        if epoch % fid_interval == 0:
            fake_val = generator(noise_val, training=False)
            real_batch = next(iter(real_val))
            real_for_fid = tf.image.resize(real_batch, [299,299]) * 0.5 + 0.5
            fake_for_fid = tf.image.resize(fake_val,    [299,299]) * 0.5 + 0.5

            fid_value = calculate_fid(real_for_fid, fake_for_fid)
            print(f"Epoch {epoch} → Val FID: {fid_value:.2f}")

            if fid_value < best_fid:
                best_fid   = fid_value
                no_improve = 0
                if save_path:
                    os.makedirs(save_path, exist_ok=True)
                    generator.save_weights(os.path.join(save_path, "best_gen.weights.h5"))
                    critic.save_weights(os.path.join(save_path, "best_crt.weights.h5"))
            else:
                no_improve += 1

            if no_improve >= patience:
                print(f"No FID improvement for {patience} epochs, stopping.")
                break

        # ----- Display Generated Images Every 5 Epochs ----- #
        if epoch % 5 == 0:
            display_generated_images(generator, latent_dim)

    # ----- Restore Best Weights After Training ----- #
    if save_path:
        best_gen_path = os.path.join(save_path, "best_gen.weights.h5")
        best_crt_path = os.path.join(save_path, "best_crt.weights.h5")
        if os.path.exists(best_gen_path) and os.path.exists(best_crt_path):
            generator.load_weights(best_gen_path)
            critic.load_weights(best_crt_path)
            print("Restored best generator and critic weights based on lowest FID.")
        else:
            print("Best weight files not found. Skipping restore.")

    return float(avg_gen_loss), gen_loss_history, critic_loss_history

With reference to the Code Cell above, we are able to determine that the Training Loop have been successfully defined and we are able to proceed to define the Optimisers with Tunable Parameter in preparation for the Optuna Tuning.

---
### 4.5.8 Defining Optimisers with Tunable Parameters

In this section, we will be defining the Common Optimisers that will be Tuned using the Optuna Tuning. The Optimisers will each have a range of values in a list so as to allow for the Optuna to search for the best Hyperparameter for the GAN. Through these parameters, it will affect how effective and efficiently the GAN will learn from the provided EMNIST Data. The relevent mathematical formula is indicated below:

---
**Optimiser Update Rule**

Purpose: Parameter Update Step in the Adam Optimiser, using Adaptive Learning Rates with Momentum.

$$
\theta \leftarrow \theta - \eta \cdot \frac{m_t}{\sqrt{v_t} + \epsilon}
$$

Where:
- $\theta$ = Model Parameters (Generator or Discriminator)
- $\eta$ = Learning Rate (Tuned)
- $m_t$ = First Moment Estimate (Mean of Gradients)
- $v_t$ = Second Moment Estimate (Variance of Gradients)
- $\epsilon$ = Small Constant for Numerical Stability

---
With the mathematical formula indicated above, we will proceed to define the Optimisers in the Code Cell.

In [ ]:
# ========== Objective Function ========== #
def objective(trial):
    # ----- Hyperparameters ----- #
    latent_dim = trial.suggest_categorical('latent_dim', [100, 128, 160])
    learning_rate = trial.suggest_float('learning_rate', 5e-5, 2e-4, log=True)
    beta_1 = trial.suggest_float('beta_1', 0.4, 0.6)

    # ----- Build Models ----- #
    generator = build_generator(latent_dim=latent_dim)
    critic = build_critic(input_shape=(28, 28, 1))

    # ----- Trigger Weight Creation ----- #
    _ = generator(tf.random.uniform([1, latent_dim], -1.0, 1.0))
    _ = critic(tf.random.normal([1, 28, 28, 1]))

    # ----- Define Optimizers ----- #
    gen_optimizer = tf.keras.optimizers.Adam(learning_rate=learning_rate, beta_1=beta_1)
    critic_optimizer = tf.keras.optimizers.Adam(learning_rate=learning_rate, beta_1=beta_1)

    # ----- Prebuild Optimizers ----- #
    _ = gen_optimizer.apply_gradients([(tf.zeros_like(v), v) for v in generator.trainable_variables])
    _ = critic_optimizer.apply_gradients([(tf.zeros_like(v), v) for v in critic.trainable_variables])

    # ----- Load Data ----- #
    dataset = load_data(X_train, batch_size=64)
    steps = min(1000, len(X_train) // 64)

    # ----- Train ----- #
    _ = train(
        dataset,
        epochs=100,
        generator=generator,
        critic=critic,
        gen_optimizer=gen_optimizer,
        critic_optimizer=critic_optimizer,
        latent_dim=latent_dim,
        steps_per_epoch=steps,
        save_path=None,
        n_critic=5
    )

    # ----- Generate Fake Images ----- #
    z = tf.random.uniform([1000, latent_dim], -1.0, 1.0)
    fake_images = generator(z, training=False)
    fake_images = (fake_images + 1.0) / 2.0

    # ----- Real Images ----- #
    real_images = X_val[:1000]

    # ----- Compute FID ----- #
    fid_score = calculate_fid(real_images, fake_images)
    return fid_score

With reference to the Code Cell above, we are able to determine that the Optimiser have been successfully defined with a range of different Parameter Values so as to provide a robust Optuna Tuning in the subsequent sections.

---
### 4.5.9 Optuna Tuning Study

In this sub-section, we will be conducting the Optuna Tuning Study on the GAN Model so as to obtain the best Hyperparameter for the GAN Model. It is expected that the Tuning takes a prolonged duration due to the nature of how the GAN works and the 2 Neural Networks involved. However, the training data will be stored so as to prevent re-running of repetitive codes. The Optuna Tuning Study will be conducted in the Code Cell below.

In [ ]:
# ========== Create or Load Study with SQLite Backend ========== #
study = optuna.create_study(
    direction="minimize",
    study_name="wgan_tuning",  # Updated for WGAN
    storage="sqlite:////content/drive/MyDrive/Colab Notebooks/DELE CA2 A/Non-Augmented GAN Tunings/wgan_optuna.db",
    load_if_exists=True
)

# ========== Trial Management ========== #
MAX_TRIALS = 50
completed_trials = len([t for t in study.trials if t.state == optuna.trial.TrialState.COMPLETE])
remaining_trials = MAX_TRIALS - completed_trials

if remaining_trials > 0:
    print(f"Resuming WGAN study: {completed_trials} completed, running {remaining_trials} more.")
    study.optimize(objective, n_trials=remaining_trials)
else:
    print(f"WGAN study already completed {MAX_TRIALS} trials. Skipping optimization.")

# ========== Output Best Trial ========== #
best_trial = study.best_trial
best_params_df = pd.DataFrame([best_trial.params])
best_params_df["Final FID"] = best_trial.value
best_params_df.style.background_gradient(cmap="Blues")

With reference to the Code Cell above, we are able to view the Best Hyperparameter obtained during the Optuna Tuning Study. This Hyperparameter will be extracted and be trained for a longer period of time so as to attempt to increase the performance and limit the loss for the GAN.

---
### 4.5.10 GAN Re-Training Operation

As obtained from the previous sub-section, we will be re-training the GAN using the best Hyperparameter obtained during the Optuna Tuning Study. This is to push the GAN Model to its limits and also to save Computational Power as we are not taking up prolonged periods of time tuning GAN Models which may not have any clear signs of good performance. The retraining of the GAN Model will be conducted in the Code Cell below.

In [ ]:
# ========== Extract Best Parameters from Study ========== #
best_params = study.best_trial.params

latent_dim   = best_params['latent_dim']
learning_rate = best_params['learning_rate']
beta_1 = best_params['beta_1']

print("Using Best Trial Parameters for WGAN:")
print(best_params)

# ========== Rebuild WGAN Models ========== #
generator = build_generator(latent_dim=latent_dim)
critic    = build_critic()

# ========== Optimisers ========== #
gen_optimizer    = tf.keras.optimizers.Adam(learning_rate=learning_rate, beta_1=beta_1)
critic_optimizer = tf.keras.optimizers.Adam(learning_rate=learning_rate, beta_1=beta_1)

# ========== Dummy Forward Pass to Trigger Variables ========== #
_ = generator(tf.zeros((1, latent_dim)))
_ = critic(tf.zeros((1, 28, 28, 1)))

_ = gen_optimizer.apply_gradients([(tf.zeros_like(var), var) for var in generator.trainable_variables])
_ = critic_optimizer.apply_gradients([(tf.zeros_like(var), var) for var in critic.trainable_variables])

# ========== Reload Dataset ========== #
dataset = load_data(X_train)

# ========== Path Setup ========== #
model_name   = "wgan"
output_dir   = f"/content/drive/MyDrive/Colab Notebooks/DELE CA2 A/final_outputs_{model_name}_noAugment"
weights_path = os.path.join(output_dir, f"{model_name}_generator_final.weights.h5")
gen_ckpt     = os.path.join(output_dir, "best_gen.weights.h5")
crit_ckpt    = os.path.join(output_dir, "best_crt.weights.h5")
loss_csv     = os.path.join(output_dir, "loss_history.csv")

os.makedirs(output_dir, exist_ok=True)

# ========== Skip if All Files Exist ========== #
if os.path.exists(gen_ckpt) and os.path.exists(crit_ckpt) and os.path.exists(loss_csv):
    print("All outputs found. Skipping training...")

    generator.load_weights(gen_ckpt)
    critic.load_weights(crit_ckpt)
    print("Generator and Critic Weights Loaded.")

    loss_df = pd.read_csv(loss_csv)
    gen_loss_history    = loss_df['gen_loss'].tolist()
    critic_loss_history = loss_df['critic_loss'].tolist()
    final_gen_loss      = gen_loss_history[-1]

else:
    # ========== Train ========== #
    final_gen_loss, gen_loss_history, critic_loss_history = train(
        dataset=dataset,
        epochs=100,
        steps_per_epoch=1546,
        generator=generator,
        critic=critic,
        gen_optimizer=gen_optimizer,
        critic_optimizer=critic_optimizer,
        latent_dim=latent_dim,
        use_early_stopping=False,
        save_path=output_dir,
        n_critic=5
    )

    # ========== Save Final Weights ========== #
    generator.save_weights(weights_path)
    print("Generator Weights Saved.")

    # ========== Save Loss History ========== #
    loss_df = pd.DataFrame({
        'epoch': list(range(1, len(gen_loss_history) + 1)),
        'gen_loss': gen_loss_history,
        'critic_loss': critic_loss_history
    })
    loss_df.to_csv(loss_csv, index=False)
    print(f"Loss history saved to {loss_csv}")

With reference to the Code Cell above, we are able to determine that the best GAN Model have been trained and saved successfully and we are able to proceed with the Model Evaluation in the next sub-sections where we determine how well the GAN Model performed based on various metrics and evaluation methodologies.

---
### 4.5.11 GAN Model Evaluation

In this sub-section, we will be evaluating the GAN's Generator performance on various aspects. We will be utilising different methodologies and metrics to attempt to quantify the performance of the GAN Generator while also conducting Visual Inspection on the result output of the GAN. The Evaluation will be conducted in the following sections.

---
#### 4.5.11.1 Visual Grid of Samples

In this sub-section, we will be visualising the Grid of Samples of the Synthesised Data generated by the GAN Model. A list of observations will be made before the utilisation of Quantitative Metrics to evaluate the GAN Generator's general performance. As the observations made are purely from professional opinion, it will most likely not be used as a Basis of Comparison between GAN Models due to differing opinions unless in extreme cases. The visualisation will be conducted in the Code Cell below.

In [ ]:
def plot_generated_images(generator, latent_dim, n_rows=5, n_cols=5, save_path=None, title="WGAN Generated Images (Non-Augmented)", seed=42):
    # ----- Generate Latent Vectors ----- #
    if seed is not None:
        tf.random.set_seed(seed)
    noise = tf.random.normal([n_rows * n_cols, latent_dim])
    gen_images = generator(noise, training=False)

    # ----- Rescale from [-1, 1] to [0, 1] ----- #
    gen_images = (gen_images + 1.0) / 2.0
    gen_images = tf.clip_by_value(gen_images, 0.0, 1.0)

    # ----- Shape Check ----- #
    assert gen_images.shape[-1] == 1, "Expected single-channel (grayscale) output"

    # ----- Sample Plot ----- #
    fig, axes = plt.subplots(n_rows, n_cols, figsize=(n_cols, n_rows))
    for idx, ax in enumerate(axes.flat):
        ax.imshow(gen_images[idx, :, :, 0], cmap='gray')
        ax.axis('off')

    plt.suptitle(title, fontsize=16)
    plt.tight_layout()

    if save_path:
        plt.savefig(save_path, dpi=300)
        print(f"Saved Generated Image Grid to: {save_path}")

    plt.show()

# ========== Call Plot Function ========== #
plot_generated_images(generator, latent_dim=160, n_rows=10, n_cols=16)

With reference to the output of the Visual Grid of Samples above, we are able to make the following observations:

**Basic Structure of Letters is Somewhat Captured**

- A few characters loosely resemble letters like "G", "C", "O", or "S".

- Stroke presence is consistent, but structural precision is lacking.

**High Blur and Noise**

- Many samples are heavily pixelated or contain grainy textures.

- Characters often appear “puffy” or overexposed, with bleeding edges.

**Poor Character Consistency**

- Several outputs display malformed or overly compressed letters.

- Letters often contain distortions or broken connections, hindering clarity.

**Lack of Diversity**

- Characters tend to follow similar round shapes repeatedly.

- Frequent duplication of loopy or circular structures suggests mode collapse may be occurring.

With the observations indicated above, we will proceed to conduct the Loss Curve Analysis in the next sub-section.

---
#### 4.5.11.2 Loss Curve Over Epochs

In this sub-section, we will be plotting the GAN Loss Curve over Epochs to evaluate the training behaviour of the GAN. This curve provides insight into whether the GAN is experiencing underfitting, overfitting, or achieving a stable training dynamic between the generator and discriminator.

Although GANs do not minimise a single unified loss in the traditional sense, the trajectory of the generator and discriminator losses can indicate whether the adversarial training process is converging appropriately.

If the generator loss remains high while discriminator loss quickly drops to near-zero, this may indicate **underfitting**, where the generator is not learning effectively to produce plausible images.

If the discriminator loss increases while generator loss sharply decreases, this may suggest **overfitting** or **mode collapse**, where the generator exploits narrow weaknesses in the discriminator without general improvement.

A relatively stable oscillation or convergence between both losses typically signifies a **well-balanced** training process, where the generator and discriminator are learning in tandem.

The Potential Observations and Corresponding Action Plans are outlined below:

**Underfitting Loss Curve**

- Increase training epochs to allow generator more time to learn.

- Simplify the discriminator to reduce overpowering the generator.

- Adjust the learning rate or apply label smoothing.

- Consider adding batch normalisation in the generator.

**Overfitting or Mode Collapse Loss Curve**

- Introduce or increase dropout in the discriminator.

- Apply input noise or label flipping to the discriminator.

- Evaluate diversity of generated samples regularly.

**Balanced/Ideal Loss Curve**

- Maintain current architecture and hyperparameters.

- Proceed with full-scale image generation.

- Evaluate both qualitative (visual inspection) and quantitative metrics (e.g. Inception Score or FID).

With these potential training behaviours and action plan outlined, we will proceed to visualise the GAN loss dynamics in the code cell below.

In [ ]:
# ========== Figure Size Configuration ========== #
plt.figure(figsize=(10, 6))

# ========== Plot Training Losses ========== #
plt.plot(gen_loss_history, label="Generator Loss", linewidth=2, color='blue')
plt.plot(critic_loss_history, label="Critic Loss", linewidth=2, color='red')

# ========== Labels and Title ========== #
plt.xlabel("Epoch", fontsize=12)
plt.ylabel("Loss", fontsize=12)
plt.title("WGAN Training Loss Curve (Non-Augmented)", fontsize=16)

# ========== Grid, Legend, and Styling ========== #
plt.grid(True, linestyle='--', alpha=0.6)
plt.legend(fontsize=12)
plt.xticks(fontsize=10)
plt.yticks(fontsize=10)

# ========== Display the Plot ========== #
plt.tight_layout()
plt.show()

With reference to the output of the Loss Curve above, we are able to make the following observations:

- Training is stable and effective, with generator improving and critic performing as expected.

- WGAN's behaviour aligns with theory, showing smooth critic loss and meaningful (non-saturating) generator updates.

- Data augmentation could further improve generalisability, but current training is healthy even without it.

With the observations indicated above, we will proceed to conduct the Quantitative Metrics Evaluation in the next sub-section.

---
#### 4.5.11.3 Quantitative Metrics Evaluation

In this sub-section we will be utilising Quantitative Metrics such as FID and KID to tabulate the results of the performance and quantify the performance of the GAN Generator. We will subsequently be use these metrics to conduct inter-model evaluation after tuning and training all the GANs. The metrics that we will be utilising and their respective formulas are indicated below:

---
**Fréchet Inception Distance**

Purpose: Measures the Distance between the Real-Image and Generated-Image Distributions in Inception Feature Space.

$$
\mathrm{FID} \;=\;\|\mu_r - \mu_f\|^2
\;+\;\mathrm{Tr}\Bigl(\Sigma_r + \Sigma_f - 2\,(\Sigma_r\,\Sigma_f)^{\tfrac12}\Bigr)
$$


Where:
- $\mu_r = \mathbb{E}[f(x)]$, $\Sigma_r = \mathrm{Cov}[f(x)]$ for Real Images $x$.  
- $\mu_f = \mathbb{E}[f(\hat x)]$, $\Sigma_f = \mathrm{Cov}[f(\hat x)]$ for Generated Images $\hat x$.  
- $f(\cdot)$ = Map from Image to its InceptionV3 'pooling=avg' Features.  

---
**Diversity (t-SNE Spread)**

Purpose: Quantifies how 'wide' the 2D t-SNE Embedding of Generated Samples.

$$
\mathrm{Spread}
\;=\;
\bigl(\max_i\,z_i^{(1)} - \min_i\,z_i^{(1)}\bigr)
\;\times\;
\bigl(\max_i\,z_i^{(2)} - \min_i\,z_i^{(2)}\bigr)
$$

Where:
- $z_i = (z_i^{(1)}, z_i^{(2)})$ = 2-dimensional t-SNE Embedding of $i$th Generated Image.

---
**Mode Collapse Risk**

Purpose: Flags when too many Generated Images are Nearly Identical (mode collapse).

$$
\text{ModeCollapseRisk} =
\begin{cases}
\text{Low}, & \dfrac{\bigl|\{\mathrm{unique\_rounded}(x_i)\}\bigr|}{N} > \tau,\\
\text{High}, & \text{otherwise}.
\end{cases}
$$

Where:
- $x_i$ = $N$ Generated Samples.  
- $\mathrm{unique\_rounded}(x_i)$ = Rounds Pixels to Detect Duplicates.  
- $\tau=0.9$ = Uniqueness Threshold (90%).

---
**Perceptual Path Length**

Purpose: Measures Sensitively of Generator's Outputs (in VGG16 Feature Space) move when the Latent Code $z$ is Perturbed.

$$
\mathrm{PPL}
\;=\;
\mathbb{E}_{z,\delta z}\Bigl[\,
\|\phi\bigl(G(z + \epsilon\,\delta z)\bigr)\;-\;\phi\bigl(G(z)\bigr)\|_2^2
\Bigr]
$$

Where:  
- $G(z)$ = GAN Generator Mapping $z\in\mathbb{R}^{\mathrm{latent\_dim}}$ to an Image.  
- $\phi(\cdot)$ = VGG16 'pooling=avg' Feature Extractor.  
- $\delta z\sim\mathcal{N}(0,I)$, $\epsilon$ = Small Constant.

---
**Kernel Inception Distance**

Purpose: An Unbiased Estimator of the Squared Maximum Mean Discrepancy (MMD) between Real and Generated Inception Features.

$$
\mathrm{KID}
\;=\;
\frac{1}{m(m-1)}\sum_{i\neq j} k\bigl(\phi(x_i),\phi(x_j)\bigr)
\;+\;
\frac{1}{n(n-1)}\sum_{i\neq j} k\bigl(\phi(\hat x_i),\phi(\hat x_j)\bigr)
\;-\;
\frac{2}{mn}\sum_{i=1}^m\sum_{j=1}^n k\bigl(\phi(x_i),\phi(\hat x_j)\bigr)
$$

Where:  
- $x_i$ ($i=1\ldots m$) = Real Images.
- $\hat x_j$ ($j=1\ldots n$) = Generated Images.  
- $\phi(\cdot)$ = InceptionV3 Feature Extractor (pooling='avg').  
- $k(u,v) = \bigl(\frac{u^\top v}{d}+1\bigr)^3$ = degree-3 Polynomial Kernel on $d$-dimensional Features.

---

With the mathematical and metrics indicated above, we will proceed to conduct the Quantitave Metrics Evalution in the Code Cell below.

In [ ]:
# ========== Preload InceptionV3 and VGG16 Models ========== #
inception = InceptionV3(include_top=False, pooling='avg', input_shape=(299, 299, 3))
vgg_model = VGG16(include_top=False, weights='imagenet', input_shape=(128, 128, 3))

# ========== Preprocess for Inception ========== #
def preprocess_images_for_inception(images):
    images = (images + 1.0) * 127.5  # Scale from [-1, 1] to [0, 255]
    images = tf.image.resize(images, [299, 299])
    if images.shape[-1] == 1:
        images = tf.image.grayscale_to_rgb(images)
    return preprocess_input(images)

# ========== Compute FID ========== #
def calculate_fid(real_images, fake_images, batch_size=50):
    real_pp = preprocess_images_for_inception(real_images)
    fake_pp = preprocess_images_for_inception(fake_images)

    act1 = inception.predict(real_pp, batch_size=batch_size, verbose=0)
    act2 = inception.predict(fake_pp, batch_size=batch_size, verbose=0)

    mu1, sigma1 = np.mean(act1, axis=0), np.cov(act1, rowvar=False)
    mu2, sigma2 = np.mean(act2, axis=0), np.cov(act2, rowvar=False)

    diff = mu1 - mu2
    covmean = sqrtm(sigma1 @ sigma2)
    if np.iscomplexobj(covmean):
        covmean = covmean.real

    fid = diff @ diff + np.trace(sigma1 + sigma2 - 2 * covmean)
    return round(fid, 4)

# ========== Compute KID ========== #
def polynomial_kernel(X, Y):
    d = X.shape[1]
    return (np.dot(X, Y.T) / d + 1) ** 3

def calculate_kid(real_images, fake_images, batch_size=128):
    real_pp = preprocess_images_for_inception(real_images)
    fake_pp = preprocess_images_for_inception(fake_images)

    real_features = inception.predict(real_pp, batch_size=batch_size, verbose=0)
    fake_features = inception.predict(fake_pp, batch_size=batch_size, verbose=0)

    m = real_features.shape[0]
    n = fake_features.shape[0]

    k_rr = polynomial_kernel(real_features, real_features)
    k_gg = polynomial_kernel(fake_features, fake_features)
    k_rg = polynomial_kernel(real_features, fake_features)

    np.fill_diagonal(k_rr, 0)
    np.fill_diagonal(k_gg, 0)

    mmd = (k_rr.sum() / (m * (m - 1)) +
           k_gg.sum() / (n * (n - 1)) -
           2 * k_rg.mean())
    return round(mmd, 4)

# ========== Compute t-SNE Spread (Diversity) ========== #
def calculate_tsne_spread(images):
    flat = images.reshape(images.shape[0], -1)
    tsne = TSNE(n_components=2, random_state=42)
    proj = tsne.fit_transform(flat)
    x_range = proj[:, 0].max() - proj[:, 0].min()
    y_range = proj[:, 1].max() - proj[:, 1].min()
    return round(x_range * y_range, 2)

# ========== Assess Mode Collapse ========== #
def assess_mode_collapse(images):
    unique = np.unique(np.round(images), axis=0).shape[0]
    return "Low" if unique > 0.9 * images.shape[0] else "High"

# ========== Compute Perceptual Path Length (PPL) ========== #
def calculate_ppl(generator, latent_dim=160, num_samples=50, epsilon=1e-2):
    distances = []
    for _ in range(num_samples):
        z1 = tf.random.normal([1, latent_dim])
        z2 = z1 + epsilon * tf.random.normal([1, latent_dim])
        img1 = generator(z1, training=False)
        img2 = generator(z2, training=False)

        img1 = tf.image.resize(tf.image.grayscale_to_rgb((img1 + 1.0) * 127.5), (128, 128))
        img2 = tf.image.resize(tf.image.grayscale_to_rgb((img2 + 1.0) * 127.5), (128, 128))

        f1 = vgg_model(vgg_preprocess(img1))
        f2 = vgg_model(vgg_preprocess(img2))
        d = tf.reduce_mean(tf.square(f1 - f2)).numpy()
        distances.append(d)
    return round(np.mean(distances), 4)

# ========== Evaluate GAN Performance ========== #
def evaluate_gan_model(generator, latent_dim, X_val, gen_loss_history, disc_loss_history, model_name="WGAN"):
    # ----- Generate Fake Images ----- #
    noise = tf.random.normal([1000, latent_dim])
    fake_images = generator(noise, training=False)
    fake_images = tf.clip_by_value((fake_images + 1) / 2.0, 0.0, 1.0)
    fake_np = fake_images.numpy()

    # ----- Select and preprocess 1000 Real Images from Validation Set ----- #
    real_images = tf.convert_to_tensor(X_val[:1000], dtype=tf.float32)
    real_images = tf.clip_by_value(real_images, 0.0, 1.0)

    # ----- Compute Metrics ----- #
    fid_value = calculate_fid(real_images, fake_images)
    kid_value = calculate_kid(real_images, fake_images)
    tsne_spread_value = calculate_tsne_spread(fake_np)
    collapse_risk_value = assess_mode_collapse(fake_np)
    ppl_score_value = calculate_ppl(generator, latent_dim)
    mean_gen_loss_value = round(np.mean(gen_loss_history), 4)
    std_gen_loss_value = round(np.std(gen_loss_history), 4)
    mean_disc_loss_value = round(np.mean(disc_loss_history), 4)
    std_disc_loss_value = round(np.std(disc_loss_history), 4)

    # ----- Store as Global Vars (Optional) ----- #
    prefix = model_name.upper()
    globals()[f"{prefix}_FID"] = fid_value
    globals()[f"{prefix}_KID"] = kid_value
    globals()[f"{prefix}_TSNE"] = tsne_spread_value
    globals()[f"{prefix}_MODE_COLLAPSE"] = collapse_risk_value
    globals()[f"{prefix}_PPL"] = ppl_score_value
    globals()[f"{prefix}_GEN_LOSS_MEAN"] = mean_gen_loss_value
    globals()[f"{prefix}_GEN_LOSS_STD"] = std_gen_loss_value
    globals()[f"{prefix}_DISC_LOSS_MEAN"] = mean_disc_loss_value
    globals()[f"{prefix}_DISC_LOSS_STD"] = std_disc_loss_value

    # ----- Final Summary Row ----- #
    row = {
        "Model": model_name,
        "FID Score": fid_value,
        "KID Score": kid_value,
        "Mean Generator Loss": mean_gen_loss_value,
        "Std Generator Loss": std_gen_loss_value,
        "Mean Discriminator Loss": mean_disc_loss_value,
        "Std Discriminator Loss": std_disc_loss_value,
        "Mode Collapse Risk": collapse_risk_value,
        "Visual Quality": "Poor",
        "Diversity (t-SNE Spread)": tsne_spread_value,
        "PPL": ppl_score_value
    }

    return pd.DataFrame([row])[[
        "Model", "FID Score", "KID Score",
        "Mean Generator Loss", "Std Generator Loss",
        "Mean Discriminator Loss", "Std Discriminator Loss",
        "Mode Collapse Risk", "Visual Quality",
        "Diversity (t-SNE Spread)", "PPL"
    ]]

# ========== Create DataFrame ========== #
df = evaluate_gan_model(generator, latent_dim, X_val, gen_loss_history, critic_loss_history, model_name="WGAN (Non-Augmented)")

# ========== Display DataFrame ========== #
df.style.background_gradient(cmap="Blues")

With reference to the output of the Quantitative Metrics above, we are able to determine the performance of the GAN. This will be used as a basis of comparison to the other GAN Models.

---
#### 4.5.11.4 t-SNE of Generated Samples

In this sub-section, we visualise the t-SNE projection of generated samples to evaluate the latent diversity learned by the unconditional GAN. Although our GAN is not class-conditioned, t-SNE remains a valuable tool to assess whether the generator is producing varied outputs that reflect meaningful use of the latent space. While GANs are trained in high-dimensional spaces, t-SNE enables us to observe 2D structural patterns such as sample clustering, separation, and density, which provide indirect insights into the diversity and generalisation capabilities of the generator. The visualisation will be conducted in the Code Cell below.

In [ ]:
# ========== Generate New Images from Random Noise ========== #
latent_dim = 160
noise = tf.random.normal([500, latent_dim])
generated_images = generator(noise, training=False)
generated_images = (generated_images + 1) / 2.0

def plot_tsne_embeddings(generated_images, save_path=None, title="t-SNE of WGAN Generated Samples (Non-Augmented)"):
    # ----- Convert to [0, 1] Range ----- #
    if tf.reduce_max(generated_images).numpy() > 1.0:
        raise ValueError("Images should be in [-1, 1] range before scaling.")

    images_rescaled = (generated_images + 1.0) / 2.0
    flat_images = images_rescaled.numpy().reshape(images_rescaled.shape[0], -1)

    # ---- Run t-SNE ----- #
    tsne = TSNE(n_components=2, perplexity=30, learning_rate='auto', init='pca', random_state=42)
    tsne_proj = tsne.fit_transform(flat_images)

    # ----- Plot t-SNE ----- #
    plt.figure(figsize=(12, 6))
    sns.scatterplot(
        x=tsne_proj[:, 0],
        y=tsne_proj[:, 1],
        s=20,
        alpha=0.9,
        edgecolor='none'
    )
    plt.title(title, fontsize=14, weight='bold')
    plt.xlabel("t-SNE Dimension 1")
    plt.ylabel("t-SNE Dimension 2")
    plt.grid(True, linestyle='--', alpha=0.3)
    plt.tight_layout()

    if save_path:
        plt.savefig(save_path, dpi=300)
        print(f"t-SNE plot saved to: {save_path}")

    plt.show()

plot_tsne_embeddings(generated_images=generated_images)

With reference to the t-SNE plot of WGAN generated samples (Non-Augmented) above, we are able to make the following observations:

**Good Coverage of Latent Space**

- The distribution forms a wide elliptical shape across both axes.

- Suggests that WGAN is effectively exploring the latent manifold and generating diverse samples.

**Uniform Dispersion**

- Data points are moderately spread with no severe clustering or empty regions.

- Indicates stable training and minimal collapse during generation.

**No Clear Structure**

- The points are randomly scattered with no visible semantic organisation.

- Expected for unsupervised settings where labels are not imposed on generation.

**Low Outlier Presence**

- Outlier density is minimal, with most points conforming to the main body of the distribution.

- Implies that WGAN is less prone to mode-hopping or unstable generation.

With the observations indicated above, we will proceed to train the next GAN Model in the next sub-section.

---
## 4.6 Deep Regret Analytic GAN Model Training

In this sub-section, we will be training a Deep Regret Analytic GAN (DRAGAN) for the EMNIST Dataset. The LSGAN mainly replace the Binary Crossentropy Loss used in classical GANs with a Least Squares Loss. Through this replacement, the LSGAN is more likely able to have a Stabalise GAN Training, Produce Higher Quality Images, and Encourage the Generator more Accurate Results. The relevant mathematical formulas are indicated below:

---
**Generator Loss:**

Purpose: Produce Outputs Classified as Real.

$$
\mathcal{L}_G = -\mathbb{E}_{z \sim p_z(z)} [\log D(G(z))]
$$

Where:
- $c = 1$ = Desired Discriminator Output for Generated Images  
- $D(G(z))$ = Discriminator's Output for Generated Data  
- $z \sim p_z$ = Sample from Latent Space

---
**Discriminator Loss:**

Purpose: Discriminator Minimises the Error between Predictions and True Labels.

$$
\mathcal{L}_D = \frac{1}{2} \mathbb{E}_{x \sim p_{\text{data}}}[(D(x) - b)^2] + \frac{1}{2} \mathbb{E}_{z \sim p_z}[(D(G(z)) - a)^2]
$$

Where:  
- $D(x)$ = Discriminator's Output for Real Data  
- $D(G(z))$ = Discriminator's Output for Generated Data  
- $a = 0$ = Label for Fake Data  
- $b = 1$ = Label for Real Data  
- $x \sim p_{\text{data}}$ = Sample from Real Data Distribution  
- $z \sim p_z$ = Sample from Latent Noise Distribution  

---
**Pertubed Input Sampling:**

Purpose: Add noise to real images to compute gradient penalty near the data manifold.

$$
\hat{x} = x + \alpha \cdot \sigma_x \cdot \epsilon
$$
Where:
- $x$ = Real Data Sample
- $\alpha \sim \mathcal{U}(0, 1)$ = Uniform Random Variable
- $\epsilon \sim \mathcal{N}(0, 1)$ = Standard Normal Noise
- $\sigma_x$ = Standard Deviation of Data Batch (per pixel)

---
**Minimax Objective with DRAGAN Penalty:**

Purpose: LSGAN Seeks to Solve the Objective.

$$
\min_G \max_D \; V(D, G) = \mathbb{E}_{x \sim p_{data}} [\log D(x)] + \mathbb{E}_{z \sim p_z(z)} [\log(1 - D(G(z)))] - \lambda \cdot \mathbb{E}_{\hat{x} \sim p_{perturbed}} \left[ \left( \|\nabla_{\hat{x}} D(\hat{x}) \|_2 - 1 \right)^2 \right]
$$
Where:
- $V(D, G)$ = Value Function for Adversarial Training
---
With the relevant mathematical formulas indicated above, we will proceed to train the LSGAN in this sub-section for the purpose of it's potential for smoother training process and also to serve as a comparison for our other GAN Models.

---
### 4.6.1 Defining Data Pre-Processing Function

In this sub-section, we will be defining the Data Pre-Processing Function which will allow for consistent Data Pipeline to be built and modifed accordingly should the need arise such as when we need to add Spectral Normalisation. This will ensure that the Data's integrity is not affected and we are able to make fair and un-biased assumptions. The Data Pre-Processing Function is defined in the Code Cell below:

In [ ]:
# ========== Data Pre-Processing Function ========== #
def load_data(X_train, batch_size=32):
    buffer_size = X_train.shape[0]

    dataset = tf.data.Dataset.from_tensor_slices(X_train)
    dataset = dataset.shuffle(buffer_size)
    dataset = dataset.batch(batch_size)
    dataset = dataset.prefetch(tf.data.AUTOTUNE)

    return dataset

With reference to the Code Cell above, we are able to determine that the Data Pre-Processing Function have been defined successfully and we are able to move on to Defining Callback Functions in the next sub-section.

---
### 4.6.2 Defining Callback Functions

In this sub-section, we will be pre-defining the various Callbacks. This is to ensure consistency throughout the model and also to increase the GAN Accuracy and optimise the Computation Cost for training each GAN Model. These Callbacks will be main used during the GAN Training in the next sub-section. The revelant mathematical formula for the callbacks and logic are indicated below:

---
**Learning Rate Scheduler:**

Purpose: Reduces Learning Rate During Training to Fine-tune Model Convergence.

$$\text{If } epoch \mod 10 = 0 \Rightarrow \eta_{new} = \frac{1}{2} \cdot \eta$$

Where:

- $\eta$ = current learning rate

- $\eta_{\text{new}}$ = updated learning rate

- $epoch \bmod 10$ checks if the epoch is a multiple of 10

---
**ReduceLROnPlateau:**

Purpose: Automatically Reduces the Learning Rate when Validation Performance Plateaus.

$$
\text{If no improvement in } val\_loss \text{ for 10 epochs: } \eta_{\text{new}} = 0.5 \cdot \eta
$$

Where:

- $\eta_{\text{new}}$ = reduced learning rate

---
**EarlyStopping:**

Purpose: Prevents Wastage of Computational Power if Model Does Not Improve.

$$
\text{If } val\_loss \text{ does not improve for 25 epochs, stop training and restore best weights}
$$

---

We will proceed to Define the Callbacks in the Code Cell below in preparation for the GAN Model Training.

In [ ]:
# ========== Define Learning Rate Scheduler ========== #
lr_scheduler = LearningRateScheduler(
    lambda epoch, lr: lr * 0.95 if epoch % 10 == 0 else lr,
    verbose=1
)

# ========== Define Reduce LR on Plateau ========== #
reduce_lr = ReduceLROnPlateau(
    monitor='loss',
    factor=0.5,
    patience=10,
    verbose=1,
    min_lr=1e-6
)

# ========== Define Early Stopping ========== #
early_stop = EarlyStopping(
    monitor='loss',
    patience=25,
    verbose=1,
    restore_best_weights=False
)

# ========== Define Display Generated Images ========== #
def display_generated_images(generator, latent_dim, n=7):
    noise = tf.random.normal([n * n, latent_dim])
    generated_images = generator(noise, training=False)
    generated_images = (generated_images + 1.0) / 2.0

    fig, axes = plt.subplots(n, n, figsize=(n, n))
    for i in range(n):
        for j in range(n):
            img = generated_images[i * n + j, :, :, 0]
            axes[i, j].imshow(img, cmap='gray')
            axes[i, j].axis('off')

    plt.tight_layout()
    plt.show()

# ========== Define FID Calculator ========== #
def calculate_fid(real_images, fake_images, batch_size=10):

    # ----- Scale to [0,255] and Resize to 299×299 ----- #
    real = (real_images + 1.0) * 127.5
    fake = (fake_images + 1.0) * 127.5
    real = tf.image.resize(real, (299,299))
    fake = tf.image.resize(fake, (299,299))

    # ----- If Grayscale, Convert to RGB ----- #
    if real.shape[-1] == 1:
        real = tf.image.grayscale_to_rgb(real)
        fake = tf.image.grayscale_to_rgb(fake)

    # ----- Preprocess for Inception (–1 to +1) ----- #
    real_pp = preprocess_input(real)
    fake_pp = preprocess_input(fake)

    # ----- Extract Features in Batches ----- #
    def _get_acts(x):
        acts = []
        n = x.shape[0]
        for i in range(0, n, batch_size):
            chunk = x[i:i+batch_size]
            acts.append(_inception_model(chunk, training=False).numpy())
        return np.vstack(acts)

    act_real = _get_acts(real_pp)
    act_fake = _get_acts(fake_pp)

    # ----- Compute Statistics ----- #
    mu1, sigma1 = act_real.mean(axis=0), np.cov(act_real, rowvar=False)
    mu2, sigma2 = act_fake.mean(axis=0), np.cov(act_fake, rowvar=False)
    diff    = mu1 - mu2
    covmean = sqrtm(sigma1.dot(sigma2))
    if np.iscomplexobj(covmean):
        covmean = covmean.real
    fid = diff.dot(diff) + np.trace(sigma1 + sigma2 - 2*covmean)
    return float(np.round(fid,4))

# ========== Confirmation Message ========== #
print("DRAGAN Callbacks and Visualisation Setup Complete")

With reference to the Code Cell above, we are able to verify that that the Callbacks have been successfully defined and the various parameters are set so as to ensure a High Accuracy and Computing Efficient GAN Model Training.

---
### 4.6.3 Defining DRAGAN Generator Architecture

In this section, we will be defining the DRAGAN's Generator Architecture. The Generator will be trying to make realistic generated images in the attempt to bypass the Discriminator. Subsequently, the generated images from the Generator will be evaluated based on a few evaluation metrics such as Inception Score. The relevent mathematical formulas for the Loss Function of the Vanilla GAN Generator is indicated below:

---
**Generator Loss:**

Purpose: Produce Outputs Classified as Real.

$$
\mathcal{L}_G = \frac{1}{2} \mathbb{E}_{z \sim p_z}[(D(G(z)) - c)^2]
$$

Where:
- $c = 1$ = Desired Discriminator Output for Generated Images  
- $D(G(z))$ = Discriminator's Output for Generated Data  
- $z \sim p_z$ = Sample from Latent Space

---
With the relevant Generator Loss Formula indicated above, we will proceed to define the GAN Generator's Architecture in the Code Cell below and the Lost Function.

In [ ]:
# ========== Generator Function ========== #
def build_generator(latent_dim=100, use_dropout=True):
    model = tf.keras.Sequential(name="generator")

    # ----- Project and Reshape ----- #
    model.add(tf.keras.layers.Dense(7 * 7 * 256, use_bias=False, input_shape=(latent_dim,)))
    model.add(tf.keras.layers.BatchNormalization())
    model.add(tf.keras.layers.ReLU())
    model.add(tf.keras.layers.Reshape((7, 7, 256)))

    # ----- Upsample 1 ----- #
    model.add(tf.keras.layers.Conv2DTranspose(128, kernel_size=5, strides=1, padding='same', use_bias=False))
    model.add(tf.keras.layers.BatchNormalization())
    model.add(tf.keras.layers.ReLU())
    if use_dropout:
        model.add(tf.keras.layers.Dropout(0.3))

    # ----- Upsample 2 ----- #
    model.add(tf.keras.layers.Conv2DTranspose(64, kernel_size=5, strides=2, padding='same', use_bias=False))
    model.add(tf.keras.layers.BatchNormalization())
    model.add(tf.keras.layers.ReLU())
    if use_dropout:
        model.add(tf.keras.layers.Dropout(0.3))

    # ----- Final Output to 28x28x1 ----- #
    model.add(tf.keras.layers.Conv2DTranspose(1, kernel_size=5, strides=2, padding='same', activation='tanh'))

    return model

# ========== DRAGAN Generator Loss (BCE) ========== #
bce = tf.keras.losses.BinaryCrossentropy(from_logits=True)

def generator_loss(fake_output):
    return bce(tf.ones_like(fake_output), fake_output)

With reference to the Code Cell above, we are able to verify that the GAN's Generator Architecture have been successfully defined and we are able to proceed to define the GAN's Discriminator Architecture.

---
### 4.6.4 Defining DRAGAN Discriminator Architecture

In this section, we will be defining the Architecture of the Discriminator of the GAN. The Discriminator will be the Neural Network that will attempt to tell the Generated Images apart from the Actual Images. This will allow the GAN to improve overtime and also improve itself overtime. The mathematical function for the Discriminator Loss Function is indicated below:

---
**Discriminator Loss:**

Purpose: Discriminator Minimises the Least Squares Error between Predictions and True Labels.

$$
\mathcal{L}_D = \frac{1}{2} \mathbb{E}_{x \sim p_{\text{data}}}[(D(x) - b)^2] + \frac{1}{2} \mathbb{E}_{z \sim p_z}[(D(G(z)) - a)^2]
$$

Where:  
- $D(x)$ = Discriminator's Output for Real Data  
- $D(G(z))$ = Discriminator's Output for Generated Data  
- $a = 0$ = Label for Fake Data  
- $b = 1$ = Label for Real Data  
- $x \sim p_{\text{data}}$ = Sample from Real Data Distribution  
- $z \sim p_z$ = Sample from Latent Noise Distribution  
---

With the mathematical formula for the Loss Function indicated above, we will proceed to define the GAN's Discriminator Architecture and the Loss Function in the Code Cell below.

In [ ]:
# ========== Discriminator Function ========== #
def build_discriminator(input_shape=(28, 28, 1)):
    model = tf.keras.Sequential(name="discriminator")

    model.add(tf.keras.layers.Conv2D(64, kernel_size=5, strides=2, padding='same', input_shape=input_shape))
    model.add(tf.keras.layers.LeakyReLU(0.2))
    model.add(tf.keras.layers.Dropout(0.3))

    model.add(tf.keras.layers.Conv2D(128, kernel_size=5, strides=2, padding='same'))
    model.add(tf.keras.layers.LeakyReLU(0.2))
    model.add(tf.keras.layers.Dropout(0.3))

    model.add(tf.keras.layers.Flatten())
    model.add(tf.keras.layers.Dense(1))

    return model

# ========== Discriminator Loss Function ========== #
bce = tf.keras.losses.BinaryCrossentropy(from_logits=True)

def discriminator_loss(real_output, fake_output):
    real_labels = tf.ones_like(real_output)
    fake_labels = tf.zeros_like(fake_output)

    real_loss = bce(real_labels, real_output)
    fake_loss = bce(fake_labels, fake_output)

    return real_loss + fake_loss

With reference to the Code Cell above, we are able to verify that the GAN's Discriminator Architecture have been successfully defined and we are able to proceed to define the GAN's Training Steps.

---
### 4.6.5 Defining Training Step

In this section, we will be Defining the Training Step function so as to train both the Generator and the Discriminator at the same time. The Training Step will compute the gradients for both the Generator and the Discriminator at the same time. The relevant mathematical formulas for the Training Step is indicated below:

---
**Generator Loss:**

Purpose: Produce Outputs Classified as Real.

$$
\mathcal{L}_G = \frac{1}{2} \mathbb{E}_{z \sim p_z}[(D(G(z)) - c)^2]
$$

Where:
- $c = 1$ = Desired Discriminator Output for Generated Images  
- $D(G(z))$ = Discriminator's Output for Generated Data  
- $z \sim p_z$ = Sample from Latent Space

---
**Discriminator Loss:**

Purpose: Discriminator Minimises the Least Squares Error between Predictions and True Labels.

$$
\mathcal{L}_D = \frac{1}{2} \mathbb{E}_{x \sim p_{\text{data}}}[(D(x) - b)^2] + \frac{1}{2} \mathbb{E}_{z \sim p_z}[(D(G(z)) - a)^2]
$$

Where:  
- $D(x)$ = Discriminator's Output for Real Data  
- $D(G(z))$ = Discriminator's Output for Generated Data  
- $a = 0$ = Label for Fake Data  
- $b = 1$ = Label for Real Data  
- $x \sim p_{\text{data}}$ = Sample from Real Data Distribution  
- $z \sim p_z$ = Sample from Latent Noise Distribution  

---
**Minimax Objective with DRAGAN Penalty:**

Purpose: LSGAN Seeks to Solve the Objective.

$$
\min_G \max_D \; V(D, G) = \mathbb{E}_{x \sim p_{data}} [\log D(x)] + \mathbb{E}_{z \sim p_z(z)} [\log(1 - D(G(z)))] - \lambda \cdot \mathbb{E}_{\hat{x} \sim p_{perturbed}} \left[ \left( \|\nabla_{\hat{x}} D(\hat{x}) \|_2 - 1 \right)^2 \right]
$$
Where:
- $V(D, G)$ = Value Function for Adversarial Training
---

With the relevant mathematical formulas indicated above, we will proceed to define the Training Step in the Code Cell below.

In [ ]:
# ========== Training Step Function ========== #
@tf.function
def train_step(real_images, generator, discriminator,
               gen_optimizer, disc_optimizer,
               latent_dim, gp_weight=10.0):
    batch_size = tf.shape(real_images)[0]

    # ----- Latent Noise (L2-Normalised) ----- #
    noise = tf.random.normal([batch_size, latent_dim])
    noise = tf.math.l2_normalize(noise, axis=1)

    # ----- Instance Noise on Real Images ----- #
    real_noisy = real_images + tf.random.normal(tf.shape(real_images)) * 0.05

    # ----- Compute Losses & Gradients ----- #
    with tf.GradientTape(persistent=True) as tape:
        # ----- Generate Fake Images ----- #
        fake_images = generator(noise, training=True)

        # ----- Discriminator Outputs ----- #
        real_output = discriminator(real_noisy, training=True)
        fake_output = discriminator(fake_images, training=True)

        # ----- Adversarial BCE Losses with Label Smoothing ----- #
        real_labels = tf.ones_like(real_output) * 0.9  # label smoothing
        fake_labels = tf.zeros_like(fake_output)
        real_loss = bce(real_labels, real_output)
        fake_loss = bce(fake_labels, fake_output)
        disc_loss = real_loss + fake_loss

        # ----- Generator Loss ----- #
        gen_loss = bce(tf.ones_like(fake_output), fake_output)

        # ----- DRAGAN Gradient Penalty (Local Perturbation) ----- #
        alpha = tf.random.uniform([batch_size, 1, 1, 1], 0.0, 1.0)
        noise_perturb = 0.5 * tf.random.normal(tf.shape(real_images))
        x_hat = real_images + alpha * noise_perturb
        with tf.GradientTape() as gp_tape:
            gp_tape.watch(x_hat)
            pred_hat = discriminator(x_hat, training=True)
        grads = gp_tape.gradient(pred_hat, x_hat)
        grads = tf.reshape(grads, [batch_size, -1])
        grads_norm = tf.sqrt(tf.reduce_sum(tf.square(grads), axis=1) + 1e-12)
        gp = tf.reduce_mean((grads_norm - 1.0) ** 2)

        # ----- Add Gradient Penalty to Discriminator Loss ----- #
        disc_loss += gp_weight * gp

    # ----- Apply Gradients ----- #
    gen_grads = tape.gradient(gen_loss, generator.trainable_variables)
    disc_grads = tape.gradient(disc_loss, discriminator.trainable_variables)
    gen_optimizer.apply_gradients(zip(gen_grads, generator.trainable_variables))
    disc_optimizer.apply_gradients(zip(disc_grads, discriminator.trainable_variables))

    del tape

    return gen_loss, disc_loss, real_output, fake_output

With reference to the Code Cell above, we are able to determine that the Training Step have been successfully defined and we are able to proceed to prepare the Training Loop in the next section.

---
### 4.6.6 Defining Training Loop

In this section, we will be defining the Training Loop for the GAN Training. The Training Loop will go through each of the Epoch and train both the Generator and Discriminator simultaneously. The Training Loop will also track the Loss for both the Generator and the Discriminator for each of the Epoch trained. The relevant mathematical formulas are indicated below:

---
**Epoch Loss Averaging:**

Purpose: These are the Average Generator and Discriminator Losses over all Batches in Epoch $t$.

$$
\bar{L}_G^{(t)} = \frac{1}{N} \sum_{i=1}^{N} L_G^{(i)} \\
\bar{L}_D^{(t)} = \frac{1}{N} \sum_{i=1}^{N} L_D^{(i)}
$$

Where:
- $\bar{L}_G^{(t)}$ = Average Generator Loss at Epoch $t$  
- $\bar{L}_D^{(t)}$ = Average Discriminator Loss at Epoch $t$  
- $N$ = Number of Batches in the Dataset  
- $L_G^{(i)}$ = Generator Loss on Batch $i$  
- $L_D^{(i)}$ = Discriminator Loss on Batch $i$

---
**Epoch Iteration:**

Purpose: Describes the Training Loop Logic: for Each Epoch $t$, perform `train_step` for Every Mini-batch.

$$
\text{for } t = 1 \text{ to } T:
\quad \text{for each batch } (x^{(i)}):
\quad \text{train_step}(x^{(i)})
$$

Where:
- $T$: Total Number of Epochs  
- $x^{(i)}$: Real Batch $i$ from the Dataset  
- `train_step`: Function that Updates $G$ and $D$ using that Batch

---
**Generated Image Output:**

Purpose: Sample Random Noise $z$ and Generate Synthetic Image $\hat{x}$ from the Generator. This is used for Visual Monitoring of Model Quality.

$$
z \sim p_z(z) \\
\hat{x} = G(z)
$$

Where:
- $z$: Random Latent Vector  
- $G(z)$: Generated Image from Generator

---
With the mathematical formulas indicated above, we will proceed to define the Training Loop in the Code Cell below in preparation of the GAN Training.

In [ ]:
# ========== Training Loop Function ========== #
def train(dataset, epochs, generator, discriminator, gen_optimizer, disc_optimizer, latent_dim, save_path=None, use_early_stopping=True, steps_per_epoch=None, gp_weight=10.0):

    # ----- Prepare Fixed Real Batch from Validation Set X_val for FID ----- #
    real_val = tf.data.Dataset.from_tensor_slices(X_val).batch(50).take(1)

    # ----- Prepare Fixed Noise Batch for FID ----- #
    noise_val = tf.random.normal([50, latent_dim])
    noise_val = tf.math.l2_normalize(noise_val, axis=1)

    # ----- Early Stopping ----- #
    best_fid     = float('inf')
    no_improve   = 0
    patience     = 10
    fid_interval = 1

    # ----- Dummy Model ----- #
    dummy_input  = Input(shape=(1,))
    dummy_output = Lambda(lambda x: x)(dummy_input)
    dummy_model  = Model(dummy_input, dummy_output)
    dummy_model.compile(optimizer=Adam(1e-4), loss='mse')

    early_stop.set_model(dummy_model)
    reduce_lr.set_model(dummy_model)
    lr_scheduler.set_model(dummy_model)
    early_stop.on_train_begin({})
    reduce_lr.on_train_begin({})
    lr_scheduler.on_train_begin({})

    # ----- Track Loss Histories ----- #
    gen_loss_history  = []
    disc_loss_history = []

    for epoch in range(1, epochs + 1):
        print(f"\nEpoch {epoch}/{epochs}")
        gen_losses, disc_losses = [], []

        # ----- Per‐batch Training ----- #
        for i, real_images in enumerate(dataset):
            if steps_per_epoch and i >= steps_per_epoch:
                break

            gen_loss, disc_loss, _, _ = train_step(
                real_images,
                generator,
                discriminator,
                gen_optimizer,
                disc_optimizer,
                latent_dim,
                gp_weight=gp_weight
            )
            gen_losses.append(gen_loss)
            disc_losses.append(disc_loss)

        # ----- Compute Averages ----- #
        avg_gen_loss  = tf.reduce_mean(gen_losses)
        avg_disc_loss = tf.reduce_mean(disc_losses)
        print(f"Generator Loss: {avg_gen_loss:.4f} | Discriminator Loss: {avg_disc_loss:.4f}")

        gen_loss_history.append(float(avg_gen_loss))
        disc_loss_history.append(float(avg_disc_loss))

        # ----- Callback on Generator Loss ----- #
        logs = {'loss': float(avg_gen_loss)}
        if use_early_stopping:
            early_stop.on_epoch_end(epoch=epoch, logs=logs)
            if early_stop.stopped_epoch > 0:
                print(f"Early stopping triggered at epoch {epoch}")
                break
        reduce_lr.on_epoch_end(epoch=epoch, logs=logs)
        lr_scheduler.on_epoch_end(epoch=epoch, logs=logs)

        # ----- FID Evaluation & Early Stopping ----- #
        if epoch % fid_interval == 0:
            fake_val   = generator(noise_val, training=False)
            real_batch = next(iter(real_val))

            # ----- Resize & Rescale to [0,1] ----- #
            real_for_fid = tf.image.resize(real_batch, [299,299]) * 0.5 + 0.5
            fake_for_fid = tf.image.resize(fake_val,    [299,299]) * 0.5 + 0.5
            fid_value = calculate_fid(real_for_fid, fake_for_fid)

            print(f"Epoch {epoch} → Val FID: {fid_value:.2f}")

            if fid_value < best_fid:
                best_fid   = fid_value
                no_improve = 0
                if save_path:
                    os.makedirs(save_path, exist_ok=True)
                    generator.save_weights(os.path.join(save_path, "best_gen.weights.h5"))
                    discriminator.save_weights(os.path.join(save_path, "best_disc.weights.h5"))
            else:
                no_improve += 1

            if no_improve >= patience:
                print(f"No FID improvement for {patience} epochs, stopping.")
                break

        # ----- Display Every 5 Epochs ----- #
        if epoch % 5 == 0:
            display_generated_images(generator, latent_dim)

    # ----- Restore Best Weights After Training ----- #
    if save_path:
        best_gen_path = os.path.join(save_path, "best_gen.weights.h5")
        best_disc_path = os.path.join(save_path, "best_disc.weights.h5")
        if os.path.exists(best_gen_path) and os.path.exists(best_disc_path):
            generator.load_weights(best_gen_path)
            discriminator.load_weights(best_disc_path)
            print("Restored best generator and discriminator weights based on lowest FID.")
        else:
            print("Best weight files not found. Skipping restore.")

    return avg_gen_loss, gen_loss_history, disc_loss_history

With reference to the Code Cell above, we are able to determine that the Training Loop have been successfully defined and we are able to proceed to define the Optimisers with Tunable Parameter in preparation for the Optuna Tuning.

---
### 4.6.7 Defining Optimisers with Tunable Parameters

In this section, we will be defining the Common Optimisers that will be Tuned using the Optuna Tuning. The Optimisers will each have a range of values in a list so as to allow for the Optuna to search for the best Hyperparameter for the GAN. Through these parameters, it will affect how effective and efficiently the GAN will learn from the provided EMNIST Data. The relevent mathematical formula is indicated below:

---
**Optimiser Update Rule**

Purpose: Parameter Update Step in the Adam Optimiser, using Adaptive Learning Rates with Momentum.

$$
\theta \leftarrow \theta - \eta \cdot \frac{m_t}{\sqrt{v_t} + \epsilon}
$$

Where:
- $\theta$ = Model Parameters (Generator or Discriminator)
- $\eta$ = Learning Rate (Tuned)
- $m_t$ = First Moment Estimate (Mean of Gradients)
- $v_t$ = Second Moment Estimate (Variance of Gradients)
- $\epsilon$ = Small Constant for Numerical Stability

---
With the mathematical formula indicated above, we will proceed to define the Optimisers in the Code Cell.

In [ ]:
# ========== Objective Function ========== #
def objective(trial):
    # ----- Hyperparameters ----- #
    latent_dim   = trial.suggest_categorical('latent_dim', [100, 128, 160])
    learning_rate = trial.suggest_float('learning_rate', 5e-5, 2e-4, log=True)
    beta_1        = trial.suggest_float('beta_1', 0.4, 0.6)

    # ----- Build Models (Unconditional) ----- #
    generator     = build_generator(latent_dim=latent_dim)
    discriminator = build_discriminator()

    # ----- Trigger Weight Creation ----- #
    _ = generator(tf.random.uniform([1, latent_dim], -1.0, 1.0))
    _ = discriminator(tf.random.normal([1, 28, 28, 1]))

    # ----- Define Optimizers ----- #
    gen_optimizer  = tf.keras.optimizers.Adam(learning_rate=learning_rate, beta_1=beta_1)
    disc_optimizer = tf.keras.optimizers.Adam(learning_rate=learning_rate, beta_1=beta_1)

    # ----- Prebuild Optimizer Variables ----- #
    _ = gen_optimizer.apply_gradients([(tf.zeros_like(v), v) for v in generator.trainable_variables])
    _ = disc_optimizer.apply_gradients([(tf.zeros_like(v), v) for v in discriminator.trainable_variables])

    # ----- Load Training Data (Unconditional) ----- #
    dataset = load_data(X_train, batch_size=64)
    steps   = min(1000, len(X_train) // 64)

    # ----- Train DRAGAN ----- #
    _ = train(
        dataset=dataset,
        epochs=50,
        generator=generator,
        discriminator=discriminator,
        gen_optimizer=gen_optimizer,
        disc_optimizer=disc_optimizer,
        latent_dim=latent_dim,
        steps_per_epoch=steps,
        save_path=None,
        gp_weight=10.0
    )

    # ----- Generate Fake Images ----- #
    z = tf.random.uniform([1000, latent_dim], -1.0, 1.0)
    fake_images = generator(z, training=False)
    fake_images = (fake_images + 1.0) / 2.0  # Rescale to [0, 1]

    # ----- Real Images from Validation Set ----- #
    real_images = X_val[:1000]

    # ----- Compute FID Score ----- #
    fid_score = calculate_fid(real_images, fake_images)
    return fid_score

With reference to the Code Cell above, we are able to determine that the Optimiser have been successfully defined with a range of different Parameter Values so as to provide a robust Optuna Tuning in the subsequent sections.

---
### 4.6.8 Optuna Tuning Study

In this sub-section, we will be conducting the Optuna Tuning Study on the GAN Model so as to obtain the best Hyperparameter for the GAN Model. It is expected that the Tuning takes a prolonged duration due to the nature of how the GAN works and the 2 Neural Networks involved. However, the training data will be stored so as to prevent re-running of repetitive codes. The Optuna Tuning Study will be conducted in the Code Cell below.

In [ ]:
# ========== Create or Load Study with SQLite Backend ========== #
study = optuna.create_study(
    direction="minimize",
    study_name="dragan_tuning",
    storage="sqlite:////content/drive/MyDrive/Colab Notebooks/DELE CA2 A/Non-Augmented GAN Tunings/dragan_optuna.db",
    load_if_exists=True
)

# ========== Trial Management ========== #
MAX_TRIALS = 50
completed_trials = len([t for t in study.trials if t.state == optuna.trial.TrialState.COMPLETE])
remaining_trials = MAX_TRIALS - completed_trials

if remaining_trials > 0:
    print(f"Resuming DRAGAN study: {completed_trials} completed, running {remaining_trials} more.")
    study.optimize(objective, n_trials=remaining_trials)
else:
    print(f"DRAGAN study already completed {MAX_TRIALS} trials. Skipping optimization.")

# ========== Output Best Trial ========== #
best_trial = study.best_trial
best_params_df = pd.DataFrame([best_trial.params])
best_params_df["Final FID"] = best_trial.value
best_params_df.style.background_gradient(cmap="Blues")

With reference to the Code Cell above, we are able to view the Best Hyperparameter obtained during the Optuna Tuning Study. This Hyperparameter will be extracted and be trained for a longer period of time so as to attempt to increase the performance and limit the loss for the GAN.

---
### 4.6.9 GAN Re-Training Operation

As obtained from the previous sub-section, we will be re-training the GAN using the best Hyperparameter obtained during the Optuna Tuning Study. This is to push the GAN Model to its limits and also to save Computational Power as we are not taking up prolonged periods of time tuning GAN Models which may not have any clear signs of good performance. The retraining of the GAN Model will be conducted in the Code Cell below.

In [ ]:
# ========== Extract Best Parameters from Study ========== #
best_params = study.best_trial.params

latent_dim   = best_params['latent_dim']
learning_rate = best_params['learning_rate']
beta_1        = best_params['beta_1']

print("Using Best Trial Parameters for DRAGAN:")
print(best_params)

# ========== Rebuild DRAGAN Models ========== #
generator = build_generator(latent_dim=latent_dim)
discriminator = build_discriminator()

# ========== Initialise Optimisers ========== #
gen_optimizer  = tf.keras.optimizers.Adam(learning_rate=learning_rate, beta_1=beta_1)
disc_optimizer = tf.keras.optimizers.Adam(learning_rate=learning_rate, beta_1=beta_1)

_ = gen_optimizer.apply_gradients([(tf.zeros_like(v), v) for v in generator.trainable_variables])
_ = disc_optimizer.apply_gradients([(tf.zeros_like(v), v) for v in discriminator.trainable_variables])

# ========== Reload Dataset ========== #
dataset = load_data(X_train)

# ========== Define Paths ========== #
model_name   = "dragan"
output_dir   = f"/content/drive/MyDrive/Colab Notebooks/DELE CA2 A/final_outputs_{model_name}_noAugment"
weights_path = os.path.join(output_dir, f"{model_name}_generator_final.weights.h5")
gen_ckpt     = os.path.join(output_dir, "best_gen.weights.h5")
disc_ckpt    = os.path.join(output_dir, "best_disc.weights.h5")
loss_csv     = os.path.join(output_dir, "loss_history.csv")

os.makedirs(output_dir, exist_ok=True)

# ========== Skip if Already Trained ========== #
if os.path.exists(gen_ckpt) and os.path.exists(disc_ckpt) and os.path.exists(loss_csv):
    print("All outputs found. Skipping training...")

    generator.load_weights(gen_ckpt)
    discriminator.load_weights(disc_ckpt)
    print("Generator and Discriminator Weights Loaded.")

    loss_df = pd.read_csv(loss_csv)
    gen_loss_history  = loss_df['gen_loss'].tolist()
    disc_loss_history = loss_df['disc_loss'].tolist()
    final_gen_loss    = gen_loss_history[-1]

else:
    # ========== Train DRAGAN ========== #
    final_gen_loss, gen_loss_history, disc_loss_history = train(
        dataset=dataset,
        epochs=100,
        steps_per_epoch=1546,
        generator=generator,
        discriminator=discriminator,
        gen_optimizer=gen_optimizer,
        disc_optimizer=disc_optimizer,
        latent_dim=latent_dim,
        use_early_stopping=False,
        save_path=output_dir,
        gp_weight=10.0
    )

    # ========== Save Final Generator Weights ========== #
    generator.save_weights(weights_path)
    print("Generator Weights Saved.")

    # ========== Save Loss History ========== #
    loss_df = pd.DataFrame({
        'epoch': list(range(1, len(gen_loss_history)+1)),
        'gen_loss': gen_loss_history,
        'disc_loss': disc_loss_history
    })
    loss_df.to_csv(loss_csv, index=False)
    print(f"Loss history saved to {loss_csv}")

With reference to the Code Cell above, we are able to determine that the best GAN Model have been trained and saved successfully and we are able to proceed with the Model Evaluation in the next sub-sections where we determine how well the GAN Model performed based on various metrics and evaluation methodologies.

---
### 4.6.10 GAN Model Evaluation

In this sub-section, we will be evaluating the GAN's Generator performance on various aspects. We will be utilising different methodologies and metrics to attempt to quantify the performance of the GAN Generator while also conducting Visual Inspection on the result output of the GAN. The Evaluation will be conducted in the following sections.

---
#### 4.6.10.1 Visual Grid of Samples

In this sub-section, we will be visualising the Grid of Samples of the Synthesised Data generated by the GAN Model. A list of observations will be made before the utilisation of Quantitative Metrics to evaluate the GAN Generator's general performance. As the observations made are purely from professional opinion, it will most likely not be used as a Basis of Comparison between GAN Models due to differing opinions unless in extreme cases. The visualisation will be conducted in the Code Cell below.

In [ ]:
def plot_generated_images(generator, latent_dim, n_rows=5, n_cols=5, save_path=None, title="DRAGAN Generated Images", seed=42):
    # ----- Generate Latent Vectors ----- #
    if seed is not None:
        tf.random.set_seed(seed)
    noise = tf.random.normal([n_rows * n_cols, latent_dim])
    gen_images = generator(noise, training=False)

    # ----- Rescale from [-1, 1] to [0, 1] ----- #
    gen_images = (gen_images + 1.0) / 2.0
    gen_images = tf.clip_by_value(gen_images, 0.0, 1.0)

    # ----- Shape Check ----- #
    assert gen_images.shape[-1] == 1, "Expected single-channel (grayscale) output"

    # ----- Sample Plot ----- #
    fig, axes = plt.subplots(n_rows, n_cols, figsize=(n_cols, n_rows))
    for idx, ax in enumerate(axes.flat):
        ax.imshow(gen_images[idx, :, :, 0], cmap='gray')
        ax.axis('off')

    plt.suptitle(title, fontsize=16)
    plt.tight_layout()

    if save_path:
        plt.savefig(save_path, dpi=300)
        print(f"Saved Generated Image Grid to: {save_path}")

    plt.show()

# ========== Call Plot Function ========== #
plot_generated_images(generator, latent_dim=100, n_rows=10, n_cols=16)

With reference to the output of the Visual Grid of Samples above, we are able to make the following observations:

**Basic Structure of Letters is Somewhat Captured**

- Several characters resemble familiar letters such as "E", "A", "R", "L", or "P".

- Stroke alignment and proportions are generally consistent with handwritten EMNIST characters.

**High Blur and Noise**

- A few samples exhibit faint ghosting or residual noise in the black background.

- Some strokes bleed slightly or show inconsistent thickness, particularly near curves.

**Poor Character Consistency**

- Certain letters like "K", "X", or "G" show unusual formation or missing segments.

- A few glyphs display extra strokes or disconnected lines that reduce interpretability.

With the observations indicated above, we will proceed to conduct the Loss Curve Analysis in the next sub-section.

---
#### 4.6.10.2 Loss Curve Over Epochs

In this sub-section, we will be plotting the GAN Loss Curve over Epochs to evaluate the training behaviour of the GAN. This curve provides insight into whether the GAN is experiencing underfitting, overfitting, or achieving a stable training dynamic between the generator and discriminator.

Although GANs do not minimise a single unified loss in the traditional sense, the trajectory of the generator and discriminator losses can indicate whether the adversarial training process is converging appropriately.

If the generator loss remains high while discriminator loss quickly drops to near-zero, this may indicate **underfitting**, where the generator is not learning effectively to produce plausible images.

If the discriminator loss increases while generator loss sharply decreases, this may suggest **overfitting** or **mode collapse**, where the generator exploits narrow weaknesses in the discriminator without general improvement.

A relatively stable oscillation or convergence between both losses typically signifies a **well-balanced** training process, where the generator and discriminator are learning in tandem.

The Potential Observations and Corresponding Action Plans are outlined below:

**Underfitting Loss Curve**

- Increase training epochs to allow generator more time to learn.

- Simplify the discriminator to reduce overpowering the generator.

- Adjust the learning rate or apply label smoothing.

- Consider adding batch normalisation in the generator.

**Overfitting or Mode Collapse Loss Curve**

- Introduce or increase dropout in the discriminator.

- Apply input noise or label flipping to the discriminator.

- Evaluate diversity of generated samples regularly.

**Balanced/Ideal Loss Curve**

- Maintain current architecture and hyperparameters.

- Proceed with full-scale image generation.

- Evaluate both qualitative (visual inspection) and quantitative metrics (e.g. Inception Score or FID).

With these potential training behaviours and action plan outlined, we will proceed to visualise the GAN loss dynamics in the code cell below.

In [ ]:
# =========== Figure Size Configuration =========== #
plt.figure(figsize=(10, 6))

# =========== Plot Training Losses =========== #
plt.plot(gen_loss_history, label="Generator Loss", linewidth=2, color='blue')
plt.plot(disc_loss_history, label="Discriminator Loss", linewidth=2, color='red')

# =========== Labels and Title =========== #
plt.xlabel("Epoch", fontsize=12)
plt.ylabel("Loss", fontsize=12)
plt.title("DRAGAN Training Loss Curve (Non-Augmented)", fontsize=16)

# =========== Grid, Legend, and Styling =========== #
plt.grid(True, linestyle='--', alpha=0.6)
plt.legend(fontsize=12)
plt.xticks(fontsize=10)
plt.yticks(fontsize=10)

# =========== Plot Display =========== #
plt.tight_layout()
plt.show()

With reference to the Loss Curve above, we are able to make the following observations:

- DRAGAN demonstrates initial instability, but achieves excellent long-term stability.

- Generator and discriminator converge to a balanced state, reflecting effective use of gradient penalty.

- Even without augmentation, DRAGAN handles the non-augmented data well and avoids erratic loss behaviour.

With the observations indicated above, we will proceed with conducting the Quantitative Metrics Evaluation in the next sub-section.

---
#### 4.6.10.3 Quantitative Metrics Evaluation

In this sub-section we will be utilising Quantitative Metrics such as FID and KID to tabulate the results of the performance and quantify the performance of the GAN Generator. We will subsequently be use these metrics to conduct inter-model evaluation after tuning and training all the GANs. The metrics that we will be utilising and their respective formulas are indicated below:

---
**Fréchet Inception Distance**

Purpose: Measures the Distance between the Real-Image and Generated-Image Distributions in Inception Feature Space.

$$
\mathrm{FID} \;=\;\|\mu_r - \mu_f\|^2
\;+\;\mathrm{Tr}\Bigl(\Sigma_r + \Sigma_f - 2\,(\Sigma_r\,\Sigma_f)^{\tfrac12}\Bigr)
$$


Where:
- $\mu_r = \mathbb{E}[f(x)]$, $\Sigma_r = \mathrm{Cov}[f(x)]$ for Real Images $x$.  
- $\mu_f = \mathbb{E}[f(\hat x)]$, $\Sigma_f = \mathrm{Cov}[f(\hat x)]$ for Generated Images $\hat x$.  
- $f(\cdot)$ = Map from Image to its InceptionV3 'pooling=avg' Features.  

---
**Diversity (t-SNE Spread)**

Purpose: Quantifies how 'wide' the 2D t-SNE Embedding of Generated Samples.

$$
\mathrm{Spread}
\;=\;
\bigl(\max_i\,z_i^{(1)} - \min_i\,z_i^{(1)}\bigr)
\;\times\;
\bigl(\max_i\,z_i^{(2)} - \min_i\,z_i^{(2)}\bigr)
$$

Where:
- $z_i = (z_i^{(1)}, z_i^{(2)})$ = 2-dimensional t-SNE Embedding of $i$th Generated Image.

---
**Mode Collapse Risk**

Purpose: Flags when too many Generated Images are Nearly Identical (mode collapse).

$$
\text{ModeCollapseRisk} =
\begin{cases}
\text{Low}, & \dfrac{\bigl|\{\mathrm{unique\_rounded}(x_i)\}\bigr|}{N} > \tau,\\
\text{High}, & \text{otherwise}.
\end{cases}
$$

Where:
- $x_i$ = $N$ Generated Samples.  
- $\mathrm{unique\_rounded}(x_i)$ = Rounds Pixels to Detect Duplicates.  
- $\tau=0.9$ = Uniqueness Threshold (90%).

---
**Perceptual Path Length**

Purpose: Measures Sensitively of Generator's Outputs (in VGG16 Feature Space) move when the Latent Code $z$ is Perturbed.

$$
\mathrm{PPL}
\;=\;
\mathbb{E}_{z,\delta z}\Bigl[\,
\|\phi\bigl(G(z + \epsilon\,\delta z)\bigr)\;-\;\phi\bigl(G(z)\bigr)\|_2^2
\Bigr]
$$

Where:  
- $G(z)$ = GAN Generator Mapping $z\in\mathbb{R}^{\mathrm{latent\_dim}}$ to an Image.  
- $\phi(\cdot)$ = VGG16 'pooling=avg' Feature Extractor.  
- $\delta z\sim\mathcal{N}(0,I)$, $\epsilon$ = Small Constant.

---
**Kernel Inception Distance**

Purpose: An Unbiased Estimator of the Squared Maximum Mean Discrepancy (MMD) between Real and Generated Inception Features.

$$
\mathrm{KID}
\;=\;
\frac{1}{m(m-1)}\sum_{i\neq j} k\bigl(\phi(x_i),\phi(x_j)\bigr)
\;+\;
\frac{1}{n(n-1)}\sum_{i\neq j} k\bigl(\phi(\hat x_i),\phi(\hat x_j)\bigr)
\;-\;
\frac{2}{mn}\sum_{i=1}^m\sum_{j=1}^n k\bigl(\phi(x_i),\phi(\hat x_j)\bigr)
$$

Where:  
- $x_i$ ($i=1\ldots m$) = Real Images.
- $\hat x_j$ ($j=1\ldots n$) = Generated Images.  
- $\phi(\cdot)$ = InceptionV3 Feature Extractor (pooling='avg').  
- $k(u,v) = \bigl(\frac{u^\top v}{d}+1\bigr)^3$ = degree-3 Polynomial Kernel on $d$-dimensional Features.

---

With the mathematical and metrics indicated above, we will proceed to conduct the Quantitave Metrics Evalution in the Code Cell below.

In [ ]:
# ========== Preload InceptionV3 and VGG16 Models ========== #
inception = InceptionV3(include_top=False, pooling='avg', input_shape=(299, 299, 3))
vgg_model = VGG16(include_top=False, weights='imagenet', input_shape=(128, 128, 3))

# ========== Preprocess for Inception ========== #
def preprocess_images_for_inception(images):
    images = (images + 1.0) * 127.5  # Scale from [-1, 1] to [0, 255]
    images = tf.image.resize(images, [299, 299])
    if images.shape[-1] == 1:
        images = tf.image.grayscale_to_rgb(images)
    return preprocess_input(images)

# ========== Compute FID ========== #
def calculate_fid(real_images, fake_images, batch_size=50):
    real_pp = preprocess_images_for_inception(real_images)
    fake_pp = preprocess_images_for_inception(fake_images)

    act1 = inception.predict(real_pp, batch_size=batch_size, verbose=0)
    act2 = inception.predict(fake_pp, batch_size=batch_size, verbose=0)

    mu1, sigma1 = np.mean(act1, axis=0), np.cov(act1, rowvar=False)
    mu2, sigma2 = np.mean(act2, axis=0), np.cov(act2, rowvar=False)

    diff = mu1 - mu2
    covmean = sqrtm(sigma1 @ sigma2)
    if np.iscomplexobj(covmean):
        covmean = covmean.real

    fid = diff @ diff + np.trace(sigma1 + sigma2 - 2 * covmean)
    return round(fid, 4)

# ========== Compute KID ========== #
def polynomial_kernel(X, Y):
    d = X.shape[1]
    return (np.dot(X, Y.T) / d + 1) ** 3

def calculate_kid(real_images, fake_images, batch_size=128):
    real_pp = preprocess_images_for_inception(real_images)
    fake_pp = preprocess_images_for_inception(fake_images)

    real_features = inception.predict(real_pp, batch_size=batch_size, verbose=0)
    fake_features = inception.predict(fake_pp, batch_size=batch_size, verbose=0)

    m = real_features.shape[0]
    n = fake_features.shape[0]

    k_rr = polynomial_kernel(real_features, real_features)
    k_gg = polynomial_kernel(fake_features, fake_features)
    k_rg = polynomial_kernel(real_features, fake_features)

    np.fill_diagonal(k_rr, 0)
    np.fill_diagonal(k_gg, 0)

    mmd = (k_rr.sum() / (m * (m - 1)) +
           k_gg.sum() / (n * (n - 1)) -
           2 * k_rg.mean())
    return round(mmd, 4)

# ========== Compute t-SNE Spread (Diversity) ========== #
def calculate_tsne_spread(images):
    flat = images.reshape(images.shape[0], -1)
    tsne = TSNE(n_components=2, random_state=42)
    proj = tsne.fit_transform(flat)
    x_range = proj[:, 0].max() - proj[:, 0].min()
    y_range = proj[:, 1].max() - proj[:, 1].min()
    return round(x_range * y_range, 2)

# ========== Assess Mode Collapse ========== #
def assess_mode_collapse(images):
    unique = np.unique(np.round(images), axis=0).shape[0]
    return "Low" if unique > 0.9 * images.shape[0] else "High"

# ========== Compute Perceptual Path Length (PPL) ========== #
def calculate_ppl(generator, latent_dim=160, num_samples=50, epsilon=1e-2):
    distances = []
    for _ in range(num_samples):
        z1 = tf.random.normal([1, latent_dim])
        z2 = z1 + epsilon * tf.random.normal([1, latent_dim])
        img1 = generator(z1, training=False)
        img2 = generator(z2, training=False)

        img1 = tf.image.resize(tf.image.grayscale_to_rgb((img1 + 1.0) * 127.5), (128, 128))
        img2 = tf.image.resize(tf.image.grayscale_to_rgb((img2 + 1.0) * 127.5), (128, 128))

        f1 = vgg_model(vgg_preprocess(img1))
        f2 = vgg_model(vgg_preprocess(img2))
        d = tf.reduce_mean(tf.square(f1 - f2)).numpy()
        distances.append(d)
    return round(np.mean(distances), 4)

# ========== Evaluate GAN Performance ========== #
def evaluate_gan_model(generator, latent_dim, X_val, gen_loss_history, disc_loss_history, model_name="DRAGAN"):
    # ----- Generate Fake Images ----- #
    noise = tf.random.normal([1000, latent_dim])
    fake_images = generator(noise, training=False)
    fake_images = tf.clip_by_value((fake_images + 1) / 2.0, 0.0, 1.0)
    fake_np = fake_images.numpy()

    # ----- Select and preprocess 1000 Real Images from Validation Set ----- #
    real_images = tf.convert_to_tensor(X_val[:1000], dtype=tf.float32)
    real_images = tf.clip_by_value(real_images, 0.0, 1.0)

    # ----- Compute Metrics ----- #
    fid_value = calculate_fid(real_images, fake_images)
    kid_value = calculate_kid(real_images, fake_images)
    tsne_spread_value = calculate_tsne_spread(fake_np)
    collapse_risk_value = assess_mode_collapse(fake_np)
    ppl_score_value = calculate_ppl(generator, latent_dim)
    mean_gen_loss_value = round(np.mean(gen_loss_history), 4)
    std_gen_loss_value = round(np.std(gen_loss_history), 4)
    mean_disc_loss_value = round(np.mean(disc_loss_history), 4)
    std_disc_loss_value = round(np.std(disc_loss_history), 4)

    # ----- Store as Global Vars (Optional) ----- #
    prefix = model_name.upper()
    globals()[f"{prefix}_FID"] = fid_value
    globals()[f"{prefix}_KID"] = kid_value
    globals()[f"{prefix}_TSNE"] = tsne_spread_value
    globals()[f"{prefix}_MODE_COLLAPSE"] = collapse_risk_value
    globals()[f"{prefix}_PPL"] = ppl_score_value
    globals()[f"{prefix}_GEN_LOSS_MEAN"] = mean_gen_loss_value
    globals()[f"{prefix}_GEN_LOSS_STD"] = std_gen_loss_value
    globals()[f"{prefix}_DISC_LOSS_MEAN"] = mean_disc_loss_value
    globals()[f"{prefix}_DISC_LOSS_STD"] = std_disc_loss_value

    # ----- Final Summary Row ----- #
    row = {
        "Model": model_name,
        "FID Score": fid_value,
        "KID Score": kid_value,
        "Mean Generator Loss": mean_gen_loss_value,
        "Std Generator Loss": std_gen_loss_value,
        "Mean Discriminator Loss": mean_disc_loss_value,
        "Std Discriminator Loss": std_disc_loss_value,
        "Mode Collapse Risk": collapse_risk_value,
        "Visual Quality": "Very Good",
        "Diversity (t-SNE Spread)": tsne_spread_value,
        "PPL": ppl_score_value
    }

    return pd.DataFrame([row])[[
        "Model", "FID Score", "KID Score",
        "Mean Generator Loss", "Std Generator Loss",
        "Mean Discriminator Loss", "Std Discriminator Loss",
        "Mode Collapse Risk", "Visual Quality",
        "Diversity (t-SNE Spread)", "PPL"
    ]]

# ========== Create DataFrame ========== #
df = evaluate_gan_model(generator, latent_dim, X_val, gen_loss_history, disc_loss_history, model_name="DRAGAN (Non-Augmented)")

# ========== Display DataFrame ========== #
df.style.background_gradient(cmap="Blues")

With reference to the output of the Quantitative Metrics above, we are able to determine the performance of the GAN. This will be used as a basis of comparison to the other GAN Models.

---
#### 4.6.10.4 t-SNE of Generated Samples

In this sub-section, we visualise the t-SNE projection of generated samples to evaluate the latent diversity learned by the unconditional GAN. Although our GAN is not class-conditioned, t-SNE remains a valuable tool to assess whether the generator is producing varied outputs that reflect meaningful use of the latent space. While GANs are trained in high-dimensional spaces, t-SNE enables us to observe 2D structural patterns such as sample clustering, separation, and density, which provide indirect insights into the diversity and generalisation capabilities of the generator. The visualisation will be conducted in the Code Cell below.

In [ ]:
# ========== Generate New Images from Random Noise ========== #
latent_dim = 100
noise = tf.random.normal([500, latent_dim])
generated_images = generator(noise, training=False)
generated_images = (generated_images + 1) / 2.0

def plot_tsne_embeddings(generated_images, save_path=None, title="t-SNE of DRAGAN Generated Samples (Non-Augmented)"):
    # ----- Convert to [0, 1] Range ----- #
    if tf.reduce_max(generated_images).numpy() > 1.0:
        raise ValueError("Images should be in [-1, 1] range before scaling.")

    images_rescaled = (generated_images + 1.0) / 2.0
    flat_images = images_rescaled.numpy().reshape(images_rescaled.shape[0], -1)

    # ----- Run t-SNE ----- #
    tsne = TSNE(n_components=2, perplexity=30, learning_rate='auto', init='pca', random_state=42)
    tsne_proj = tsne.fit_transform(flat_images)

    # ----- Plot t-SNE ----- #
    plt.figure(figsize=(12, 6))
    sns.scatterplot(
        x=tsne_proj[:, 0],
        y=tsne_proj[:, 1],
        s=20,
        alpha=0.9,
        edgecolor='none'
    )
    plt.title(title, fontsize=14, weight='bold')
    plt.xlabel("t-SNE Dimension 1")
    plt.ylabel("t-SNE Dimension 2")
    plt.grid(True, linestyle='--', alpha=0.3)
    plt.tight_layout()

    if save_path:
        plt.savefig(save_path, dpi=300)
        print(f"t-SNE plot saved to: {save_path}")

    plt.show()

plot_tsne_embeddings(generated_images=generated_images)

With reference to the t-SNE plot of DRAGAN generated samples (Non-Augmented) above, we are able to make the following observations:

**Stable Spread Across Both Axes**

- The points are fairly dispersed in a dense horizontal band, showing a wide latent space coverage.

- Indicates stable discriminator gradients as expected from DRAGAN's gradient penalty mechanism.

**Moderate Uniformity in Distribution**

- Clusters are minimal, and there are no significant empty zones or tight clumping.

- This hints at reasonable diversity in sample generation, without strong mode collapse.

**Less Vertical Variance**

- The vertical spread is slightly more constrained compared to WGAN variants.

- May suggest lower variability in certain abstract feature directions, possibly due to DRAGAN's local gradient constraints.

**Mild Presence of Outliers**

- A few points drift further away from the central cluster, but are not extreme.

- This implies occasional irregular samples, which could be artefacts of the DRAGAN noise regularisation.

With the observations indicated above, we will proceed to train the next GAN Model in the next sub-section.

---
## 4.7 Information Maximising GAN Model Training

In this sub-section, we will be training a Information Maximising GAN (InfoGAN) for the EMNIST Dataset. The InfoGAN is capable of giving us control and interpretability over the Image Generation process. InfoGAN is also able to learn sematically meaningful factors without the need of labels unlike Conditional GANs. The relevant mathematical formulas are indicated below:

---
**Generator Loss:**

Purpose: Produce Outputs Classified as Real.

$$
\mathcal{L}_G = \frac{1}{2} \mathbb{E}_{z \sim p_z}[(D(G(z)) - c)^2]
$$

Where:
- $c = 1$ = Desired Discriminator Output for Generated Images  
- $D(G(z))$ = Discriminator's Output for Generated Data  
- $z \sim p_z$ = Sample from Latent Space

---
**Discriminator Loss:**

Purpose: Discriminator Minimises the Error between Predictions and True Labels.

$$
\mathcal{L}_D = \frac{1}{2} \mathbb{E}_{x \sim p_{\text{data}}}[(D(x) - b)^2] + \frac{1}{2} \mathbb{E}_{z \sim p_z}[(D(G(z)) - a)^2]
$$

Where:  
- $D(x)$ = Discriminator's Output for Real Data  
- $D(G(z))$ = Discriminator's Output for Generated Data  
- $a = 0$ = Label for Fake Data  
- $b = 1$ = Label for Real Data  
- $x \sim p_{\text{data}}$ = Sample from Real Data Distribution  
- $z \sim p_z$ = Sample from Latent Noise Distribution  

---
**Latent Code Regularisation**

Purpose: Encourage Generator to Learn Disentangled, Interpretable Latent Factors by Maximising Mutual Information between Latent Code and Generated Output.

$$
\mathcal{L}_{\text{Info}} = -\lambda \cdot \mathbb{E}_{z \sim p(z), c \sim p(c)} \left[ \log Q(c \mid G(z, c)) \right]
$$

Where:
- $Q(c \mid x)$ = Auxiliary Network Approximates Posterior $P(c \mid x)$
- $G(z, c)$ = Generated Sample
- $\lambda$ = Mutual Information Weight Hyperparameter

---
**Total Objective:**

Purpose: LSGAN Seeks to Solve the Objective.

$$
\min_{G, Q} \max_D \; \mathcal{L}_{\text{InfoGAN}} = \mathcal{L}_{\text{GAN}} + \mathcal{L}_{\text{Info}}
$$

Where:
- $G$ = Generator
- $Q$ = Auxiliary Network
- $D$ = Discriminator

---
**Variational Lower Bound on Mutual Information**

Purpose: True Mutual Information $I(c; G(z, c))$ is Intractable, Maximise a Variational Lower Bound using $Q(c \mid x)$.

$$
I(c; G(z, c)) \geq \mathbb{E}_{z \sim p(z), c \sim p(c)} \left[ \log Q(c \mid G(z, c)) \right] + H(c)
$$
Where:
- $I(c; G(z, c))$ = Mutual Information
- $H(c)$ = Entropy of $c$ (constant if $p(c)$ is fixed)

---
With the relevant mathematical formulas indicated above, we will proceed to train the InfoGAN in this sub-section for the purpose of it's potential for smoother training process and also to serve as a comparison for our other GAN Models.

---
### 4.7.1 Defining Data Pre-Processing Function

In this sub-section, we will be defining the Data Pre-Processing Function which will allow for consistent Data Pipeline to be built and modifed accordingly should the need arise such as when we need to add Spectral Normalisation. This will ensure that the Data's integrity is not affected and we are able to make fair and un-biased assumptions. The Data Pre-Processing Function is defined in the Code Cell below:

In [ ]:
# =========== Data Pre-Processing Function =========== #
def load_data(X_train, batch_size=32):
    buffer_size = X_train.shape[0]

    dataset = tf.data.Dataset.from_tensor_slices(X_train)
    dataset = dataset.shuffle(buffer_size)
    dataset = dataset.batch(batch_size)
    dataset = dataset.prefetch(tf.data.AUTOTUNE)

    return dataset

With reference to the Code Cell above, we are able to determine that the Data Pre-Processing Function have been defined successfully and we are able to move on to Defining Callback Functions in the next sub-section.

---
### 4.7.2 Defining Callback Functions

In this sub-section, we will be pre-defining the various Callbacks. This is to ensure consistency throughout the model and also to increase the GAN Accuracy and optimise the Computation Cost for training each GAN Model. These Callbacks will be main used during the GAN Training in the next sub-section. The revelant mathematical formula for the callbacks and logic are indicated below:

---
**Learning Rate Scheduler:**

Purpose: Reduces Learning Rate During Training to Fine-tune Model Convergence.

$$\text{If } epoch \mod 10 = 0 \Rightarrow \eta_{new} = \frac{1}{2} \cdot \eta$$

Where:

- $\eta$ = current learning rate

- $\eta_{\text{new}}$ = updated learning rate

- $epoch \bmod 10$ checks if the epoch is a multiple of 10

---
**ReduceLROnPlateau:**

Purpose: Automatically Reduces the Learning Rate when Validation Performance Plateaus.

$$
\text{If no improvement in } val\_loss \text{ for 10 epochs: } \eta_{\text{new}} = 0.5 \cdot \eta
$$

Where:

- $\eta_{\text{new}}$ = reduced learning rate

---
**EarlyStopping:**

Purpose: Prevents Wastage of Computational Power if Model Does Not Improve.

$$
\text{If } val\_loss \text{ does not improve for 25 epochs, stop training and restore best weights}
$$

---

We will proceed to Define the Callbacks in the Code Cell below in preparation for the GAN Model Training.

In [ ]:
# =========== Define Learning Rate Scheduler =========== #
lr_scheduler = LearningRateScheduler(
    lambda epoch, lr: lr * 0.95 if epoch % 10 == 0 else lr,
    verbose=1
)

# =========== Define Reduce LR on Plateau =========== #
reduce_lr = ReduceLROnPlateau(
    monitor='loss',
    factor=0.5,
    patience=10,
    verbose=1,
    min_lr=1e-6
)

# =========== Define Early Stopping =========== #
early_stop = EarlyStopping(
    monitor='loss',
    patience=25,
    verbose=1,
    restore_best_weights=False
)

# =========== Define Display Generated Images =========== #
def display_generated_images(generator, latent_dim, code_dim=2, n=7):
    total = n * n
    z = tf.random.normal([total, latent_dim])
    c = tf.random.uniform([total, code_dim], minval=-1.0, maxval=1.0)

    # ----- Normalise Both z and c ----- #
    z = tf.math.l2_normalize(z, axis=1)
    c = tf.math.l2_normalize(c, axis=1)

    zc = tf.concat([z, c], axis=1)
    generated_images = generator(zc, training=False)
    generated_images = (generated_images + 1.0) / 2.0

    fig, axes = plt.subplots(n, n, figsize=(n, n))
    for i in range(n):
        for j in range(n):
            img = generated_images[i * n + j, :, :, 0]
            axes[i, j].imshow(img, cmap='gray')
            axes[i, j].axis('off')

    plt.tight_layout()
    plt.show()

# =========== Define FID Calculator =========== #
def calculate_fid(real_images, fake_images, batch_size=10):

    # 1) Scale to [0,255] and resize to 299×299
    real = (real_images + 1.0) * 127.5
    fake = (fake_images + 1.0) * 127.5
    real = tf.image.resize(real, (299,299))
    fake = tf.image.resize(fake, (299,299))

    # ----- If Grayscale, Convert to RGB ----- #
    if real.shape[-1] == 1:
        real = tf.image.grayscale_to_rgb(real)
        fake = tf.image.grayscale_to_rgb(fake)

    # ----- Preprocess for Inception (–1 to +1) ----- #
    real_pp = preprocess_input(real)
    fake_pp = preprocess_input(fake)

    # ----- Extract Features in Batches ----- #
    def _get_acts(x):
        acts = []
        n = x.shape[0]
        for i in range(0, n, batch_size):
            chunk = x[i:i+batch_size]
            acts.append(_inception_model(chunk, training=False).numpy())
        return np.vstack(acts)

    act_real = _get_acts(real_pp)
    act_fake = _get_acts(fake_pp)

    # ----- Compute Statistics ----- #
    mu1, sigma1 = act_real.mean(axis=0), np.cov(act_real, rowvar=False)
    mu2, sigma2 = act_fake.mean(axis=0), np.cov(act_fake, rowvar=False)
    diff    = mu1 - mu2
    covmean = sqrtm(sigma1.dot(sigma2))
    if np.iscomplexobj(covmean):
        covmean = covmean.real
    fid = diff.dot(diff) + np.trace(sigma1 + sigma2 - 2*covmean)
    return float(np.round(fid,4))

# =========== Confirmation Message =========== #
print("InfoGAN Callbacks and Visualisation Setup Complete")

With reference to the Code Cell above, we are able to verify that that the Callbacks have been successfully defined and the various parameters are set so as to ensure a High Accuracy and Computing Efficient GAN Model Training.

---
### 4.7.3 Defining InfoGAN Generator Architecture

In this section, we will be defining the InfoGAN's Generator Architecture. The Generator will be trying to make realistic generated images in the attempt to bypass the Discriminator. Subsequently, the generated images from the Generator will be evaluated based on a few evaluation metrics such as Inception Score. The relevent mathematical formulas for the Loss Function of the Vanilla GAN Generator is indicated below:

---
**Generator Loss:**

Purpose: Produce Outputs Classified as Real.

$$
\mathcal{L}_G = \frac{1}{2} \mathbb{E}_{z \sim p_z}[(D(G(z)) - c)^2]
$$

Where:
- $c = 1$ = Desired Discriminator Output for Generated Images  
- $D(G(z))$ = Discriminator's Output for Generated Data  
- $z \sim p_z$ = Sample from Latent Space

---
With the relevant Generator Loss Formula indicated above, we will proceed to define the GAN Generator's Architecture in the Code Cell below and the Lost Function.

In [ ]:
# =========== Generator Function =========== #
def build_generator(latent_dim=100, code_dim=2, use_dropout=True):
    input_dim = latent_dim + code_dim
    model = tf.keras.Sequential(name="generator")

    # ----- Project and Reshape ----- #
    model.add(tf.keras.layers.Dense(7 * 7 * 256, use_bias=False, input_shape=(input_dim,)))
    model.add(tf.keras.layers.BatchNormalization())
    model.add(tf.keras.layers.ReLU())
    model.add(tf.keras.layers.Reshape((7, 7, 256)))

    # ----- Upsample 1 ----- #
    model.add(tf.keras.layers.Conv2DTranspose(128, kernel_size=5, strides=1, padding='same', use_bias=False))
    model.add(tf.keras.layers.BatchNormalization())
    model.add(tf.keras.layers.ReLU())
    if use_dropout:
        model.add(tf.keras.layers.Dropout(0.3))

    # ----- Upsample 2 ----- #
    model.add(tf.keras.layers.Conv2DTranspose(64, kernel_size=5, strides=2, padding='same', use_bias=False))
    model.add(tf.keras.layers.BatchNormalization())
    model.add(tf.keras.layers.ReLU())
    if use_dropout:
        model.add(tf.keras.layers.Dropout(0.3))

    # ----- Final Output to 28x28x1 ----- #
    model.add(tf.keras.layers.Conv2DTranspose(1, kernel_size=5, strides=2, padding='same', activation='tanh'))

    return model

# =========== Generator Loss =========== #
bce = tf.keras.losses.BinaryCrossentropy(from_logits=True)

def generator_loss(fake_output):
    return bce(tf.ones_like(fake_output), fake_output)

With reference to the Code Cell above, we are able to verify that the GAN's Generator Architecture have been successfully defined and we are able to proceed to define the GAN's Discriminator Architecture.

---
### 4.7.4 Defining InfoGAN Discriminator Architecture

In this section, we will be defining the Architecture of the Discriminator of the GAN. The Discriminator will be the Neural Network that will attempt to tell the Generated Images apart from the Actual Images. This will allow the GAN to improve overtime and also improve itself overtime. The mathematical function for the Discriminator Loss Function is indicated below:

---
**Discriminator Loss:**

Purpose: Discriminator Minimises the Error between Predictions and True Labels.

$$
\mathcal{L}_D = \frac{1}{2} \mathbb{E}_{x \sim p_{\text{data}}}[(D(x) - b)^2] + \frac{1}{2} \mathbb{E}_{z \sim p_z}[(D(G(z)) - a)^2]
$$

Where:  
- $D(x)$ = Discriminator's Output for Real Data  
- $D(G(z))$ = Discriminator's Output for Generated Data  
- $a = 0$ = Label for Fake Data  
- $b = 1$ = Label for Real Data  
- $x \sim p_{\text{data}}$ = Sample from Real Data Distribution  
- $z \sim p_z$ = Sample from Latent Noise Distribution  
---

With the mathematical formula for the Loss Function indicated above, we will proceed to define the GAN's Discriminator Architecture and the Loss Function in the Code Cell below.

In [ ]:
# =========== Discriminator Function =========== #
def build_discriminator(input_shape=(28, 28, 1), code_dim=2):
    image_input = tf.keras.Input(shape=input_shape)

    # ----- Shared Convolutional Layers ----- #
    x = tf.keras.layers.Conv2D(64, kernel_size=5, strides=2, padding='same')(image_input)
    x = tf.keras.layers.LeakyReLU(0.2)(x)
    x = tf.keras.layers.Dropout(0.3)(x)

    x = tf.keras.layers.Conv2D(128, kernel_size=5, strides=2, padding='same')(x)
    x = tf.keras.layers.LeakyReLU(0.2)(x)
    x = tf.keras.layers.Dropout(0.3)(x)

    x = tf.keras.layers.Flatten()(x)

    # ----- Output 1: Real/Fake (Adversarial) ----- #
    real_fake_output = tf.keras.layers.Dense(1, name="real_fake")(x)

    # ----- Output 2: Q-network for Predicting Latent Code ----- #
    code_output = tf.keras.layers.Dense(code_dim, name="code")(x)

    return tf.keras.Model(inputs=image_input, outputs=[real_fake_output, code_output], name="discriminator_with_Q")

# =========== Discriminator Loss =========== #
bce = tf.keras.losses.BinaryCrossentropy(from_logits=True)

def discriminator_loss(real_output, fake_output):
    real_labels = tf.ones_like(real_output)
    fake_labels = tf.zeros_like(fake_output)

    real_loss = bce(real_labels, real_output)
    fake_loss = bce(fake_labels, fake_output)

    return real_loss + fake_loss

With reference to the Code Cell above, we are able to verify that the GAN's Discriminator Architecture have been successfully defined and we are able to proceed to define the GAN's Training Steps.

---
### 4.7.5 Defining Training Step

In this section, we will be Defining the Training Step function so as to train both the Generator and the Discriminator at the same time. The Training Step will compute the gradients for both the Generator and the Discriminator at the same time. The relevant mathematical formulas for the Training Step is indicated below:

---
**Generator Loss:**

Purpose: Produce Outputs Classified as Real.

$$
\mathcal{L}_G = \frac{1}{2} \mathbb{E}_{z \sim p_z}[(D(G(z)) - c)^2]
$$

Where:
- $c = 1$ = Desired Discriminator Output for Generated Images  
- $D(G(z))$ = Discriminator's Output for Generated Data  
- $z \sim p_z$ = Sample from Latent Space

---
**Discriminator Loss:**

Purpose: Discriminator Minimises the Error between Predictions and True Labels.

$$
\mathcal{L}_D = \frac{1}{2} \mathbb{E}_{x \sim p_{\text{data}}}[(D(x) - b)^2] + \frac{1}{2} \mathbb{E}_{z \sim p_z}[(D(G(z)) - a)^2]
$$

Where:  
- $D(x)$ = Discriminator's Output for Real Data  
- $D(G(z))$ = Discriminator's Output for Generated Data  
- $a = 0$ = Label for Fake Data  
- $b = 1$ = Label for Real Data  
- $x \sim p_{\text{data}}$ = Sample from Real Data Distribution  
- $z \sim p_z$ = Sample from Latent Noise Distribution  

---
**Total Objective:**

Purpose: LSGAN Seeks to Solve the Objective.

$$
\min_G \max_D \; \mathcal{L}_D \quad \text{and} \quad \min_G \; \mathcal{L}_G
$$

---

With the relevant mathematical formulas indicated above, we will proceed to define the Training Step in the Code Cell below.

In [ ]:
# =========== Training Step Function =========== #
@tf.function
def train_step(real_images, generator, discriminator, gen_optimizer, disc_optimizer, latent_dim, code_dim=2, lambda_mi=1.0):
    batch_size = tf.shape(real_images)[0]

    # ----- Sample & Normalisation ----- #
    z = tf.random.normal([batch_size, latent_dim])
    c = tf.random.uniform([batch_size, code_dim], -1.0, 1.0)
    z = tf.math.l2_normalize(z, axis=1)
    c = tf.math.l2_normalize(c, axis=1)
    zc = tf.concat([z, c], axis=1)

    # ----- Instance Noise on Real Images ----- #
    real_noisy = real_images + tf.random.normal(tf.shape(real_images)) * 0.05

    with tf.GradientTape(persistent=True) as tape:
        # ----- Generate Fake Images ----- #
        fake_images = generator(zc, training=True)

        # ----- Discriminator and Q-network ----- #
        real_logits, _ = discriminator(real_noisy,  training=True)
        fake_logits, predicted_c = discriminator(fake_images, training=True)

        # ----- Adversarial Losses with Label Smoothing ----- #
        real_labels = tf.ones_like(real_logits) * 0.9
        fake_labels = tf.zeros_like(fake_logits)
        d_real_loss = bce(real_labels, real_logits)
        d_fake_loss = bce(fake_labels,  fake_logits)
        disc_loss   = d_real_loss + d_fake_loss

        # ----- Generator Adversarial Loss ----- #
        adv_loss = bce(tf.ones_like(fake_logits), fake_logits)

        # ----- Mutual-Information Loss (MSE) ----- #
        mi_loss  = tf.reduce_mean(tf.square(predicted_c - c))

        # ----- Total Generator and Q Loss ----- #
        gen_loss = adv_loss + lambda_mi * mi_loss

    # ----- Update Discriminator ----- #
    code_vars  = [v for v in discriminator.trainable_variables if 'code/' in v.name]
    adv_vars   = [v for v in discriminator.trainable_variables if v not in code_vars]
    disc_grads = tape.gradient(disc_loss, adv_vars)
    disc_optimizer.apply_gradients(zip(disc_grads, adv_vars))

    # ----- Update Generator and Q-network ----- #
    gq_vars  = generator.trainable_variables + code_vars
    gq_grads = tape.gradient(gen_loss, gq_vars)
    gen_optimizer.apply_gradients(zip(gq_grads, gq_vars))

    del tape
    return gen_loss, disc_loss, adv_loss, mi_loss

With reference to the Code Cell above, we are able to determine that the Training Step have been successfully defined and we are able to proceed to prepare the Training Loop in the next section.

---
### 4.7.6 Defining Training Loop

In this section, we will be defining the Training Loop for the GAN Training. The Training Loop will go through each of the Epoch and train both the Generator and Discriminator simultaneously. The Training Loop will also track the Loss for both the Generator and the Discriminator for each of the Epoch trained. The relevant mathematical formulas are indicated below:

---
**Epoch Loss Averaging:**

Purpose: These are the Average Generator and Discriminator Losses over all Batches in Epoch $t$.

$$
\bar{L}_G^{(t)} = \frac{1}{N} \sum_{i=1}^{N} L_G^{(i)} \\
\bar{L}_D^{(t)} = \frac{1}{N} \sum_{i=1}^{N} L_D^{(i)}
$$

Where:
- $\bar{L}_G^{(t)}$ = Average Generator Loss at Epoch $t$  
- $\bar{L}_D^{(t)}$ = Average Discriminator Loss at Epoch $t$  
- $N$ = Number of Batches in the Dataset  
- $L_G^{(i)}$ = Generator Loss on Batch $i$  
- $L_D^{(i)}$ = Discriminator Loss on Batch $i$

---
**Epoch Iteration:**

Purpose: Describes the Training Loop Logic: for Each Epoch $t$, perform `train_step` for Every Mini-batch.

$$
\text{for } t = 1 \text{ to } T:
\quad \text{for each batch } (x^{(i)}):
\quad \text{train_step}(x^{(i)})
$$

Where:
- $T$: Total Number of Epochs  
- $x^{(i)}$: Real Batch $i$ from the Dataset  
- `train_step`: Function that Updates $G$ and $D$ using that Batch

---
**Generated Image Output:**

Purpose: Sample Random Noise $z$ and Generate Synthetic Image $\hat{x}$ from the Generator. This is used for Visual Monitoring of Model Quality.

$$
z \sim p_z(z) \\
\hat{x} = G(z)
$$

Where:
- $z$: Random Latent Vector  
- $G(z)$: Generated Image from Generator

---
With the mathematical formulas indicated above, we will proceed to define the Training Loop in the Code Cell below in preparation of the GAN Training.

In [ ]:
# =========== Training Loop Function =========== #
def train(dataset, epochs, generator, discriminator, gen_optimizer, disc_optimizer, latent_dim, code_dim=2, lambda_mi=1.0, save_path=None, use_early_stopping=True, steps_per_epoch=None):
    # ----- Prepare FID Reference Set from Validation Data X_val ----- #
    real_val = tf.data.Dataset.from_tensor_slices(X_val).batch(50).take(1)

    # ----- Latent Noise and Code for InfoGAN FID Evaluation ----- #
    noise_val = tf.random.normal([50, latent_dim])
    noise_val = tf.math.l2_normalize(noise_val, axis=1)
    code_val = tf.random.uniform([50, code_dim], minval=-1.0, maxval=1.0)
    code_val = tf.math.l2_normalize(code_val, axis=1)
    zc_val = tf.concat([noise_val, code_val], axis=1)

    # ----- FID Early-Stopping Parameters for InfoGAN ----- #
    best_fid = float('inf')
    no_improve = 0
    patience = 10
    fid_interval = 1

    # ----- Dummy Model ----- #
    dummy_input  = tf.keras.Input(shape=(1,))
    dummy_output = tf.keras.layers.Lambda(lambda x: x)(dummy_input)
    dummy_model  = tf.keras.Model(dummy_input, dummy_output)
    dummy_model.compile(optimizer=tf.keras.optimizers.Adam(1e-4), loss='mse')

    early_stop.set_model(dummy_model)
    reduce_lr.set_model(dummy_model)
    lr_scheduler.set_model(dummy_model)
    early_stop.on_train_begin({})
    reduce_lr.on_train_begin({})
    lr_scheduler.on_train_begin({})

    # ----- Track Loss History ----- #
    gen_loss_history  = []
    disc_loss_history = []
    mi_loss_history   = []

    for epoch in range(1, epochs + 1):
        print(f"\nEpoch {epoch}/{epochs}")
        gen_losses, disc_losses, mi_losses = [], [], []

        # ----- Per-Batch Training ----- #
        for i, real_images in enumerate(dataset):
            if steps_per_epoch and i >= steps_per_epoch:
                break

            # ----- InfoGAN Train Step ----- #
            gen_loss, disc_loss, adv_loss, mi_loss = train_step(
                real_images,
                generator,
                discriminator,
                gen_optimizer,
                disc_optimizer,
                latent_dim,
                code_dim=code_dim,
                lambda_mi=lambda_mi
            )

            gen_losses.append(gen_loss)
            disc_losses.append(disc_loss)
            mi_losses.append(mi_loss)

        # ----- Compute Average Losses Epoch ----- #
        avg_gen_loss  = tf.reduce_mean(gen_losses)
        avg_disc_loss = tf.reduce_mean(disc_losses)
        avg_mi_loss   = tf.reduce_mean(mi_losses)
        print(f"Gen Loss: {avg_gen_loss:.4f} | "
              f"Disc Loss: {avg_disc_loss:.4f} | "
              f"MI Loss: {avg_mi_loss:.4f}")

        # ----- Save to History ----- #
        gen_loss_history.append(float(avg_gen_loss))
        disc_loss_history.append(float(avg_disc_loss))
        mi_loss_history.append(float(avg_mi_loss))

        # ----- Callback Updates ----- #
        logs = {'loss': float(avg_gen_loss)}
        if use_early_stopping:
            early_stop.on_epoch_end(epoch=epoch, logs=logs)
            if early_stop.stopped_epoch > 0:
                print(f"Early stopping triggered at epoch {epoch}")
                break
        reduce_lr.on_epoch_end(epoch=epoch, logs=logs)
        lr_scheduler.on_epoch_end(epoch=epoch, logs=logs)

        # ----- FID Evaluation & Early Stopping ----- #
        if epoch % fid_interval == 0:
            fake_val = generator(zc_val, training=False)
            real_batch = next(iter(real_val))
            real_for_fid = tf.image.resize(real_batch, [299, 299]) * 0.5 + 0.5
            fake_for_fid = tf.image.resize(fake_val,    [299, 299]) * 0.5 + 0.5
            fid_value = calculate_fid(real_for_fid, fake_for_fid)

            print(f"Epoch {epoch} → Val FID: {fid_value:.2f}")

            # ----- Early-stop Logic on FID ----- #
            if fid_value < best_fid:
                best_fid   = fid_value
                no_improve = 0
                if save_path:
                    os.makedirs(save_path, exist_ok=True)
                    generator.save_weights(os.path.join(save_path, "best_gen.weights.h5"))
                    discriminator.save_weights(os.path.join(save_path, "best_disc.weights.h5"))
            else:
                no_improve += 1

            if no_improve >= patience:
                print(f"No FID improvement for {patience} epochs, stopping.")
                break

        # ----- Display Images Every 5 Epochs ----- #
        if epoch % 5 == 0:
            display_generated_images(generator, latent_dim)

    # ----- Restore Best Weights After Training ----- #
    if save_path:
        best_gen_path = os.path.join(save_path, "best_gen.weights.h5")
        best_disc_path = os.path.join(save_path, "best_disc.weights.h5")
        if os.path.exists(best_gen_path) and os.path.exists(best_disc_path):
            generator.load_weights(best_gen_path)
            discriminator.load_weights(best_disc_path)
            print("Restored best generator and discriminator weights based on lowest FID.")
        else:
            print("Best weight files not found. Skipping restore.")

    return avg_gen_loss, gen_loss_history, disc_loss_history, mi_loss_history

With reference to the Code Cell above, we are able to determine that the Training Loop have been successfully defined and we are able to proceed to define the Optimisers with Tunable Parameter in preparation for the Optuna Tuning.

---
### 4.7.7 Defining Optimisers with Tunable Parameters

In this section, we will be defining the Common Optimisers that will be Tuned using the Optuna Tuning. The Optimisers will each have a range of values in a list so as to allow for the Optuna to search for the best Hyperparameter for the GAN. Through these parameters, it will affect how effective and efficiently the GAN will learn from the provided EMNIST Data. The relevent mathematical formula is indicated below:

---
**Optimiser Update Rule**

Purpose: Parameter Update Step in the Adam Optimiser, using Adaptive Learning Rates with Momentum.

$$
\theta \leftarrow \theta - \eta \cdot \frac{m_t}{\sqrt{v_t} + \epsilon}
$$

Where:
- $\theta$ = Model Parameters (Generator or Discriminator)
- $\eta$ = Learning Rate (Tuned)
- $m_t$ = First Moment Estimate (Mean of Gradients)
- $v_t$ = Second Moment Estimate (Variance of Gradients)
- $\epsilon$ = Small Constant for Numerical Stability

---
With the mathematical formula indicated above, we will proceed to define the Optimisers in the Code Cell.

In [ ]:
def objective(trial):
    # ----- Hyperparameters ----- #
    latent_dim = trial.suggest_categorical('latent_dim', [100, 128, 160])
    learning_rate = trial.suggest_float('learning_rate', 5e-5, 2e-4, log=True)
    beta_1 = trial.suggest_float('beta_1', 0.4, 0.6)
    code_dim = 2
    lambda_mi = 1.0

    # ----- Build Models ----- #
    generator = build_generator(latent_dim=latent_dim, code_dim=code_dim)
    discriminator = build_discriminator(code_dim=code_dim)

    # ----- Trigger Weight Creation ----- #
    z = tf.random.uniform([1, latent_dim], -1.0, 1.0)
    c = tf.random.uniform([1, code_dim], -1.0, 1.0)
    zc = tf.concat([z, c], axis=1)
    _ = generator(zc)
    _ = discriminator(tf.random.normal([1, 28, 28, 1]))

    # ----- Define Optimizers ----- #
    gen_optimizer = tf.keras.optimizers.Adam(learning_rate=learning_rate, beta_1=beta_1)
    disc_optimizer = tf.keras.optimizers.Adam(learning_rate=learning_rate, beta_1=beta_1)

    # ----- Prebuild Optimizers ----- #
    _ = gen_optimizer.apply_gradients([(tf.zeros_like(v), v) for v in generator.trainable_variables])
    _ = disc_optimizer.apply_gradients([(tf.zeros_like(v), v) for v in discriminator.trainable_variables])

    # ----- Load Data ----- #
    dataset = load_data(X_train, batch_size=64)
    steps = min(1000, len(X_train) // 64)

    # ----- Train InfoGAN ----- #
    _ = train(
        dataset=dataset,
        epochs=100,
        generator=generator,
        discriminator=discriminator,
        gen_optimizer=gen_optimizer,
        disc_optimizer=disc_optimizer,
        latent_dim=latent_dim,
        code_dim=code_dim,
        lambda_mi=lambda_mi,
        steps_per_epoch=steps,
        use_early_stopping=False,
        save_path=None
    )

    # ----- Evaluate FID Score ----- #
    noise = tf.random.uniform([1000, latent_dim], minval=-1.0, maxval=1.0)
    code = tf.random.uniform([1000, code_dim], minval=-1.0, maxval=1.0)
    latent = tf.concat([noise, code], axis=1)
    fake_images = generator(latent, training=False)
    fake_images = (fake_images + 1.0) / 2.0
    real_images = X_val[:1000]
    fid_score = calculate_fid(real_images, fake_images)

    return fid_score

With reference to the Code Cell above, we are able to determine that the Optimiser have been successfully defined with a range of different Parameter Values so as to provide a robust Optuna Tuning in the subsequent sections.

---
### 4.7.8 Optuna Tuning Study

In this sub-section, we will be conducting the Optuna Tuning Study on the GAN Model so as to obtain the best Hyperparameter for the GAN Model. It is expected that the Tuning takes a prolonged duration due to the nature of how the GAN works and the 2 Neural Networks involved. However, the training data will be stored so as to prevent re-running of repetitive codes. The Optuna Tuning Study will be conducted in the Code Cell below.

In [ ]:
# ========== Create or Load Study with SQLite Backend ========== #
study = optuna.create_study(
    direction="minimize",
    study_name="infogan_tuning",
    storage="sqlite:////content/drive/MyDrive/Colab Notebooks/DELE CA2 A/Non-Augmented GAN Tunings/infogan_optuna.db",
    load_if_exists=True
)

# ========== Trial Management ========== #
MAX_TRIALS = 50
completed_trials = len([t for t in study.trials if t.state == optuna.trial.TrialState.COMPLETE])
remaining_trials = MAX_TRIALS - completed_trials

if remaining_trials > 0:
    print(f"Resuming InfoGAN study: {completed_trials} completed, running {remaining_trials} more.")
    study.optimize(objective, n_trials=remaining_trials)
else:
    print(f"InfoGAN study already completed {MAX_TRIALS} trials. Skipping optimization.")

# ========== Output Best Trial ========== #
best_trial = study.best_trial
best_params_df = pd.DataFrame([best_trial.params])
best_params_df["Final FID"] = best_trial.value
best_params_df.style.background_gradient(cmap="Blues")

With reference to the Code Cell above, we are able to view the Best Hyperparameter obtained during the Optuna Tuning Study. This Hyperparameter will be extracted and be trained for a longer period of time so as to attempt to increase the performance and limit the loss for the GAN.

---
### 4.7.9 GAN Re-Training Operation

As obtained from the previous sub-section, we will be re-training the GAN using the best Hyperparameter obtained during the Optuna Tuning Study. This is to push the GAN Model to its limits and also to save Computational Power as we are not taking up prolonged periods of time tuning GAN Models which may not have any clear signs of good performance. The retraining of the GAN Model will be conducted in the Code Cell below.

In [ ]:
# =========== Extract Best Parameters from Study =========== #
best_params = study.best_trial.params

latent_dim = best_params['latent_dim']
learning_rate = best_params['learning_rate']
beta_1 = best_params['beta_1']
code_dim = 2
lambda_mi = 1.0

print("Using Best Trial Parameters for InfoGAN:")
print(best_params)

# =========== Rebuild InfoGAN Models =========== #
generator     = build_generator(latent_dim=latent_dim, code_dim=code_dim)
discriminator = build_discriminator(code_dim=code_dim)

# =========== Initialise Optimisers =========== #
gen_optimizer  = tf.keras.optimizers.Adam(learning_rate=learning_rate, beta_1=beta_1)
disc_optimizer = tf.keras.optimizers.Adam(learning_rate=learning_rate, beta_1=beta_1)

_ = gen_optimizer.apply_gradients([(tf.zeros_like(v), v) for v in generator.trainable_variables])
_ = disc_optimizer.apply_gradients([(tf.zeros_like(v), v) for v in discriminator.trainable_variables])

# =========== Reload Preprocessed Dataset =========== #
dataset = load_data(X_train)

# =========== Define Save Paths =========== #
model_name   = "infogan"
output_dir   = f"/content/drive/MyDrive/Colab Notebooks/DELE CA2 A/final_outputs_{model_name}_noAugment"
weights_path = os.path.join(output_dir, f"{model_name}_generator_final.weights.h5")
gen_ckpt     = os.path.join(output_dir, "best_gen.weights.h5")
disc_ckpt    = os.path.join(output_dir, "best_disc.weights.h5")
loss_csv     = os.path.join(output_dir, "loss_history.csv")

os.makedirs(output_dir, exist_ok=True)

# =========== Load or Train =========== #
if os.path.exists(gen_ckpt) and os.path.exists(disc_ckpt) and os.path.exists(loss_csv):
    print("All outputs found. Skipping training...")

    generator.load_weights(gen_ckpt)
    discriminator.load_weights(disc_ckpt)
    print("Generator and Discriminator Weights Loaded.")

    loss_df = pd.read_csv(loss_csv)
    gen_loss_history  = loss_df['gen_loss'].tolist()
    disc_loss_history = loss_df['disc_loss'].tolist()
    mi_loss_history   = loss_df['mi_loss'].tolist()
    final_gen_loss    = gen_loss_history[-1]

else:
    # =========== Train InfoGAN =========== #
    final_gen_loss, gen_loss_history, disc_loss_history, mi_loss_history = train(
        dataset=dataset,
        epochs=50,
        steps_per_epoch=1546,
        generator=generator,
        discriminator=discriminator,
        gen_optimizer=gen_optimizer,
        disc_optimizer=disc_optimizer,
        latent_dim=latent_dim,
        code_dim=code_dim,
        lambda_mi=lambda_mi,
        use_early_stopping=False,
        save_path=output_dir
    )

    # =========== Save Final Generator Weights =========== #
    generator.save_weights(weights_path)
    print("Generator Weights Saved.")

    # =========== Save Loss History =========== #
    loss_df = pd.DataFrame({
        'epoch': list(range(1, len(gen_loss_history) + 1)),
        'gen_loss': gen_loss_history,
        'disc_loss': disc_loss_history,
        'mi_loss': mi_loss_history
    })
    loss_df.to_csv(loss_csv, index=False)
    print(f"Loss history saved to {loss_csv}")

With reference to the Code Cell above, we are able to determine that the best GAN Model have been trained and saved successfully and we are able to proceed with the Model Evaluation in the next sub-sections where we determine how well the GAN Model performed based on various metrics and evaluation methodologies.

---
### 4.7.10 GAN Model Evaluation

In this sub-section, we will be evaluating the GAN's Generator performance on various aspects. We will be utilising different methodologies and metrics to attempt to quantify the performance of the GAN Generator while also conducting Visual Inspection on the result output of the GAN. The Evaluation will be conducted in the following sections.

---
#### 4.7.10.1 Visual Grid of Samples

In this sub-section, we will be visualising the Grid of Samples of the Synthesised Data generated by the GAN Model. A list of observations will be made before the utilisation of Quantitative Metrics to evaluate the GAN Generator's general performance. As the observations made are purely from professional opinion, it will most likely not be used as a Basis of Comparison between GAN Models due to differing opinions unless in extreme cases. The visualisation will be conducted in the Code Cell below.

In [ ]:
def plot_generated_images(generator, latent_dim, code_dim=2, n_rows=5, n_cols=5, save_path=None, title="InfoGAN Generated Images (Non-Augmented)", seed=42):
    total_samples = n_rows * n_cols

    # ----- Generate Latent Vectors ----- #
    if seed is not None:
        tf.random.set_seed(seed)

    z = tf.random.normal([total_samples, latent_dim])
    c = tf.random.uniform([total_samples, code_dim], minval=-1.0, maxval=1.0)

    z = tf.math.l2_normalize(z, axis=1)
    c = tf.math.l2_normalize(c, axis=1)
    zc = tf.concat([z, c], axis=1)

    # ----- Generate Images ----- #
    gen_images = generator(zc, training=False)
    gen_images = (gen_images + 1.0) / 2.0
    gen_images = tf.clip_by_value(gen_images, 0.0, 1.0)

    # ----- Plot ----- #
    fig, axes = plt.subplots(n_rows, n_cols, figsize=(n_cols, n_rows))
    for idx, ax in enumerate(axes.flat):
        ax.imshow(gen_images[idx, :, :, 0], cmap='gray')
        ax.axis('off')

    plt.suptitle(title, fontsize=16)
    plt.tight_layout()

    if save_path:
        plt.savefig(save_path, dpi=300)
        print(f"Saved Generated Image Grid to: {save_path}")

    plt.show()

plot_generated_images(generator, latent_dim=100, code_dim=2, n_rows=10, n_cols=16)

With reference to the output of the Visual Grid of Samples above, we are able to make the following observations:

**Basic Structure of Letters is Somewhat Captured**

- Several characters resemble structured forms of "T", "I", or "O".

- The generated samples are generally well-aligned, and stroke quality is consistent.

**High Blur and Noise**

- Minor fuzziness is present in a few samples, but overall sharpness is maintained.

- Backgrounds remain clean with minimal artefacts or pixel noise.

**Poor Character Consistency**

- Many characters converge toward a narrow set of shapes as the majority look like "6" or "I".

- Variation in angles or structural patterns is minimal, making some letters ambiguous.

**Lack of Diversity**

- There is an overrepresentation of circular and stick-like characters.

- Several samples appear almost identical, indicating early signs of mode collapse.

With the observations indicated above, we will proceed to conduct the Loss Curve Analysis in the next sub-section.

---
#### 4.7.10.2 Loss Curve Over Epochs

In this sub-section, we will be plotting the GAN Loss Curve over Epochs to evaluate the training behaviour of the GAN. This curve provides insight into whether the GAN is experiencing underfitting, overfitting, or achieving a stable training dynamic between the generator and discriminator.

Although GANs do not minimise a single unified loss in the traditional sense, the trajectory of the generator and discriminator losses can indicate whether the adversarial training process is converging appropriately.

If the generator loss remains high while discriminator loss quickly drops to near-zero, this may indicate **underfitting**, where the generator is not learning effectively to produce plausible images.

If the discriminator loss increases while generator loss sharply decreases, this may suggest **overfitting** or **mode collapse**, where the generator exploits narrow weaknesses in the discriminator without general improvement.

A relatively stable oscillation or convergence between both losses typically signifies a **well-balanced** training process, where the generator and discriminator are learning in tandem.

The Potential Observations and Corresponding Action Plans are outlined below:

**Underfitting Loss Curve**

- Increase training epochs to allow generator more time to learn.

- Simplify the discriminator to reduce overpowering the generator.

- Adjust the learning rate or apply label smoothing.

- Consider adding batch normalisation in the generator.

**Overfitting or Mode Collapse Loss Curve**

- Introduce or increase dropout in the discriminator.

- Apply input noise or label flipping to the discriminator.

- Evaluate diversity of generated samples regularly.

**Balanced/Ideal Loss Curve**

- Maintain current architecture and hyperparameters.

- Proceed with full-scale image generation.

- Evaluate both qualitative (visual inspection) and quantitative metrics (e.g. Inception Score or FID).

With these potential training behaviours and action plan outlined, we will proceed to visualise the GAN loss dynamics in the code cell below.

In [ ]:
# =========== Figure Size Configuration =========== #
plt.figure(figsize=(10, 6))

# =========== Plot Training Losses =========== #
plt.plot(gen_loss_history, label="Generator Loss", linewidth=2, color='blue')
plt.plot(disc_loss_history, label="Discriminator Loss", linewidth=2, color='red')

# =========== Labels and Title =========== #
plt.xlabel("Epoch", fontsize=12)
plt.ylabel("Loss", fontsize=12)
plt.title("InfoGAN Training Loss Curve (Non-Augmented)", fontsize=16)

# =========== Grid, Legend, and Styling =========== #
plt.grid(True, linestyle='--', alpha=0.6)
plt.legend(fontsize=12)
plt.xticks(fontsize=10)
plt.yticks(fontsize=10)

# =========== Display the Plot =========== #
plt.tight_layout()
plt.show()

With reference to the output of the Loss Curve above, we are able to make the following observations:

- InfoGAN training appears unstable.

- Generator is losing ground to discriminator across epochs.

- Possible reason include Insufficient Learning Rate and Poor Hyperparameter Tuning.

With the observations indicated above, we will proceed to conduct the Quantitative Metrics Evaluation in the next sub-section.

---
#### 4.7.10.3 Quantitative Metrics Evaluation

In this sub-section we will be utilising Quantitative Metrics such as FID and KID to tabulate the results of the performance and quantify the performance of the GAN Generator. We will subsequently be use these metrics to conduct inter-model evaluation after tuning and training all the GANs. The metrics that we will be utilising and their respective formulas are indicated below:

---
**Fréchet Inception Distance**

Purpose: Measures the Distance between the Real-Image and Generated-Image Distributions in Inception Feature Space.

$$
\mathrm{FID} \;=\;\|\mu_r - \mu_f\|^2
\;+\;\mathrm{Tr}\Bigl(\Sigma_r + \Sigma_f - 2\,(\Sigma_r\,\Sigma_f)^{\tfrac12}\Bigr)
$$


Where:
- $\mu_r = \mathbb{E}[f(x)]$, $\Sigma_r = \mathrm{Cov}[f(x)]$ for Real Images $x$.  
- $\mu_f = \mathbb{E}[f(\hat x)]$, $\Sigma_f = \mathrm{Cov}[f(\hat x)]$ for Generated Images $\hat x$.  
- $f(\cdot)$ = Map from Image to its InceptionV3 'pooling=avg' Features.  

---
**Diversity (t-SNE Spread)**

Purpose: Quantifies how 'wide' the 2D t-SNE Embedding of Generated Samples.

$$
\mathrm{Spread}
\;=\;
\bigl(\max_i\,z_i^{(1)} - \min_i\,z_i^{(1)}\bigr)
\;\times\;
\bigl(\max_i\,z_i^{(2)} - \min_i\,z_i^{(2)}\bigr)
$$

Where:
- $z_i = (z_i^{(1)}, z_i^{(2)})$ = 2-dimensional t-SNE Embedding of $i$th Generated Image.

---
**Mode Collapse Risk**

Purpose: Flags when too many Generated Images are Nearly Identical (mode collapse).

$$
\text{ModeCollapseRisk} =
\begin{cases}
\text{Low}, & \dfrac{\bigl|\{\mathrm{unique\_rounded}(x_i)\}\bigr|}{N} > \tau,\\
\text{High}, & \text{otherwise}.
\end{cases}
$$

Where:
- $x_i$ = $N$ Generated Samples.  
- $\mathrm{unique\_rounded}(x_i)$ = Rounds Pixels to Detect Duplicates.  
- $\tau=0.9$ = Uniqueness Threshold (90%).

---
**Perceptual Path Length**

Purpose: Measures Sensitively of Generator's Outputs (in VGG16 Feature Space) move when the Latent Code $z$ is Perturbed.

$$
\mathrm{PPL}
\;=\;
\mathbb{E}_{z,\delta z}\Bigl[\,
\|\phi\bigl(G(z + \epsilon\,\delta z)\bigr)\;-\;\phi\bigl(G(z)\bigr)\|_2^2
\Bigr]
$$

Where:  
- $G(z)$ = GAN Generator Mapping $z\in\mathbb{R}^{\mathrm{latent\_dim}}$ to an Image.  
- $\phi(\cdot)$ = VGG16 'pooling=avg' Feature Extractor.  
- $\delta z\sim\mathcal{N}(0,I)$, $\epsilon$ = Small Constant.

---
**Kernel Inception Distance**

Purpose: An Unbiased Estimator of the Squared Maximum Mean Discrepancy (MMD) between Real and Generated Inception Features.

$$
\mathrm{KID}
\;=\;
\frac{1}{m(m-1)}\sum_{i\neq j} k\bigl(\phi(x_i),\phi(x_j)\bigr)
\;+\;
\frac{1}{n(n-1)}\sum_{i\neq j} k\bigl(\phi(\hat x_i),\phi(\hat x_j)\bigr)
\;-\;
\frac{2}{mn}\sum_{i=1}^m\sum_{j=1}^n k\bigl(\phi(x_i),\phi(\hat x_j)\bigr)
$$

Where:  
- $x_i$ ($i=1\ldots m$) = Real Images.
- $\hat x_j$ ($j=1\ldots n$) = Generated Images.  
- $\phi(\cdot)$ = InceptionV3 Feature Extractor (pooling='avg').  
- $k(u,v) = \bigl(\frac{u^\top v}{d}+1\bigr)^3$ = degree-3 Polynomial Kernel on $d$-dimensional Features.

---

With the mathematical and metrics indicated above, we will proceed to conduct the Quantitave Metrics Evalution in the Code Cell below.

In [ ]:
# ========== Preload InceptionV3 and VGG16 Models ========== #
inception = InceptionV3(include_top=False, pooling='avg', input_shape=(299, 299, 3))
vgg_model = VGG16(include_top=False, weights='imagenet', input_shape=(128, 128, 3))

# ========== Preprocess for Inception ========== #
def preprocess_images_for_inception(images):
    images = (images + 1.0) * 127.5  # Scale from [-1, 1] to [0, 255]
    images = tf.image.resize(images, [299, 299])
    if images.shape[-1] == 1:
        images = tf.image.grayscale_to_rgb(images)
    return preprocess_input(images)

# ========== Compute FID ========== #
def calculate_fid(real_images, fake_images, batch_size=50):
    real_pp = preprocess_images_for_inception(real_images)
    fake_pp = preprocess_images_for_inception(fake_images)

    act1 = inception.predict(real_pp, batch_size=batch_size, verbose=0)
    act2 = inception.predict(fake_pp, batch_size=batch_size, verbose=0)

    mu1, sigma1 = np.mean(act1, axis=0), np.cov(act1, rowvar=False)
    mu2, sigma2 = np.mean(act2, axis=0), np.cov(act2, rowvar=False)

    diff = mu1 - mu2
    covmean = sqrtm(sigma1 @ sigma2)
    if np.iscomplexobj(covmean):
        covmean = covmean.real

    fid = diff @ diff + np.trace(sigma1 + sigma2 - 2 * covmean)
    return round(fid, 4)

# ========== Compute KID ========== #
def polynomial_kernel(X, Y):
    d = X.shape[1]
    return (np.dot(X, Y.T) / d + 1) ** 3

def calculate_kid(real_images, fake_images, batch_size=128):
    real_pp = preprocess_images_for_inception(real_images)
    fake_pp = preprocess_images_for_inception(fake_images)

    real_features = inception.predict(real_pp, batch_size=batch_size, verbose=0)
    fake_features = inception.predict(fake_pp, batch_size=batch_size, verbose=0)

    m = real_features.shape[0]
    n = fake_features.shape[0]

    k_rr = polynomial_kernel(real_features, real_features)
    k_gg = polynomial_kernel(fake_features, fake_features)
    k_rg = polynomial_kernel(real_features, fake_features)

    np.fill_diagonal(k_rr, 0)
    np.fill_diagonal(k_gg, 0)

    mmd = (k_rr.sum() / (m * (m - 1)) +
           k_gg.sum() / (n * (n - 1)) -
           2 * k_rg.mean())
    return round(mmd, 4)

# ========== Compute t-SNE Spread (Diversity) ========== #
def calculate_tsne_spread(images):
    flat = images.reshape(images.shape[0], -1)
    tsne = TSNE(n_components=2, random_state=42)
    proj = tsne.fit_transform(flat)
    x_range = proj[:, 0].max() - proj[:, 0].min()
    y_range = proj[:, 1].max() - proj[:, 1].min()
    return round(x_range * y_range, 2)

# ========== Assess Mode Collapse ========== #
def assess_mode_collapse(images):
    unique = np.unique(np.round(images), axis=0).shape[0]
    return "Low" if unique > 0.9 * images.shape[0] else "High"

# ========== Compute Perceptual Path Length (PPL) ========== #
def calculate_ppl(generator, latent_dim=100, num_samples=50, epsilon=1e-2):
    distances = []
    for _ in range(num_samples):
        z1 = tf.random.normal([1, latent_dim])
        z2 = z1 + epsilon * tf.random.normal([1, latent_dim])
        img1 = generator(z1, training=False)
        img2 = generator(z2, training=False)

        img1 = tf.image.resize(tf.image.grayscale_to_rgb((img1 + 1.0) * 127.5), (128, 128))
        img2 = tf.image.resize(tf.image.grayscale_to_rgb((img2 + 1.0) * 127.5), (128, 128))

        f1 = vgg_model(vgg_preprocess(img1))
        f2 = vgg_model(vgg_preprocess(img2))
        d = tf.reduce_mean(tf.square(f1 - f2)).numpy()
        distances.append(d)
    return round(np.mean(distances), 4)

# ========== Evaluate GAN Performance ========== #
def evaluate_gan_model(generator, latent_dim, code_dim, X_val, gen_loss_history, disc_loss_history, model_name="InfoGAN"):

    # ----- Sample uniform noise and code in [-1, 1] ----- #
    noise = tf.random.uniform([1000, latent_dim], minval=-1.0, maxval=1.0)
    code  = tf.random.uniform([1000, code_dim],  minval=-1.0, maxval=1.0)
    z     = tf.concat([noise, code], axis=1)

    # ----- Generate Fake Images ----- #
    fake_images = generator(z, training=False)
    fake_images = tf.clip_by_value((fake_images + 1.0) / 2.0, 0.0, 1.0)
    fake_np     = fake_images.numpy()

    # ----- Select and preprocess 1000 Real Images from Validation Set ----- #
    real_images = tf.convert_to_tensor(X_val[:1000], dtype=tf.float32)
    real_images = tf.clip_by_value(real_images, 0.0, 1.0)

    # ----- Compute Metrics ----- #
    fid_value         = calculate_fid(real_images, fake_images)
    kid_value         = calculate_kid(real_images, fake_images)
    tsne_spread_value = calculate_tsne_spread(fake_np)
    collapse_risk     = assess_mode_collapse(fake_np)
    ppl_score         = calculate_ppl(generator, latent_dim + code_dim)
    mean_g_loss       = round(np.mean(gen_loss_history), 4)
    std_g_loss        = round(np.std(gen_loss_history), 4)
    mean_d_loss       = round(np.mean(disc_loss_history), 4)
    std_d_loss        = round(np.std(disc_loss_history), 4)

    # ----- Store as global vars (optional) ----- #
    prefix = model_name.upper()
    globals()[f"{prefix}_FID"]           = fid_value
    globals()[f"{prefix}_KID"]           = kid_value
    globals()[f"{prefix}_TSNE"]          = tsne_spread_value
    globals()[f"{prefix}_MODE_COLLAPSE"] = collapse_risk
    globals()[f"{prefix}_PPL"]           = ppl_score
    globals()[f"{prefix}_GEN_LOSS_MEAN"]  = mean_g_loss
    globals()[f"{prefix}_GEN_LOSS_STD"]   = std_g_loss
    globals()[f"{prefix}_DISC_LOSS_MEAN"] = mean_d_loss
    globals()[f"{prefix}_DISC_LOSS_STD"]  = std_d_loss

    # ----- Final Summary Row ----- #
    row = {
        "Model": model_name,
        "FID Score": fid_value,
        "KID Score": kid_value,
        "Mean Generator Loss": mean_g_loss,
        "Std Generator Loss": std_g_loss,
        "Mean Discriminator Loss": mean_d_loss,
        "Std Discriminator Loss": std_d_loss,
        "Mode Collapse Risk": collapse_risk,
        "Visual Quality": "Acceptable",
        "Diversity (t-SNE Spread)": tsne_spread_value,
        "PPL": ppl_score
    }

    return pd.DataFrame([row])[[
        "Model", "FID Score", "KID Score",
        "Mean Generator Loss", "Std Generator Loss",
        "Mean Discriminator Loss", "Std Discriminator Loss",
        "Mode Collapse Risk", "Visual Quality",
        "Diversity (t-SNE Spread)", "PPL"
    ]]

# ========== Create DataFrame ========== #
# for example, if your code_dim is 2:
df = evaluate_gan_model(
    generator,
    latent_dim=100,
    code_dim=2,
    X_val=X_val,
    gen_loss_history=gen_loss_history,
    disc_loss_history=disc_loss_history,
    model_name="InfoGAN (Non-Augmented)"
)

# ========== Display DataFrame ========== #
df.style.background_gradient(cmap="Blues")

With reference to the output of the Quantitative Metrics above, we are able to determine the performance of the GAN. This will be used as a basis of comparison to the other GAN Models.

---
#### 4.7.10.4 t-SNE of Generated Samples

In this sub-section, we visualise the t-SNE projection of generated samples to evaluate the latent diversity learned by the unconditional GAN. Although our GAN is not class-conditioned, t-SNE remains a valuable tool to assess whether the generator is producing varied outputs that reflect meaningful use of the latent space. While GANs are trained in high-dimensional spaces, t-SNE enables us to observe 2D structural patterns such as sample clustering, separation, and density, which provide indirect insights into the diversity and generalisation capabilities of the generator. The visualisation will be conducted in the Code Cell below.

In [ ]:
# =========== Generate z and c for InfoGAN =========== #
latent_dim = 100
code_dim = 2
z = tf.random.normal([500, latent_dim])
c = tf.random.uniform([500, code_dim], minval=-1.0, maxval=1.0)

# =========== ℓ₂-Normalise =========== #
z = tf.math.l2_normalize(z, axis=1)
c = tf.math.l2_normalize(c, axis=1)

# =========== Concatenate z and c ===========
zc = tf.concat([z, c], axis=1)

# =========== Generate Images =========== #
generated_images = generator(zc, training=False)
generated_images = (generated_images + 1) / 2.0

def plot_tsne_embeddings(generated_images, save_path=None, title="t-SNE of InfoGAN Generated Samples (Non-Augmented)"):
    # ----- Convert to [0, 1] Range ----- #
    if tf.reduce_max(generated_images).numpy() > 1.0:
        raise ValueError("Images should be in [-1, 1] range before scaling.")

    images_rescaled = (generated_images + 1.0) / 2.0
    flat_images = images_rescaled.numpy().reshape(images_rescaled.shape[0], -1)

    # ----- Run t-SNE ----- #
    tsne = TSNE(n_components=2, perplexity=30, learning_rate='auto', init='pca', random_state=42)
    tsne_proj = tsne.fit_transform(flat_images)

    # ----- Plot t-SNE ----- #
    plt.figure(figsize=(12, 6))
    sns.scatterplot(
        x=tsne_proj[:, 0],
        y=tsne_proj[:, 1],
        s=20,
        alpha=0.9,
        edgecolor='none'
    )
    plt.title(title, fontsize=14, weight='bold')
    plt.xlabel("t-SNE Dimension 1")
    plt.ylabel("t-SNE Dimension 2")
    plt.grid(True, linestyle='--', alpha=0.3)
    plt.tight_layout()

    if save_path:
        plt.savefig(save_path, dpi=300)
        print(f"t-SNE plot saved to: {save_path}")

    plt.show()

plot_tsne_embeddings(generated_images=generated_images)

With reference to the t-SNE plot of InfoGAN generated samples (Non-Augmented) above, we are able to make the following observations:

**Highly Structured and Disentangled Distribution**

- The samples form a perfect elliptical or circular manifold, suggesting that InfoGAN has learned a smooth and interpretable latent space.

- This is expected due to InfoGAN's use of mutual information regularisation, which encourages disentanglement in the latent variables.

**Exceptional Mode Coverage**

- The uniform curve indicates that all latent codes are being used effectively, with no signs of mode collapse or sparsity.

- Implies that latent traversals will result in continuous and meaningful changes in output, ideal for applications like controllable generation.

**Almost No Outliers**

- Points are tightly packed along the curve with minimal scatter outside the manifold, showing strong generator consistency.

- Such tight structure can be attributed to the semantic regularity induced by InfoGAN's auxiliary Q-network.

**Possible Over-Constraint**

- While visually impressive, the overly clean elliptical layout may reflect a model bias toward encoding structure rather than raw diversity.

- May suggest that InfoGAN has prioritised latent interpretability over visual variability, which is suitable for interpretability tasks.

With the observations indicated above, we will proceed to train the next GAN Model in the final sub-section.

---
## 4.8 Self-Attention GAN Model Training

In this sub-section, we will be training a Self-Attention GAN (SAGAN) for the EMNIST Dataset. The SAGAN adds a Self-Attention Mechanism that captures global dependencies across the training data, understands contextual relationships while preserving structural consistency. This will generally result is a more stable training as no local convolutional layers are relied on. The relevant mathematical formulas are indicated below:

---
**Generator Loss:**

Purpose: Produce Outputs Classified as Real.

$$
\mathcal{L}_G = \frac{1}{2} \mathbb{E}_{z \sim p_z}[(D(G(z)) - c)^2]
$$

Where:
- $c = 1$ = Desired Discriminator Output for Generated Images  
- $D(G(z))$ = Discriminator's Output for Generated Data  
- $z \sim p_z$ = Sample from Latent Space

---
**Discriminator Loss:**

Purpose: Discriminator Minimises the Error between Predictions and True Labels.

$$
\mathcal{L}_D = \frac{1}{2} \mathbb{E}_{x \sim p_{\text{data}}}[(D(x) - b)^2] + \frac{1}{2} \mathbb{E}_{z \sim p_z}[(D(G(z)) - a)^2]
$$

Where:  
- $D(x)$ = Discriminator's Output for Real Data  
- $D(G(z))$ = Discriminator's Output for Generated Data  
- $a = 0$ = Label for Fake Data  
- $b = 1$ = Label for Real Data  
- $x \sim p_{\text{data}}$ = Sample from Real Data Distribution  
- $z \sim p_z$ = Sample from Latent Noise Distribution  

---
**Self-Attention Output**

Purpose: Allows Generator and Discriminator to capture long-range dependencies between pixels across the image.

$$
\text{Attention}(X) = \text{Softmax}(QK^T) V
$$

Where:
- $X \in \mathbb{R}^{N \times d}$ = Input Feature Map Reshaped into Sequence
- $Q = XW_Q$ = Query Projections
- $K = XW_K$ = Key Projection
- $V = XW_V$ = Value Projection
- $W_Q, W_K, W_V \in \mathbb{R}^{d \times d'}$ = Learnable Weights
- $\text{Softmax}(QK^T) \in \mathbb{R}^{N \times N}$ = Attention Matrix

---
**Self-Attention Layer Output**

Purpose: Add Global Context to each feature by Combining Local Convolution and Self-Attention Output.

$$
O = \gamma \cdot \text{Attention}(X) + X
$$

Where:
- $\text{Attention}(X)$ = Attention-Modulated Feature Map
- $X$ = Original Input Feature Map
- $\gamma$ = Learnable Scalar to Control Attention Usage

---
**Total Objective:**

Purpose: SAGAN Seeks to Solve the Objective.

$$
\min_G \max_D \; \mathcal{L}_D \quad \text{and} \quad \min_G \; \mathcal{L}_G
$$

---
With the relevant mathematical formulas indicated above, we will proceed to train the SAGAN in this sub-section for the purpose of it's potential for smoother training process and also to serve as a comparison for our other GAN Models.

---
### 4.8.1 Defining Data Pre-Processing Function

In this sub-section, we will be defining the Data Pre-Processing Function which will allow for consistent Data Pipeline to be built and modifed accordingly should the need arise such as when we need to add Spectral Normalisation. This will ensure that the Data's integrity is not affected and we are able to make fair and un-biased assumptions. The Data Pre-Processing Function is defined in the Code Cell below:

In [ ]:
# =========== Data Pre-Processing Function =========== #
def load_data(X_train, batch_size=32):
    buffer_size = X_train.shape[0]

    dataset = tf.data.Dataset.from_tensor_slices(X_train)
    dataset = dataset.shuffle(buffer_size)
    dataset = dataset.batch(batch_size)
    dataset = dataset.prefetch(tf.data.AUTOTUNE)

    return dataset

With reference to the Code Cell above, we are able to determine that the Data Pre-Processing Function have been defined successfully and we are able to move on to Defining Callback Functions in the next sub-section.

---
### 4.8.2 Defining Callback Functions

In this sub-section, we will be pre-defining the various Callbacks. This is to ensure consistency throughout the model and also to increase the GAN Accuracy and optimise the Computation Cost for training each GAN Model. These Callbacks will be main used during the GAN Training in the next sub-section. The revelant mathematical formula for the callbacks and logic are indicated below:

---
**Learning Rate Scheduler:**

Purpose: Reduces Learning Rate During Training to Fine-tune Model Convergence.

$$\text{If } epoch \mod 10 = 0 \Rightarrow \eta_{new} = \frac{1}{2} \cdot \eta$$

Where:

- $\eta$ = current learning rate

- $\eta_{\text{new}}$ = updated learning rate

- $epoch \bmod 10$ checks if the epoch is a multiple of 10

---
**ReduceLROnPlateau:**

Purpose: Automatically Reduces the Learning Rate when Validation Performance Plateaus.

$$
\text{If no improvement in } val\_loss \text{ for 10 epochs: } \eta_{\text{new}} = 0.5 \cdot \eta
$$

Where:

- $\eta_{\text{new}}$ = reduced learning rate

---
**EarlyStopping:**

Purpose: Prevents Wastage of Computational Power if Model Does Not Improve.

$$
\text{If } val\_loss \text{ does not improve for 25 epochs, stop training and restore best weights}
$$

---

We will proceed to Define the Callbacks in the Code Cell below in preparation for the GAN Model Training.

In [ ]:
# =========== Define Learning Rate Scheduler =========== #
lr_scheduler = LearningRateScheduler(
    lambda epoch, lr: lr * 0.95 if epoch % 10 == 0 else lr,
    verbose=1
)

# =========== Define Reduce LR on Plateau =========== #
reduce_lr = ReduceLROnPlateau(
    monitor='loss',
    factor=0.5,
    patience=10,
    verbose=1,
    min_lr=1e-6
)

# =========== Define Early Stopping =========== #
early_stop = EarlyStopping(
    monitor='loss',
    patience=25,
    verbose=1,
    restore_best_weights=False
)

# =========== Define Display Generated Images =========== #
def display_generated_images(generator, latent_dim, n=7, channels=1):
    noise = tf.random.normal([n * n, latent_dim])
    generated_images = generator(noise, training=False)
    generated_images = (generated_images + 1.0) / 2.0

    fig, axes = plt.subplots(n, n, figsize=(n, n))

    for i in range(n):
        for j in range(n):
            img = generated_images[i * n + j]
            if channels == 1:
                axes[i, j].imshow(img[:, :, 0], cmap='gray')
            else:
                axes[i, j].imshow(img)
            axes[i, j].axis('off')

    plt.tight_layout()
    plt.show()

# =========== Define FID Calculator =========== #
def calculate_fid(real_images, fake_images, batch_size=10):

    # ----- Scale to [0,255] and Resize to 299×299 ----- #
    real = (real_images + 1.0) * 127.5
    fake = (fake_images + 1.0) * 127.5
    real = tf.image.resize(real, (299,299))
    fake = tf.image.resize(fake, (299,299))

    # ----- If Grayscale, Convert to RGB ----- #
    if real.shape[-1] == 1:
        real = tf.image.grayscale_to_rgb(real)
        fake = tf.image.grayscale_to_rgb(fake)

    # ----- Preprocess for Inception (–1 to +1) ----- #
    real_pp = preprocess_input(real)
    fake_pp = preprocess_input(fake)

    # ----- Extract Features in Batches ----- #
    def _get_acts(x):
        acts = []
        n = x.shape[0]
        for i in range(0, n, batch_size):
            chunk = x[i:i+batch_size]
            acts.append(_inception_model(chunk, training=False).numpy())
        return np.vstack(acts)

    act_real = _get_acts(real_pp)
    act_fake = _get_acts(fake_pp)

    # ----- Compute Statistics ----- #
    mu1, sigma1 = act_real.mean(axis=0), np.cov(act_real, rowvar=False)
    mu2, sigma2 = act_fake.mean(axis=0), np.cov(act_fake, rowvar=False)
    diff    = mu1 - mu2
    covmean = sqrtm(sigma1.dot(sigma2))
    if np.iscomplexobj(covmean):
        covmean = covmean.real
    fid = diff.dot(diff) + np.trace(sigma1 + sigma2 - 2*covmean)
    return float(np.round(fid,4))

# =========== Confirmation Message =========== #
print("InfoGAN Callbacks and Visualisation Setup Complete")

With reference to the Code Cell above, we are able to verify that that the Callbacks have been successfully defined and the various parameters are set so as to ensure a High Accuracy and Computing Efficient GAN Model Training.

---
### 4.8.3 Defining Spectral Normalisation

In this section, we will be defining the Spectral Normalisation layer for the Discriminator of the GAN. The Spectral Normalisation controls the Lipschitz constant of the Discriminator by constraining the Spectral Norm of each weight matrix. This will prevent the Discriminator from being too powerful and dominating the Generator which can cause various issues such as Mode Collaspe, Training Instability and Exploding Gradients. The relevant mathematical formula is indicated below:

---
**Spectral Normalisation (SN)**

Purpose: Controls the Lipschitz Constant of the Discriminator.

$$
\bar{W} = \frac{W}{\sigma(W)}
$$

Where:
- $W$ = Original Weight Matrix
- $\bar{W}$ = Spectrally Normalised Weight Matrix used in the Forward Pass
- $\sigma(W)$ = Spectral Norm of $W$
---
**Spectral Norm Approximation using Power Iteration**

Purpose: Controls the Lipschitz Constant of the Discriminator.

$$
\sigma(W) \approx \mathbf{u}^\top W \mathbf{v}
$$

Where:
- $\mathbf{v} \leftarrow \frac{W^\top \mathbf{u}}{\|W^\top \mathbf{u}\|_2}$
- $\mathbf{u} \leftarrow \frac{W \mathbf{v}}{\|W \mathbf{v}\|_2}$
- $\mathbf{u}, \mathbf{v}$: Approximated Unit Vectors
---
With the relevant mathematical formulas indicated, we will proceed to define the Spectral Normalisation in the Code Cell below.

In [ ]:
# =========== Spectral Normalisation Convolution Layer =========== #
class SpectralConv2D(tf.keras.layers.Layer):
    def __init__(self, filters, kernel_size, strides=1, padding='same', **kwargs):
        super().__init__()
        self.conv = tf.keras.layers.Conv2D(filters, kernel_size, strides=strides, padding=padding, use_bias=False, **kwargs)
        self.u = None

    def build(self, input_shape):
        self.conv.build(input_shape)
        self.w = self.conv.kernel
        self.u = self.add_weight(shape=(1, self.w.shape[-1]), initializer='random_normal', trainable=False, name='sn_u')

    def compute_spectral_norm(self, w):
        w_reshaped = tf.reshape(w, [-1, w.shape[-1]])
        u = self.u

        for _ in range(1):
            v = tf.linalg.l2_normalize(tf.matmul(u, tf.transpose(w_reshaped)))
            u = tf.linalg.l2_normalize(tf.matmul(v, w_reshaped))

        sigma = tf.matmul(tf.matmul(v, w_reshaped), tf.transpose(u))
        self.u.assign(u)
        return w / sigma

    def call(self, inputs):
        self.conv.kernel.assign(self.compute_spectral_norm(self.w))
        return self.conv(inputs)

# =========== Spectral Normalisation Dense Layer =========== #
class SpectralDense(tf.keras.layers.Layer):
    def __init__(self, units, activation=None):
        super().__init__()
        self.dense = tf.keras.layers.Dense(units, activation=activation, use_bias=False)
        self.u = None

    def build(self, input_shape):
        self.dense.build(input_shape)
        self.w = self.dense.kernel
        self.u = self.add_weight(shape=(1, self.w.shape[-1]), initializer='random_normal', trainable=False, name='sn_u_dense')

    def compute_spectral_norm(self, w):
        w_reshaped = tf.reshape(w, [-1, w.shape[-1]])
        u = self.u

        for _ in range(1):
            v = tf.linalg.l2_normalize(tf.matmul(u, tf.transpose(w_reshaped)))
            u = tf.linalg.l2_normalize(tf.matmul(v, w_reshaped))

        sigma = tf.matmul(tf.matmul(v, w_reshaped), tf.transpose(u))
        self.u.assign(u)
        return w / sigma

    def call(self, inputs):
        self.dense.kernel.assign(self.compute_spectral_norm(self.w))
        return self.dense(inputs)

With reference to the Code Cell above, we are able to verify that the Spectral Normalisation have been successfully defined and we are able to proceed to include this layer in the Discriminator Architecture in the next section.

---
### 4.8.3 Defining Self-Attention Layer

In this section, we will be defining the Self-Attention Layer for the SAGAN. The Self-Attention Layers will be giving the SAGAN the ability to be capable of capturing patterns and data relationships clearly which is the speciality and function of the SAGAN. The Layers will subsequently be added in to the Model Architectures alongside the Spectral Normalisation. The relevant mathematical formulas are indicated below:

---
**Self-Attention Output**

Purpose: Allows Generator and Discriminator to capture long-range dependencies between pixels across the image.

$$
\text{Attention}(X) = \text{Softmax}(QK^T) V
$$

Where:
- $X \in \mathbb{R}^{N \times d}$ = Input Feature Map Reshaped into Sequence
- $Q = XW_Q$ = Query Projections
- $K = XW_K$ = Key Projection
- $V = XW_V$ = Value Projection
- $W_Q, W_K, W_V \in \mathbb{R}^{d \times d'}$ = Learnable Weights
- $\text{Softmax}(QK^T) \in \mathbb{R}^{N \times N}$ = Attention Matrix

---
**Self-Attention Layer Output**

Purpose: Add Global Context to each feature by Combining Local Convolution and Self-Attention Output.

$$
O = \gamma \cdot \text{Attention}(X) + X
$$

Where:
- $\text{Attention}(X)$ = Attention-Modulated Feature Map
- $X$ = Original Input Feature Map
- $\gamma$ = Learnable Scalar to Control Attention Usage

---

With the mathematical formulas indicated, we will proceed to define the Self-Attention layers in the Code Cell below for the usage in the Architecture later on.

In [ ]:
# =========== Self Attention Layer =========== #
class SelfAttention(tf.keras.layers.Layer):
    def __init__(self, channels):
        super().__init__()
        self.channels = channels
        self.query = SpectralConv2D(filters=channels // 8, kernel_size=1, padding='same')
        self.key = SpectralConv2D(filters=channels // 8, kernel_size=1, padding='same')
        self.value = SpectralConv2D(filters=channels,       kernel_size=1, padding='same')
        self.gamma = tf.Variable(0.0, trainable=True, name='attn_gamma')

    def call(self, x):
        # ----- Shape Operation ----- #
        shape = tf.shape(x)
        batch_size, height, width = shape[0], shape[1], shape[2]
        flatten_dim = height * width

        # ----- Project ----- #
        f = self.query(x)
        g = self.key(x)
        h = self.value(x)

        # ----- Flatten ----- #
        f_flat = tf.reshape(f, [batch_size, flatten_dim, -1])
        g_flat = tf.reshape(g, [batch_size, flatten_dim, -1])
        h_flat = tf.reshape(h, [batch_size, flatten_dim, -1])

        # ----- Attention Map ----- #
        s    = tf.matmul(f_flat, g_flat, transpose_b=True)
        beta = tf.nn.softmax(s, axis=-1)

        # ----- Attend ----- #
        o_flat = tf.matmul(beta, h_flat)
        o = tf.reshape(o_flat, [batch_size, height, width, self.channels])

        # ----- Residual ----- #
        return self.gamma * o + x

With reference to the Code Cell above, we are able to determine that the Self-Attention Layers have been successfully defined and we are able to move on the define the Generator and Discriminator Architecture while utilising the Self-Attention Layer.

---
### 4.8.4 Defining SAGAN Generator Architecture

In this section, we will be defining the InfoGAN's Generator Architecture. The Generator will be trying to make realistic generated images in the attempt to bypass the Discriminator. Subsequently, the generated images from the Generator will be evaluated based on a few evaluation metrics such as Inception Score. The relevent mathematical formulas for the Loss Function of the Vanilla GAN Generator is indicated below:

---
**Generator Loss:**

Purpose: Produce Outputs Classified as Real.

$$
\mathcal{L}_G = \frac{1}{2} \mathbb{E}_{z \sim p_z}[(D(G(z)) - c)^2]
$$

Where:
- $c = 1$ = Desired Discriminator Output for Generated Images  
- $D(G(z))$ = Discriminator's Output for Generated Data  
- $z \sim p_z$ = Sample from Latent Space

---
With the relevant Generator Loss Formula indicated above, we will proceed to define the GAN Generator's Architecture in the Code Cell below and the Lost Function.

In [ ]:
# =========== Generator Function =========== #
def build_sagan_generator(latent_dim=100, code_dim=2, use_dropout=True):
    input_dim = latent_dim + code_dim  # z + c
    model = tf.keras.Sequential(name="sagan_generator")

    # ----- Project and Reshape ----- #
    model.add(layers.Dense(7 * 7 * 256, use_bias=False, input_shape=(input_dim,)))
    model.add(layers.BatchNormalization())
    model.add(layers.ReLU())
    model.add(layers.Reshape((7, 7, 256)))

    # ----- Upsample 1 ----- #
    model.add(layers.Conv2DTranspose(128, kernel_size=5, strides=1, padding='same', use_bias=False))
    model.add(layers.BatchNormalization())
    model.add(layers.ReLU())
    if use_dropout:
        model.add(layers.Dropout(0.3))

    # ----- Self-Attention Layer ----- #
    model.add(SelfAttention(128))

    # ----- Upsample 2 ----- #
    model.add(layers.Conv2DTranspose(64, kernel_size=5, strides=2, padding='same', use_bias=False))
    model.add(layers.BatchNormalization())
    model.add(layers.ReLU())
    if use_dropout:
        model.add(layers.Dropout(0.3))

    # ----- Final Upsample ----- #
    model.add(layers.Conv2DTranspose(1, kernel_size=5, strides=2, padding='same', activation='tanh'))

    return model

# =========== Generator Loss Function =========== #
bce = tf.keras.losses.BinaryCrossentropy(from_logits=True)

def generator_loss(fake_logits):
    return -tf.reduce_mean(fake_logits)

With reference to the Code Cell above, we are able to verify that the GAN's Generator Architecture have been successfully defined and we are able to proceed to define the GAN's Discriminator Architecture.

---
### 4.8.5 Defining SAGAN Discriminator Architecture

In this section, we will be defining the Architecture of the Discriminator of the GAN. The Discriminator will be the Neural Network that will attempt to tell the Generated Images apart from the Actual Images. This will allow the GAN to improve overtime and also improve itself overtime. The mathematical function for the Discriminator Loss Function is indicated below:

---
**Discriminator Loss:**

Purpose: Discriminator Minimises the Least Squares Error between Predictions and True Labels.

$$
\mathcal{L}_D = \frac{1}{2} \mathbb{E}_{x \sim p_{\text{data}}}[(D(x) - b)^2] + \frac{1}{2} \mathbb{E}_{z \sim p_z}[(D(G(z)) - a)^2]
$$

Where:  
- $D(x)$ = Discriminator's Output for Real Data  
- $D(G(z))$ = Discriminator's Output for Generated Data  
- $a = 0$ = Label for Fake Data  
- $b = 1$ = Label for Real Data  
- $x \sim p_{\text{data}}$ = Sample from Real Data Distribution  
- $z \sim p_z$ = Sample from Latent Noise Distribution  
---

With the mathematical formula for the Loss Function indicated above, we will proceed to define the GAN's Discriminator Architecture and the Loss Function in the Code Cell below.

In [ ]:
# =========== Discriminator Function =========== #
def build_sagan_discriminator(input_shape=(28, 28, 1), code_dim=2):
    image_input = tf.keras.Input(shape=input_shape)

    # ----- Spectral-normalized Convolution Layer ----- #
    x = SpectralConv2D(64, kernel_size=5, strides=2, padding='same')(image_input)
    x = tf.keras.layers.LeakyReLU(0.2)(x)
    x = tf.keras.layers.Dropout(0.3)(x)

    # ----- Spectral-normalized Convolution Layer ----- #
    x = SpectralConv2D(128, kernel_size=5, strides=2, padding='same')(x)
    x = tf.keras.layers.LeakyReLU(0.2)(x)
    x = tf.keras.layers.Dropout(0.3)(x)

    # ----- Self-Attention Layer ----- #
    x = SelfAttention(128)(x)

    x = tf.keras.layers.Flatten()(x)

    # ----- Output 1: Real/Fake ----- #
    real_fake_output = SpectralDense(1)(x)

    # ----- Output 2: Q-network Code Prediction ----- #
    code_output = SpectralDense(code_dim)(x)

    return tf.keras.Model(inputs=image_input,
                          outputs=[real_fake_output, code_output],
                          name="sagan_discriminator_with_Q")

# =========== Discriminator Loss Function =========== #
def discriminator_loss(real_logits, fake_logits):
    loss_real = tf.reduce_mean(tf.nn.relu(1.0 - real_logits))
    loss_fake = tf.reduce_mean(tf.nn.relu(1.0 + fake_logits))
    return loss_real + loss_fake

With reference to the Code Cell above, we are able to verify that the GAN's Discriminator Architecture have been successfully defined and we are able to proceed to define the GAN's Training Steps.

---
### 4.8.6 Defining Training Step

In this section, we will be Defining the Training Step function so as to train both the Generator and the Discriminator at the same time. The Training Step will compute the gradients for both the Generator and the Discriminator at the same time. The relevant mathematical formulas for the Training Step is indicated below:

---
**Generator Loss:**

Purpose: Produce Outputs Classified as Real.

$$
\mathcal{L}_G = \frac{1}{2} \mathbb{E}_{z \sim p_z}[(D(G(z)) - c)^2]
$$

Where:
- $c = 1$ = Desired Discriminator Output for Generated Images  
- $D(G(z))$ = Discriminator's Output for Generated Data  
- $z \sim p_z$ = Sample from Latent Space

---
**Discriminator Loss:**

Purpose: Discriminator Minimises the Least Squares Error between Predictions and True Labels.

$$
\mathcal{L}_D = \frac{1}{2} \mathbb{E}_{x \sim p_{\text{data}}}[(D(x) - b)^2] + \frac{1}{2} \mathbb{E}_{z \sim p_z}[(D(G(z)) - a)^2]
$$

Where:  
- $D(x)$ = Discriminator's Output for Real Data  
- $D(G(z))$ = Discriminator's Output for Generated Data  
- $a = 0$ = Label for Fake Data  
- $b = 1$ = Label for Real Data  
- $x \sim p_{\text{data}}$ = Sample from Real Data Distribution  
- $z \sim p_z$ = Sample from Latent Noise Distribution  

---
**Total Objective:**

Purpose: LSGAN Seeks to Solve the Objective.

$$
\min_G \max_D \; \mathcal{L}_D \quad \text{and} \quad \min_G \; \mathcal{L}_G
$$

---

With the relevant mathematical formulas indicated above, we will proceed to define the Training Step in the Code Cell below.

In [ ]:
# =========== Training Step Function =========== #
@tf.function
def train_step(real_images, generator, discriminator, gen_optimizer, disc_optimizer,
               latent_dim, code_dim=2, lambda_mi=1.0):
    batch_size = tf.shape(real_images)[0]
    bce = tf.keras.losses.BinaryCrossentropy(from_logits=True)

    # ----- Sample Latent Noise and Latent Code ----- #
    z = tf.random.normal([batch_size, latent_dim])
    c = tf.random.uniform([batch_size, code_dim], minval=-1.0, maxval=1.0)
    zc = tf.concat([z, c], axis=1)

    with tf.GradientTape(persistent=True) as tape:
        # ----- Generator Forward Pass ----- #
        fake_images = generator(zc, training=True)

        # ----- Discriminator Forward Pass ----- #
        real_output, _ = discriminator(real_images, training=True)
        fake_output, predicted_c = discriminator(fake_images, training=True)

        # ----- Discriminator Loss (Standard Binary Crossentropy) ----- #
        d_loss_real = bce(tf.ones_like(real_output), real_output)
        d_loss_fake = bce(tf.zeros_like(fake_output), fake_output)
        disc_loss = d_loss_real + d_loss_fake

        # ----- Generator Loss ----- #
        adv_loss = bce(tf.ones_like(fake_output), fake_output)

        # ----- Mutual Information Loss ----- #
        mi_loss = tf.reduce_mean(tf.square(predicted_c - c))  # For continuous latent codes

        # ----- Total Generator Loss ----- #
        gen_loss = adv_loss + lambda_mi * mi_loss

    # ----- Apply Gradients ----- #
    gen_grads = tape.gradient(gen_loss, generator.trainable_variables)
    disc_grads = tape.gradient(disc_loss, discriminator.trainable_variables)

    gen_optimizer.apply_gradients(zip(gen_grads, generator.trainable_variables))
    disc_optimizer.apply_gradients(zip(disc_grads, discriminator.trainable_variables))

    return gen_loss, disc_loss, adv_loss, mi_loss

With reference to the Code Cell above, we are able to determine that the Training Step have been successfully defined and we are able to proceed to prepare the Training Loop in the next section.

---
### 4.8.7 Defining Training Loop

In this section, we will be defining the Training Loop for the GAN Training. The Training Loop will go through each of the Epoch and train both the Generator and Discriminator simultaneously. The Training Loop will also track the Loss for both the Generator and the Discriminator for each of the Epoch trained. The relevant mathematical formulas are indicated below:

---
**Epoch Loss Averaging:**

Purpose: These are the Average Generator and Discriminator Losses over all Batches in Epoch $t$.

$$
\bar{L}_G^{(t)} = \frac{1}{N} \sum_{i=1}^{N} L_G^{(i)} \\
\bar{L}_D^{(t)} = \frac{1}{N} \sum_{i=1}^{N} L_D^{(i)}
$$

Where:
- $\bar{L}_G^{(t)}$ = Average Generator Loss at Epoch $t$  
- $\bar{L}_D^{(t)}$ = Average Discriminator Loss at Epoch $t$  
- $N$ = Number of Batches in the Dataset  
- $L_G^{(i)}$ = Generator Loss on Batch $i$  
- $L_D^{(i)}$ = Discriminator Loss on Batch $i$

---
**Epoch Iteration:**

Purpose: Describes the Training Loop Logic: for Each Epoch $t$, perform `train_step` for Every Mini-batch.

$$
\text{for } t = 1 \text{ to } T:
\quad \text{for each batch } (x^{(i)}):
\quad \text{train_step}(x^{(i)})
$$

Where:
- $T$: Total Number of Epochs  
- $x^{(i)}$: Real Batch $i$ from the Dataset  
- `train_step`: Function that Updates $G$ and $D$ using that Batch

---
**Generated Image Output:**

Purpose: Sample Random Noise $z$ and Generate Synthetic Image $\hat{x}$ from the Generator. This is used for Visual Monitoring of Model Quality.

$$
z \sim p_z(z) \\
\hat{x} = G(z)
$$

Where:
- $z$: Random Latent Vector  
- $G(z)$: Generated Image from Generator

---
With the mathematical formulas indicated above, we will proceed to define the Training Loop in the Code Cell below in preparation of the GAN Training.

In [ ]:
# =========== Training Loop Function =========== #
def train(dataset, epochs, generator, discriminator, gen_optimizer, disc_optimizer, latent_dim, code_dim=2, lambda_mi=1.0, save_path=None, use_early_stopping=True, steps_per_epoch=None, channels=1):
    # ----- Prepare FID Reference Sets Using X_val ----- #
    real_val  = tf.data.Dataset.from_tensor_slices(X_val).batch(50).take(1)
    noise_val = tf.random.normal([50, latent_dim])
    noise_val = tf.math.l2_normalize(noise_val, axis=1)
    code_val  = tf.random.uniform([50, code_dim], minval=-1.0, maxval=1.0)
    code_val  = tf.math.l2_normalize(code_val, axis=1)
    zc_val    = tf.concat([noise_val, code_val], axis=1)

    # ----- FID Early Stopping Settings ----- #
    best_fid = float('inf')
    no_improve = 0
    patience = 7
    fid_interval = 1

    # ----- Dummy Model ----- #
    dummy_input  = tf.keras.Input(shape=(1,))
    dummy_output = tf.keras.layers.Lambda(lambda x: x)(dummy_input)
    dummy_model  = tf.keras.Model(dummy_input, dummy_output)
    dummy_model.compile(optimizer=tf.keras.optimizers.Adam(1e-4), loss='mse')

    early_stop.set_model(dummy_model)
    reduce_lr.set_model(dummy_model)
    lr_scheduler.set_model(dummy_model)
    early_stop.on_train_begin({})
    reduce_lr.on_train_begin({})
    lr_scheduler.on_train_begin({})

    # ----- Track Loss History ----- #
    gen_loss_history = []
    disc_loss_history = []
    mi_loss_history = []

    for epoch in range(1, epochs + 1):
        print(f"\nEpoch {epoch}/{epochs}")
        gen_losses, disc_losses, mi_losses = [], [], []

        for i, real_images in enumerate(dataset):
            if steps_per_epoch and i >= steps_per_epoch:
                break

            # ----- SAGAN Train Step ----- #
            gen_loss, disc_loss, adv_loss, mi_loss = train_step(
                real_images, generator, discriminator,
                gen_optimizer, disc_optimizer,
                latent_dim, code_dim=code_dim, lambda_mi=lambda_mi
            )
            gen_losses.append(gen_loss)
            disc_losses.append(disc_loss)
            mi_losses.append(mi_loss)

        # ----- Epoch Statistics ----- #
        avg_gen_loss  = tf.reduce_mean(gen_losses)
        avg_disc_loss = tf.reduce_mean(disc_losses)
        avg_mi_loss   = tf.reduce_mean(mi_losses)
        print(f"Gen Loss: {avg_gen_loss:.4f} | "
              f"Disc Loss: {avg_disc_loss:.4f} | "
              f"MI Loss: {avg_mi_loss:.4f}")

        # ----- Save Losses ----- #
        gen_loss_history.append(float(avg_gen_loss))
        disc_loss_history.append(float(avg_disc_loss))
        mi_loss_history.append(float(avg_mi_loss))

        # ----- Run Callbacks ----- #
        logs = {'loss': float(avg_gen_loss)}
        if use_early_stopping:
            early_stop.on_epoch_end(epoch=epoch, logs=logs)
            if early_stop.stopped_epoch > 0:
                print(f"Early stopping triggered at epoch {epoch}")
                break
        reduce_lr.on_epoch_end(epoch=epoch, logs=logs)
        lr_scheduler.on_epoch_end(epoch=epoch, logs=logs)

        # ----- FID evaluation & early stopping ----- #
        if epoch % fid_interval == 0:
            fake_val    = generator(zc_val, training=False)
            real_batch  = next(iter(real_val))
            real_for_fid = tf.image.resize(real_batch, [299,299]) * 0.5 + 0.5
            fake_for_fid = tf.image.resize(fake_val,    [299,299]) * 0.5 + 0.5
            fid_value    = calculate_fid(real_for_fid, fake_for_fid)
            print(f"Epoch {epoch} → FID: {fid_value:.2f}")

            if fid_value < best_fid:
                best_fid   = fid_value
                no_improve = 0
                if save_path:
                    os.makedirs(save_path, exist_ok=True)
                    generator.save_weights(os.path.join(save_path, "best_gen.weights.h5"))
                    discriminator.save_weights(os.path.join(save_path, "best_disc.weights.h5"))
            else:
                no_improve += 1

            if no_improve >= patience:
                print(f"No FID improvement for {patience} epochs, stopping.")
                break

        # ----- Visualisation Every 10 Epochs ----- #
        if epoch % 10 == 0:
            display_generated_images(generator, latent_dim + code_dim, channels=channels)

    # ----- Restore Best Weights After Training ----- #
    if save_path:
        best_gen_path = os.path.join(save_path, "best_gen.weights.h5")
        best_disc_path = os.path.join(save_path, "best_disc.weights.h5")
        if os.path.exists(best_gen_path) and os.path.exists(best_disc_path):
            generator.load_weights(best_gen_path)
            discriminator.load_weights(best_disc_path)
            print("Restored best generator and discriminator weights based on lowest FID.")
        else:
            print("Best weight files not found. Skipping restore.")

    return avg_gen_loss, gen_loss_history, disc_loss_history, mi_loss_history

With reference to the Code Cell above, we are able to determine that the Training Loop have been successfully defined and we are able to proceed to define the Optimisers with Tunable Parameter in preparation for the Optuna Tuning.

---
### 4.8.8 Defining Optimisers with Tunable Parameters

In this section, we will be defining the Common Optimisers that will be Tuned using the Optuna Tuning. The Optimisers will each have a range of values in a list so as to allow for the Optuna to search for the best Hyperparameter for the GAN. Through these parameters, it will affect how effective and efficiently the GAN will learn from the provided EMNIST Data. The relevent mathematical formula is indicated below:

---
**Optimiser Update Rule**

Purpose: Parameter Update Step in the Adam Optimiser, using Adaptive Learning Rates with Momentum.

$$
\theta \leftarrow \theta - \eta \cdot \frac{m_t}{\sqrt{v_t} + \epsilon}
$$

Where:
- $\theta$ = Model Parameters (Generator or Discriminator)
- $\eta$ = Learning Rate (Tuned)
- $m_t$ = First Moment Estimate (Mean of Gradients)
- $v_t$ = Second Moment Estimate (Variance of Gradients)
- $\epsilon$ = Small Constant for Numerical Stability

---
With the mathematical formula indicated above, we will proceed to define the Optimisers in the Code Cell.

In [ ]:
# ========== Objective Function ========== #
def objective(trial):
    # ----- Hyperparameters ----- #
    latent_dim = trial.suggest_categorical('latent_dim', [100, 128, 160])
    learning_rate = trial.suggest_float('learning_rate', 5e-5, 2e-4, log=True)
    beta_1 = trial.suggest_float('beta_1', 0.4, 0.6)
    code_dim = 2
    lambda_mi = 1.0

    # ----- Build Models ----- #
    generator = build_sagan_generator(latent_dim=latent_dim, code_dim=code_dim)
    discriminator = build_sagan_discriminator(code_dim=code_dim)

    # ----- Trigger Weight Creation ----- #
    z = tf.random.uniform([1, latent_dim], -1.0, 1.0)
    c = tf.random.uniform([1, code_dim], -1.0, 1.0)
    zc = tf.concat([z, c], axis=1)
    _ = generator(zc)
    _ = discriminator(tf.random.normal([1, 28, 28, 1]))

    # ----- Define Optimizers ----- #
    gen_optimizer = tf.keras.optimizers.Adam(learning_rate=learning_rate, beta_1=beta_1)
    disc_optimizer = tf.keras.optimizers.Adam(learning_rate=learning_rate, beta_1=beta_1)

    # ----- Prebuild Optimizer States ----- #
    _ = gen_optimizer.apply_gradients([(tf.zeros_like(v), v) for v in generator.trainable_variables])
    _ = disc_optimizer.apply_gradients([(tf.zeros_like(v), v) for v in discriminator.trainable_variables])

    # ----- Load EMNIST Dataset ----- #
    dataset = load_data(X_train, batch_size=64)
    steps = min(1000, len(X_train) // 64)

    # ----- Train InfoGAN-SAGAN ----- #
    _ = train(
        dataset=dataset,
        epochs=100,
        generator=generator,
        discriminator=discriminator,
        gen_optimizer=gen_optimizer,
        disc_optimizer=disc_optimizer,
        latent_dim=latent_dim,
        code_dim=code_dim,
        lambda_mi=lambda_mi,
        steps_per_epoch=steps,
        use_early_stopping=False,
        save_path=None,
        channels=1
    )

    # ----- Evaluate FID Score ----- #
    z = tf.random.uniform([1000, latent_dim], minval=-1.0, maxval=1.0)
    c = tf.random.uniform([1000, code_dim], minval=-1.0, maxval=1.0)
    zc = tf.concat([z, c], axis=1)
    fake_images = generator(zc, training=False)
    fake_images = (fake_images + 1.0) / 2.0

    real_images = X_val[:1000]

    fid_score = calculate_fid(real_images, fake_images)

    return fid_score

With reference to the Code Cell above, we are able to determine that the Optimiser have been successfully defined with a range of different Parameter Values so as to provide a robust Optuna Tuning in the subsequent sections.

---
### 4.8.9 Optuna Tuning Study

In this sub-section, we will be conducting the Optuna Tuning Study on the GAN Model so as to obtain the best Hyperparameter for the GAN Model. It is expected that the Tuning takes a prolonged duration due to the nature of how the GAN works and the 2 Neural Networks involved. However, the training data will be stored so as to prevent re-running of repetitive codes. The Optuna Tuning Study will be conducted in the Code Cell below.

In [ ]:
# =========== Create or Load Study with SQLite Backend =========== #
study = optuna.create_study(
    direction="minimize",
    study_name="sagan_tuning",
    storage="sqlite:////content/drive/MyDrive/Colab Notebooks/DELE CA2 A/Non-Augmented GAN Tunings/sagan_optuna.db",
    load_if_exists=True
)

# =========== Trial Management =========== #
MAX_TRIALS = 50
completed_trials = len([t for t in study.trials if t.state == optuna.trial.TrialState.COMPLETE])
remaining_trials = MAX_TRIALS - completed_trials

if remaining_trials > 0:
    print(f"Resuming SAGAN study: {completed_trials} completed, running {remaining_trials} more.")
    study.optimize(objective, n_trials=remaining_trials)
else:
    print(f"SAGAN study already completed {MAX_TRIALS} trials. Skipping optimization.")

# =========== Output Best Trial =========== #
best_trial = study.best_trial
best_params_df = pd.DataFrame([best_trial.params])
best_params_df["Final FID"] = best_trial.value
best_params_df.style.background_gradient(cmap="Blues")

With reference to the Code Cell above, we are able to view the Best Hyperparameter obtained during the Optuna Tuning Study. This Hyperparameter will be extracted and be trained for a longer period of time so as to attempt to increase the performance and limit the loss for the GAN.

---
### 4.8.10 GAN Re-Training Operation

As obtained from the previous sub-section, we will be re-training the GAN using the best Hyperparameter obtained during the Optuna Tuning Study. This is to push the GAN Model to its limits and also to save Computational Power as we are not taking up prolonged periods of time tuning GAN Models which may not have any clear signs of good performance. The retraining of the GAN Model will be conducted in the Code Cell below.

In [ ]:
# =========== Extract Best Parameters from Study =========== #
best_params = study.best_trial.params

latent_dim  = best_params['latent_dim']
learning_rate = best_params['learning_rate']
beta_1      = best_params['beta_1']
code_dim    = 2
lambda_mi   = 1.0

print("Using Best Trial Parameters for SAGAN:")
print(best_params)

# =========== Rebuild SAGAN Models =========== #
generator     = build_sagan_generator(latent_dim=latent_dim, code_dim=code_dim)
discriminator = build_sagan_discriminator(code_dim=code_dim)

gen_optimizer  = tf.keras.optimizers.Adam(learning_rate=learning_rate, beta_1=beta_1)
disc_optimizer = tf.keras.optimizers.Adam(learning_rate=learning_rate, beta_1=beta_1)

_ = gen_optimizer.apply_gradients([(tf.zeros_like(v), v) for v in generator.trainable_variables])
_ = disc_optimizer.apply_gradients([(tf.zeros_like(v), v) for v in discriminator.trainable_variables])

# =========== Reload Preprocessed Dataset =========== #
dataset = load_data(X_train)

# =========== Define Save Paths =========== #
model_name   = "sagan"
output_dir   = f"/content/drive/MyDrive/Colab Notebooks/DELE CA2 A/final_outputs_{model_name}_noAugment"
weights_path = os.path.join(output_dir, f"{model_name}_generator_final.weights.h5")
gen_ckpt     = os.path.join(output_dir, "best_gen.weights.h5")
disc_ckpt    = os.path.join(output_dir, "best_disc.weights.h5")
loss_csv     = os.path.join(output_dir, "loss_history.csv")

os.makedirs(output_dir, exist_ok=True)

# =========== Load or Train =========== #
if os.path.exists(gen_ckpt) and os.path.exists(disc_ckpt) and os.path.exists(loss_csv):
    print("All outputs found. Skipping training...")

    generator.load_weights(gen_ckpt)
    discriminator.load_weights(disc_ckpt)
    print("SAGAN Generator and Discriminator Weights Loaded.")

    loss_df = pd.read_csv(loss_csv)
    gen_loss_history  = loss_df['gen_loss'].tolist()
    disc_loss_history = loss_df['disc_loss'].tolist()
    mi_loss_history   = loss_df['mi_loss'].tolist()
    final_gen_loss    = gen_loss_history[-1]

else:
    # =========== Train SAGAN =========== #
    final_gen_loss, gen_loss_history, disc_loss_history, mi_loss_history = train(
        dataset=dataset,
        epochs=50,
        steps_per_epoch=1546,
        generator=generator,
        discriminator=discriminator,
        gen_optimizer=gen_optimizer,
        disc_optimizer=disc_optimizer,
        latent_dim=latent_dim,
        code_dim=code_dim,
        lambda_mi=lambda_mi,
        use_early_stopping=False,
        save_path=output_dir,
        channels=1
    )

    # =========== Save Generator Weights =========== #
    generator.save_weights(weights_path)
    print("SAGAN Generator Weights Saved.")

    # =========== Save Loss History =========== #
    loss_df = pd.DataFrame({
        'epoch': list(range(1, len(gen_loss_history) + 1)),
        'gen_loss': gen_loss_history,
        'disc_loss': disc_loss_history,
        'mi_loss': mi_loss_history
    })
    loss_df.to_csv(loss_csv, index=False)
    print(f"Loss history saved to {loss_csv}")

With reference to the Code Cell above, we are able to determine that the best GAN Model have been trained and saved successfully and we are able to proceed with the Model Evaluation in the next sub-sections where we determine how well the GAN Model performed based on various metrics and evaluation methodologies.

---
### 4.8.11 GAN Model Evaluation

In this sub-section, we will be evaluating the GAN's Generator performance on various aspects. We will be utilising different methodologies and metrics to attempt to quantify the performance of the GAN Generator while also conducting Visual Inspection on the result output of the GAN. The Evaluation will be conducted in the following sections.

---
#### 4.8.11.1 Visual Grid of Samples

In this sub-section, we will be visualising the Grid of Samples of the Synthesised Data generated by the GAN Model. A list of observations will be made before the utilisation of Quantitative Metrics to evaluate the GAN Generator's general performance. As the observations made are purely from professional opinion, it will most likely not be used as a Basis of Comparison between GAN Models due to differing opinions unless in extreme cases. The visualisation will be conducted in the Code Cell below.

In [ ]:
def plot_generated_images(generator, latent_dim, code_dim=2, n_rows=5, n_cols=5, save_path=None, title="SAGAN Generated Images (Non-Augmented)", seed=42):
    total_samples = n_rows * n_cols

    # ----- Generate Latent Vectors ----- #
    if seed is not None:
        tf.random.set_seed(seed)

    z = tf.random.normal([total_samples, latent_dim])
    c = tf.random.uniform([total_samples, code_dim], minval=-1.0, maxval=1.0)

    z = tf.math.l2_normalize(z, axis=1)
    c = tf.math.l2_normalize(c, axis=1)
    zc = tf.concat([z, c], axis=1)

    # ----- Generate Images ----- #
    gen_images = generator(zc, training=False)
    gen_images = (gen_images + 1.0) / 2.0
    gen_images = tf.clip_by_value(gen_images, 0.0, 1.0)

    # ----- Plot ----- #
    fig, axes = plt.subplots(n_rows, n_cols, figsize=(n_cols, n_rows))
    for idx, ax in enumerate(axes.flat):
        ax.imshow(gen_images[idx, :, :, 0], cmap='gray')
        ax.axis('off')

    plt.suptitle(title, fontsize=16)
    plt.tight_layout()

    if save_path:
        plt.savefig(save_path, dpi=300)
        print(f"Saved Generated Image Grid to: {save_path}")

    plt.show()

plot_generated_images(generator, latent_dim=160, code_dim=2, n_rows=10, n_cols=16)

With reference to the output of the Visual Grid of Samples above, we are able to make the following observations:

**Basic Structure of Letters is Not Captured**

- Most generated characters resemble the digit "1" or the uppercase letter "I".

- Stroke thickness is consistent and centred, maintaining EMNIST-like formatting.

**High Blur and Noise**

- Several samples are faint, with incomplete or faded strokes.

- Some characters show disconnected or ghost-like fragments, reducing clarity.

**Poor Character Consistency**

- The outputs are highly repetitive and lack character differentiation.

- There are virtually no complex or looped characters present.

**Lack of Diversity**

- An extreme case of mode collapse is observed as nearly all samples look like the same vertical stroke.

- Very limited variety in shape, angle, or complexity across the entire grid.

With the observations indicated above, we will proceed to conduct the Loss Curve Analysis in the next sub-section.

---
#### 4.8.11.2 Loss Curve Over Epochs

In this sub-section, we will be plotting the GAN Loss Curve over Epochs to evaluate the training behaviour of the GAN. This curve provides insight into whether the GAN is experiencing underfitting, overfitting, or achieving a stable training dynamic between the generator and discriminator.

Although GANs do not minimise a single unified loss in the traditional sense, the trajectory of the generator and discriminator losses can indicate whether the adversarial training process is converging appropriately.

If the generator loss remains high while discriminator loss quickly drops to near-zero, this may indicate **underfitting**, where the generator is not learning effectively to produce plausible images.

If the discriminator loss increases while generator loss sharply decreases, this may suggest **overfitting** or **mode collapse**, where the generator exploits narrow weaknesses in the discriminator without general improvement.

A relatively stable oscillation or convergence between both losses typically signifies a **well-balanced** training process, where the generator and discriminator are learning in tandem.

The Potential Observations and Corresponding Action Plans are outlined below:

**Underfitting Loss Curve**

- Increase training epochs to allow generator more time to learn.

- Simplify the discriminator to reduce overpowering the generator.

- Adjust the learning rate or apply label smoothing.

- Consider adding batch normalisation in the generator.

**Overfitting or Mode Collapse Loss Curve**

- Introduce or increase dropout in the discriminator.

- Apply input noise or label flipping to the discriminator.

- Evaluate diversity of generated samples regularly.

**Balanced/Ideal Loss Curve**

- Maintain current architecture and hyperparameters.

- Proceed with full-scale image generation.

- Evaluate both qualitative (visual inspection) and quantitative metrics (e.g. Inception Score or FID).

With these potential training behaviours and action plan outlined, we will proceed to visualise the GAN loss dynamics in the code cell below.

In [ ]:
# =========== Figure Size Configuration =========== #
plt.figure(figsize=(10, 6))

# =========== Plot Training Losses =========== #
plt.plot(gen_loss_history, label="Generator Loss", linewidth=2, color='blue')
plt.plot(disc_loss_history, label="Discriminator Loss", linewidth=2, color='red')

# =========== Labels and Title =========== #
plt.xlabel("Epoch", fontsize=12)
plt.ylabel("Loss", fontsize=12)
plt.title("SAGAN Training Loss Curve (Non-Augmented)", fontsize=16)

# =========== Grid, Legend, and Styling =========== #
plt.grid(True, linestyle='--', alpha=0.6)
plt.legend(fontsize=12)
plt.xticks(fontsize=10)
plt.yticks(fontsize=10)

# =========== Display the Plot =========== #
plt.tight_layout()
plt.show()

With reference to the output of the Loss Curve above, we are able to make the following observations:

- Excellent training dynamics (i.e. Early generator improvement and sustained learning)

- No instability, no signs of collapse or domination.

- Shows that self-attention improves convergence and stability, even without data augmentation.

With the observations indicated above, we will proceed to conduct the Quantitative Metrics Evaluation in the next sub-section.

---
#### 4.8.11.3 Quantitative Metrics Evaluation

In this sub-section we will be utilising Quantitative Metrics such as FID and KID to tabulate the results of the performance and quantify the performance of the GAN Generator. We will subsequently be use these metrics to conduct inter-model evaluation after tuning and training all the GANs. The metrics that we will be utilising and their respective formulas are indicated below:

---
**Fréchet Inception Distance**

Purpose: Measures the Distance between the Real-Image and Generated-Image Distributions in Inception Feature Space.

$$
\mathrm{FID} \;=\;\|\mu_r - \mu_f\|^2
\;+\;\mathrm{Tr}\Bigl(\Sigma_r + \Sigma_f - 2\,(\Sigma_r\,\Sigma_f)^{\tfrac12}\Bigr)
$$


Where:
- $\mu_r = \mathbb{E}[f(x)]$, $\Sigma_r = \mathrm{Cov}[f(x)]$ for Real Images $x$.  
- $\mu_f = \mathbb{E}[f(\hat x)]$, $\Sigma_f = \mathrm{Cov}[f(\hat x)]$ for Generated Images $\hat x$.  
- $f(\cdot)$ = Map from Image to its InceptionV3 'pooling=avg' Features.  

---
**Diversity (t-SNE Spread)**

Purpose: Quantifies how 'wide' the 2D t-SNE Embedding of Generated Samples.

$$
\mathrm{Spread}
\;=\;
\bigl(\max_i\,z_i^{(1)} - \min_i\,z_i^{(1)}\bigr)
\;\times\;
\bigl(\max_i\,z_i^{(2)} - \min_i\,z_i^{(2)}\bigr)
$$

Where:
- $z_i = (z_i^{(1)}, z_i^{(2)})$ = 2-dimensional t-SNE Embedding of $i$th Generated Image.

---
**Mode Collapse Risk**

Purpose: Flags when too many Generated Images are Nearly Identical (mode collapse).

$$
\text{ModeCollapseRisk} =
\begin{cases}
\text{Low}, & \dfrac{\bigl|\{\mathrm{unique\_rounded}(x_i)\}\bigr|}{N} > \tau,\\
\text{High}, & \text{otherwise}.
\end{cases}
$$

Where:
- $x_i$ = $N$ Generated Samples.  
- $\mathrm{unique\_rounded}(x_i)$ = Rounds Pixels to Detect Duplicates.  
- $\tau=0.9$ = Uniqueness Threshold (90%).

---
**Perceptual Path Length**

Purpose: Measures Sensitively of Generator's Outputs (in VGG16 Feature Space) move when the Latent Code $z$ is Perturbed.

$$
\mathrm{PPL}
\;=\;
\mathbb{E}_{z,\delta z}\Bigl[\,
\|\phi\bigl(G(z + \epsilon\,\delta z)\bigr)\;-\;\phi\bigl(G(z)\bigr)\|_2^2
\Bigr]
$$

Where:  
- $G(z)$ = GAN Generator Mapping $z\in\mathbb{R}^{\mathrm{latent\_dim}}$ to an Image.  
- $\phi(\cdot)$ = VGG16 'pooling=avg' Feature Extractor.  
- $\delta z\sim\mathcal{N}(0,I)$, $\epsilon$ = Small Constant.

---
**Kernel Inception Distance**

Purpose: An Unbiased Estimator of the Squared Maximum Mean Discrepancy (MMD) between Real and Generated Inception Features.

$$
\mathrm{KID}
\;=\;
\frac{1}{m(m-1)}\sum_{i\neq j} k\bigl(\phi(x_i),\phi(x_j)\bigr)
\;+\;
\frac{1}{n(n-1)}\sum_{i\neq j} k\bigl(\phi(\hat x_i),\phi(\hat x_j)\bigr)
\;-\;
\frac{2}{mn}\sum_{i=1}^m\sum_{j=1}^n k\bigl(\phi(x_i),\phi(\hat x_j)\bigr)
$$

Where:  
- $x_i$ ($i=1\ldots m$) = Real Images.
- $\hat x_j$ ($j=1\ldots n$) = Generated Images.  
- $\phi(\cdot)$ = InceptionV3 Feature Extractor (pooling='avg').  
- $k(u,v) = \bigl(\frac{u^\top v}{d}+1\bigr)^3$ = degree-3 Polynomial Kernel on $d$-dimensional Features.

---

With the mathematical and metrics indicated above, we will proceed to conduct the Quantitave Metrics Evalution in the Code Cell below.

In [ ]:
# ========== Preload InceptionV3 and VGG16 Models ========== #
inception = InceptionV3(include_top=False, pooling='avg', input_shape=(299, 299, 3))
vgg_model = VGG16(include_top=False, weights='imagenet', input_shape=(128, 128, 3))

# ========== Preprocess for Inception ========== #
def preprocess_images_for_inception(images):
    images = (images + 1.0) * 127.5
    images = tf.image.resize(images, [299, 299])
    if images.shape[-1] == 1:
        images = tf.image.grayscale_to_rgb(images)
    return preprocess_input(images)

# ========== Compute FID ========== #
def calculate_fid(real_images, fake_images, batch_size=50):
    real_pp = preprocess_images_for_inception(real_images)
    fake_pp = preprocess_images_for_inception(fake_images)

    act1 = inception.predict(real_pp, batch_size=batch_size, verbose=0)
    act2 = inception.predict(fake_pp, batch_size=batch_size, verbose=0)

    mu1, sigma1 = np.mean(act1, axis=0), np.cov(act1, rowvar=False)
    mu2, sigma2 = np.mean(act2, axis=0), np.cov(act2, rowvar=False)

    diff = mu1 - mu2
    covmean = sqrtm(sigma1 @ sigma2)
    if np.iscomplexobj(covmean):
        covmean = covmean.real

    fid = diff @ diff + np.trace(sigma1 + sigma2 - 2 * covmean)
    return round(fid, 4)

# ========== Compute KID ========== #
def polynomial_kernel(X, Y):
    d = X.shape[1]
    return (np.dot(X, Y.T) / d + 1) ** 3

def calculate_kid(real_images, fake_images, batch_size=128):
    real_pp = preprocess_images_for_inception(real_images)
    fake_pp = preprocess_images_for_inception(fake_images)

    real_features = inception.predict(real_pp, batch_size=batch_size, verbose=0)
    fake_features = inception.predict(fake_pp, batch_size=batch_size, verbose=0)

    m = real_features.shape[0]
    n = fake_features.shape[0]

    k_rr = polynomial_kernel(real_features, real_features)
    k_gg = polynomial_kernel(fake_features, fake_features)
    k_rg = polynomial_kernel(real_features, fake_features)

    np.fill_diagonal(k_rr, 0)
    np.fill_diagonal(k_gg, 0)

    mmd = (k_rr.sum() / (m * (m - 1)) +
           k_gg.sum() / (n * (n - 1)) -
           2 * k_rg.mean())
    return round(mmd, 4)

# ========== Compute t-SNE Spread (Diversity) ========== #
def calculate_tsne_spread(images):
    flat = images.reshape(images.shape[0], -1)
    tsne = TSNE(n_components=2, random_state=42)
    proj = tsne.fit_transform(flat)
    x_range = proj[:, 0].max() - proj[:, 0].min()
    y_range = proj[:, 1].max() - proj[:, 1].min()
    return round(x_range * y_range, 2)

# ========== Assess Mode Collapse ========== #
def assess_mode_collapse(images):
    unique = np.unique(np.round(images), axis=0).shape[0]
    return "Low" if unique > 0.9 * images.shape[0] else "High"

# ========== Compute Perceptual Path Length (PPL) ========== #
def calculate_ppl(generator, latent_dim=160, num_samples=50, epsilon=1e-2):
    distances = []
    for _ in range(num_samples):
        z1 = tf.random.normal([1, latent_dim])
        z2 = z1 + epsilon * tf.random.normal([1, latent_dim])
        img1 = generator(z1, training=False)
        img2 = generator(z2, training=False)

        img1 = tf.image.resize(tf.image.grayscale_to_rgb((img1 + 1.0) * 127.5), (128, 128))
        img2 = tf.image.resize(tf.image.grayscale_to_rgb((img2 + 1.0) * 127.5), (128, 128))

        f1 = vgg_model(vgg_preprocess(img1))
        f2 = vgg_model(vgg_preprocess(img2))
        d = tf.reduce_mean(tf.square(f1 - f2)).numpy()
        distances.append(d)
    return round(np.mean(distances), 4)

# ========== Evaluate GAN Performance ========== #
def evaluate_gan_model(generator, latent_dim, code_dim, X_val,
                       gen_loss_history, disc_loss_history,
                       model_name="SAGAM"):

    # ----- Sample uniform noise and code in [-1, 1] ----- #
    noise = tf.random.uniform([1000, latent_dim], minval=-1.0, maxval=1.0)
    code  = tf.random.uniform([1000, code_dim],  minval=-1.0, maxval=1.0)
    z     = tf.concat([noise, code], axis=1)

    # ----- Generate Fake Images ----- #
    fake_images = generator(z, training=False)
    fake_images = tf.clip_by_value((fake_images + 1.0) / 2.0, 0.0, 1.0)
    fake_np     = fake_images.numpy()

    # ----- Select and preprocess 1000 Real Images from Validation Set ----- #
    real_images = tf.convert_to_tensor(X_val[:1000], dtype=tf.float32)
    real_images = tf.clip_by_value(real_images, 0.0, 1.0)

    # ----- Compute Metrics ----- #
    fid_value         = calculate_fid(real_images, fake_images)
    kid_value         = calculate_kid(real_images, fake_images)
    tsne_spread_value = calculate_tsne_spread(fake_np)
    collapse_risk     = assess_mode_collapse(fake_np)
    ppl_score         = calculate_ppl(generator, latent_dim + code_dim)
    mean_g_loss       = round(np.mean(gen_loss_history), 4)
    std_g_loss        = round(np.std(gen_loss_history), 4)
    mean_d_loss       = round(np.mean(disc_loss_history), 4)
    std_d_loss        = round(np.std(disc_loss_history), 4)

    # ----- Store as global vars (optional) ----- #
    prefix = model_name.upper()
    globals()[f"{prefix}_FID"]           = fid_value
    globals()[f"{prefix}_KID"]           = kid_value
    globals()[f"{prefix}_TSNE"]          = tsne_spread_value
    globals()[f"{prefix}_MODE_COLLAPSE"] = collapse_risk
    globals()[f"{prefix}_PPL"]           = ppl_score
    globals()[f"{prefix}_GEN_LOSS_MEAN"]  = mean_g_loss
    globals()[f"{prefix}_GEN_LOSS_STD"]   = std_g_loss
    globals()[f"{prefix}_DISC_LOSS_MEAN"] = mean_d_loss
    globals()[f"{prefix}_DISC_LOSS_STD"]  = std_d_loss

    # ----- Final Summary Row ----- #
    row = {
        "Model": model_name,
        "FID Score": fid_value,
        "KID Score": kid_value,
        "Mean Generator Loss": mean_g_loss,
        "Std Generator Loss": std_g_loss,
        "Mean Discriminator Loss": mean_d_loss,
        "Std Discriminator Loss": std_d_loss,
        "Mode Collapse Risk": collapse_risk,
        "Visual Quality": "Extremely Poor",
        "Diversity (t-SNE Spread)": tsne_spread_value,
        "PPL": ppl_score
    }

    return pd.DataFrame([row])[[
        "Model", "FID Score", "KID Score",
        "Mean Generator Loss", "Std Generator Loss",
        "Mean Discriminator Loss", "Std Discriminator Loss",
        "Mode Collapse Risk", "Visual Quality",
        "Diversity (t-SNE Spread)", "PPL"
    ]]

# ========== Create DataFrame ========== #
df = evaluate_gan_model(
    generator,
    latent_dim=160,
    code_dim=2,
    X_val=X_val,
    gen_loss_history=gen_loss_history,
    disc_loss_history=disc_loss_history,
    model_name="SAGAN (Non-Augmented)"
)

# ========== Display DataFrame ========== #
df.style.background_gradient(cmap="Blues")

With reference to the output of the Quantitative Metrics above, we are able to determine the performance of the GAN. This will be used as a basis of comparison to the other GAN Models.

---
#### 4.8.11.4 t-SNE of Generated Samples

In this sub-section, we visualise the t-SNE projection of generated samples to evaluate the latent diversity learned by the unconditional GAN. Although our GAN is not class-conditioned, t-SNE remains a valuable tool to assess whether the generator is producing varied outputs that reflect meaningful use of the latent space. While GANs are trained in high-dimensional spaces, t-SNE enables us to observe 2D structural patterns such as sample clustering, separation, and density, which provide indirect insights into the diversity and generalisation capabilities of the generator. The visualisation will be conducted in the Code Cell below.

In [ ]:
# =========== Generate z and c for InfoGAN =========== #
latent_dim = 160
code_dim = 2
z = tf.random.normal([500, latent_dim])
c = tf.random.uniform([500, code_dim], minval=-1.0, maxval=1.0)

# =========== ℓ₂-Normalise =========== #
z = tf.math.l2_normalize(z, axis=1)
c = tf.math.l2_normalize(c, axis=1)

# =========== Concatenate z and c ===========
zc = tf.concat([z, c], axis=1)

# =========== Generate Images =========== #
generated_images = generator(zc, training=False)
generated_images = (generated_images + 1) / 2.0

def plot_tsne_embeddings(generated_images, save_path=None, title="t-SNE of SAGAN Generated Samples (Non-Augmented)"):
    # ----- Convert to [0, 1] Range ----- #
    if tf.reduce_max(generated_images).numpy() > 1.0:
        raise ValueError("Images should be in [-1, 1] range before scaling.")

    images_rescaled = (generated_images + 1.0) / 2.0
    flat_images = images_rescaled.numpy().reshape(images_rescaled.shape[0], -1)

    # ----- Run t-SNE ----- #
    tsne = TSNE(n_components=2, perplexity=30, learning_rate='auto', init='pca', random_state=42)
    tsne_proj = tsne.fit_transform(flat_images)

    # ----- Plot t-SNE ----- #
    plt.figure(figsize=(12, 6))
    sns.scatterplot(
        x=tsne_proj[:, 0],
        y=tsne_proj[:, 1],
        s=20,
        alpha=0.9,
        edgecolor='none'
    )
    plt.title(title, fontsize=14, weight='bold')
    plt.xlabel("t-SNE Dimension 1")
    plt.ylabel("t-SNE Dimension 2")
    plt.grid(True, linestyle='--', alpha=0.3)
    plt.tight_layout()

    if save_path:
        plt.savefig(save_path, dpi=300)
        print(f"t-SNE plot saved to: {save_path}")

    plt.show()

plot_tsne_embeddings(generated_images=generated_images)

With reference to the t-SNE plot of SAGAN generated samples (Non-Augmented) above, we are able to make the following observations:

**Globally Structured but Slightly Irregular**

- The distribution forms an approximate elliptical ring, similar to InfoGAN, suggesting that SAGAN has also learned a well-disentangled latent space.

- However, the circularity is slightly asymmetric, indicating some minor inconsistencies in sample spread.

**Strong Use of Attention Mechanism**

- The globally coherent layout may reflect SAGAN's self-attention mechanism, which enables the model to capture long-range dependencies and maintain structural uniformity in generation.

- Such structure implies that the generator is attending to consistent global features, even in a non-label-conditioned setting.

**High Diversity without Outliers**

- Unlike some GANs that produce scattered t-SNE plots, the SAGAN plot shows very few outlier points, indicating stable mode usage and effective latent navigation.

- Diversity is maintained within the curve, suggesting a balanced generative process.

**Slight Curvature Bias**

- The stretching along the horizontal axis suggests a potential latent axis bias, where certain dimensions may dominate latent transformations.

- Still, the manifold shape remains compact and continuous, showing a good trade-off between flexibility and control.

With the observations indicated above, we will proceed to the next section to conduct GAN Model Training on Augmented Data.